
# 예금보험공사 RAG 챗봇 데이터 수집 파이프라인

이 노트북은 예금보험공사·금융안심포털의 **42개 URL Manifest**를 기준으로 다음 작업을 수행합니다.

1. Manifest 검증 및 실행 대상 선택
2. 정적 HTML 수집
3. 이미지·첨부파일·링크 메타데이터 추출
4. 동적 조회 페이지의 초기 HTML·폼·스크립트 구조 진단
5. 결정론적 HTML → Markdown/JSON 변환
6. 응답 메타데이터·해시·실행 로그 저장
7. 페이지 문서와 기본 청크 JSONL 생성
8. 결과 전체 ZIP 다운로드

## 수집 정책

- `include_full`: 공개 본문 전체 수집
- `include_partial`: 공개 페이지의 초기 구조만 진단하며, 검색 조건을 임의 생성하지 않음
- `link_only`: 로그인·본인인증·신청·자가진단 결과를 수집하지 않고 서비스 설명과 공식 URL만 저장
- `review`: 사람의 최종 결정 전에는 실행하지 않음
- `exclude`: 요청 자체를 보내지 않음

> 이 노트북은 로그인 우회, 본인인증 자동화, 개인정보 조회, 신청 제출을 수행하지 않습니다.  
> 기본 요청 간격과 낮은 동시성으로 실행되며, `robots.txt` 정책을 확인합니다.

## v2 수정 사항

이번 버전에서는 `BI-004` 동적 페이지 처리 중 발생한 아래 오류를 수정했습니다.

```text
missing ), unterminated subpattern at position 0
```

원인은 동적 스크립트 탐지용 정규표현식에 역슬래시가 중복 입력된 것이었습니다.

수정 전:

```python
r"(ajax|fetch\\s*\\(|XMLHttpRequest|\\.do\\b|pageIndex|pageNo|search)"
```

수정 후:

```python
r"(?:ajax|fetch\s*\(|XMLHttpRequest|\.do\b|pageIndex|pageNo|search)"
```

추가 기능:

- `RUN_ONLY_URL_IDS` 특정 URL 단독 실행
- 동적 페이지 정규표현식 자체 테스트
- `BI-004` 진단 함수 합성 HTML 테스트
- 존재하지 않는 `url_id` 입력 검증

## FIXED 버전의 수정 범위

이 노트북은 기존 실행 순서를 변경하지 않았습니다.

```text
설정 → Manifest → HTTP 수집 → 원본 저장 → 파싱 → 자산 처리
→ 동적 페이지 진단 → 로그 → 검수 → 문서/청크 생성 → ZIP
```

기존 결과 분석에서 확인된 다음 문제만 수정했습니다.

- `div`, `span`, `dd` 직접 텍스트 누락
- FAQ의 질문 `dt` 누락
- 제목이 `퀵메뉴`로 저장되는 문제
- 공통 UI 문구 혼입
- `rowspan`, `colspan` 표 구조 붕괴
- `gfn_downloadFile()` 첨부파일 버튼 미인식
- 문자 수만 기준으로 FAQ·표가 잘리는 청킹
- BI-004가 첫 페이지만 수집됐다는 사실이 메타데이터에 남지 않는 문제

## BI-004 전체 페이지 수집 적용 버전

이 버전은 `보험금 지급대상 금융회사(BI-004)`를 첫 페이지만
저장하지 않고 다음 방식으로 전체 공개 페이지를 수집합니다.

```text
첫 페이지 GET
→ 마지막 페이지 인덱스 확인
→ curPage를 변경하며 POST 반복
→ 페이지별 원본 HTML 저장
→ 각 페이지 표 추출
→ 중복 행·중복 페이지 검사
→ 전체 표로 BI-004 문서 재생성
```

전체 수집이 정상 완료되면 다음 메타데이터가 기록됩니다.

```json
{
  "collection_scope": "all_public_pages",
  "is_complete": true,
  "page_count": 5,
  "row_count": 42
}
```

페이지 요청이 실패하거나 같은 페이지가 반복되면
`is_complete`를 `false`로 기록하고 로그에 실패 원인을 남깁니다.

In [ ]:

# Colab 런타임에 필요한 라이브러리 설치
%pip -q install "requests>=2.32,<3" "beautifulsoup4>=4.12,<5" "lxml>=5,<7" "pandas>=2.2,<4" "tqdm>=4.66,<5"



## 1. 실행 환경과 설정

기본 설정은 정적 페이지·자산 페이지·링크 전용 레코드를 처리합니다.

- 특정 업무만 실행하려면 `SELECT_BUSINESS_FUNCTIONS`에 업무명을 입력합니다.
- 테스트 실행은 `MAX_URLS = 3`처럼 제한합니다.
- 동적 공개 페이지를 브라우저로 렌더링한 초기 화면까지 저장하려면 `ENABLE_PLAYWRIGHT_SNAPSHOT = True`로 바꿉니다.
- Playwright도 검색 폼 제출이나 로그인은 수행하지 않습니다.


In [ ]:

from __future__ import annotations

import hashlib
import json
import mimetypes
import os
import re
import shutil
import subprocess
import sys
import time
from dataclasses import dataclass, asdict
from datetime import datetime, timezone, timedelta
from pathlib import Path
from typing import Any, Iterable
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup, Tag
from IPython.display import display
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

KST = timezone(timedelta(hours=9))

# -------------------------
# 사용자 실행 설정
# -------------------------
SELECT_BUSINESS_FUNCTIONS: list[str] = []
# 예: ["예금자보호제도", "예금보험금 안내"]

RUN_ONLY_URL_IDS: list[str] = []
# BI-004 오류 수정만 먼저 검증하려면:
# RUN_ONLY_URL_IDS = ["BI-004"]
# 전체 실행하려면 빈 리스트 []를 사용합니다.

RUN_DECISIONS = {"include_full", "include_partial", "link_only"}
RUN_WAVES = {"W1_STATIC", "W1_ASSET", "W2_DYNAMIC", "META"}

MAX_URLS: int | None = None
# 최초 테스트 권장: 3
# 전체 실행: None

REQUEST_DELAY_SECONDS = 1.2
REQUEST_TIMEOUT_SECONDS = 25
MAX_RETRIES = 3
MAX_ASSET_BYTES = 25 * 1024 * 1024

RESPECT_ROBOTS_TXT = True
DOWNLOAD_ATTACHMENTS = True
DOWNLOAD_IMAGES = True
DOWNLOAD_VIDEOS = False

ENABLE_PLAYWRIGHT_SNAPSHOT = False
PLAYWRIGHT_WAIT_MS = 1500

# JavaScript gfn_downloadFile(...) 버튼을 실제 파일로 내려받을지 여부
# 처음에는 False로 파싱 품질부터 검증하고, 이후 True로 시험하는 것을 권장합니다.
DOWNLOAD_JS_ATTACHMENTS_WITH_PLAYWRIGHT = False

# BI-004 지급대상 금융회사 공개 목록의 전체 페이지를 POST로 수집합니다.
# 전체 수집을 원하지 않을 때만 False로 변경합니다.
FETCH_BI004_ALL_PAGES = True

# 사이트 오류로 비정상적으로 많은 페이지가 계산되는 상황을 막는 안전 한도
BI004_MAX_PAGES = 100
BI004_PAGE_SIZE = 10

CREATE_BASELINE_CHUNKS = True
CHUNK_MAX_CHARS = 1200
CHUNK_OVERLAP_CHARS = 150

USER_AGENT = (
    "KDIC-RAG-Academic-Crawler/0.1 "
    "(low-rate public-document collection; no authentication automation)"
)

RUN_ID = datetime.now(KST).strftime("%Y%m%d_%H%M%S")
RUNTIME_ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
OUTPUT_ROOT = RUNTIME_ROOT / f"kdic_crawl_output_{RUN_ID}"

PATHS = {
    "raw_html": OUTPUT_ROOT / "raw" / "html",
    "raw_assets": OUTPUT_ROOT / "raw" / "assets",
    "response_meta": OUTPUT_ROOT / "raw" / "response_meta",
    "parsed_markdown": OUTPUT_ROOT / "parsed" / "markdown",
    "parsed_json": OUTPUT_ROOT / "parsed" / "json",
    "dynamic_diagnostics": OUTPUT_ROOT / "diagnostics" / "dynamic",
    "screenshots": OUTPUT_ROOT / "diagnostics" / "screenshots",
    "logs": OUTPUT_ROOT / "logs",
    "processed": OUTPUT_ROOT / "processed",
    "manifest": OUTPUT_ROOT / "manifest",
}

for path in PATHS.values():
    path.mkdir(parents=True, exist_ok=True)

print("실행 ID:", RUN_ID)
print("결과 경로:", OUTPUT_ROOT)


## 2. 42개 URL Manifest 불러오기

In [ ]:

# 현재 프로젝트의 42개 Manifest를 노트북 내부에 압축하여 포함했습니다.
# 별도 CSV 업로드 없이 기본 Manifest를 자동 생성합니다.

import base64
import gzip

EMBEDDED_MANIFEST_GZIP_B64 = """H4sIAHmDVGoC/+1dbW8TWbL+PtL8h1akXSXaXhzbCS/3w1xlgJmJBmaZwOzofrIcuxO8OG3T3Q6T/TAy0EGGZJawNwYH7FznEkiCgtYJDnF2g66Un+Pu1v6FW3XqdLvbbjs2hLCajYRiu19On656TtVTdeoc/vmP/0to0lRETokZJRlJxMXxjJqQJVWNTGTkmJZIyaIWVSYlLdJ6Ih2dlCJaQktKoprKKDEpAm2IKjQYkaNTkphSEpMJOZqMpJUEfNdmRDmlTEWTiT9L8UhciiVUbMX+ElGkqOq0OpOWxAlJi12PqJoS1aTJGTihqJICjaUmEvDIaEa7DvfczCQUKS5GVRX6mE4lE7EZMZaSNUnWImosBc0o0nRCutW4NKZEbyUjt6LTEv+qalEto4rST2kppkHXUhktndFEWYLvvFNOi5mpqaiCL6JJ6uefBcULV34/OBgUzUKuXsuZywvGm6pVqJnlovFQb3OUflh59uO6pqXV/wgEbt26depGPBE7lVJO3VACajoQT8Obw8tqgSvw5+qMquHnpSlNDahSEnp6NabIp+Ip/hRs9Umh/mbHvLMpWnndzBXEhBxLZuISaCyZFM0ns8ZmVTDXcsZmzSwV4Ntts/xIqO/u17cqxkpRgDvNpxtCvZI1HryA/gnY2lxJMPM5405VsP5SMEtVcy0rosASsQhqSkSxSqqmRq5rU0kRXyHCT08HQUzwTwZNQgci46k4Ck78MRi5em3k2uh5lGIE5AgyF81nIKSa8M21y5eE3wn1nU1zpSLAO5krVfioQZfhsLGxaN3NGr9UoB+WXsE3N+8VQ0PCD2OXxKBZqhlze0JDHLaQhaD5eAceUH+zL5jVHHygVvIrgrGn03UCnDX3lgRTXzVezYqffxYixYZ6Uqz/0V40fKLZZs3WaxXoclAw5rPWo6KVf2282kA91itFY3ZBMEv75l7BR1DwqgL9Np7PGy9A8eXbxst3Vh4e+mjBfJIz5lbxGlRYccd6Og/34EPMuTJHkFnSQZDZeu2+lS8YDxYRI8bLzQZGwoSR8HsMfnzQ3dvCF97Hf4FvZT5d7AoyhBS3abiamVJOUOPYA7cdELjAPeI2/3fWeFHoN0tZ6/H9g10SGJw42LVym6i3tVX4B2KC74Lx31sDgnGnYD6pguqHSPVDH656+G79Nfehqr+mTMJnPHaCgW4xQHJHS4JOgM7QsX7SPhgCC46h3lEw8HTjrY7myFzNNmzAMAFhuCcg9NXflOuVkqnvQ4dB9V+NfA9/jb/VQD9GeQ/usO7tWPkN6IK5/bofVQDv9gDe4K0OiMWuwtPNvdeiYM6tQqMAzCpoCZDs8X4DfQ6cJhKy6sZTbCowPm4Tia+iN79TolPaSDo5g+DhEsrnzLlN6+Gmpdc+NngmojebQIMdjsDhT+Br+t3cgeyCPWwHWl1PXyvfEOE9dTAU4HOy5ts8GRhUnnmnQlcLdDnor3AbDUw5D1IQrGe643tq5luQXzlnVtbRe4G3Mp5vIlL6AHWnCXWnP5ydYIs9MRTAyuEkBaBirj06RrQwE/Np0MJI5fN9FCdwhHplCXporM0LocHQsDGrC5yCom15/ALNiWC3YD4BED25Z/wCstgpMuJxZ9NYyTr6rol2I+fMZ4t+xNbdKjWBtsTdTNUszQNizhBiznwIYuiZ/UChzKXaQDvUxBOB8T+nx5Oy46fGY8oV/G2HL3+YVtQ2wJF+YohxPulZaKLrOwvYEeBygJTTMARBsIAO8OIMVy5AGXM7xpssxxVSPDOvG3MFQMyEpEgyRKmx1BTEkgmM6+wHqTcSaQc5/I949dvRK+xMGgLB8Rk7uiR0AynEMWvmH6CEfXTq6btAnRfByNd3Xh/smn9fADIBWoavpX0AG3wWbrNTxsNZNC34Qi0KZzplDeL4AEIMqj0rfjnqikG5PMHG0Ljp7TD4IfI65lyR+bw93dSL4HS6MhIX0urU6Dj3JFdSqnohNiWfBDUtpiN0KmiLmQlXaBfXQFPgCgAT81kIPegr0AR9C4wCZ6V2QyUIYotlq5A33ugezdkK//yzc4SU0IcihT2QOycEjO95Ov0esFFSsU+OGOhWDFNd7w+ZkatXL147SsSEmuTeHjI8jAGSKRgQVNwtGW+zSDQpmIVXhjNgblhr8ElH9+bhCxMaNe8w3OAgoSb8YaiBFzKWs//ZGx4u/pROyifmw595hHrJidRy/fCiXNe13MEuhpvARu0DA6T6PPtwVB8k1Q8dlWtxHWIP9onK3yMA/hKQ8nViWsIg+CtZ1dqgBEYBUnDmvIX623lz+YUDGaADWiIKqFmpQDcE+KhvVQ92HQwY+ip+mQXhPF1k1lZnfF4vIXCMv64ieJCU1LcroAnkf/Q4gVPh+IwcnQKUJFOpG5m08xNfNqJdl+RIOhmduaUkJq9rhCrvDW50AQCVmYgiqZmkpoozkir+GIpc+K/vRi57McZ7ZGwvmkVdIMQFqH8+4OprRpe5lPXjtaJgzYOeKm6dMY+zDaq8z2/mgWgzI6IotU/0b6EFNW6ImE8g/M46T2FjFhEaEn/4jpEfCqk9QTQ3mJ1OeY7xoJtBIMCN7RcC3e2+sKd4iQXWo/JNdG3s+9eZ+IlJa2PSwi0mDeH3aoOZK7ePc2mjYbgYYks1aNfQd4EoVTbwoVyR+Gs73+BCwTAhJ3QEyHFna5AP3ZtH64q9YYc8j3b4EqNvHNxl3VpatJVKPWyXuMlEA1GA0iSi6AKiDa3eNSn5fUb1g9bxp3LaQ4uldI4fWmGgTl4suOBVLMOL4euQDgT79bd0Q18QMcsGvyCcrG+/EzAk4udJpSx76MZGvZo1yuu8LdAshN9cQJ5gPDhE2At/Aux1jy6yXNqkgqZrVFPil7Xr8ROIdWu9ACMPAXvBg92ww925NtpSd4+u2ewE02Rh3cCEAMsRrFqLOv1qtmwOaxsmdA19PJ8I0Dbv/YJ9IOWjApcLdFVP3vEbJS19k1DRQXaVVfzVO8RmGhZuk14UXTqwvQhx164YGFKutRK4oH5rcd9paUCwDZ7Hv+qNZ0Ev+XPsOVG9iImGB6sIvNPi5WuUiapsAFLNeyss1V0pWEsFG129ncKY9NkCT1UD7lxXuC8Q6AoIMDug78aUorI/aMu6ZWD9mKDH+RZbQwNeCJIimbTtrCDG3XQXHv5h7BIMW85iaIw2Kx1iioqpl8XpRFxK+UAwwmpbVB8k0h1TkhaNR7WoiHCEQCKj0h2NEx8lJUFAGzosCm2nVK4zNmx31/EYmDqaBHXZsjMEqdCngNQh+SsPnk6yVn4QGToVFHyE6nKA0AA0ygoxtirM9DxcAuSI3F9S4gpjxFwBGT02RakqYdiYW69j2NicrzpLkAl/Asi4nHy3wEFyhcTqJCRsBU+XtOmQmI/CyEdon7ddyfDgOcLJ0HvixHcOvx00+D3dzsZfVuPK19PjNzpMyLfMlBV0c3kTnTOWtlCaBxw/+GcnA1SGuKCGmICzEDEbC+tsJvWop8Bw1B/sovaaFcbYA9MZsoeNClYvMPjqNRjpLBB7jfioV/IikRqq04Pwv1apV3TmSOEdENPFMk1+tHUvoBBP0BUaJH0PH6W+r/3hihAc7Fqv11Lp4OC/c4XFId5W6J1F2CH3TtGYW7XTt0xeoJj6VpUlbPPvEABBAsDpI3AMni54hz0ODeAzwMu/YOmFUoGA3RYkCSUQjccVVQqMsI8RTZMva5pyEmZ3Qk9DzNhbEHK5CEbiYBcJQu1+S6KnuWZnecFc07l+4T4Qg+hpiPsOOrhdxaflsJJUoPYd1sFkhiU5oRCh68xHQJeT06FOOckc9yxvI8PjJ6SO4LMTPuh1CILO1H0v6Z5+c3sdCdv8PEi5KUzqrC1nZlMvAhAx4LTy6/V/FOzJk6imRWPXp7D+3i86cs6qBEzX5QTOeOqWnExF455LXcGS6+hHocEdEOqZqG9PcBr3w6VYMuIBpbtF5vDRMTJZotOH4Y34DBM+z/5L4bMHZKaVo0elT+UEhLIs58bu+vUi0quEZp5dQL7YHTRbRdjvV7QCqMBMFRZRu9tFhgYSZ24JGOnLTUTqECH13DEi1RvneQL/jrjUFBV5+tW0ovlkAI7fZXfIAHTps48YaJ1Jnu1Fc0WAA4/rPcCyK1eWgcOtdAj/2QKPvy8YL1+zyx+sWvfKgCZrdtEAWRb3mRCdtT3DDGDA24/RFDKaCkeM8h6bGajvFgHxXaNsTEpeit5KjU1mkoeDrI+bPnocSMC6vSlyOGEENVdkc1nm20VWHlnWITAGaf3Wjup04MwDfX6JTpay4S/BmmWCd2c+6bCdk0ZA6yxzDYYBsyf17SyoRrBuL2BWGuyBsVISk9JkNIkVDbSIMCHHpZ+aTGoyId/gxpSuZhdxIONJXFGYimWYzYzAnRnJZU/xgk+QDeWlqCzspSGNyftXG2wEU9xCwoQAl8EBmzfuV5sLb+0kRjvc0c1Y0lW0Ht+HaJotZYPXwVSIQwAoOx8MHiPqsTx5acPUy9bdUtdQv6pltPPXY93DnNIHNvBWivVd9D8EY9HG6do8t5bNwKY+YjUP0n4yEi35fKdVgZq1faQz/cEg7jwJwYC4jsrxhDwZUVIZTVLaI5rOczDD9/FURo7TBX7J/WNAcxM/AGj5hl5caNQ6z9+3idrh8kCb+I1AguZgsWKU9vkgYMkb1r491bSmm8uzWCixgfm7EE0NBEPHShMwvciTiZ1n0ptR/UdVG4ulT0o02qR6h9qWaBCkAIHP90EvzlS5SxOcAzDa/rSKU29zRZYtRCKK6/z4nXbmN0QzBMHwR04E9ZQCmmIhDo90TlJA7xm4IGjYG9NhRkK8WeNWK9Xn0aFf3pkXomH2nJ0VcQ0GCNB4wdYKMy7VSATRvEJw6DjQ5TP9hB6rVnHWcVCPefmktZQ/JOTmMCTc4fTD98mJ2Ghc1mbG1KTWOdJu8bSMmEVScnJGBC7D6o91oO12kSwaeAgR/+c+uICSqW81phPQ5XJIEq/ivpUq/LCEQUBFl2+zhxHDScjgQ6MxLTEtRaD7E5HYdSl2Q/Q4S+qL91Bnr+tzO1x9+eK1EQ+mHRrwahbQ6+ki+10zX5bsGtbf2a+GC1Qb9R5ElRkhB74I5wCeIKm7dvTCCSRcDv4SF/8wR4hl9RsVex0oXMhWhfnPmhHe+8QztHqKxgpcvV0F0bKF8T6TmZ7yW7cWcQE8TakEhz/ZVCsNxx6mWrFasotKIzbYP3BuzW07j2qOrbk8aKhtlTYa1u0q4o9smb2alZLdh5UH8eQk8w+eBvqHofWf+X4aA406Q2q2aYLO8bjhoHhhhCqDtnRcrbeCQnKWCxx6bHkTM5+OGXSd67zHhjIeSI4nteifprTAJfgyAl8wV+N872rLDXonoT9sbOf4wwV6Oo6qsz+fG8SaPByjMO5xOcObOzDOWETA1gAMOOr/duy35z3db7haxjbBGvJVA6uPjDc7xkLBfFzlISU3EBx3HKEfA2NNEBs+rMDHoylKB4KZ2MaibXf+14EKSCXHzFwzSEIEktBRggSG4hu7ZE345/49gapym+bje4DNqDyRGpVvXlBjqowGpeu6jYYvpPwoOQW09RRSsiJk1k8YRbx02HYraOX/lmXmhdwhhalMGeRLXL63nMWVLbg7kiQDOKK4o5EqKdOJmNSVN8QVJQw2/KZIXFJjSiKNWz59BA/owG0Y6z1odBFPEVo8GBUHsSU3rGVaJLtu18Pyom23eMl5ti44Odh1jUXeL5bLclqjPti64IABlIYJpeH3QmmHRdOKlMSdswK4XPpbRY4B2KIAxJOyoKYIYBirS4ZPhfEPc38ECqfERBeYZp03YBsB/bLIalj1mrWcswljn51idlkj0bvmGRpdQJJVXO/HAoP5rHBmEMz9bwZEATqHwUE1a6/HflRslJtgHBAeIqAMHY05g74+3QDsglsQ6Giv1uu8IsXHYtPKDP44HFf28jaWRaMt1ER3L4D2QCdYuUXZXprO1uU3FuTzlfjEkZBmYLyQX7fyZXQ3xJS49ROj8T9FY2w+jTxTWuK5M3oy38ONH3XA1nQVmq6xi38cvfijc8bZZs6aX2CxBxsCPENHO27QikBXiATjtJloDbclWv4ysQqPzPw7bpK6YFstLbCZMSy+L/rpnsvc5TyHCW3DR4I2speI/K3ZXmF2RZW/VN4PXphqw3R2pYj5W9aHg13qBQtUWt/iV44tEgibfiYouEVjT309WOwEro5N2NL1lrMgnE4TnE4fCZz4I8HxuqYBuofT2ORRwIk9/QRGbgwwkXwYjNxN9Id/ZpvVgC5QfpjWLwy4MHWGMHWme0xRMgCiiVoO58te4hZ5WCm6pQO3w3maZwsBzwokRvc7k3s1Df8UbSIjxwMIsK/k+AVpXLuQlCeRz3e3PJdPNjUINCW5qFOYFqgUG1ViLOXI6RUPrLffYaoG+SZ7Uz7D0cR53Gw+nU4ilUcK3guX75DZOh5233covW+7wMkWmb9QD8FrK+O3x/3yAqMmXhRhXqQZRs7zHOp/lgB8tmsA+9eIuyLTpiVzh9UPuwLQNqlYX9bmiZCag2K7TvrhrPHynW17+Lw9K8u5Ddac20NuoHjzWFL8L2MDO8HLDhazHt7WkAqvGu6Gq1nFKqZDHlYEO353LGJnMfOZxJLuHfK4/Rog65z4Dd9HuJQ15u6DxJmDnysDgvyPtTONc6vIlpZ1nqy0t5mkU1R0gCvP3W1SkxRkwBeCaw/m87wcm7oQSyonOxu0Lww53bqzwZKO68QxHqVEGoxgmnkSUCflPG01aQee4UGec6VMfhWeB/dhKaiz+MV8WrH+UnMvggFtNu97MDRIUAt1DTV/I9aCn66NGMLlipLWEDL/zmshHH2D5lj9FTk20N+ebsA7QDvLL9oWXrrQwvfwQbTY1as0on0bpJ2uWbkFIBKkRCZoKEi4CLfBBZtrZ0Fnv/E2B5DCSiOgzOz8wAfbo9Y2m+wRL8rowSyNTqsT0eQPspIcTU4muzdQfbj3IlKvVjmwdBOcfVFzVq/aiVkcnWvsW1+fz51wECWuP0c3aqwtoFDZlZ3l2teHLfLaOetusb432+e49Y79czfWwIO3z7xdTN6gNanxJzQ22/GNfmz++cldf4sJosLtah1eC9kh39ujYNwpiIL1dIUV2hmVCqCRlcjz1+d2kubJcV8n3HqT+CXTDhdbn+jdKtwLYDRIfo/w1QIWreCQC9GQG+raFPsMKHNNh+atgo5FbJxr9FNwZhMQJj2bs2OytNB2887pVCyAGzTACAmcjyqSpIxJkwlVG2H/4UMgCV99wyG2XwAWxtuTUJ4e0ENZ2atfd/lMNqXqx0a+tm0nV7utmaq9kWcsJatO5f3Rzay2ypvHaVQysoTM0E8BuIPva75NGE8IMpfJNx6dFzpoyt6soYp1b5y30njrO0zZjEraWzGw0sWi3gjMPbqnBSOO9gWnhu7/AYEZ+Ef+YwAA"""

MANIFEST_PATH = RUNTIME_ROOT / "kdic_crawl_manifest_42.csv"
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH.write_bytes(
    gzip.decompress(base64.b64decode(EMBEDDED_MANIFEST_GZIP_B64))
)

print("Manifest 생성 완료:", MANIFEST_PATH)
print("크기:", MANIFEST_PATH.stat().st_size, "bytes")


In [ ]:

# 필요하면 사용자 수정본 CSV로 교체할 수 있습니다.
USE_CUSTOM_MANIFEST_UPLOAD = False

if USE_CUSTOM_MANIFEST_UPLOAD:
    from google.colab import files

    uploaded = files.upload()
    csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
    if len(csv_names) != 1:
        raise ValueError("CSV 파일을 정확히 1개 업로드해야 합니다.")

    MANIFEST_PATH = Path("/content") / csv_names[0]
    MANIFEST_PATH.write_bytes(uploaded[csv_names[0]])
    print("사용자 Manifest로 교체:", MANIFEST_PATH)


In [ ]:

REQUIRED_COLUMNS = {
    "url_id",
    "business_function",
    "page_title",
    "source_url",
    "site_name",
    "normalized_decision",
    "decision_reason",
    "page_type",
    "fetch_strategy",
    "parser_profile",
    "auth_required",
    "asset_policy",
    "content_scope",
    "crawl_wave",
}

manifest_df = pd.read_csv(MANIFEST_PATH, encoding="utf-8-sig", dtype=str).fillna("")

missing_columns = sorted(REQUIRED_COLUMNS - set(manifest_df.columns))
if missing_columns:
    raise ValueError(f"Manifest 필수 열이 없습니다: {missing_columns}")

if manifest_df["url_id"].duplicated().any():
    duplicates = manifest_df.loc[
        manifest_df["url_id"].duplicated(keep=False), ["url_id", "source_url"]
    ]
    raise ValueError(f"url_id 중복:\n{duplicates}")

if manifest_df["source_url"].duplicated().any():
    duplicates = manifest_df.loc[
        manifest_df["source_url"].duplicated(keep=False), ["url_id", "source_url"]
    ]
    raise ValueError(f"source_url 중복:\n{duplicates}")

invalid_urls = manifest_df[
    ~manifest_df["source_url"].str.match(r"^https?://", na=False)
]
if not invalid_urls.empty:
    raise ValueError(
        "HTTP/HTTPS가 아닌 URL이 있습니다:\n"
        + invalid_urls[["url_id", "source_url"]].to_string(index=False)
    )

# 실행 대상 필터
target_df = manifest_df.copy()

if SELECT_BUSINESS_FUNCTIONS:
    target_df = target_df[
        target_df["business_function"].isin(SELECT_BUSINESS_FUNCTIONS)
    ]

if RUN_ONLY_URL_IDS:
    known_url_ids = set(manifest_df["url_id"])
    unknown_url_ids = sorted(set(RUN_ONLY_URL_IDS) - known_url_ids)

    if unknown_url_ids:
        raise ValueError(
            f"Manifest에 존재하지 않는 url_id입니다: {unknown_url_ids}"
        )

    target_df = target_df[
        target_df["url_id"].isin(RUN_ONLY_URL_IDS)
    ]

target_df = target_df[
    target_df["normalized_decision"].isin(RUN_DECISIONS)
    & target_df["crawl_wave"].isin(RUN_WAVES)
].copy()

if MAX_URLS is not None:
    target_df = target_df.head(MAX_URLS).copy()

print("전체 Manifest:", len(manifest_df))
print("이번 실행 대상:", len(target_df))

display(
    manifest_df.groupby(
        ["business_function", "normalized_decision"],
        dropna=False,
    ).size().rename("count").reset_index()
)

display(
    target_df[
        [
            "url_id",
            "business_function",
            "page_title",
            "normalized_decision",
            "crawl_wave",
            "fetch_strategy",
        ]
    ].head(20)
)


## 3. 공통 유틸리티

In [ ]:

ATTACHMENT_EXTENSIONS = {
    ".pdf", ".hwp", ".hwpx", ".doc", ".docx",
    ".xls", ".xlsx", ".ppt", ".pptx", ".zip",
    ".csv", ".txt",
}

IMAGE_EXTENSIONS = {
    ".png", ".jpg", ".jpeg", ".gif", ".webp", ".svg", ".bmp",
}

VIDEO_EXTENSIONS = {
    ".mp4", ".webm", ".mov", ".avi", ".mkv",
}

NOISE_SELECTORS = [
    "script", "style", "noscript", "template",
    "header", "footer", "nav", "aside",
    ".skip", ".skip-navigation", ".skipnav",
    ".gnb", ".lnb", ".snb", ".breadcrumb-wrap",
    ".footer", ".header", ".quick-menu", ".quick",
    ".util", ".utility", ".pagination",
    "[aria-hidden='true']",
]

MAIN_SELECTORS_BY_SITE = {
    "예금보험공사": [
        "#contents", "#content", ".contents", ".content",
        ".sub-content", ".sub_contents", "main", "article",
    ],
    "금융안심포털": [
        "#contents", "#content", ".contents", ".content",
        ".sub-content", ".sub_contents", "main", "article",
    ],
}

BREADCRUMB_SELECTORS = [
    ".breadcrumb", ".breadcrumbs", ".location", ".location-wrap",
    ".path", ".sub-location", "nav[aria-label*='breadcrumb' i]",
]


def now_kst_iso() -> str:
    return datetime.now(KST).isoformat(timespec="seconds")


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def sha256_text(text: str) -> str:
    return sha256_bytes(text.encode("utf-8"))


def normalize_space(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


def normalize_multiline(text: str) -> str:
    lines = [normalize_space(line) for line in (text or "").splitlines()]
    cleaned: list[str] = []

    for line in lines:
        if not line:
            if cleaned and cleaned[-1] != "":
                cleaned.append("")
            continue
        cleaned.append(line)

    while cleaned and cleaned[-1] == "":
        cleaned.pop()

    return "\n".join(cleaned)


def safe_name(value: str, max_length: int = 90) -> str:
    value = normalize_space(value)
    value = re.sub(r'[\\/:*?"<>|]+', "_", value)
    value = re.sub(r"_+", "_", value).strip("._ ")
    return (value or "untitled")[:max_length]


def ensure_parent(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)


def write_json(path: Path, data: Any) -> None:
    ensure_parent(path)
    path.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def append_jsonl(path: Path, data: dict[str, Any]) -> None:
    ensure_parent(path)
    with path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(data, ensure_ascii=False) + "\n")


def absolute_url(base_url: str, candidate: str | None) -> str | None:
    if not candidate:
        return None

    candidate = candidate.strip()
    if not candidate or candidate.startswith(("#", "javascript:", "mailto:", "tel:")):
        return None

    return urljoin(base_url, candidate)


def extension_from_url(url: str) -> str:
    path = urlparse(url).path
    return Path(path).suffix.lower()


def classify_asset_url(url: str) -> str:
    extension = extension_from_url(url)

    if extension in ATTACHMENT_EXTENSIONS:
        return "attachment"
    if extension in IMAGE_EXTENSIONS:
        return "image"
    if extension in VIDEO_EXTENSIONS:
        return "video"
    return "link"


def row_to_clean_dict(row: pd.Series) -> dict[str, str]:
    return {
        str(key): "" if pd.isna(value) else str(value)
        for key, value in row.to_dict().items()
    }


## 4. HTTP 세션·재시도·robots.txt 검사

In [ ]:

def create_session() -> requests.Session:
    retry_policy = Retry(
        total=MAX_RETRIES,
        connect=MAX_RETRIES,
        read=MAX_RETRIES,
        status=MAX_RETRIES,
        backoff_factor=0.8,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset({"GET", "HEAD"}),
        respect_retry_after_header=True,
        raise_on_status=False,
    )

    adapter = HTTPAdapter(
        max_retries=retry_policy,
        pool_connections=2,
        pool_maxsize=2,
    )

    session = requests.Session()
    session.headers.update(
        {
            "User-Agent": USER_AGENT,
            "Accept": (
                "text/html,application/xhtml+xml,application/xml;q=0.9,"
                "image/avif,image/webp,*/*;q=0.8"
            ),
            "Accept-Language": "ko-KR,ko;q=0.9,en;q=0.5",
            "Connection": "keep-alive",
        }
    )
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


SESSION = create_session()
ROBOTS_CACHE: dict[str, RobotFileParser | None] = {}


def get_robots_parser(url: str) -> RobotFileParser | None:
    parsed = urlparse(url)
    base = f"{parsed.scheme}://{parsed.netloc}"

    if base in ROBOTS_CACHE:
        return ROBOTS_CACHE[base]

    robots_url = urljoin(base, "/robots.txt")

    try:
        response = SESSION.get(
            robots_url,
            timeout=REQUEST_TIMEOUT_SECONDS,
            allow_redirects=True,
        )

        if response.status_code >= 400:
            ROBOTS_CACHE[base] = None
            return None

        parser = RobotFileParser()
        parser.set_url(robots_url)
        parser.parse(response.text.splitlines())
        ROBOTS_CACHE[base] = parser
        return parser

    except requests.RequestException:
        ROBOTS_CACHE[base] = None
        return None


def robots_allows(url: str) -> tuple[bool, str]:
    if not RESPECT_ROBOTS_TXT:
        return True, "robots_check_disabled"

    parser = get_robots_parser(url)
    if parser is None:
        return True, "robots_unavailable_assumed_allowed"

    allowed = parser.can_fetch(USER_AGENT, url)
    return allowed, "allowed" if allowed else "disallowed"


def choose_response_encoding(response: requests.Response) -> str:
    content_type = response.headers.get("Content-Type", "")
    declared = response.encoding

    # requests의 기본 ISO-8859-1 추정은 한국어 HTML에서 잘못될 수 있습니다.
    if not declared or declared.lower() == "iso-8859-1":
        apparent = response.apparent_encoding
        if apparent:
            return apparent

    return declared or "utf-8"


## 5. HTML 수집과 응답 메타데이터 저장

In [ ]:

@dataclass
class FetchResult:
    requested_url: str
    final_url: str
    status_code: int
    ok: bool
    content_type: str
    encoding: str
    elapsed_seconds: float
    collected_at: str
    content_length: int
    raw_sha256: str
    redirect_history: list[dict[str, Any]]
    selected_headers: dict[str, str]
    body: bytes


def fetch_url(url: str) -> FetchResult:
    allowed, robots_status = robots_allows(url)
    if not allowed:
        raise PermissionError(f"robots.txt에서 수집을 허용하지 않습니다: {url}")

    started = time.perf_counter()

    response = SESSION.get(
        url,
        timeout=REQUEST_TIMEOUT_SECONDS,
        allow_redirects=True,
    )

    elapsed = time.perf_counter() - started
    encoding = choose_response_encoding(response)
    response.encoding = encoding

    selected_header_names = {
        "Content-Type",
        "Content-Length",
        "Last-Modified",
        "ETag",
        "Cache-Control",
        "Content-Disposition",
    }

    selected_headers = {
        key: value
        for key, value in response.headers.items()
        if key in selected_header_names
    }
    selected_headers["Robots-Check"] = robots_status

    return FetchResult(
        requested_url=url,
        final_url=response.url,
        status_code=response.status_code,
        ok=response.ok,
        content_type=response.headers.get("Content-Type", ""),
        encoding=encoding,
        elapsed_seconds=round(elapsed, 3),
        collected_at=now_kst_iso(),
        content_length=len(response.content),
        raw_sha256=sha256_bytes(response.content),
        redirect_history=[
            {
                "status_code": item.status_code,
                "url": item.url,
                "location": item.headers.get("Location"),
            }
            for item in response.history
        ],
        selected_headers=selected_headers,
        body=response.content,
    )


def save_fetch_result(
    manifest_row: dict[str, str],
    result: FetchResult,
) -> tuple[Path, Path]:
    business_dir = safe_name(manifest_row["business_function"])
    url_id = manifest_row["url_id"]

    html_path = PATHS["raw_html"] / business_dir / f"{url_id}.html"
    meta_path = PATHS["response_meta"] / business_dir / f"{url_id}.json"

    ensure_parent(html_path)
    html_path.write_bytes(result.body)

    metadata = asdict(result)
    metadata.pop("body", None)
    metadata["url_id"] = url_id
    metadata["business_function"] = manifest_row["business_function"]
    metadata["page_title_manifest"] = manifest_row["page_title"]
    metadata["fetch_strategy"] = manifest_row["fetch_strategy"]
    metadata["parser_profile"] = manifest_row["parser_profile"]
    metadata["raw_html_path"] = str(html_path.relative_to(OUTPUT_ROOT))

    write_json(meta_path, metadata)
    return html_path, meta_path


## 6. 결정론적 본문·표·링크·이미지 추출

In [ ]:
from bs4 import Comment, NavigableString

# 실제 본문 밖에서 반복되는 UI 요소
FIXED_NOISE_SELECTORS = [
    "script", "style", "noscript", "template",
    ".floatTop", ".floatBottom", ".paging", ".pagination",
    ".sr-only", ".skip", ".skipnav", ".skip-navigation",
    ".loading", ".waitPage", "#waitPage",
]

FIXED_NOISE_TEXTS = {
    "퀵메뉴", "글자 크기", "글자확대", "글자축소",
    "KOR", "ENG", "상단으로 이동", "검색 초기화",
    "좌우로 움직여보세요", "표 더보기",
}

FAQ_PREFIX_RE = re.compile(r"^\s*질문\s*")
ANSWER_PREFIX_RE = re.compile(r"^\s*답변\s*")
OPEN_SUFFIX_RE = re.compile(r"\s*열기\s*$")
DOWNLOAD_CALL_RE = re.compile(
    r"gfn_downloadFile\(\s*'([^']+)'\s*,\s*'([^']+)'\s*\)"
)
MOVE_URL_RE = re.compile(r"gfn_moveUrl\(\s*'([^']+)'\s*\)")

BLOCK_TAGS = {
    "div", "section", "article", "aside", "header", "footer", "main",
    "p", "ul", "ol", "dl", "table", "figure",
    "h1", "h2", "h3", "h4", "h5", "h6",
}


def fixed_norm(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


def fixed_safe_text(node: Tag) -> str:
    """원본 DOM을 훼손하지 않고 노이즈를 제거한 텍스트를 반환합니다."""
    fragment = BeautifulSoup(str(node), "lxml")
    root = fragment.body.find() if fragment.body else fragment.find()
    if not root:
        return ""

    for child in root.select(
        ".sr-only,.hide,script,style,noscript,template,.floatTop,.floatBottom"
    ):
        child.decompose()

    for image in root.find_all("img"):
        image.replace_with(fixed_norm(image.get("alt", "")))

    for br in root.find_all("br"):
        br.replace_with(" ")

    return fixed_norm(root.get_text(" ", strip=True))


def fixed_clean_question(text: str) -> str:
    text = FAQ_PREFIX_RE.sub("", fixed_norm(text))
    text = OPEN_SUFFIX_RE.sub("", text)
    return fixed_norm(text)


def fixed_clean_answer(text: str) -> str:
    return fixed_norm(ANSWER_PREFIX_RE.sub("", fixed_norm(text)))


def fixed_clean_term(text: str) -> str:
    text = OPEN_SUFFIX_RE.sub("", fixed_norm(text))
    text = re.sub(r"^\s*\d{1,2}\s+(?=\D)", "", text)
    return fixed_norm(text)


def fixed_infer_file_format(button: Tag) -> str:
    search_text = (
        " ".join(button.get("class", [])).lower()
        + " "
        + fixed_norm(button.get_text(" ", strip=True)).lower()
    )
    for keyword, file_format in [
        ("icohwp", "HWP"), ("hwp", "HWP"),
        ("icopdf", "PDF"), ("pdf", "PDF"),
        ("xlsx", "XLSX"), ("excel", "XLSX"),
        ("docx", "DOCX"), ("doc", "DOC"),
    ]:
        if keyword in search_text:
            return file_format
    return "FILE"


def fixed_cell_text(cell: Tag) -> str:
    fragment = BeautifulSoup(str(cell), "lxml")
    root = fragment.body.find() if fragment.body else fragment.find()
    if not root:
        return ""

    for child in root.select("script,style,noscript,template,.sr-only"):
        child.decompose()

    for button in root.find_all("button"):
        file_format = fixed_infer_file_format(button)
        label = fixed_norm(button.get_text(" ", strip=True))
        label = re.sub(r"다운로드$", "", label).strip()
        replacement = (
            f"{label} {file_format} 다운로드".strip()
            if label
            else f"{file_format} 다운로드"
        )
        button.replace_with(replacement)

    for image in root.find_all("img"):
        image.replace_with(fixed_norm(image.get("alt", "")))

    for br in root.find_all("br"):
        br.replace_with(" ")

    return fixed_norm(root.get_text(" ", strip=True))


def fixed_expand_table(table: Tag) -> tuple[list[str], list[list[str]]]:
    """rowspan/colspan을 펼쳐 각 행이 같은 열 수를 갖도록 만듭니다."""
    grid: list[list[str]] = []
    active_rowspans: dict[int, list[Any]] = {}
    header_flags: list[bool] = []
    max_columns = 0

    for tr in table.find_all("tr"):
        row: list[str] = []
        column = 0
        cells = tr.find_all(["th", "td"], recursive=False)

        def fill_active() -> None:
            nonlocal column
            while column in active_rowspans:
                remaining, value = active_rowspans[column]
                while len(row) <= column:
                    row.append("")
                row[column] = value
                remaining -= 1
                if remaining <= 0:
                    del active_rowspans[column]
                else:
                    active_rowspans[column] = [remaining, value]
                column += 1

        fill_active()
        header_flags.append(any(cell.name == "th" for cell in cells))

        for cell in cells:
            fill_active()
            text = fixed_cell_text(cell)
            rowspan = max(1, int(cell.get("rowspan", 1) or 1))
            colspan = max(1, int(cell.get("colspan", 1) or 1))

            for offset in range(colspan):
                target_column = column + offset
                while len(row) <= target_column:
                    row.append("")
                row[target_column] = text
                if rowspan > 1:
                    active_rowspans[target_column] = [rowspan - 1, text]

            column += colspan

        if active_rowspans:
            final_active_column = max(active_rowspans)
            while column <= final_active_column:
                if column in active_rowspans:
                    fill_active()
                else:
                    while len(row) <= column:
                        row.append("")
                    column += 1

        max_columns = max(max_columns, len(row))
        if row and any(row):
            grid.append(row)

    if not grid:
        return [], []

    for row in grid:
        row.extend([""] * (max_columns - len(row)))

    if table.thead:
        header_count = len(table.thead.find_all("tr", recursive=False))
    else:
        header_count = 1 if header_flags and header_flags[0] else 0

    if header_count == 0:
        header_count = 1

    header_rows = grid[:header_count]
    rows = grid[header_count:]
    headers: list[str] = []

    for column in range(max_columns):
        values: list[str] = []
        for header_row in header_rows:
            value = fixed_norm(header_row[column])
            if value and value not in values:
                values.append(value)
        headers.append(" / ".join(values) if values else f"열{column + 1}")

    # 중복 헤더 이름을 고유하게 만듭니다.
    counts: dict[str, int] = {}
    unique_headers: list[str] = []
    for raw_header in headers:
        header = raw_header or "열"
        counts[header] = counts.get(header, 0) + 1
        unique_headers.append(
            header if counts[header] == 1 else f"{header} {counts[header]}"
        )

    return unique_headers, rows


def fixed_extract_attachments(root: Tag, base_url: str) -> list[dict[str, Any]]:
    attachments: list[dict[str, Any]] = []
    seen: set[Any] = set()

    # KDIC 사이트의 JavaScript 다운로드 버튼
    for button in root.find_all("button", onclick=True):
        match = DOWNLOAD_CALL_RE.search(button.get("onclick", ""))
        if not match:
            continue

        key = (match.group(1), match.group(2))
        if key in seen:
            continue
        seen.add(key)

        file_format = fixed_infer_file_format(button)
        label = fixed_norm(button.get_text(" ", strip=True))
        label = re.sub(r"다운로드$", "", label).strip()

        row_context = ""
        parent_row = button.find_parent("tr")
        if parent_row:
            values: list[str] = []
            for cell in parent_row.find_all(["th", "td"], recursive=False):
                if button in cell.descendants:
                    continue
                value = fixed_cell_text(cell)
                if value and "다운로드" not in value and value not in values:
                    values.append(value)
            row_context = " | ".join(values[:4])

        display_name = label or row_context or f"{file_format} 첨부파일"
        if file_format not in display_name.upper():
            display_name = f"{display_name} ({file_format})"

        attachment_id = sha256_text(
            f"{match.group(1)}|{match.group(2)}"
        )[:16]

        attachments.append(
            {
                "attachment_id": attachment_id,
                "display_name": display_name,
                "file_format": file_format,
                "download_method": "gfn_downloadFile",
                "token1": match.group(1),
                "token2": match.group(2),
                "row_context": row_context,
                "button_text": fixed_norm(button.get_text(" ", strip=True)),
                "source_url": base_url,
                "download_status": "metadata_only",
            }
        )

    # href에 직접 연결된 첨부파일
    attachment_extensions = {
        ".pdf", ".hwp", ".hwpx", ".doc", ".docx",
        ".xls", ".xlsx", ".ppt", ".pptx", ".zip", ".csv",
    }

    for anchor in root.find_all("a", href=True):
        href = anchor.get("href", "")
        url = absolute_url(base_url, href)
        if not url:
            continue

        extension = Path(urlparse(url).path).suffix.lower()
        if extension not in attachment_extensions:
            continue

        key = ("href", url)
        if key in seen:
            continue
        seen.add(key)

        attachments.append(
            {
                "attachment_id": sha256_text(url)[:16],
                "display_name": (
                    fixed_norm(anchor.get_text(" ", strip=True))
                    or Path(urlparse(url).path).name
                ),
                "file_format": extension.lstrip(".").upper(),
                "download_method": "href",
                "url": url,
                "source_url": base_url,
                "download_status": "metadata_only",
            }
        )

    return attachments


def fixed_extract_links(root: Tag, base_url: str) -> list[dict[str, Any]]:
    links: list[dict[str, Any]] = []
    seen: set[Any] = set()

    for anchor in root.find_all("a", href=True):
        href = anchor.get("href", "").strip()
        if not href or href.startswith(("#", "javascript:", "mailto:", "tel:")):
            continue

        url = urljoin(base_url, href)
        text = fixed_norm(anchor.get_text(" ", strip=True))
        if not text or text in FIXED_NOISE_TEXTS:
            continue

        key = (url, text)
        if key in seen:
            continue
        seen.add(key)
        links.append({"url": url, "anchor_text": text})

    for button in root.find_all("button", onclick=True):
        match = MOVE_URL_RE.search(button.get("onclick", ""))
        if not match:
            continue
        url = urljoin(base_url, match.group(1))
        text = fixed_norm(button.get_text(" ", strip=True))
        key = (url, text)
        if key not in seen:
            seen.add(key)
            links.append(
                {"url": url, "anchor_text": text, "link_type": "button"}
            )

    return links


def fixed_extract_images(root: Tag, base_url: str) -> list[dict[str, Any]]:
    images: list[dict[str, Any]] = []
    seen: set[str] = set()

    for image in root.find_all("img"):
        source = (
            image.get("src")
            or image.get("data-src")
            or image.get("data-original")
        )
        url = absolute_url(base_url, source)
        if not url or url in seen:
            continue
        seen.add(url)

        alt = fixed_norm(image.get("alt", ""))
        filename = Path(urlparse(url).path).name
        lowered = filename.lower()
        image_role = "decorative"

        if "webtoon" in lowered:
            image_role = "supplementary_visual"
        elif any(keyword in lowered for keyword in ["proc", "process", "step", "flow"]):
            image_role = "process_diagram"
        elif alt:
            image_role = "content_image"

        images.append(
            {
                "url": url,
                "alt": alt,
                "filename": filename,
                "image_role": image_role,
            }
        )

    return images


class FixedStructureParser:
    def __init__(self, page_type: str = "static_page"):
        self.blocks: list[dict[str, Any]] = []
        self.page_type = page_type

    def add(self, block: dict[str, Any] | None) -> None:
        if not block:
            return

        if (
            self.blocks
            and block.get("type") in {"paragraph", "heading"}
            and self.blocks[-1].get("type") == block.get("type")
            and self.blocks[-1].get("text") == block.get("text")
        ):
            return

        self.blocks.append(block)

    def parse(self, root: Tag) -> list[dict[str, Any]]:
        faq_dls = root.select(
            ".accoWrap .accodian > dl, .accodian.con.ico > dl"
        )

        if self.page_type == "faq" or faq_dls:
            faq_nodes = {id(node) for node in faq_dls}

            for child in root.find_all(recursive=False):
                if not isinstance(child, Tag):
                    self.process_node(child)
                    continue

                child_dls = child.select("dl")
                if any(id(dl) in faq_nodes for dl in child_dls):
                    continue
                self.process_node(child)

            for dl in faq_dls:
                self.parse_faq(dl)
        else:
            for child in root.find_all(recursive=False):
                self.process_node(child)

        return self.blocks

    def parse_faq(self, dl: Tag) -> None:
        dt = dl.find("dt", recursive=False)
        dd = dl.find("dd", recursive=False)
        if not dt or not dd:
            return

        question = fixed_clean_question(fixed_safe_text(dt))
        answer_parser = FixedStructureParser("static_page")
        answer_parser.process_node(dd)

        if not answer_parser.blocks:
            answer_text = fixed_clean_answer(fixed_safe_text(dd))
            answer_parser.add({"type": "paragraph", "text": answer_text})
        else:
            for block in answer_parser.blocks:
                if "text" in block:
                    block["text"] = fixed_clean_answer(block["text"])
                    break

        answer_text = fixed_blocks_text(answer_parser.blocks)
        self.add(
            {
                "type": "faq",
                "question": question,
                "answer_blocks": answer_parser.blocks,
                "answer_text": answer_text,
            }
        )

    def parse_definition_list(self, dl: Tag) -> None:
        children = [
            child
            for child in dl.find_all(recursive=False)
            if isinstance(child, Tag)
        ]
        index = 0

        while index < len(children):
            if children[index].name != "dt":
                self.process_node(children[index])
                index += 1
                continue

            dt = children[index]
            dd = (
                children[index + 1]
                if index + 1 < len(children)
                and children[index + 1].name == "dd"
                else None
            )
            term = fixed_clean_term(fixed_safe_text(dt))

            if dd:
                definition_parser = FixedStructureParser("static_page")
                definition_parser.process_node(dd)
                if not definition_parser.blocks:
                    definition_parser.add(
                        {"type": "paragraph", "text": fixed_safe_text(dd)}
                    )

                self.add(
                    {
                        "type": "definition",
                        "term": term,
                        "definition_blocks": definition_parser.blocks,
                        "definition_text": fixed_blocks_text(
                            definition_parser.blocks
                        ),
                    }
                )
                index += 2
            else:
                self.add({"type": "heading", "level": 3, "text": term})
                index += 1

    def process_node(self, node: Any) -> None:
        if isinstance(node, Comment):
            return

        if isinstance(node, NavigableString):
            text = fixed_norm(str(node))
            if text and text not in FIXED_NOISE_TEXTS:
                self.add({"type": "paragraph", "text": text})
            return

        if not isinstance(node, Tag):
            return

        name = node.name.lower()
        classes = set(node.get("class", []))

        if (
            name in {"script", "style", "noscript", "template"}
            or classes & {
                "floatTop", "floatBottom", "paging", "pagination", "sr-only"
            }
        ):
            return

        if name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
            text = fixed_safe_text(node)
            if text and text not in FIXED_NOISE_TEXTS:
                self.add(
                    {"type": "heading", "level": int(name[1]), "text": text}
                )
            return

        if name == "p":
            text = fixed_safe_text(node)
            if text and text not in FIXED_NOISE_TEXTS:
                self.add({"type": "paragraph", "text": text})
            return

        if name == "table":
            headers, rows = fixed_expand_table(node)
            if headers or rows:
                self.add(
                    {
                        "type": "table",
                        "headers": headers,
                        "rows": rows,
                        "row_count": len(rows),
                    }
                )
            return

        if name in {"ul", "ol"}:
            items: list[str] = []
            for li in node.find_all("li", recursive=False):
                fragment = BeautifulSoup(str(li), "lxml")
                copied_li = fragment.body.find("li") if fragment.body else None
                if not copied_li:
                    continue
                for nested in copied_li.find_all(["ul", "ol"]):
                    nested.decompose()
                text = fixed_safe_text(copied_li)
                if text:
                    items.append(text)

            if items:
                self.add(
                    {"type": "list", "ordered": name == "ol", "items": items}
                )

            for li in node.find_all("li", recursive=False):
                for nested in li.find_all(["ul", "ol"], recursive=False):
                    self.process_node(nested)
            return

        if name == "dl":
            self.parse_definition_list(node)
            return

        if name == "figure":
            caption = node.find("figcaption")
            if caption:
                text = fixed_safe_text(caption)
                if text:
                    self.add({"type": "paragraph", "text": text})
            return

        # div, span 등이 가진 직접 텍스트와 하위 블록의 순서를 보존합니다.
        inline_parts: list[str] = []

        def flush_inline() -> None:
            nonlocal inline_parts
            text = fixed_norm(" ".join(inline_parts))
            inline_parts = []
            if text and text not in FIXED_NOISE_TEXTS:
                self.add({"type": "paragraph", "text": text})

        for child in node.children:
            if isinstance(child, Comment):
                continue
            if isinstance(child, NavigableString):
                text = fixed_norm(str(child))
                if text:
                    inline_parts.append(text)
            elif isinstance(child, Tag):
                child_name = child.name.lower()
                if child_name == "br":
                    inline_parts.append(" ")
                elif child_name in BLOCK_TAGS:
                    flush_inline()
                    self.process_node(child)
                else:
                    text = fixed_safe_text(child)
                    if text:
                        inline_parts.append(text)

        flush_inline()


def fixed_render_blocks(blocks: list[dict[str, Any]]) -> str:
    lines: list[str] = []

    for block in blocks:
        block_type = block.get("type")

        if block_type == "heading":
            level = min(max(int(block.get("level", 2)), 2), 6)
            lines.append(f"{'#' * level} {block.get('text', '')}")
        elif block_type == "paragraph":
            lines.append(block.get("text", ""))
        elif block_type == "list":
            for index, item in enumerate(block.get("items", []), start=1):
                prefix = f"{index}. " if block.get("ordered") else "- "
                lines.append(prefix + item)
        elif block_type == "definition":
            lines.append("### " + block.get("term", ""))
            lines.append(fixed_render_blocks(block.get("definition_blocks", [])))
        elif block_type == "faq":
            lines.append("### Q. " + block.get("question", ""))
            lines.append(fixed_render_blocks(block.get("answer_blocks", [])))
        elif block_type == "table":
            headers = [
                fixed_norm(header).replace("|", "\\|")
                for header in block.get("headers", [])
            ]
            if headers:
                lines.append("| " + " | ".join(headers) + " |")
                lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
                for row in block.get("rows", []):
                    normalized_row = [
                        fixed_norm(value).replace("|", "\\|")
                        for value in row
                    ]
                    normalized_row.extend(
                        [""] * (len(headers) - len(normalized_row))
                    )
                    lines.append(
                        "| " + " | ".join(normalized_row[: len(headers)]) + " |"
                    )

        lines.append("")

    return re.sub(r"\n{3,}", "\n\n", "\n".join(lines)).strip()


def fixed_blocks_text(blocks: list[dict[str, Any]]) -> str:
    values: list[str] = []

    for block in blocks:
        block_type = block.get("type")
        if block_type in {"heading", "paragraph"}:
            values.append(block.get("text", ""))
        elif block_type == "list":
            values.extend(block.get("items", []))
        elif block_type == "definition":
            values.extend(
                [block.get("term", ""), block.get("definition_text", "")]
            )
        elif block_type == "faq":
            values.extend(
                [block.get("question", ""), block.get("answer_text", "")]
            )
        elif block_type == "table":
            values.extend(block.get("headers", []))
            for row in block.get("rows", []):
                values.extend(row)

    return fixed_norm(" ".join(values))


def fixed_manifest_title(manifest_title: str, business_function: str) -> str:
    # Manifest에서 사람이 확인한 전체 경로명을 사용합니다.
    # HTML 내부의 "퀵메뉴" 같은 잘못된 제목을 선택하지 않습니다.
    return fixed_norm(manifest_title) or business_function


def fixed_manifest_breadcrumb(
    manifest_title: str,
    business_function: str,
) -> list[str]:
    parts = [
        fixed_norm(part)
        for part in re.split(r"\s*>\s*", manifest_title or "")
        if fixed_norm(part)
    ]
    result: list[str] = []
    for value in [business_function] + parts:
        if value and value not in result:
            result.append(value)
    return result


def parse_html_document(
    html_bytes: bytes,
    final_url: str,
    manifest_row: dict[str, str],
    encoding: str,
) -> dict[str, Any]:
    html_text = html_bytes.decode(encoding, errors="replace")
    soup = BeautifulSoup(html_text, "lxml")

    # 실제 KDIC 본문 컨테이너를 우선합니다.
    root = (
        soup.select_one(".contents")
        or soup.select_one("#contents")
        or soup.select_one("#content")
        or soup.select_one("main")
        or soup.body
    )
    if not root:
        raise ValueError("본문 루트를 찾지 못했습니다.")

    # 버튼은 노이즈 제거 전에 추출해야 합니다.
    attachments = fixed_extract_attachments(root, final_url)
    images = fixed_extract_images(root, final_url)

    for selector in FIXED_NOISE_SELECTORS:
        for node in root.select(selector):
            node.decompose()

    links = fixed_extract_links(root, final_url)
    source_text = fixed_norm(root.get_text(" ", strip=True))

    structure_parser = FixedStructureParser(
        manifest_row.get("page_type", "static_page")
    )
    blocks = structure_parser.parse(root)
    markdown = fixed_render_blocks(blocks)
    content_text = fixed_blocks_text(blocks)

    # UI 문구 마지막 안전 제거
    for phrase in FIXED_NOISE_TEXTS:
        markdown = re.sub(
            rf"(?m)^\s*{re.escape(phrase)}\s*$",
            "",
            markdown,
        )
    markdown = re.sub(r"\n{3,}", "\n\n", markdown).strip()

    content_text = fixed_norm(
        re.sub(
            "|".join(re.escape(phrase) for phrase in FIXED_NOISE_TEXTS),
            " ",
            content_text,
        )
    )

    faq_count = sum(1 for block in blocks if block.get("type") == "faq")
    table_count = sum(1 for block in blocks if block.get("type") == "table")
    noise_hits = [
        phrase for phrase in FIXED_NOISE_TEXTS if phrase in markdown
    ]
    retention_ratio = round(
        len(content_text) / max(1, len(source_text)),
        3,
    )

    quality_reasons: list[str] = []
    if len(content_text) < 80:
        quality_reasons.append("본문이 80자 미만")
    if retention_ratio < 0.70:
        quality_reasons.append("본문 보존율 70% 미만")
    if noise_hits:
        quality_reasons.append("공통 UI 문구 잔존")
    if manifest_row.get("page_type") == "faq" and faq_count == 0:
        quality_reasons.append("FAQ 질문-답변 추출 실패")

    if any(
        reason in {"공통 UI 문구 잔존", "FAQ 질문-답변 추출 실패"}
        for reason in quality_reasons
    ):
        quality_status = "fail"
    elif quality_reasons:
        quality_status = "review"
    else:
        quality_status = "pass"

    parsed_hash = sha256_text(markdown)

    document = {
        "doc_id": manifest_row["url_id"],
        "parent_doc_id": manifest_row["url_id"],
        "record_type": "page",
        "title": fixed_manifest_title(
            manifest_row["page_title"],
            manifest_row["business_function"],
        ),
        "manifest_title": manifest_row["page_title"],
        "html_title": (
            fixed_norm(soup.title.get_text(" ", strip=True))
            if soup.title
            else ""
        ),
        "source_url": manifest_row["source_url"],
        "final_url": final_url,
        "site_name": manifest_row["site_name"],
        "business_function": manifest_row["business_function"],
        "target_business_function": manifest_row.get(
            "target_business_function",
            manifest_row["business_function"],
        ),
        "page_type": manifest_row["page_type"],
        "parser_profile": manifest_row["parser_profile"],
        "breadcrumb": fixed_manifest_breadcrumb(
            manifest_row["page_title"],
            manifest_row["business_function"],
        ),
        "content_markdown": markdown,
        "content_text": content_text,
        "parsed_content_sha256": parsed_hash,
        "content_hash": parsed_hash,
        "blocks": blocks,
        "links": links,
        "attachments": attachments,
        "images": images,
        "videos": [],
        "quality": {
            "status": quality_status,
            "reasons": quality_reasons,
            "source_text_chars": len(source_text),
            "parsed_text_chars": len(content_text),
            "retention_ratio": retention_ratio,
            "faq_count": faq_count,
            "table_count": table_count,
            "attachment_button_count": len(attachments),
            "noise_hits": noise_hits,
        },
        "parsed_at": now_kst_iso(),
    }

    if manifest_row["url_id"] == "BI-004":
        document["dynamic_collection"] = {
            "collection_scope": "initial_public_page_only",
            "is_complete": False,
            "current_page_count": 1,
        }

    return document

## 7. 첨부파일·이미지 다운로드

In [ ]:
CONTENT_TYPE_EXTENSIONS = {
    "application/pdf": ".pdf",
    "application/zip": ".zip",
    "application/vnd.openxmlformats-officedocument.wordprocessingml.document": ".docx",
    "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet": ".xlsx",
    "application/vnd.openxmlformats-officedocument.presentationml.presentation": ".pptx",
    "image/jpeg": ".jpg",
    "image/png": ".png",
    "image/gif": ".gif",
    "image/webp": ".webp",
    "image/svg+xml": ".svg",
}


def filename_from_response(
    url: str,
    response: requests.Response,
    fallback_stem: str,
) -> str:
    disposition = response.headers.get("Content-Disposition", "")
    filename_star = re.search(
        r"filename\*=UTF-8''([^;]+)",
        disposition,
        flags=re.IGNORECASE,
    )
    filename_plain = re.search(
        r'filename="?([^";]+)"?',
        disposition,
        flags=re.IGNORECASE,
    )

    if filename_star:
        from urllib.parse import unquote
        name = unquote(filename_star.group(1))
    elif filename_plain:
        name = filename_plain.group(1)
    else:
        name = Path(urlparse(url).path).name

    name = safe_name(name, max_length=120)
    extension = Path(name).suffix.lower()

    if not extension:
        content_type = response.headers.get("Content-Type", "").split(";")[0].strip()
        extension = CONTENT_TYPE_EXTENSIONS.get(content_type)
        if not extension:
            extension = mimetypes.guess_extension(content_type) or ""
        name = f"{safe_name(fallback_stem)}{extension}"

    return name


def download_binary_asset(
    url: str,
    output_dir: Path,
    fallback_stem: str,
) -> dict[str, Any]:
    output_dir.mkdir(parents=True, exist_ok=True)

    with SESSION.get(
        url,
        timeout=REQUEST_TIMEOUT_SECONDS,
        allow_redirects=True,
        stream=True,
    ) as response:
        response.raise_for_status()

        content_length = response.headers.get("Content-Length")
        if content_length and int(content_length) > MAX_ASSET_BYTES:
            raise ValueError("파일이 제한 용량을 초과합니다.")

        filename = filename_from_response(url, response, fallback_stem)
        output_path = output_dir / filename

        hasher = hashlib.sha256()
        written = 0

        with output_path.open("wb") as file:
            for chunk in response.iter_content(chunk_size=64 * 1024):
                if not chunk:
                    continue
                written += len(chunk)
                if written > MAX_ASSET_BYTES:
                    output_path.unlink(missing_ok=True)
                    raise ValueError("다운로드 중 제한 용량을 초과했습니다.")
                hasher.update(chunk)
                file.write(chunk)

        return {
            "source_url": url,
            "final_url": response.url,
            "filename": filename,
            "relative_path": str(output_path.relative_to(OUTPUT_ROOT)),
            "content_type": response.headers.get("Content-Type", ""),
            "size_bytes": written,
            "sha256": hasher.hexdigest(),
            "downloaded_at": now_kst_iso(),
            "download_status": "downloaded",
        }


def ensure_attachment_playwright() -> None:
    try:
        import playwright  # noqa: F401
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "playwright"],
            check=True,
        )
    subprocess.run(
        [sys.executable, "-m", "playwright", "install", "chromium"],
        check=True,
    )


def download_js_attachments_for_document(
    document: dict[str, Any],
    output_dir: Path,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    js_attachments = [
        item
        for item in document.get("attachments", [])
        if item.get("download_method") == "gfn_downloadFile"
    ]
    if not js_attachments:
        return [], []

    ensure_attachment_playwright()
    from playwright.sync_api import sync_playwright

    downloaded: list[dict[str, Any]] = []
    failures: list[dict[str, Any]] = []
    output_dir.mkdir(parents=True, exist_ok=True)

    with sync_playwright() as playwright:
        browser = playwright.chromium.launch(headless=True)
        context = browser.new_context(
            accept_downloads=True,
            user_agent=USER_AGENT,
            locale="ko-KR",
        )
        page = context.new_page()

        try:
            page.goto(
                document["source_url"],
                wait_until="domcontentloaded",
                timeout=REQUEST_TIMEOUT_SECONDS * 1000,
            )
            page.wait_for_function(
                "typeof gfn_downloadFile === 'function'",
                timeout=REQUEST_TIMEOUT_SECONDS * 1000,
            )

            for item in js_attachments:
                try:
                    with page.expect_download(timeout=60_000) as download_info:
                        page.evaluate(
                            """
                            ([token1, token2]) => {
                                gfn_downloadFile(token1, token2);
                            }
                            """,
                            [item["token1"], item["token2"]],
                        )

                    download = download_info.value
                    filename = safe_name(download.suggested_filename, max_length=120)
                    output_path = output_dir / filename
                    download.save_as(str(output_path))

                    downloaded.append(
                        {
                            **item,
                            "filename": filename,
                            "relative_path": str(output_path.relative_to(OUTPUT_ROOT)),
                            "size_bytes": output_path.stat().st_size,
                            "sha256": sha256_bytes(output_path.read_bytes()),
                            "download_status": "downloaded",
                            "error": "",
                        }
                    )
                except Exception as error:
                    failures.append(
                        {
                            "asset_type": "attachment",
                            "display_name": item.get("display_name", ""),
                            "download_method": "gfn_downloadFile",
                            "error_type": type(error).__name__,
                            "error": str(error),
                        }
                    )
        finally:
            context.close()
            browser.close()

    return downloaded, failures


def download_document_assets(
    document: dict[str, Any],
    manifest_row: dict[str, str],
) -> dict[str, list[dict[str, Any]]]:
    business_dir = safe_name(manifest_row["business_function"])
    asset_root = PATHS["raw_assets"] / business_dir / manifest_row["url_id"]

    results = {
        "attachments": [],
        "attachment_metadata": [],
        "images": [],
        "videos": [],
        "failures": [],
    }

    should_download_attachments = (
        DOWNLOAD_ATTACHMENTS
        and (
            manifest_row["asset_policy"] == "download_attachments"
            or manifest_row["page_type"] == "attachment_page"
        )
    )

    # JS 버튼은 URL이 없으므로 우선 메타데이터로 항상 보존합니다.
    for item in document.get("attachments", []):
        if item.get("download_method") == "gfn_downloadFile":
            results["attachment_metadata"].append(item)

    # 직접 href 첨부파일
    if should_download_attachments:
        for index, item in enumerate(document.get("attachments", []), start=1):
            if item.get("download_method") != "href" or not item.get("url"):
                continue
            try:
                result = download_binary_asset(
                    item["url"],
                    asset_root / "attachments",
                    f"{manifest_row['url_id']}_attachment_{index:03d}",
                )
                result.update(item)
                results["attachments"].append(result)
            except Exception as error:
                results["failures"].append(
                    {
                        "asset_type": "attachment",
                        "url": item.get("url", ""),
                        "error_type": type(error).__name__,
                        "error": str(error),
                    }
                )

    # 선택적으로 JavaScript 버튼 실제 다운로드
    if should_download_attachments and DOWNLOAD_JS_ATTACHMENTS_WITH_PLAYWRIGHT:
        downloaded, failures = download_js_attachments_for_document(
            document,
            asset_root / "attachments",
        )
        results["attachments"].extend(downloaded)
        results["failures"].extend(failures)

    should_download_images = (
        DOWNLOAD_IMAGES
        and (
            manifest_row["crawl_wave"] == "W1_ASSET"
            or manifest_row["page_type"] in {
                "process_page", "video_page", "attachment_page"
            }
        )
    )

    if should_download_images:
        for index, item in enumerate(document.get("images", []), start=1):
            if item.get("image_role") == "decorative":
                continue
            try:
                result = download_binary_asset(
                    item["url"],
                    asset_root / "images",
                    f"{manifest_row['url_id']}_image_{index:03d}",
                )
                result.update(item)
                results["images"].append(result)
            except Exception as error:
                results["failures"].append(
                    {
                        "asset_type": "image",
                        "url": item.get("url", ""),
                        "error_type": type(error).__name__,
                        "error": str(error),
                    }
                )

    return results

## 8. 동적 조회 페이지 진단

In [ ]:
DYNAMIC_SCRIPT_KEYWORD_PATTERN = re.compile(
    r"(?:ajax|fetch\s*\(|XMLHttpRequest|\.do\b|pageIndex|pageNo|search)",
    flags=re.IGNORECASE,
)


def extract_dynamic_diagnostic(
    html_bytes: bytes,
    final_url: str,
    encoding: str,
) -> dict[str, Any]:
    """공개 동적 페이지의 폼과 스크립트 구조를 진단합니다."""
    html_text = html_bytes.decode(encoding, errors="replace")
    soup = BeautifulSoup(html_text, "lxml")

    forms = []

    for form_index, form in enumerate(
        soup.find_all("form"),
        start=1,
    ):
        inputs = []

        for field in form.select(
            "input, select, textarea, button"
        ):
            name = normalize_space(field.get("name", ""))
            field_type = normalize_space(
                field.get("type", field.name or "")
            )

            if not name and field.name != "button":
                continue

            record = {
                "tag": field.name,
                "name": name,
                "type": field_type,
                "value": normalize_space(
                    field.get("value", "")
                ),
            }

            if field.name == "select":
                record["options"] = [
                    {
                        "value": normalize_space(
                            option.get("value", "")
                        ),
                        "text": normalize_space(
                            option.get_text(" ", strip=True)
                        ),
                    }
                    for option in field.find_all("option")
                ]

            inputs.append(record)

        forms.append(
            {
                "form_index": form_index,
                "method": normalize_space(
                    form.get("method", "GET")
                ).upper(),
                "action": (
                    absolute_url(
                        final_url,
                        form.get("action"),
                    )
                    or final_url
                ),
                "id": normalize_space(form.get("id", "")),
                "name": normalize_space(
                    form.get("name", "")
                ),
                "inputs": inputs,
            }
        )

    script_sources = []

    for script in soup.find_all("script", src=True):
        source = absolute_url(
            final_url,
            script.get("src"),
        )

        if source and source not in script_sources:
            script_sources.append(source)

    inline_script_hints = []

    for script in soup.find_all("script"):
        if script.get("src"):
            continue

        text = script.get_text("\n", strip=True)

        if (
            text
            and DYNAMIC_SCRIPT_KEYWORD_PATTERN.search(text)
        ):
            inline_script_hints.append(text[:1200])

    return {
        "final_url": final_url,
        "forms": forms,
        "script_sources": script_sources,
        "inline_script_hints": inline_script_hints[:20],
        "diagnosed_at": now_kst_iso(),
        "safety_note": (
            "공개 페이지 구조만 진단합니다. "
            "로그인·본인인증·개인정보 조회·신청 제출은 수행하지 않습니다."
        ),
    }


def bi004_extract_base_payload(
    html_bytes: bytes,
    encoding: str,
) -> dict[str, str]:
    """BI-004 검색 폼의 기본 공개 검색 조건을 추출합니다."""
    soup = BeautifulSoup(
        html_bytes.decode(encoding, errors="replace"),
        "lxml",
    )
    form = soup.select_one("form#srchForm")

    if not form:
        raise ValueError(
            "BI-004 검색 폼 form#srchForm을 찾지 못했습니다."
        )

    payload: dict[str, str] = {}

    for field in form.select(
        "input[name], select[name], textarea[name]"
    ):
        name = normalize_space(field.get("name", ""))
        if not name:
            continue

        if field.name == "select":
            selected = field.find("option", selected=True)
            if selected is None:
                selected = field.find("option")
            value = (
                selected.get("value", "")
                if selected
                else ""
            )

        elif field.name == "textarea":
            value = field.get_text("", strip=False)

        else:
            field_type = normalize_space(
                field.get("type", "text")
            ).lower()

            if (
                field_type in {"checkbox", "radio"}
                and not field.has_attr("checked")
            ):
                continue

            value = field.get("value", "")

        payload[name] = str(value or "")

    # 전체 금융권역·전체 회사 조회
    payload["fncSareaCd"] = ""
    payload["searchInfnstNm"] = ""

    return payload


def bi004_last_internal_page_index(
    html_bytes: bytes,
    encoding: str,
) -> int:
    """
    BI-004는 curPage가 0부터 시작합니다.
    화면의 1페이지는 curPage=0, 화면의 2페이지는 curPage=1입니다.
    """
    soup = BeautifulSoup(
        html_bytes.decode(encoding, errors="replace"),
        "lxml",
    )

    page_indexes: list[int] = []

    for node in soup.select(
        ".paging [onclick*='chgPagingNo']"
    ):
        onclick = node.get("onclick", "")
        match = re.search(
            r"chgPagingNo\(\s*(\d+)\s*\)",
            onclick,
        )
        if match:
            page_indexes.append(int(match.group(1)))

    return max(page_indexes, default=0)


def bi004_displayed_page_number(
    html_bytes: bytes,
    encoding: str,
) -> int | None:
    soup = BeautifulSoup(
        html_bytes.decode(encoding, errors="replace"),
        "lxml",
    )

    current = soup.select_one(
        ".paging strong[title*='현재 페이지'] span"
    ) or soup.select_one(".paging strong span")

    if not current:
        return None

    text = normalize_space(current.get_text(" ", strip=True))
    return int(text) if text.isdigit() else None


def bi004_parse_result_table(
    html_bytes: bytes,
    encoding: str,
) -> tuple[list[str], list[list[str]]]:
    """금융회사 조회 결과 표 하나를 구조화합니다."""
    soup = BeautifulSoup(
        html_bytes.decode(encoding, errors="replace"),
        "lxml",
    )

    root = (
        soup.select_one(".contents")
        or soup.select_one("#contents")
        or soup.select_one("main")
        or soup.body
    )

    if not root:
        raise ValueError("BI-004 본문 영역을 찾지 못했습니다.")

    candidate_tables = root.find_all("table")
    target_table = None

    for table in candidate_tables:
        table_text = normalize_space(
            table.get_text(" ", strip=True)
        )
        if (
            "금융권역" in table_text
            and "금융회사명" in table_text
        ):
            target_table = table
            break

    if target_table is None:
        raise ValueError(
            "BI-004 금융회사 결과 표를 찾지 못했습니다."
        )

    headers, rows = fixed_expand_table(target_table)

    cleaned_rows = [
        [normalize_space(value) for value in row]
        for row in rows
        if any(normalize_space(value) for value in row)
    ]

    if not headers:
        raise ValueError("BI-004 결과 표의 헤더가 없습니다.")

    return headers, cleaned_rows


def bi004_replace_document_table(
    document: dict[str, Any],
    headers: list[str],
    rows: list[list[str]],
) -> None:
    """첫 페이지 표를 전체 페이지 통합 표로 교체합니다."""
    new_blocks: list[dict[str, Any]] = []
    replaced = False

    for block in document.get("blocks", []):
        if block.get("type") != "table":
            new_blocks.append(block)
            continue

        block_headers = " ".join(
            block.get("headers", [])
        )

        if (
            not replaced
            and "금융권역" in block_headers
            and "금융회사명" in block_headers
        ):
            new_blocks.append(
                {
                    "type": "table",
                    "headers": headers,
                    "rows": rows,
                    "row_count": len(rows),
                    "table_role": (
                        "deposit_insurance_payment_target_companies"
                    ),
                }
            )
            replaced = True
        else:
            new_blocks.append(block)

    if not replaced:
        new_blocks.append(
            {
                "type": "table",
                "headers": headers,
                "rows": rows,
                "row_count": len(rows),
                "table_role": (
                    "deposit_insurance_payment_target_companies"
                ),
            }
        )

    document["blocks"] = new_blocks
    document["content_markdown"] = fixed_render_blocks(
        new_blocks
    )
    document["content_text"] = fixed_blocks_text(
        new_blocks
    )

    parsed_hash = sha256_text(
        document["content_markdown"]
    )
    document["parsed_content_sha256"] = parsed_hash
    document["content_hash"] = parsed_hash

    quality = document.setdefault("quality", {})
    quality["parsed_text_chars"] = len(
        document["content_text"]
    )
    quality["table_count"] = sum(
        1
        for block in new_blocks
        if block.get("type") == "table"
    )


def collect_bi004_all_pages(
    first_result: FetchResult,
    manifest_row: dict[str, str],
) -> dict[str, Any]:
    """
    첫 GET 응답을 1페이지로 사용하고,
    curPage=1부터 마지막 내부 인덱스까지 POST 요청합니다.
    """
    first_headers, first_rows = bi004_parse_result_table(
        first_result.body,
        first_result.encoding,
    )

    last_index = bi004_last_internal_page_index(
        first_result.body,
        first_result.encoding,
    )
    page_count = last_index + 1

    if page_count < 1:
        raise ValueError("BI-004 페이지 수가 1보다 작습니다.")

    if page_count > BI004_MAX_PAGES:
        raise ValueError(
            f"BI-004 페이지 수 {page_count}가 "
            f"안전 한도 {BI004_MAX_PAGES}를 초과했습니다."
        )

    base_payload = bi004_extract_base_payload(
        first_result.body,
        first_result.encoding,
    )

    pages_dir = (
        PATHS["dynamic_diagnostics"]
        / "BI-004_pages"
    )
    pages_dir.mkdir(parents=True, exist_ok=True)

    combined_rows: list[list[str]] = []
    seen_rows: set[tuple[str, ...]] = set()
    seen_page_hashes: dict[str, int] = {}
    page_records: list[dict[str, Any]] = []
    failures: list[dict[str, Any]] = []

    def register_page(
        internal_index: int,
        body: bytes,
        encoding: str,
        status_code: int,
        final_url: str,
        request_method: str,
        request_payload: dict[str, str] | None,
    ) -> None:
        displayed_page = bi004_displayed_page_number(
            body,
            encoding,
        )
        headers, rows = bi004_parse_result_table(
            body,
            encoding,
        )

        if headers != first_headers:
            failures.append(
                {
                    "internal_page_index": internal_index,
                    "reason": "페이지별 표 헤더가 서로 다름",
                    "headers": headers,
                }
            )

        row_hash = sha256_text(
            json.dumps(
                rows,
                ensure_ascii=False,
                sort_keys=True,
            )
        )

        if row_hash in seen_page_hashes:
            failures.append(
                {
                    "internal_page_index": internal_index,
                    "reason": "다른 페이지와 동일한 표 결과 반복",
                    "same_as_internal_index": (
                        seen_page_hashes[row_hash]
                    ),
                }
            )
        else:
            seen_page_hashes[row_hash] = internal_index

        page_filename = (
            f"page_{internal_index + 1:03d}.html"
        )
        page_path = pages_dir / page_filename
        page_path.write_bytes(body)

        page_meta = {
            "internal_page_index": internal_index,
            "displayed_page_number": displayed_page,
            "status_code": status_code,
            "request_method": request_method,
            "request_payload": request_payload,
            "final_url": final_url,
            "row_count": len(rows),
            "row_hash": row_hash,
            "raw_sha256": sha256_bytes(body),
            "raw_html_path": str(
                page_path.relative_to(OUTPUT_ROOT)
            ),
        }
        page_records.append(page_meta)

        expected_displayed_page = internal_index + 1
        if (
            displayed_page is not None
            and displayed_page != expected_displayed_page
        ):
            failures.append(
                {
                    "internal_page_index": internal_index,
                    "reason": (
                        "요청 페이지와 화면 현재 페이지 불일치"
                    ),
                    "expected_displayed_page": (
                        expected_displayed_page
                    ),
                    "actual_displayed_page": displayed_page,
                }
            )

        for row in rows:
            key = tuple(row)
            if key not in seen_rows:
                seen_rows.add(key)
                combined_rows.append(row)

    # GET으로 확보한 첫 페이지
    register_page(
        internal_index=0,
        body=first_result.body,
        encoding=first_result.encoding,
        status_code=first_result.status_code,
        final_url=first_result.final_url,
        request_method="GET",
        request_payload=None,
    )

    # 화면 2페이지부터 마지막 페이지까지 반복 POST
    for internal_index in range(1, page_count):
        payload = dict(base_payload)
        payload["curPage"] = str(internal_index)
        payload["pageSize"] = str(BI004_PAGE_SIZE)

        try:
            response = SESSION.post(
                manifest_row["source_url"],
                data=payload,
                headers={
                    "Referer": manifest_row["source_url"],
                },
                timeout=REQUEST_TIMEOUT_SECONDS,
                allow_redirects=True,
            )

            response.raise_for_status()

            encoding = choose_response_encoding(response)
            response.encoding = encoding

            register_page(
                internal_index=internal_index,
                body=response.content,
                encoding=encoding,
                status_code=response.status_code,
                final_url=response.url,
                request_method="POST",
                request_payload=payload,
            )

        except Exception as error:
            failures.append(
                {
                    "internal_page_index": internal_index,
                    "reason": "페이지 요청 또는 파싱 실패",
                    "error_type": type(error).__name__,
                    "error": str(error),
                }
            )

        time.sleep(REQUEST_DELAY_SECONDS)

    fetched_indexes = {
        record["internal_page_index"]
        for record in page_records
    }
    expected_indexes = set(range(page_count))

    missing_indexes = sorted(
        expected_indexes - fetched_indexes
    )

    if missing_indexes:
        failures.append(
            {
                "reason": "수집되지 않은 페이지 존재",
                "missing_internal_indexes": missing_indexes,
            }
        )

    is_complete = (
        not failures
        and len(page_records) == page_count
        and len(combined_rows) > 0
    )

    all_rows_csv = (
        PATHS["dynamic_diagnostics"]
        / "BI-004_all_rows.csv"
    )
    pd.DataFrame(
        combined_rows,
        columns=first_headers,
    ).to_csv(
        all_rows_csv,
        index=False,
        encoding="utf-8-sig",
    )

    return {
        "headers": first_headers,
        "rows": combined_rows,
        "collection": {
            "collection_scope": (
                "all_public_pages"
                if is_complete
                else "partial_public_pages"
            ),
            "is_complete": is_complete,
            "page_index_base": 0,
            "expected_page_count": page_count,
            "fetched_page_count": len(page_records),
            "row_count": len(combined_rows),
            "page_size": BI004_PAGE_SIZE,
            "search_filters": {
                "fncSareaCd": "",
                "searchInfnstNm": "",
            },
            "pages": page_records,
            "failures": failures,
            "all_rows_csv_path": str(
                all_rows_csv.relative_to(OUTPUT_ROOT)
            ),
            "collected_at": now_kst_iso(),
        },
    }


def ensure_playwright_installed() -> None:
    try:
        import playwright  # noqa: F401
        return
    except ImportError:
        pass

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "playwright"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "playwright", "install", "chromium"],
        check=True,
    )


def save_playwright_snapshot(
    url: str,
    url_id: str,
) -> dict[str, Any] | None:
    if not ENABLE_PLAYWRIGHT_SNAPSHOT:
        return None

    ensure_playwright_installed()
    from playwright.sync_api import sync_playwright

    html_path = PATHS["dynamic_diagnostics"] / f"{url_id}_rendered.html"
    screenshot_path = PATHS["screenshots"] / f"{url_id}.png"
    ensure_parent(html_path)
    ensure_parent(screenshot_path)

    with sync_playwright() as playwright:
        browser = playwright.chromium.launch(headless=True)
        page = browser.new_page(
            user_agent=USER_AGENT,
            locale="ko-KR",
            viewport={"width": 1440, "height": 1200},
        )

        response = page.goto(
            url,
            wait_until="domcontentloaded",
            timeout=REQUEST_TIMEOUT_SECONDS * 1000,
        )
        page.wait_for_timeout(PLAYWRIGHT_WAIT_MS)

        html = page.content()
        html_path.write_text(html, encoding="utf-8")
        page.screenshot(path=str(screenshot_path), full_page=True)

        result = {
            "status_code": response.status if response else None,
            "final_url": page.url,
            "rendered_html_path": str(html_path.relative_to(OUTPUT_ROOT)),
            "screenshot_path": str(screenshot_path.relative_to(OUTPUT_ROOT)),
            "captured_at": now_kst_iso(),
        }

        browser.close()
        return result

### 동적 페이지 진단 코드 자체 테스트

실제 웹 요청 전에 다음을 검사합니다.

- `fetch(...)`
- `XMLHttpRequest`
- `.do` 엔드포인트
- `pageIndex`, `pageNo`, `search`
- 합성 HTML의 폼·인라인 스크립트 추출

이 셀이 실패하면 실제 크롤링을 실행하지 마십시오.

In [ ]:
_DYNAMIC_PATTERN_TEST_CASES = {
    "fetch('/api/list')": True,
    "const xhr = new XMLHttpRequest();": True,
    "url: '/board/selectList.do'": True,
    "pageIndex = 2": True,
    "pageNo = 3": True,
    "searchKeyword": True,
    "일반 안내 문장입니다.": False,
}

for sample_text, expected in _DYNAMIC_PATTERN_TEST_CASES.items():
    actual = bool(DYNAMIC_SCRIPT_KEYWORD_PATTERN.search(sample_text))
    assert actual == expected, (
        f"동적 스크립트 정규표현식 테스트 실패: "
        f"text={sample_text!r}, expected={expected}, actual={actual}"
    )

_DYNAMIC_TEST_HTML = """
<!doctype html>
<html lang="ko">
<body>
  <form id="searchForm" method="post" action="/search/list.do">
    <input type="hidden" name="pageIndex" value="1">
    <input type="text" name="searchKeyword" value="">
    <select name="category">
      <option value="">전체</option>
      <option value="bank">은행</option>
    </select>
  </form>

  <script>
    function loadData() {
      fetch('/api/list?pageNo=1');
    }
  </script>
</body>
</html>
""".encode("utf-8")

_dynamic_test_result = extract_dynamic_diagnostic(
    html_bytes=_DYNAMIC_TEST_HTML,
    final_url="https://example.org/dynamic/page",
    encoding="utf-8",
)

assert len(_dynamic_test_result["forms"]) == 1
assert _dynamic_test_result["forms"][0]["method"] == "POST"
assert _dynamic_test_result["forms"][0]["action"] == (
    "https://example.org/search/list.do"
)
assert any(
    item["name"] == "pageIndex"
    for item in _dynamic_test_result["forms"][0]["inputs"]
)
assert len(_dynamic_test_result["inline_script_hints"]) == 1

print("동적 페이지 진단 자체 테스트: 통과")
print("정규표현식:", DYNAMIC_SCRIPT_KEYWORD_PATTERN.pattern)


_BI004_PAGINATION_TEST_HTML = """
<!doctype html>
<html lang="ko">
<body>
  <form id="srchForm" method="post">
    <select name="fncSareaCd">
      <option value="" selected>전체</option>
    </select>
    <input name="searchInfnstNm" type="text" value="">
  </form>

  <div class="contents">
    <table>
      <thead>
        <tr>
          <th>금융권역</th>
          <th>금융회사명</th>
          <th>상태</th>
        </tr>
      </thead>
      <tbody>
        <tr>
          <td>상호저축은행</td>
          <td>예시은행</td>
          <td>파산</td>
        </tr>
      </tbody>
    </table>

    <div class="paging">
      <strong title="현재 페이지"><span>1</span></strong>
      <a onclick="chgPagingNo(1);">2</a>
      <a onclick="chgPagingNo(2);">3</a>
      <a class="btnLast" onclick="chgPagingNo(2);">
        마지막 페이지
      </a>
    </div>
  </div>
</body>
</html>
""".encode("utf-8")

assert bi004_last_internal_page_index(
    _BI004_PAGINATION_TEST_HTML,
    "utf-8",
) == 2

assert bi004_displayed_page_number(
    _BI004_PAGINATION_TEST_HTML,
    "utf-8",
) == 1

_test_headers, _test_rows = bi004_parse_result_table(
    _BI004_PAGINATION_TEST_HTML,
    "utf-8",
)

assert _test_headers == [
    "금융권역",
    "금융회사명",
    "상태",
]
assert len(_test_rows) == 1
assert _test_rows[0][1] == "예시은행"

_test_payload = bi004_extract_base_payload(
    _BI004_PAGINATION_TEST_HTML,
    "utf-8",
)
assert _test_payload["fncSareaCd"] == ""
assert _test_payload["searchInfnstNm"] == ""

print("BI-004 페이지네이션 자체 테스트: 통과")


## 9. URL 유형별 처리기

In [ ]:

def save_parsed_document(
    document: dict[str, Any],
    manifest_row: dict[str, str],
) -> tuple[Path, Path]:
    business_dir = safe_name(manifest_row["business_function"])
    url_id = manifest_row["url_id"]

    markdown_path = (
        PATHS["parsed_markdown"]
        / business_dir
        / f"{url_id}.md"
    )
    json_path = (
        PATHS["parsed_json"]
        / business_dir
        / f"{url_id}.json"
    )

    ensure_parent(markdown_path)
    markdown_path.write_text(
        document["content_markdown"],
        encoding="utf-8",
    )
    write_json(json_path, document)

    return markdown_path, json_path


def process_link_only(
    manifest_row: dict[str, str],
) -> dict[str, Any]:
    record = {
        "doc_id": manifest_row["url_id"],
        "record_type": "link_only",
        "title": manifest_row["page_title"],
        "source_url": manifest_row["source_url"],
        "site_name": manifest_row["site_name"],
        "business_function": manifest_row["business_function"],
        "target_business_function": manifest_row.get(
            "target_business_function",
            manifest_row["business_function"],
        ),
        "page_type": manifest_row["page_type"],
        "auth_required": manifest_row["auth_required"],
        "content_scope": manifest_row["content_scope"],
        "description": manifest_row.get("content_summary", ""),
        "decision_reason": manifest_row["decision_reason"],
        "created_at": now_kst_iso(),
    }

    output_path = (
        PATHS["parsed_json"]
        / safe_name(manifest_row["business_function"])
        / f"{manifest_row['url_id']}.json"
    )
    write_json(output_path, record)

    return {
        "url_id": manifest_row["url_id"],
        "business_function": manifest_row["business_function"],
        "page_title": manifest_row["page_title"],
        "source_url": manifest_row["source_url"],
        "normalized_decision": manifest_row["normalized_decision"],
        "crawl_wave": manifest_row["crawl_wave"],
        "status": "link_metadata_created",
        "status_code": None,
        "content_chars": len(record["description"]),
        "attachment_count": 0,
        "image_count": 0,
        "asset_failure_count": 0,
        "error_type": "",
        "error": "",
        "finished_at": now_kst_iso(),
    }


def process_static_or_asset(
    manifest_row: dict[str, str],
) -> dict[str, Any]:
    result = fetch_url(manifest_row["source_url"])
    html_path, meta_path = save_fetch_result(manifest_row, result)

    if not result.ok:
        raise requests.HTTPError(
            f"HTTP {result.status_code}: {result.final_url}"
        )

    if "html" not in result.content_type.lower():
        raise ValueError(
            f"HTML 응답이 아닙니다: {result.content_type}"
        )

    document = parse_html_document(
        html_bytes=result.body,
        final_url=result.final_url,
        manifest_row=manifest_row,
        encoding=result.encoding,
    )

    asset_results = download_document_assets(
        document=document,
        manifest_row=manifest_row,
    )
    document["downloaded_assets"] = asset_results

    markdown_path, json_path = save_parsed_document(
        document=document,
        manifest_row=manifest_row,
    )

    return {
        "url_id": manifest_row["url_id"],
        "business_function": manifest_row["business_function"],
        "page_title": document["title"],
        "source_url": manifest_row["source_url"],
        "final_url": result.final_url,
        "normalized_decision": manifest_row["normalized_decision"],
        "crawl_wave": manifest_row["crawl_wave"],
        "status": (
            "parse_success"
            if document["content_text"]
            else "empty_content"
        ),
        "quality_status": document.get("quality", {}).get("status", ""),
        "quality_reasons": " | ".join(document.get("quality", {}).get("reasons", [])),
        "retention_ratio": document.get("quality", {}).get("retention_ratio", 0),
        "faq_count": document.get("quality", {}).get("faq_count", 0),
        "table_count": document.get("quality", {}).get("table_count", 0),
        "attachment_button_count": document.get("quality", {}).get("attachment_button_count", 0),
        "status_code": result.status_code,
        "content_chars": len(document["content_text"]),
        "block_count": len(document["blocks"]),
        "link_count": len(document["links"]),
        "attachment_count": len(document["attachments"]),
        "image_count": len(document["images"]),
        "downloaded_attachment_count": len(asset_results["attachments"]),
        "downloaded_image_count": len(asset_results["images"]),
        "asset_failure_count": len(asset_results["failures"]),
        "raw_html_path": str(html_path.relative_to(OUTPUT_ROOT)),
        "response_meta_path": str(meta_path.relative_to(OUTPUT_ROOT)),
        "parsed_markdown_path": str(markdown_path.relative_to(OUTPUT_ROOT)),
        "parsed_json_path": str(json_path.relative_to(OUTPUT_ROOT)),
        "raw_sha256": result.raw_sha256,
        "parsed_content_sha256": document["parsed_content_sha256"],
        "error_type": "",
        "error": "",
        "finished_at": now_kst_iso(),
    }


def process_dynamic_diagnostic(
    manifest_row: dict[str, str],
) -> dict[str, Any]:
    result = fetch_url(manifest_row["source_url"])
    html_path, meta_path = save_fetch_result(
        manifest_row,
        result,
    )

    if not result.ok:
        raise requests.HTTPError(
            f"HTTP {result.status_code}: {result.final_url}"
        )

    diagnostic = extract_dynamic_diagnostic(
        html_bytes=result.body,
        final_url=result.final_url,
        encoding=result.encoding,
    )
    diagnostic["url_id"] = manifest_row["url_id"]
    diagnostic["page_title"] = manifest_row["page_title"]
    diagnostic["business_function"] = (
        manifest_row["business_function"]
    )
    diagnostic["source_url"] = manifest_row["source_url"]

    playwright_snapshot = save_playwright_snapshot(
        url=manifest_row["source_url"],
        url_id=manifest_row["url_id"],
    )
    if playwright_snapshot:
        diagnostic["playwright_snapshot"] = (
            playwright_snapshot
        )

    document = parse_html_document(
        html_bytes=result.body,
        final_url=result.final_url,
        manifest_row=manifest_row,
        encoding=result.encoding,
    )

    dynamic_status = "dynamic_diagnostic_created"
    dynamic_page_count = 1
    dynamic_row_count = 0
    dynamic_is_complete = False

    if (
        manifest_row["url_id"] == "BI-004"
        and FETCH_BI004_ALL_PAGES
    ):
        collection_result = collect_bi004_all_pages(
            first_result=result,
            manifest_row=manifest_row,
        )

        bi004_replace_document_table(
            document=document,
            headers=collection_result["headers"],
            rows=collection_result["rows"],
        )

        document["dynamic_collection"] = (
            collection_result["collection"]
        )
        diagnostic["bi004_full_collection"] = (
            collection_result["collection"]
        )

        dynamic_page_count = collection_result[
            "collection"
        ]["fetched_page_count"]
        dynamic_row_count = collection_result[
            "collection"
        ]["row_count"]
        dynamic_is_complete = collection_result[
            "collection"
        ]["is_complete"]

        dynamic_status = (
            "dynamic_full_collection_created"
            if dynamic_is_complete
            else "dynamic_collection_incomplete"
        )

    else:
        document["dynamic_collection"] = {
            "collection_scope": (
                "initial_public_page_only"
            ),
            "is_complete": False,
            "expected_page_count": None,
            "fetched_page_count": 1,
            "row_count": None,
            "page_size": BI004_PAGE_SIZE,
        }
        diagnostic["bi004_full_collection"] = (
            document["dynamic_collection"]
        )

    diagnostic_path = (
        PATHS["dynamic_diagnostics"]
        / f"{manifest_row['url_id']}.json"
    )
    write_json(diagnostic_path, diagnostic)

    markdown_path, json_path = save_parsed_document(
        document=document,
        manifest_row=manifest_row,
    )

    collection_failures = (
        document.get("dynamic_collection", {})
        .get("failures", [])
    )

    return {
        "url_id": manifest_row["url_id"],
        "business_function": (
            manifest_row["business_function"]
        ),
        "page_title": document["title"],
        "source_url": manifest_row["source_url"],
        "final_url": result.final_url,
        "normalized_decision": (
            manifest_row["normalized_decision"]
        ),
        "crawl_wave": manifest_row["crawl_wave"],
        "status": dynamic_status,
        "quality_status": (
            document.get("quality", {}).get(
                "status",
                "",
            )
        ),
        "quality_reasons": " | ".join(
            document.get("quality", {}).get(
                "reasons",
                [],
            )
        ),
        "retention_ratio": (
            document.get("quality", {}).get(
                "retention_ratio",
                0,
            )
        ),
        "faq_count": (
            document.get("quality", {}).get(
                "faq_count",
                0,
            )
        ),
        "table_count": (
            document.get("quality", {}).get(
                "table_count",
                0,
            )
        ),
        "attachment_button_count": (
            document.get("quality", {}).get(
                "attachment_button_count",
                0,
            )
        ),
        "status_code": result.status_code,
        "content_chars": len(
            document["content_text"]
        ),
        "form_count": len(diagnostic["forms"]),
        "script_source_count": len(
            diagnostic["script_sources"]
        ),
        "dynamic_is_complete": (
            dynamic_is_complete
        ),
        "dynamic_page_count": (
            dynamic_page_count
        ),
        "dynamic_row_count": (
            dynamic_row_count
        ),
        "dynamic_failure_count": len(
            collection_failures
        ),
        "attachment_count": len(
            document["attachments"]
        ),
        "image_count": len(
            document["images"]
        ),
        "asset_failure_count": 0,
        "raw_html_path": str(
            html_path.relative_to(OUTPUT_ROOT)
        ),
        "response_meta_path": str(
            meta_path.relative_to(OUTPUT_ROOT)
        ),
        "parsed_markdown_path": str(
            markdown_path.relative_to(OUTPUT_ROOT)
        ),
        "parsed_json_path": str(
            json_path.relative_to(OUTPUT_ROOT)
        ),
        "dynamic_diagnostic_path": str(
            diagnostic_path.relative_to(OUTPUT_ROOT)
        ),
        "error_type": (
            ""
            if dynamic_is_complete
            or not FETCH_BI004_ALL_PAGES
            else "IncompleteDynamicCollection"
        ),
        "error": (
            ""
            if dynamic_is_complete
            or not FETCH_BI004_ALL_PAGES
            else json.dumps(
                collection_failures,
                ensure_ascii=False,
            )
        ),
        "finished_at": now_kst_iso(),
    }

def process_manifest_row(
    manifest_row: dict[str, str],
) -> dict[str, Any]:
    decision = manifest_row["normalized_decision"]
    strategy = manifest_row["fetch_strategy"]

    if decision == "link_only":
        return process_link_only(manifest_row)

    if strategy == "dynamic_http_then_playwright":
        return process_dynamic_diagnostic(manifest_row)

    return process_static_or_asset(manifest_row)


## 10. 크롤링 실행

In [ ]:

CRAWL_RESULTS_JSONL = PATHS["logs"] / "crawl_results.jsonl"
CRAWL_RESULTS_CSV = PATHS["logs"] / "crawl_results.csv"
USED_MANIFEST_PATH = PATHS["manifest"] / "manifest_used.csv"

target_df.to_csv(
    USED_MANIFEST_PATH,
    index=False,
    encoding="utf-8-sig",
)

run_results: list[dict[str, Any]] = []

for _, row in tqdm(
    target_df.iterrows(),
    total=len(target_df),
    desc="크롤링 진행",
):
    manifest_row = row_to_clean_dict(row)
    started_at = now_kst_iso()

    try:
        result = process_manifest_row(manifest_row)
        result["started_at"] = started_at

    except Exception as error:
        result = {
            "url_id": manifest_row["url_id"],
            "business_function": manifest_row["business_function"],
            "page_title": manifest_row["page_title"],
            "source_url": manifest_row["source_url"],
            "normalized_decision": manifest_row["normalized_decision"],
            "crawl_wave": manifest_row["crawl_wave"],
            "status": "failed",
            "status_code": None,
            "content_chars": 0,
            "attachment_count": 0,
            "image_count": 0,
            "asset_failure_count": 0,
            "error_type": type(error).__name__,
            "error": str(error),
            "started_at": started_at,
            "finished_at": now_kst_iso(),
        }

    run_results.append(result)
    append_jsonl(CRAWL_RESULTS_JSONL, result)

    # link_only는 실제 HTTP 요청을 보내지 않으므로 대기하지 않습니다.
    if manifest_row["normalized_decision"] != "link_only":
        time.sleep(REQUEST_DELAY_SECONDS)

results_df = pd.DataFrame(run_results)
results_df.to_csv(
    CRAWL_RESULTS_CSV,
    index=False,
    encoding="utf-8-sig",
)

print("실행 완료")

for optional_column, default_value in {
    "dynamic_is_complete": None,
    "dynamic_page_count": None,
    "dynamic_row_count": None,
    "dynamic_failure_count": None,
}.items():
    if optional_column not in results_df.columns:
        results_df[optional_column] = default_value

display(
    results_df[
        [
            "url_id",
            "business_function",
            "status",
            "status_code",
            "content_chars",
            "attachment_count",
            "image_count",
            "asset_failure_count",
            "dynamic_is_complete",
            "dynamic_page_count",
            "dynamic_row_count",
            "dynamic_failure_count",
            "error_type",
        ]
    ]
)


## 11. 결과 검수

In [ ]:
QUALITY_REPORT_CSV = PATHS["logs"] / "quality_report.csv"

if results_df.empty:
    print("실행 결과가 없습니다.")
else:
    print("수집 상태별 건수")
    display(
        results_df.groupby("status")
        .size()
        .rename("count")
        .reset_index()
        .sort_values("count", ascending=False)
    )

    failures = results_df[results_df["status"] == "failed"].copy()
    empty_contents = results_df[
        results_df["status"].isin({"empty_content"})
        | (
            results_df["status"].isin(
                {"parse_success", "dynamic_diagnostic_created"}
            )
            & (results_df["content_chars"].fillna(0).astype(int) < 80)
        )
    ].copy()

    quality_columns = [
        "url_id", "business_function", "page_title", "status",
        "quality_status", "quality_reasons", "retention_ratio",
        "content_chars", "faq_count", "table_count",
        "attachment_button_count", "attachment_count",
        "image_count", "asset_failure_count", "error_type", "error",
    ]
    available_quality_columns = [
        column for column in quality_columns if column in results_df.columns
    ]
    quality_report_df = results_df[available_quality_columns].copy()
    quality_report_df.to_csv(
        QUALITY_REPORT_CSV,
        index=False,
        encoding="utf-8-sig",
    )

    print("실패:", len(failures))
    print("빈 본문·매우 짧은 본문:", len(empty_contents))

    if "quality_status" in results_df.columns:
        print("파싱 품질 상태")
        display(
            results_df.groupby("quality_status", dropna=False)
            .size()
            .rename("count")
            .reset_index()
        )

        review_required = results_df[
            results_df["quality_status"].isin({"review", "fail"})
        ]
        if not review_required.empty:
            print("사람 검수 또는 파서 수정 필요")
            display(
                review_required[
                    [
                        "url_id", "page_title", "quality_status",
                        "quality_reasons", "retention_ratio",
                        "content_chars",
                    ]
                ]
            )

    if not failures.empty:
        display(
            failures[
                ["url_id", "page_title", "source_url", "error_type", "error"]
            ]
        )

    success_statuses = {
        "parse_success",
        "dynamic_diagnostic_created",
        "link_metadata_created",
    }
    success_count = results_df["status"].isin(success_statuses).sum()
    success_rate = success_count / len(results_df) * 100
    print(f"수집 처리 성공률: {success_count}/{len(results_df)} ({success_rate:.1f}%)")
    print("품질 보고서:", QUALITY_REPORT_CSV)

## 12. 페이지 문서 JSONL과 기본 청크 생성

In [ ]:
DOCUMENTS_JSONL = PATHS["processed"] / "documents.jsonl"
CHUNKS_JSONL = PATHS["processed"] / "chunks.jsonl"


def iter_parsed_json_files() -> Iterable[Path]:
    yield from sorted(PATHS["parsed_json"].rglob("*.json"))


def build_document_record(data: dict[str, Any]) -> dict[str, Any] | None:
    if data.get("record_type") == "link_only":
        return {
            "doc_id": data["doc_id"],
            "parent_doc_id": data["doc_id"],
            "record_type": "link_only",
            "indexable": False,
            "title": data["title"],
            "source_url": data["source_url"],
            "site_name": data["site_name"],
            "business_function": data["business_function"],
            "target_business_function": data["target_business_function"],
            "page_type": data["page_type"],
            "content_markdown": data.get("description", ""),
            "content_text": data.get("description", ""),
            "content_hash": sha256_text(data.get("description", "")),
            "blocks": [],
            "attachments": [],
            "images": [],
            "links": [
                {
                    "url": data["source_url"],
                    "anchor_text": data["title"],
                    "link_type": "official_service",
                }
            ],
            "metadata": {
                "auth_required": data.get("auth_required", ""),
                "content_scope": data.get("content_scope", ""),
                "decision_reason": data.get("decision_reason", ""),
            },
        }

    if not data.get("content_markdown"):
        return None

    return {
        "doc_id": data["doc_id"],
        "parent_doc_id": data.get("parent_doc_id", data["doc_id"]),
        "record_type": "page",
        "indexable": True,
        "title": data.get("title", ""),
        "source_url": data.get("source_url", ""),
        "final_url": data.get("final_url", ""),
        "site_name": data.get("site_name", ""),
        "business_function": data.get("business_function", ""),
        "target_business_function": data.get(
            "target_business_function",
            data.get("business_function", ""),
        ),
        "page_type": data.get("page_type", ""),
        "breadcrumb": data.get("breadcrumb", []),
        "content_markdown": data.get("content_markdown", ""),
        "content_text": data.get("content_text", ""),
        "content_hash": data.get(
            "parsed_content_sha256",
            sha256_text(data.get("content_markdown", "")),
        ),
        "blocks": data.get("blocks", []),
        "attachments": data.get("attachments", []),
        "images": data.get("images", []),
        "links": data.get("links", []),
        "quality": data.get("quality", {}),
        "dynamic_collection": data.get("dynamic_collection"),
        "metadata": {
            "parser_profile": data.get("parser_profile", ""),
            "attachment_count": len(data.get("attachments", [])),
            "image_count": len(data.get("images", [])),
            "parsed_at": data.get("parsed_at", ""),
        },
    }


def render_table_chunk(
    headers: list[str],
    rows: list[list[str]],
) -> str:
    block = {"type": "table", "headers": headers, "rows": rows}
    return fixed_render_blocks([block])


def split_table_block(
    block: dict[str, Any],
    max_chars: int,
) -> list[str]:
    headers = block.get("headers", [])
    rows = block.get("rows", [])
    all_text = render_table_chunk(headers, rows)
    if len(all_text) <= max_chars:
        return [all_text]

    chunks: list[str] = []
    current_rows: list[list[str]] = []

    for row in rows:
        candidate_rows = current_rows + [row]
        candidate_text = render_table_chunk(headers, candidate_rows)

        if current_rows and len(candidate_text) > max_chars:
            chunks.append(render_table_chunk(headers, current_rows))
            current_rows = [row]
        else:
            current_rows = candidate_rows

    if current_rows:
        chunks.append(render_table_chunk(headers, current_rows))

    return chunks


def structure_aware_chunks(
    document: dict[str, Any],
    max_chars: int = 1600,
) -> list[dict[str, Any]]:
    if not document.get("indexable", True):
        return []

    intermediate: list[dict[str, str]] = []
    current_parts: list[str] = []
    current_heading = ""

    def flush_section() -> None:
        nonlocal current_parts
        content = "\n\n".join(
            part for part in current_parts if part
        ).strip()
        current_parts = []
        if content:
            intermediate.append(
                {
                    "section_title": current_heading,
                    "chunk_type": "section",
                    "content": content,
                }
            )

    for block in document.get("blocks", []):
        block_type = block.get("type")

        if block_type == "heading":
            flush_section()
            current_heading = block.get("text", "")
            current_parts = [fixed_render_blocks([block])]
            continue

        if block_type == "faq":
            flush_section()
            intermediate.append(
                {
                    "section_title": block.get("question", ""),
                    "chunk_type": "faq",
                    "content": fixed_render_blocks([block]),
                }
            )
            continue

        if block_type == "table":
            flush_section()
            for table_text in split_table_block(block, max_chars):
                intermediate.append(
                    {
                        "section_title": current_heading,
                        "chunk_type": "table",
                        "content": table_text,
                    }
                )
            continue

        block_text = fixed_render_blocks([block])
        candidate = "\n\n".join(current_parts + [block_text]).strip()
        if current_parts and len(candidate) > max_chars:
            flush_section()
        current_parts.append(block_text)

    flush_section()

    records: list[dict[str, Any]] = []
    for index, item in enumerate(intermediate):
        content = item["content"].strip()
        if not content:
            continue

        records.append(
            {
                "chunk_id": f"{document['doc_id']}_chunk_{index:03d}",
                "parent_doc_id": document["doc_id"],
                "chunk_index": index,
                "title": document["title"],
                "section_title": item["section_title"],
                "chunk_type": item["chunk_type"],
                "business_function": document["business_function"],
                "target_business_function": document[
                    "target_business_function"
                ],
                "page_type": document["page_type"],
                "source_url": document["source_url"],
                "content": content,
                "content_hash": sha256_text(content),
            }
        )

    return records


documents: list[dict[str, Any]] = []

for json_path in iter_parsed_json_files():
    data = json.loads(json_path.read_text(encoding="utf-8"))
    document = build_document_record(data)
    if document:
        documents.append(document)

with DOCUMENTS_JSONL.open("w", encoding="utf-8") as file:
    for document in documents:
        file.write(json.dumps(document, ensure_ascii=False) + "\n")

chunk_records: list[dict[str, Any]] = []

if CREATE_BASELINE_CHUNKS:
    for document in documents:
        chunk_records.extend(
            structure_aware_chunks(
                document,
                max_chars=max(1600, CHUNK_MAX_CHARS),
            )
        )

with CHUNKS_JSONL.open("w", encoding="utf-8") as file:
    for chunk in chunk_records:
        file.write(json.dumps(chunk, ensure_ascii=False) + "\n")

print("페이지 문서:", len(documents))
print("구조 기반 청크:", len(chunk_records))

if chunk_records:
    chunk_df = pd.DataFrame(chunk_records)
    display(
        chunk_df.groupby("chunk_type")
        .size()
        .rename("count")
        .reset_index()
    )

# 핵심 회귀 검사: 기존 분석에서 누락됐던 내용 확인
by_id = {document["doc_id"]: document for document in documents}
validation_rows: list[dict[str, Any]] = []

if "DP-001" in by_id:
    validation_rows.append(
        {
            "검증": "DP-001 보호한도 핵심 문구",
            "통과": "1억원" in by_id["DP-001"].get("content_text", ""),
        }
    )

if "UN-003" in by_id:
    un003_text = by_id["UN-003"].get("content_text", "")
    validation_rows.append(
        {
            "검증": "UN-003 미수령금 종류",
            "통과": all(
                keyword in un003_text
                for keyword in ["예금보험금", "개산지급금", "파산배당금"]
            ),
        }
    )

for faq_id in ["DP-005", "DP-006", "MT-005", "HP-002"]:
    if faq_id in by_id:
        faq_count = sum(
            1
            for block in by_id[faq_id].get("blocks", [])
            if block.get("type") == "faq"
        )
        validation_rows.append(
            {"검증": f"{faq_id} FAQ 질문-답변", "통과": faq_count > 0}
        )

if validation_rows:
    validation_df = pd.DataFrame(validation_rows)
    display(validation_df)
    failed = validation_df[~validation_df["통과"]]
    if not failed.empty:
        print("주의: 아래 항목은 Markdown 직접 검수가 필요합니다.")
        display(failed)

print("documents.jsonl:", DOCUMENTS_JSONL)
print("chunks.jsonl:", CHUNKS_JSONL)

## 13. 결과 ZIP 생성 및 다운로드

In [ ]:

archive_path = Path(
    shutil.make_archive(
        base_name=str(OUTPUT_ROOT),
        format="zip",
        root_dir=OUTPUT_ROOT.parent,
        base_dir=OUTPUT_ROOT.name,
    )
)

print("결과 ZIP:", archive_path)
print("ZIP 크기:", f"{archive_path.stat().st_size / 1024 / 1024:.2f} MB")

try:
    from google.colab import files
    files.download(str(archive_path))
except Exception as error:
    print("자동 다운로드를 시작하지 못했습니다.")
    print("Colab 왼쪽 파일 패널에서 직접 다운로드하세요:", archive_path)
    print("오류:", error)



## 결과 폴더 구조

```text
kdic_crawl_output_<실행시각>/
├── manifest/
│   └── manifest_used.csv
├── raw/
│   ├── html/
│   ├── assets/
│   └── response_meta/
├── parsed/
│   ├── markdown/
│   └── json/
├── diagnostics/
│   ├── dynamic/
│   └── screenshots/
├── processed/
│   ├── documents.jsonl
│   └── chunks.jsonl
└── logs/
    ├── crawl_results.csv
    └── crawl_results.jsonl
```

## 권장 실행 순서

1. `MAX_URLS = 3`으로 테스트
2. 실패·빈 본문·자산 오류 확인
3. 사이트별 본문 선택자와 제거 선택자 조정
4. `MAX_URLS = None`으로 전체 실행
5. `Review_Queue` 결정 후 Manifest 수정본을 업로드하여 재실행
6. 동적 조회 페이지는 진단 JSON을 검토한 뒤 공개 HTTP 요청 전용 수집기를 별도로 작성

## BI-004만 먼저 재검증하는 방법

설정 셀:

```python
RUN_ONLY_URL_IDS = ["BI-004"]
MAX_URLS = None
```

정상 결과:

```text
status = dynamic_diagnostic_created
status_code = 200
```

생성되어야 하는 파일:

```text
parsed/markdown/예금보험금 안내/BI-004.md
parsed/json/예금보험금 안내/BI-004.json
diagnostics/dynamic/BI-004.json
raw/html/예금보험금 안내/BI-004.html
raw/response_meta/예금보험금 안내/BI-004.json
```

전체를 다시 실행할 때:

```python
RUN_ONLY_URL_IDS = []
MAX_URLS = None
```

## FIXED 버전에서 추가로 확인할 파일

```text
logs/quality_report.csv
```

이 파일은 HTTP 성공 여부와 별개로 다음을 보여줍니다.

- 본문 보존율
- FAQ 질문·답변 개수
- 표 개수
- 첨부파일 버튼 인식 개수
- 공통 UI 문구 잔존 여부
- 자동 판정 `pass`, `review`, `fail`

첨부파일 버튼은 `parsed/json/*/*.json`의 `attachments`에서 확인할 수 있습니다.
`DOWNLOAD_JS_ATTACHMENTS_WITH_PLAYWRIGHT = False`인 경우 실제 파일은 내려받지 않고 다운로드 메타데이터만 저장합니다.

## BI-004 전체 페이지 수집 결과 확인

기본 설정은 다음과 같습니다.

```python
FETCH_BI004_ALL_PAGES = True
```

정상 실행 시 `crawl_results.csv`의 BI-004 행:

```text
status = dynamic_full_collection_created
dynamic_is_complete = True
dynamic_page_count = 실제 전체 페이지 수
dynamic_row_count = 중복 제거 후 전체 금융회사 수
dynamic_failure_count = 0
```

페이지별 원본:

```text
diagnostics/dynamic/BI-004_pages/
├── page_001.html
├── page_002.html
└── ...
```

통합 목록:

```text
diagnostics/dynamic/BI-004_all_rows.csv
```

최종 JSON의 `dynamic_collection.is_complete`가 `true`일 때만
BI-004를 전체 공개 목록으로 사용합니다.


# 예금보험공사 RAG 챗봇 데이터 수집 파이프라인

이 노트북은 예금보험공사·금융안심포털의 **42개 URL Manifest**를 기준으로 다음 작업을 수행합니다.

1. Manifest 검증 및 실행 대상 선택
2. 정적 HTML 수집
3. 이미지·첨부파일·링크 메타데이터 추출
4. 동적 조회 페이지의 초기 HTML·폼·스크립트 구조 진단
5. 결정론적 HTML → Markdown/JSON 변환
6. 응답 메타데이터·해시·실행 로그 저장
7. 페이지 문서와 기본 청크 JSONL 생성
8. 결과 전체 ZIP 다운로드

## 수집 정책

- `include_full`: 공개 본문 전체 수집
- `include_partial`: 공개 페이지의 초기 구조만 진단하며, 검색 조건을 임의 생성하지 않음
- `link_only`: 로그인·본인인증·신청·자가진단 결과를 수집하지 않고 서비스 설명과 공식 URL만 저장
- `review`: 사람의 최종 결정 전에는 실행하지 않음
- `exclude`: 요청 자체를 보내지 않음

> 이 노트북은 로그인 우회, 본인인증 자동화, 개인정보 조회, 신청 제출을 수행하지 않습니다.  
> 기본 요청 간격과 낮은 동시성으로 실행되며, `robots.txt` 정책을 확인합니다.

## v2 수정 사항

이번 버전에서는 `BI-004` 동적 페이지 처리 중 발생한 아래 오류를 수정했습니다.

```text
missing ), unterminated subpattern at position 0
```

원인은 동적 스크립트 탐지용 정규표현식에 역슬래시가 중복 입력된 것이었습니다.

수정 전:

```python
r"(ajax|fetch\\s*\\(|XMLHttpRequest|\\.do\\b|pageIndex|pageNo|search)"
```

수정 후:

```python
r"(?:ajax|fetch\s*\(|XMLHttpRequest|\.do\b|pageIndex|pageNo|search)"
```

추가 기능:

- `RUN_ONLY_URL_IDS` 특정 URL 단독 실행
- 동적 페이지 정규표현식 자체 테스트
- `BI-004` 진단 함수 합성 HTML 테스트
- 존재하지 않는 `url_id` 입력 검증

## FIXED 버전의 수정 범위

이 노트북은 기존 실행 순서를 변경하지 않았습니다.

```text
설정 → Manifest → HTTP 수집 → 원본 저장 → 파싱 → 자산 처리
→ 동적 페이지 진단 → 로그 → 검수 → 문서/청크 생성 → ZIP
```

기존 결과 분석에서 확인된 다음 문제만 수정했습니다.

- `div`, `span`, `dd` 직접 텍스트 누락
- FAQ의 질문 `dt` 누락
- 제목이 `퀵메뉴`로 저장되는 문제
- 공통 UI 문구 혼입
- `rowspan`, `colspan` 표 구조 붕괴
- `gfn_downloadFile()` 첨부파일 버튼 미인식
- 문자 수만 기준으로 FAQ·표가 잘리는 청킹
- BI-004가 첫 페이지만 수집됐다는 사실이 메타데이터에 남지 않는 문제

## BI-004 전체 페이지 수집 적용 버전

이 버전은 `보험금 지급대상 금융회사(BI-004)`를 첫 페이지만
저장하지 않고 다음 방식으로 전체 공개 페이지를 수집합니다.

```text
첫 페이지 GET
→ 마지막 페이지 인덱스 확인
→ curPage를 변경하며 POST 반복
→ 페이지별 원본 HTML 저장
→ 각 페이지 표 추출
→ 중복 행·중복 페이지 검사
→ 전체 표로 BI-004 문서 재생성
```

전체 수집이 정상 완료되면 다음 메타데이터가 기록됩니다.

```json
{
  "collection_scope": "all_public_pages",
  "is_complete": true,
  "page_count": 5,
  "row_count": 42
}
```

페이지 요청이 실패하거나 같은 페이지가 반복되면
`is_complete`를 `false`로 기록하고 로그에 실패 원인을 남깁니다.


# 예금보험공사 RAG 챗봇 데이터 수집 파이프라인

이 노트북은 예금보험공사·금융안심포털의 **42개 URL Manifest**를 기준으로 다음 작업을 수행합니다.

1. Manifest 검증 및 실행 대상 선택
2. 정적 HTML 수집
3. 이미지·첨부파일·링크 메타데이터 추출
4. 동적 조회 페이지의 초기 HTML·폼·스크립트 구조 진단
5. 결정론적 HTML → Markdown/JSON 변환
6. 응답 메타데이터·해시·실행 로그 저장
7. 페이지 문서와 기본 청크 JSONL 생성
8. 결과 전체 ZIP 다운로드

## 수집 정책

- `include_full`: 공개 본문 전체 수집
- `include_partial`: 공개 페이지의 초기 구조만 진단하며, 검색 조건을 임의 생성하지 않음
- `link_only`: 로그인·본인인증·신청·자가진단 결과를 수집하지 않고 서비스 설명과 공식 URL만 저장
- `review`: 사람의 최종 결정 전에는 실행하지 않음
- `exclude`: 요청 자체를 보내지 않음

> 이 노트북은 로그인 우회, 본인인증 자동화, 개인정보 조회, 신청 제출을 수행하지 않습니다.  
> 기본 요청 간격과 낮은 동시성으로 실행되며, `robots.txt` 정책을 확인합니다.

## v2 수정 사항

이번 버전에서는 `BI-004` 동적 페이지 처리 중 발생한 아래 오류를 수정했습니다.

```text
missing ), unterminated subpattern at position 0
```

원인은 동적 스크립트 탐지용 정규표현식에 역슬래시가 중복 입력된 것이었습니다.

수정 전:

```python
r"(ajax|fetch\\s*\\(|XMLHttpRequest|\\.do\\b|pageIndex|pageNo|search)"
```

수정 후:

```python
r"(?:ajax|fetch\s*\(|XMLHttpRequest|\.do\b|pageIndex|pageNo|search)"
```

추가 기능:

- `RUN_ONLY_URL_IDS` 특정 URL 단독 실행
- 동적 페이지 정규표현식 자체 테스트
- `BI-004` 진단 함수 합성 HTML 테스트
- 존재하지 않는 `url_id` 입력 검증

## FIXED 버전의 수정 범위

이 노트북은 기존 실행 순서를 변경하지 않았습니다.

```text
설정 → Manifest → HTTP 수집 → 원본 저장 → 파싱 → 자산 처리
→ 동적 페이지 진단 → 로그 → 검수 → 문서/청크 생성 → ZIP
```

기존 결과 분석에서 확인된 다음 문제만 수정했습니다.

- `div`, `span`, `dd` 직접 텍스트 누락
- FAQ의 질문 `dt` 누락
- 제목이 `퀵메뉴`로 저장되는 문제
- 공통 UI 문구 혼입
- `rowspan`, `colspan` 표 구조 붕괴
- `gfn_downloadFile()` 첨부파일 버튼 미인식
- 문자 수만 기준으로 FAQ·표가 잘리는 청킹
- BI-004가 첫 페이지만 수집됐다는 사실이 메타데이터에 남지 않는 문제

## BI-004 전체 페이지 수집 적용 버전

이 버전은 `보험금 지급대상 금융회사(BI-004)`를 첫 페이지만
저장하지 않고 다음 방식으로 전체 공개 페이지를 수집합니다.

```text
첫 페이지 GET
→ 마지막 페이지 인덱스 확인
→ curPage를 변경하며 POST 반복
→ 페이지별 원본 HTML 저장
→ 각 페이지 표 추출
→ 중복 행·중복 페이지 검사
→ 전체 표로 BI-004 문서 재생성
```

전체 수집이 정상 완료되면 다음 메타데이터가 기록됩니다.

```json
{
  "collection_scope": "all_public_pages",
  "is_complete": true,
  "page_count": 5,
  "row_count": 42
}
```

페이지 요청이 실패하거나 같은 페이지가 반복되면
`is_complete`를 `false`로 기록하고 로그에 실패 원인을 남깁니다.

In [ ]:

# Colab 런타임에 필요한 라이브러리 설치
%pip -q install "requests>=2.32,<3" "beautifulsoup4>=4.12,<5" "lxml>=5,<7" "pandas>=2.2,<4" "tqdm>=4.66,<5"



## 1. 실행 환경과 설정

기본 설정은 정적 페이지·자산 페이지·링크 전용 레코드를 처리합니다.

- 특정 업무만 실행하려면 `SELECT_BUSINESS_FUNCTIONS`에 업무명을 입력합니다.
- 테스트 실행은 `MAX_URLS = 3`처럼 제한합니다.
- 동적 공개 페이지를 브라우저로 렌더링한 초기 화면까지 저장하려면 `ENABLE_PLAYWRIGHT_SNAPSHOT = True`로 바꿉니다.
- Playwright도 검색 폼 제출이나 로그인은 수행하지 않습니다.


In [ ]:

from __future__ import annotations

import hashlib
import json
import mimetypes
import os
import re
import shutil
import subprocess
import sys
import time
from dataclasses import dataclass, asdict
from datetime import datetime, timezone, timedelta
from pathlib import Path
from typing import Any, Iterable
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup, Tag
from IPython.display import display
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

KST = timezone(timedelta(hours=9))

# -------------------------
# 사용자 실행 설정
# -------------------------
SELECT_BUSINESS_FUNCTIONS: list[str] = []
# 예: ["예금자보호제도", "예금보험금 안내"]

RUN_ONLY_URL_IDS: list[str] = []
# BI-004 오류 수정만 먼저 검증하려면:
# RUN_ONLY_URL_IDS = ["BI-004"]
# 전체 실행하려면 빈 리스트 []를 사용합니다.

RUN_DECISIONS = {"include_full", "include_partial", "link_only"}
RUN_WAVES = {"W1_STATIC", "W1_ASSET", "W2_DYNAMIC", "META"}

MAX_URLS: int | None = None
# 최초 테스트 권장: 3
# 전체 실행: None

REQUEST_DELAY_SECONDS = 1.2
REQUEST_TIMEOUT_SECONDS = 25
MAX_RETRIES = 3
MAX_ASSET_BYTES = 25 * 1024 * 1024

RESPECT_ROBOTS_TXT = True
DOWNLOAD_ATTACHMENTS = True
DOWNLOAD_IMAGES = True
DOWNLOAD_VIDEOS = False

ENABLE_PLAYWRIGHT_SNAPSHOT = False
PLAYWRIGHT_WAIT_MS = 1500

# JavaScript gfn_downloadFile(...) 버튼을 실제 파일로 내려받을지 여부
# 처음에는 False로 파싱 품질부터 검증하고, 이후 True로 시험하는 것을 권장합니다.
DOWNLOAD_JS_ATTACHMENTS_WITH_PLAYWRIGHT = False

# BI-004 지급대상 금융회사 공개 목록의 전체 페이지를 POST로 수집합니다.
# 전체 수집을 원하지 않을 때만 False로 변경합니다.
FETCH_BI004_ALL_PAGES = True

# 사이트 오류로 비정상적으로 많은 페이지가 계산되는 상황을 막는 안전 한도
BI004_MAX_PAGES = 100
BI004_PAGE_SIZE = 10

CREATE_BASELINE_CHUNKS = True
CHUNK_MAX_CHARS = 1200
CHUNK_OVERLAP_CHARS = 150

USER_AGENT = (
    "KDIC-RAG-Academic-Crawler/0.1 "
    "(low-rate public-document collection; no authentication automation)"
)

RUN_ID = datetime.now(KST).strftime("%Y%m%d_%H%M%S")
RUNTIME_ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
OUTPUT_ROOT = RUNTIME_ROOT / f"kdic_crawl_output_{RUN_ID}"

PATHS = {
    "raw_html": OUTPUT_ROOT / "raw" / "html",
    "raw_assets": OUTPUT_ROOT / "raw" / "assets",
    "response_meta": OUTPUT_ROOT / "raw" / "response_meta",
    "parsed_markdown": OUTPUT_ROOT / "parsed" / "markdown",
    "parsed_json": OUTPUT_ROOT / "parsed" / "json",
    "dynamic_diagnostics": OUTPUT_ROOT / "diagnostics" / "dynamic",
    "screenshots": OUTPUT_ROOT / "diagnostics" / "screenshots",
    "logs": OUTPUT_ROOT / "logs",
    "processed": OUTPUT_ROOT / "processed",
    "manifest": OUTPUT_ROOT / "manifest",
}

for path in PATHS.values():
    path.mkdir(parents=True, exist_ok=True)

print("실행 ID:", RUN_ID)
print("결과 경로:", OUTPUT_ROOT)


## 2. 42개 URL Manifest 불러오기

In [ ]:

# 현재 프로젝트의 42개 Manifest를 노트북 내부에 압축하여 포함했습니다.
# 별도 CSV 업로드 없이 기본 Manifest를 자동 생성합니다.

import base64
import gzip

EMBEDDED_MANIFEST_GZIP_B64 = """H4sIAHmDVGoC/+1dbW8TWbL+PtL8h1akXSXaXhzbCS/3w1xlgJmJBmaZwOzofrIcuxO8OG3T3Q6T/TAy0EGGZJawNwYH7FznEkiCgtYJDnF2g66Un+Pu1v6FW3XqdLvbbjs2hLCajYRiu19On656TtVTdeoc/vmP/0to0lRETokZJRlJxMXxjJqQJVWNTGTkmJZIyaIWVSYlLdJ6Ih2dlCJaQktKoprKKDEpAm2IKjQYkaNTkphSEpMJOZqMpJUEfNdmRDmlTEWTiT9L8UhciiVUbMX+ElGkqOq0OpOWxAlJi12PqJoS1aTJGTihqJICjaUmEvDIaEa7DvfczCQUKS5GVRX6mE4lE7EZMZaSNUnWImosBc0o0nRCutW4NKZEbyUjt6LTEv+qalEto4rST2kppkHXUhktndFEWYLvvFNOi5mpqaiCL6JJ6uefBcULV34/OBgUzUKuXsuZywvGm6pVqJnlovFQb3OUflh59uO6pqXV/wgEbt26depGPBE7lVJO3VACajoQT8Obw8tqgSvw5+qMquHnpSlNDahSEnp6NabIp+Ip/hRs9Umh/mbHvLMpWnndzBXEhBxLZuISaCyZFM0ns8ZmVTDXcsZmzSwV4Ntts/xIqO/u17cqxkpRgDvNpxtCvZI1HryA/gnY2lxJMPM5405VsP5SMEtVcy0rosASsQhqSkSxSqqmRq5rU0kRXyHCT08HQUzwTwZNQgci46k4Ck78MRi5em3k2uh5lGIE5AgyF81nIKSa8M21y5eE3wn1nU1zpSLAO5krVfioQZfhsLGxaN3NGr9UoB+WXsE3N+8VQ0PCD2OXxKBZqhlze0JDHLaQhaD5eAceUH+zL5jVHHygVvIrgrGn03UCnDX3lgRTXzVezYqffxYixYZ6Uqz/0V40fKLZZs3WaxXoclAw5rPWo6KVf2282kA91itFY3ZBMEv75l7BR1DwqgL9Np7PGy9A8eXbxst3Vh4e+mjBfJIz5lbxGlRYccd6Og/34EPMuTJHkFnSQZDZeu2+lS8YDxYRI8bLzQZGwoSR8HsMfnzQ3dvCF97Hf4FvZT5d7AoyhBS3abiamVJOUOPYA7cdELjAPeI2/3fWeFHoN0tZ6/H9g10SGJw42LVym6i3tVX4B2KC74Lx31sDgnGnYD6pguqHSPVDH656+G79Nfehqr+mTMJnPHaCgW4xQHJHS4JOgM7QsX7SPhgCC46h3lEw8HTjrY7myFzNNmzAMAFhuCcg9NXflOuVkqnvQ4dB9V+NfA9/jb/VQD9GeQ/usO7tWPkN6IK5/bofVQDv9gDe4K0OiMWuwtPNvdeiYM6tQqMAzCpoCZDs8X4DfQ6cJhKy6sZTbCowPm4Tia+iN79TolPaSDo5g+DhEsrnzLlN6+Gmpdc+NngmojebQIMdjsDhT+Br+t3cgeyCPWwHWl1PXyvfEOE9dTAU4HOy5ts8GRhUnnmnQlcLdDnor3AbDUw5D1IQrGe643tq5luQXzlnVtbRe4G3Mp5vIlL6AHWnCXWnP5ydYIs9MRTAyuEkBaBirj06RrQwE/Np0MJI5fN9FCdwhHplCXporM0LocHQsDGrC5yCom15/ALNiWC3YD4BED25Z/wCstgpMuJxZ9NYyTr6rol2I+fMZ4t+xNbdKjWBtsTdTNUszQNizhBiznwIYuiZ/UChzKXaQDvUxBOB8T+nx5Oy46fGY8oV/G2HL3+YVtQ2wJF+YohxPulZaKLrOwvYEeBygJTTMARBsIAO8OIMVy5AGXM7xpssxxVSPDOvG3MFQMyEpEgyRKmx1BTEkgmM6+wHqTcSaQc5/I949dvRK+xMGgLB8Rk7uiR0AynEMWvmH6CEfXTq6btAnRfByNd3Xh/smn9fADIBWoavpX0AG3wWbrNTxsNZNC34Qi0KZzplDeL4AEIMqj0rfjnqikG5PMHG0Ljp7TD4IfI65lyR+bw93dSL4HS6MhIX0urU6Dj3JFdSqnohNiWfBDUtpiN0KmiLmQlXaBfXQFPgCgAT81kIPegr0AR9C4wCZ6V2QyUIYotlq5A33ugezdkK//yzc4SU0IcihT2QOycEjO95Ov0esFFSsU+OGOhWDFNd7w+ZkatXL147SsSEmuTeHjI8jAGSKRgQVNwtGW+zSDQpmIVXhjNgblhr8ElH9+bhCxMaNe8w3OAgoSb8YaiBFzKWs//ZGx4u/pROyifmw595hHrJidRy/fCiXNe13MEuhpvARu0DA6T6PPtwVB8k1Q8dlWtxHWIP9onK3yMA/hKQ8nViWsIg+CtZ1dqgBEYBUnDmvIX623lz+YUDGaADWiIKqFmpQDcE+KhvVQ92HQwY+ip+mQXhPF1k1lZnfF4vIXCMv64ieJCU1LcroAnkf/Q4gVPh+IwcnQKUJFOpG5m08xNfNqJdl+RIOhmduaUkJq9rhCrvDW50AQCVmYgiqZmkpoozkir+GIpc+K/vRi57McZ7ZGwvmkVdIMQFqH8+4OprRpe5lPXjtaJgzYOeKm6dMY+zDaq8z2/mgWgzI6IotU/0b6EFNW6ImE8g/M46T2FjFhEaEn/4jpEfCqk9QTQ3mJ1OeY7xoJtBIMCN7RcC3e2+sKd4iQXWo/JNdG3s+9eZ+IlJa2PSwi0mDeH3aoOZK7ePc2mjYbgYYks1aNfQd4EoVTbwoVyR+Gs73+BCwTAhJ3QEyHFna5AP3ZtH64q9YYc8j3b4EqNvHNxl3VpatJVKPWyXuMlEA1GA0iSi6AKiDa3eNSn5fUb1g9bxp3LaQ4uldI4fWmGgTl4suOBVLMOL4euQDgT79bd0Q18QMcsGvyCcrG+/EzAk4udJpSx76MZGvZo1yuu8LdAshN9cQJ5gPDhE2At/Aux1jy6yXNqkgqZrVFPil7Xr8ROIdWu9ACMPAXvBg92ww925NtpSd4+u2ewE02Rh3cCEAMsRrFqLOv1qtmwOaxsmdA19PJ8I0Dbv/YJ9IOWjApcLdFVP3vEbJS19k1DRQXaVVfzVO8RmGhZuk14UXTqwvQhx164YGFKutRK4oH5rcd9paUCwDZ7Hv+qNZ0Ev+XPsOVG9iImGB6sIvNPi5WuUiapsAFLNeyss1V0pWEsFG129ncKY9NkCT1UD7lxXuC8Q6AoIMDug78aUorI/aMu6ZWD9mKDH+RZbQwNeCJIimbTtrCDG3XQXHv5h7BIMW85iaIw2Kx1iioqpl8XpRFxK+UAwwmpbVB8k0h1TkhaNR7WoiHCEQCKj0h2NEx8lJUFAGzosCm2nVK4zNmx31/EYmDqaBHXZsjMEqdCngNQh+SsPnk6yVn4QGToVFHyE6nKA0AA0ygoxtirM9DxcAuSI3F9S4gpjxFwBGT02RakqYdiYW69j2NicrzpLkAl/Asi4nHy3wEFyhcTqJCRsBU+XtOmQmI/CyEdon7ddyfDgOcLJ0HvixHcOvx00+D3dzsZfVuPK19PjNzpMyLfMlBV0c3kTnTOWtlCaBxw/+GcnA1SGuKCGmICzEDEbC+tsJvWop8Bw1B/sovaaFcbYA9MZsoeNClYvMPjqNRjpLBB7jfioV/IikRqq04Pwv1apV3TmSOEdENPFMk1+tHUvoBBP0BUaJH0PH6W+r/3hihAc7Fqv11Lp4OC/c4XFId5W6J1F2CH3TtGYW7XTt0xeoJj6VpUlbPPvEABBAsDpI3AMni54hz0ODeAzwMu/YOmFUoGA3RYkCSUQjccVVQqMsI8RTZMva5pyEmZ3Qk9DzNhbEHK5CEbiYBcJQu1+S6KnuWZnecFc07l+4T4Qg+hpiPsOOrhdxaflsJJUoPYd1sFkhiU5oRCh68xHQJeT06FOOckc9yxvI8PjJ6SO4LMTPuh1CILO1H0v6Z5+c3sdCdv8PEi5KUzqrC1nZlMvAhAx4LTy6/V/FOzJk6imRWPXp7D+3i86cs6qBEzX5QTOeOqWnExF455LXcGS6+hHocEdEOqZqG9PcBr3w6VYMuIBpbtF5vDRMTJZotOH4Y34DBM+z/5L4bMHZKaVo0elT+UEhLIs58bu+vUi0quEZp5dQL7YHTRbRdjvV7QCqMBMFRZRu9tFhgYSZ24JGOnLTUTqECH13DEi1RvneQL/jrjUFBV5+tW0ovlkAI7fZXfIAHTps48YaJ1Jnu1Fc0WAA4/rPcCyK1eWgcOtdAj/2QKPvy8YL1+zyx+sWvfKgCZrdtEAWRb3mRCdtT3DDGDA24/RFDKaCkeM8h6bGajvFgHxXaNsTEpeit5KjU1mkoeDrI+bPnocSMC6vSlyOGEENVdkc1nm20VWHlnWITAGaf3Wjup04MwDfX6JTpay4S/BmmWCd2c+6bCdk0ZA6yxzDYYBsyf17SyoRrBuL2BWGuyBsVISk9JkNIkVDbSIMCHHpZ+aTGoyId/gxpSuZhdxIONJXFGYimWYzYzAnRnJZU/xgk+QDeWlqCzspSGNyftXG2wEU9xCwoQAl8EBmzfuV5sLb+0kRjvc0c1Y0lW0Ht+HaJotZYPXwVSIQwAoOx8MHiPqsTx5acPUy9bdUtdQv6pltPPXY93DnNIHNvBWivVd9D8EY9HG6do8t5bNwKY+YjUP0n4yEi35fKdVgZq1faQz/cEg7jwJwYC4jsrxhDwZUVIZTVLaI5rOczDD9/FURo7TBX7J/WNAcxM/AGj5hl5caNQ6z9+3idrh8kCb+I1AguZgsWKU9vkgYMkb1r491bSmm8uzWCixgfm7EE0NBEPHShMwvciTiZ1n0ptR/UdVG4ulT0o02qR6h9qWaBCkAIHP90EvzlS5SxOcAzDa/rSKU29zRZYtRCKK6/z4nXbmN0QzBMHwR04E9ZQCmmIhDo90TlJA7xm4IGjYG9NhRkK8WeNWK9Xn0aFf3pkXomH2nJ0VcQ0GCNB4wdYKMy7VSATRvEJw6DjQ5TP9hB6rVnHWcVCPefmktZQ/JOTmMCTc4fTD98mJ2Ghc1mbG1KTWOdJu8bSMmEVScnJGBC7D6o91oO12kSwaeAgR/+c+uICSqW81phPQ5XJIEq/ivpUq/LCEQUBFl2+zhxHDScjgQ6MxLTEtRaD7E5HYdSl2Q/Q4S+qL91Bnr+tzO1x9+eK1EQ+mHRrwahbQ6+ki+10zX5bsGtbf2a+GC1Qb9R5ElRkhB74I5wCeIKm7dvTCCSRcDv4SF/8wR4hl9RsVex0oXMhWhfnPmhHe+8QztHqKxgpcvV0F0bKF8T6TmZ7yW7cWcQE8TakEhz/ZVCsNxx6mWrFasotKIzbYP3BuzW07j2qOrbk8aKhtlTYa1u0q4o9smb2alZLdh5UH8eQk8w+eBvqHofWf+X4aA406Q2q2aYLO8bjhoHhhhCqDtnRcrbeCQnKWCxx6bHkTM5+OGXSd67zHhjIeSI4nteifprTAJfgyAl8wV+N872rLDXonoT9sbOf4wwV6Oo6qsz+fG8SaPByjMO5xOcObOzDOWETA1gAMOOr/duy35z3db7haxjbBGvJVA6uPjDc7xkLBfFzlISU3EBx3HKEfA2NNEBs+rMDHoylKB4KZ2MaibXf+14EKSCXHzFwzSEIEktBRggSG4hu7ZE345/49gapym+bje4DNqDyRGpVvXlBjqowGpeu6jYYvpPwoOQW09RRSsiJk1k8YRbx02HYraOX/lmXmhdwhhalMGeRLXL63nMWVLbg7kiQDOKK4o5EqKdOJmNSVN8QVJQw2/KZIXFJjSiKNWz59BA/owG0Y6z1odBFPEVo8GBUHsSU3rGVaJLtu18Pyom23eMl5ti44Odh1jUXeL5bLclqjPti64IABlIYJpeH3QmmHRdOKlMSdswK4XPpbRY4B2KIAxJOyoKYIYBirS4ZPhfEPc38ECqfERBeYZp03YBsB/bLIalj1mrWcswljn51idlkj0bvmGRpdQJJVXO/HAoP5rHBmEMz9bwZEATqHwUE1a6/HflRslJtgHBAeIqAMHY05g74+3QDsglsQ6Giv1uu8IsXHYtPKDP44HFf28jaWRaMt1ER3L4D2QCdYuUXZXprO1uU3FuTzlfjEkZBmYLyQX7fyZXQ3xJS49ROj8T9FY2w+jTxTWuK5M3oy38ONH3XA1nQVmq6xi38cvfijc8bZZs6aX2CxBxsCPENHO27QikBXiATjtJloDbclWv4ysQqPzPw7bpK6YFstLbCZMSy+L/rpnsvc5TyHCW3DR4I2speI/K3ZXmF2RZW/VN4PXphqw3R2pYj5W9aHg13qBQtUWt/iV44tEgibfiYouEVjT309WOwEro5N2NL1lrMgnE4TnE4fCZz4I8HxuqYBuofT2ORRwIk9/QRGbgwwkXwYjNxN9Id/ZpvVgC5QfpjWLwy4MHWGMHWme0xRMgCiiVoO58te4hZ5WCm6pQO3w3maZwsBzwokRvc7k3s1Df8UbSIjxwMIsK/k+AVpXLuQlCeRz3e3PJdPNjUINCW5qFOYFqgUG1ViLOXI6RUPrLffYaoG+SZ7Uz7D0cR53Gw+nU4ilUcK3guX75DZOh5233covW+7wMkWmb9QD8FrK+O3x/3yAqMmXhRhXqQZRs7zHOp/lgB8tmsA+9eIuyLTpiVzh9UPuwLQNqlYX9bmiZCag2K7TvrhrPHynW17+Lw9K8u5Ddac20NuoHjzWFL8L2MDO8HLDhazHt7WkAqvGu6Gq1nFKqZDHlYEO353LGJnMfOZxJLuHfK4/Rog65z4Dd9HuJQ15u6DxJmDnysDgvyPtTONc6vIlpZ1nqy0t5mkU1R0gCvP3W1SkxRkwBeCaw/m87wcm7oQSyonOxu0Lww53bqzwZKO68QxHqVEGoxgmnkSUCflPG01aQee4UGec6VMfhWeB/dhKaiz+MV8WrH+UnMvggFtNu97MDRIUAt1DTV/I9aCn66NGMLlipLWEDL/zmshHH2D5lj9FTk20N+ebsA7QDvLL9oWXrrQwvfwQbTY1as0on0bpJ2uWbkFIBKkRCZoKEi4CLfBBZtrZ0Fnv/E2B5DCSiOgzOz8wAfbo9Y2m+wRL8rowSyNTqsT0eQPspIcTU4muzdQfbj3IlKvVjmwdBOcfVFzVq/aiVkcnWvsW1+fz51wECWuP0c3aqwtoFDZlZ3l2teHLfLaOetusb432+e49Y79czfWwIO3z7xdTN6gNanxJzQ22/GNfmz++cldf4sJosLtah1eC9kh39ujYNwpiIL1dIUV2hmVCqCRlcjz1+d2kubJcV8n3HqT+CXTDhdbn+jdKtwLYDRIfo/w1QIWreCQC9GQG+raFPsMKHNNh+atgo5FbJxr9FNwZhMQJj2bs2OytNB2887pVCyAGzTACAmcjyqSpIxJkwlVG2H/4UMgCV99wyG2XwAWxtuTUJ4e0ENZ2atfd/lMNqXqx0a+tm0nV7utmaq9kWcsJatO5f3Rzay2ypvHaVQysoTM0E8BuIPva75NGE8IMpfJNx6dFzpoyt6soYp1b5y30njrO0zZjEraWzGw0sWi3gjMPbqnBSOO9gWnhu7/AYEZ+Ef+YwAA"""

MANIFEST_PATH = RUNTIME_ROOT / "kdic_crawl_manifest_42.csv"
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH.write_bytes(
    gzip.decompress(base64.b64decode(EMBEDDED_MANIFEST_GZIP_B64))
)

print("Manifest 생성 완료:", MANIFEST_PATH)
print("크기:", MANIFEST_PATH.stat().st_size, "bytes")


In [ ]:

# 필요하면 사용자 수정본 CSV로 교체할 수 있습니다.
USE_CUSTOM_MANIFEST_UPLOAD = False

if USE_CUSTOM_MANIFEST_UPLOAD:
    from google.colab import files

    uploaded = files.upload()
    csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
    if len(csv_names) != 1:
        raise ValueError("CSV 파일을 정확히 1개 업로드해야 합니다.")

    MANIFEST_PATH = Path("/content") / csv_names[0]
    MANIFEST_PATH.write_bytes(uploaded[csv_names[0]])
    print("사용자 Manifest로 교체:", MANIFEST_PATH)


In [ ]:

REQUIRED_COLUMNS = {
    "url_id",
    "business_function",
    "page_title",
    "source_url",
    "site_name",
    "normalized_decision",
    "decision_reason",
    "page_type",
    "fetch_strategy",
    "parser_profile",
    "auth_required",
    "asset_policy",
    "content_scope",
    "crawl_wave",
}

manifest_df = pd.read_csv(MANIFEST_PATH, encoding="utf-8-sig", dtype=str).fillna("")

missing_columns = sorted(REQUIRED_COLUMNS - set(manifest_df.columns))
if missing_columns:
    raise ValueError(f"Manifest 필수 열이 없습니다: {missing_columns}")

if manifest_df["url_id"].duplicated().any():
    duplicates = manifest_df.loc[
        manifest_df["url_id"].duplicated(keep=False), ["url_id", "source_url"]
    ]
    raise ValueError(f"url_id 중복:\n{duplicates}")

if manifest_df["source_url"].duplicated().any():
    duplicates = manifest_df.loc[
        manifest_df["source_url"].duplicated(keep=False), ["url_id", "source_url"]
    ]
    raise ValueError(f"source_url 중복:\n{duplicates}")

invalid_urls = manifest_df[
    ~manifest_df["source_url"].str.match(r"^https?://", na=False)
]
if not invalid_urls.empty:
    raise ValueError(
        "HTTP/HTTPS가 아닌 URL이 있습니다:\n"
        + invalid_urls[["url_id", "source_url"]].to_string(index=False)
    )

# 실행 대상 필터
target_df = manifest_df.copy()

if SELECT_BUSINESS_FUNCTIONS:
    target_df = target_df[
        target_df["business_function"].isin(SELECT_BUSINESS_FUNCTIONS)
    ]

if RUN_ONLY_URL_IDS:
    known_url_ids = set(manifest_df["url_id"])
    unknown_url_ids = sorted(set(RUN_ONLY_URL_IDS) - known_url_ids)

    if unknown_url_ids:
        raise ValueError(
            f"Manifest에 존재하지 않는 url_id입니다: {unknown_url_ids}"
        )

    target_df = target_df[
        target_df["url_id"].isin(RUN_ONLY_URL_IDS)
    ]

target_df = target_df[
    target_df["normalized_decision"].isin(RUN_DECISIONS)
    & target_df["crawl_wave"].isin(RUN_WAVES)
].copy()

if MAX_URLS is not None:
    target_df = target_df.head(MAX_URLS).copy()

print("전체 Manifest:", len(manifest_df))
print("이번 실행 대상:", len(target_df))

display(
    manifest_df.groupby(
        ["business_function", "normalized_decision"],
        dropna=False,
    ).size().rename("count").reset_index()
)

display(
    target_df[
        [
            "url_id",
            "business_function",
            "page_title",
            "normalized_decision",
            "crawl_wave",
            "fetch_strategy",
        ]
    ].head(20)
)


## 3. 공통 유틸리티

In [ ]:

ATTACHMENT_EXTENSIONS = {
    ".pdf", ".hwp", ".hwpx", ".doc", ".docx",
    ".xls", ".xlsx", ".ppt", ".pptx", ".zip",
    ".csv", ".txt",
}

IMAGE_EXTENSIONS = {
    ".png", ".jpg", ".jpeg", ".gif", ".webp", ".svg", ".bmp",
}

VIDEO_EXTENSIONS = {
    ".mp4", ".webm", ".mov", ".avi", ".mkv",
}

NOISE_SELECTORS = [
    "script", "style", "noscript", "template",
    "header", "footer", "nav", "aside",
    ".skip", ".skip-navigation", ".skipnav",
    ".gnb", ".lnb", ".snb", ".breadcrumb-wrap",
    ".footer", ".header", ".quick-menu", ".quick",
    ".util", ".utility", ".pagination",
    "[aria-hidden='true']",
]

MAIN_SELECTORS_BY_SITE = {
    "예금보험공사": [
        "#contents", "#content", ".contents", ".content",
        ".sub-content", ".sub_contents", "main", "article",
    ],
    "금융안심포털": [
        "#contents", "#content", ".contents", ".content",
        ".sub-content", ".sub_contents", "main", "article",
    ],
}

BREADCRUMB_SELECTORS = [
    ".breadcrumb", ".breadcrumbs", ".location", ".location-wrap",
    ".path", ".sub-location", "nav[aria-label*='breadcrumb' i]",
]


def now_kst_iso() -> str:
    return datetime.now(KST).isoformat(timespec="seconds")


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def sha256_text(text: str) -> str:
    return sha256_bytes(text.encode("utf-8"))


def normalize_space(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


def normalize_multiline(text: str) -> str:
    lines = [normalize_space(line) for line in (text or "").splitlines()]
    cleaned: list[str] = []

    for line in lines:
        if not line:
            if cleaned and cleaned[-1] != "":
                cleaned.append("")
            continue
        cleaned.append(line)

    while cleaned and cleaned[-1] == "":
        cleaned.pop()

    return "\n".join(cleaned)


def safe_name(value: str, max_length: int = 90) -> str:
    value = normalize_space(value)
    value = re.sub(r'[\\/:*?"<>|]+', "_", value)
    value = re.sub(r"_+", "_", value).strip("._ ")
    return (value or "untitled")[:max_length]


def ensure_parent(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)


def write_json(path: Path, data: Any) -> None:
    ensure_parent(path)
    path.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def append_jsonl(path: Path, data: dict[str, Any]) -> None:
    ensure_parent(path)
    with path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(data, ensure_ascii=False) + "\n")


def absolute_url(base_url: str, candidate: str | None) -> str | None:
    if not candidate:
        return None

    candidate = candidate.strip()
    if not candidate or candidate.startswith(("#", "javascript:", "mailto:", "tel:")):
        return None

    return urljoin(base_url, candidate)


def extension_from_url(url: str) -> str:
    path = urlparse(url).path
    return Path(path).suffix.lower()


def classify_asset_url(url: str) -> str:
    extension = extension_from_url(url)

    if extension in ATTACHMENT_EXTENSIONS:
        return "attachment"
    if extension in IMAGE_EXTENSIONS:
        return "image"
    if extension in VIDEO_EXTENSIONS:
        return "video"
    return "link"


def row_to_clean_dict(row: pd.Series) -> dict[str, str]:
    return {
        str(key): "" if pd.isna(value) else str(value)
        for key, value in row.to_dict().items()
    }


## 4. HTTP 세션·재시도·robots.txt 검사

In [ ]:

def create_session() -> requests.Session:
    retry_policy = Retry(
        total=MAX_RETRIES,
        connect=MAX_RETRIES,
        read=MAX_RETRIES,
        status=MAX_RETRIES,
        backoff_factor=0.8,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset({"GET", "HEAD"}),
        respect_retry_after_header=True,
        raise_on_status=False,
    )

    adapter = HTTPAdapter(
        max_retries=retry_policy,
        pool_connections=2,
        pool_maxsize=2,
    )

    session = requests.Session()
    session.headers.update(
        {
            "User-Agent": USER_AGENT,
            "Accept": (
                "text/html,application/xhtml+xml,application/xml;q=0.9,"
                "image/avif,image/webp,*/*;q=0.8"
            ),
            "Accept-Language": "ko-KR,ko;q=0.9,en;q=0.5",
            "Connection": "keep-alive",
        }
    )
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


SESSION = create_session()
ROBOTS_CACHE: dict[str, RobotFileParser | None] = {}


def get_robots_parser(url: str) -> RobotFileParser | None:
    parsed = urlparse(url)
    base = f"{parsed.scheme}://{parsed.netloc}"

    if base in ROBOTS_CACHE:
        return ROBOTS_CACHE[base]

    robots_url = urljoin(base, "/robots.txt")

    try:
        response = SESSION.get(
            robots_url,
            timeout=REQUEST_TIMEOUT_SECONDS,
            allow_redirects=True,
        )

        if response.status_code >= 400:
            ROBOTS_CACHE[base] = None
            return None

        parser = RobotFileParser()
        parser.set_url(robots_url)
        parser.parse(response.text.splitlines())
        ROBOTS_CACHE[base] = parser
        return parser

    except requests.RequestException:
        ROBOTS_CACHE[base] = None
        return None


def robots_allows(url: str) -> tuple[bool, str]:
    if not RESPECT_ROBOTS_TXT:
        return True, "robots_check_disabled"

    parser = get_robots_parser(url)
    if parser is None:
        return True, "robots_unavailable_assumed_allowed"

    allowed = parser.can_fetch(USER_AGENT, url)
    return allowed, "allowed" if allowed else "disallowed"


def choose_response_encoding(response: requests.Response) -> str:
    content_type = response.headers.get("Content-Type", "")
    declared = response.encoding

    # requests의 기본 ISO-8859-1 추정은 한국어 HTML에서 잘못될 수 있습니다.
    if not declared or declared.lower() == "iso-8859-1":
        apparent = response.apparent_encoding
        if apparent:
            return apparent

    return declared or "utf-8"


## 5. HTML 수집과 응답 메타데이터 저장

In [ ]:

@dataclass
class FetchResult:
    requested_url: str
    final_url: str
    status_code: int
    ok: bool
    content_type: str
    encoding: str
    elapsed_seconds: float
    collected_at: str
    content_length: int
    raw_sha256: str
    redirect_history: list[dict[str, Any]]
    selected_headers: dict[str, str]
    body: bytes


def fetch_url(url: str) -> FetchResult:
    allowed, robots_status = robots_allows(url)
    if not allowed:
        raise PermissionError(f"robots.txt에서 수집을 허용하지 않습니다: {url}")

    started = time.perf_counter()

    response = SESSION.get(
        url,
        timeout=REQUEST_TIMEOUT_SECONDS,
        allow_redirects=True,
    )

    elapsed = time.perf_counter() - started
    encoding = choose_response_encoding(response)
    response.encoding = encoding

    selected_header_names = {
        "Content-Type",
        "Content-Length",
        "Last-Modified",
        "ETag",
        "Cache-Control",
        "Content-Disposition",
    }

    selected_headers = {
        key: value
        for key, value in response.headers.items()
        if key in selected_header_names
    }
    selected_headers["Robots-Check"] = robots_status

    return FetchResult(
        requested_url=url,
        final_url=response.url,
        status_code=response.status_code,
        ok=response.ok,
        content_type=response.headers.get("Content-Type", ""),
        encoding=encoding,
        elapsed_seconds=round(elapsed, 3),
        collected_at=now_kst_iso(),
        content_length=len(response.content),
        raw_sha256=sha256_bytes(response.content),
        redirect_history=[
            {
                "status_code": item.status_code,
                "url": item.url,
                "location": item.headers.get("Location"),
            }
            for item in response.history
        ],
        selected_headers=selected_headers,
        body=response.content,
    )


def save_fetch_result(
    manifest_row: dict[str, str],
    result: FetchResult,
) -> tuple[Path, Path]:
    business_dir = safe_name(manifest_row["business_function"])
    url_id = manifest_row["url_id"]

    html_path = PATHS["raw_html"] / business_dir / f"{url_id}.html"
    meta_path = PATHS["response_meta"] / business_dir / f"{url_id}.json"

    ensure_parent(html_path)
    html_path.write_bytes(result.body)

    metadata = asdict(result)
    metadata.pop("body", None)
    metadata["url_id"] = url_id
    metadata["business_function"] = manifest_row["business_function"]
    metadata["page_title_manifest"] = manifest_row["page_title"]
    metadata["fetch_strategy"] = manifest_row["fetch_strategy"]
    metadata["parser_profile"] = manifest_row["parser_profile"]
    metadata["raw_html_path"] = str(html_path.relative_to(OUTPUT_ROOT))

    write_json(meta_path, metadata)
    return html_path, meta_path


## 6. 결정론적 본문·표·링크·이미지 추출

In [ ]:
from bs4 import Comment, NavigableString

# 실제 본문 밖에서 반복되는 UI 요소
FIXED_NOISE_SELECTORS = [
    "script", "style", "noscript", "template",
    ".floatTop", ".floatBottom", ".paging", ".pagination",
    ".sr-only", ".skip", ".skipnav", ".skip-navigation",
    ".loading", ".waitPage", "#waitPage",
]

FIXED_NOISE_TEXTS = {
    "퀵메뉴", "글자 크기", "글자확대", "글자축소",
    "KOR", "ENG", "상단으로 이동", "검색 초기화",
    "좌우로 움직여보세요", "표 더보기",
}

FAQ_PREFIX_RE = re.compile(r"^\s*질문\s*")
ANSWER_PREFIX_RE = re.compile(r"^\s*답변\s*")
OPEN_SUFFIX_RE = re.compile(r"\s*열기\s*$")
DOWNLOAD_CALL_RE = re.compile(
    r"gfn_downloadFile\(\s*'([^']+)'\s*,\s*'([^']+)'\s*\)"
)
MOVE_URL_RE = re.compile(r"gfn_moveUrl\(\s*'([^']+)'\s*\)")

BLOCK_TAGS = {
    "div", "section", "article", "aside", "header", "footer", "main",
    "p", "ul", "ol", "dl", "table", "figure",
    "h1", "h2", "h3", "h4", "h5", "h6",
}


def fixed_norm(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


def fixed_safe_text(node: Tag) -> str:
    """원본 DOM을 훼손하지 않고 노이즈를 제거한 텍스트를 반환합니다."""
    fragment = BeautifulSoup(str(node), "lxml")
    root = fragment.body.find() if fragment.body else fragment.find()
    if not root:
        return ""

    for child in root.select(
        ".sr-only,.hide,script,style,noscript,template,.floatTop,.floatBottom"
    ):
        child.decompose()

    for image in root.find_all("img"):
        image.replace_with(fixed_norm(image.get("alt", "")))

    for br in root.find_all("br"):
        br.replace_with(" ")

    return fixed_norm(root.get_text(" ", strip=True))


def fixed_clean_question(text: str) -> str:
    text = FAQ_PREFIX_RE.sub("", fixed_norm(text))
    text = OPEN_SUFFIX_RE.sub("", text)
    return fixed_norm(text)


def fixed_clean_answer(text: str) -> str:
    return fixed_norm(ANSWER_PREFIX_RE.sub("", fixed_norm(text)))


def fixed_clean_term(text: str) -> str:
    text = OPEN_SUFFIX_RE.sub("", fixed_norm(text))
    text = re.sub(r"^\s*\d{1,2}\s+(?=\D)", "", text)
    return fixed_norm(text)


def fixed_infer_file_format(button: Tag) -> str:
    search_text = (
        " ".join(button.get("class", [])).lower()
        + " "
        + fixed_norm(button.get_text(" ", strip=True)).lower()
    )
    for keyword, file_format in [
        ("icohwp", "HWP"), ("hwp", "HWP"),
        ("icopdf", "PDF"), ("pdf", "PDF"),
        ("xlsx", "XLSX"), ("excel", "XLSX"),
        ("docx", "DOCX"), ("doc", "DOC"),
    ]:
        if keyword in search_text:
            return file_format
    return "FILE"


def fixed_cell_text(cell: Tag) -> str:
    fragment = BeautifulSoup(str(cell), "lxml")
    root = fragment.body.find() if fragment.body else fragment.find()
    if not root:
        return ""

    for child in root.select("script,style,noscript,template,.sr-only"):
        child.decompose()

    for button in root.find_all("button"):
        file_format = fixed_infer_file_format(button)
        label = fixed_norm(button.get_text(" ", strip=True))
        label = re.sub(r"다운로드$", "", label).strip()
        replacement = (
            f"{label} {file_format} 다운로드".strip()
            if label
            else f"{file_format} 다운로드"
        )
        button.replace_with(replacement)

    for image in root.find_all("img"):
        image.replace_with(fixed_norm(image.get("alt", "")))

    for br in root.find_all("br"):
        br.replace_with(" ")

    return fixed_norm(root.get_text(" ", strip=True))


def fixed_expand_table(table: Tag) -> tuple[list[str], list[list[str]]]:
    """rowspan/colspan을 펼쳐 각 행이 같은 열 수를 갖도록 만듭니다."""
    grid: list[list[str]] = []
    active_rowspans: dict[int, list[Any]] = {}
    header_flags: list[bool] = []
    max_columns = 0

    for tr in table.find_all("tr"):
        row: list[str] = []
        column = 0
        cells = tr.find_all(["th", "td"], recursive=False)

        def fill_active() -> None:
            nonlocal column
            while column in active_rowspans:
                remaining, value = active_rowspans[column]
                while len(row) <= column:
                    row.append("")
                row[column] = value
                remaining -= 1
                if remaining <= 0:
                    del active_rowspans[column]
                else:
                    active_rowspans[column] = [remaining, value]
                column += 1

        fill_active()
        header_flags.append(any(cell.name == "th" for cell in cells))

        for cell in cells:
            fill_active()
            text = fixed_cell_text(cell)
            rowspan = max(1, int(cell.get("rowspan", 1) or 1))
            colspan = max(1, int(cell.get("colspan", 1) or 1))

            for offset in range(colspan):
                target_column = column + offset
                while len(row) <= target_column:
                    row.append("")
                row[target_column] = text
                if rowspan > 1:
                    active_rowspans[target_column] = [rowspan - 1, text]

            column += colspan

        if active_rowspans:
            final_active_column = max(active_rowspans)
            while column <= final_active_column:
                if column in active_rowspans:
                    fill_active()
                else:
                    while len(row) <= column:
                        row.append("")
                    column += 1

        max_columns = max(max_columns, len(row))
        if row and any(row):
            grid.append(row)

    if not grid:
        return [], []

    for row in grid:
        row.extend([""] * (max_columns - len(row)))

    if table.thead:
        header_count = len(table.thead.find_all("tr", recursive=False))
    else:
        header_count = 1 if header_flags and header_flags[0] else 0

    if header_count == 0:
        header_count = 1

    header_rows = grid[:header_count]
    rows = grid[header_count:]
    headers: list[str] = []

    for column in range(max_columns):
        values: list[str] = []
        for header_row in header_rows:
            value = fixed_norm(header_row[column])
            if value and value not in values:
                values.append(value)
        headers.append(" / ".join(values) if values else f"열{column + 1}")

    # 중복 헤더 이름을 고유하게 만듭니다.
    counts: dict[str, int] = {}
    unique_headers: list[str] = []
    for raw_header in headers:
        header = raw_header or "열"
        counts[header] = counts.get(header, 0) + 1
        unique_headers.append(
            header if counts[header] == 1 else f"{header} {counts[header]}"
        )

    return unique_headers, rows


def fixed_extract_attachments(root: Tag, base_url: str) -> list[dict[str, Any]]:
    attachments: list[dict[str, Any]] = []
    seen: set[Any] = set()

    # KDIC 사이트의 JavaScript 다운로드 버튼
    for button in root.find_all("button", onclick=True):
        match = DOWNLOAD_CALL_RE.search(button.get("onclick", ""))
        if not match:
            continue

        key = (match.group(1), match.group(2))
        if key in seen:
            continue
        seen.add(key)

        file_format = fixed_infer_file_format(button)
        label = fixed_norm(button.get_text(" ", strip=True))
        label = re.sub(r"다운로드$", "", label).strip()

        row_context = ""
        parent_row = button.find_parent("tr")
        if parent_row:
            values: list[str] = []
            for cell in parent_row.find_all(["th", "td"], recursive=False):
                if button in cell.descendants:
                    continue
                value = fixed_cell_text(cell)
                if value and "다운로드" not in value and value not in values:
                    values.append(value)
            row_context = " | ".join(values[:4])

        display_name = label or row_context or f"{file_format} 첨부파일"
        if file_format not in display_name.upper():
            display_name = f"{display_name} ({file_format})"

        attachment_id = sha256_text(
            f"{match.group(1)}|{match.group(2)}"
        )[:16]

        attachments.append(
            {
                "attachment_id": attachment_id,
                "display_name": display_name,
                "file_format": file_format,
                "download_method": "gfn_downloadFile",
                "token1": match.group(1),
                "token2": match.group(2),
                "row_context": row_context,
                "button_text": fixed_norm(button.get_text(" ", strip=True)),
                "source_url": base_url,
                "download_status": "metadata_only",
            }
        )

    # href에 직접 연결된 첨부파일
    attachment_extensions = {
        ".pdf", ".hwp", ".hwpx", ".doc", ".docx",
        ".xls", ".xlsx", ".ppt", ".pptx", ".zip", ".csv",
    }

    for anchor in root.find_all("a", href=True):
        href = anchor.get("href", "")
        url = absolute_url(base_url, href)
        if not url:
            continue

        extension = Path(urlparse(url).path).suffix.lower()
        if extension not in attachment_extensions:
            continue

        key = ("href", url)
        if key in seen:
            continue
        seen.add(key)

        attachments.append(
            {
                "attachment_id": sha256_text(url)[:16],
                "display_name": (
                    fixed_norm(anchor.get_text(" ", strip=True))
                    or Path(urlparse(url).path).name
                ),
                "file_format": extension.lstrip(".").upper(),
                "download_method": "href",
                "url": url,
                "source_url": base_url,
                "download_status": "metadata_only",
            }
        )

    return attachments


def fixed_extract_links(root: Tag, base_url: str) -> list[dict[str, Any]]:
    links: list[dict[str, Any]] = []
    seen: set[Any] = set()

    for anchor in root.find_all("a", href=True):
        href = anchor.get("href", "").strip()
        if not href or href.startswith(("#", "javascript:", "mailto:", "tel:")):
            continue

        url = urljoin(base_url, href)
        text = fixed_norm(anchor.get_text(" ", strip=True))
        if not text or text in FIXED_NOISE_TEXTS:
            continue

        key = (url, text)
        if key in seen:
            continue
        seen.add(key)
        links.append({"url": url, "anchor_text": text})

    for button in root.find_all("button", onclick=True):
        match = MOVE_URL_RE.search(button.get("onclick", ""))
        if not match:
            continue
        url = urljoin(base_url, match.group(1))
        text = fixed_norm(button.get_text(" ", strip=True))
        key = (url, text)
        if key not in seen:
            seen.add(key)
            links.append(
                {"url": url, "anchor_text": text, "link_type": "button"}
            )

    return links


def fixed_extract_images(root: Tag, base_url: str) -> list[dict[str, Any]]:
    images: list[dict[str, Any]] = []
    seen: set[str] = set()

    for image in root.find_all("img"):
        source = (
            image.get("src")
            or image.get("data-src")
            or image.get("data-original")
        )
        url = absolute_url(base_url, source)
        if not url or url in seen:
            continue
        seen.add(url)

        alt = fixed_norm(image.get("alt", ""))
        filename = Path(urlparse(url).path).name
        lowered = filename.lower()
        image_role = "decorative"

        if "webtoon" in lowered:
            image_role = "supplementary_visual"
        elif any(keyword in lowered for keyword in ["proc", "process", "step", "flow"]):
            image_role = "process_diagram"
        elif alt:
            image_role = "content_image"

        images.append(
            {
                "url": url,
                "alt": alt,
                "filename": filename,
                "image_role": image_role,
            }
        )

    return images


class FixedStructureParser:
    def __init__(self, page_type: str = "static_page"):
        self.blocks: list[dict[str, Any]] = []
        self.page_type = page_type

    def add(self, block: dict[str, Any] | None) -> None:
        if not block:
            return

        if (
            self.blocks
            and block.get("type") in {"paragraph", "heading"}
            and self.blocks[-1].get("type") == block.get("type")
            and self.blocks[-1].get("text") == block.get("text")
        ):
            return

        self.blocks.append(block)

    def parse(self, root: Tag) -> list[dict[str, Any]]:
        faq_dls = root.select(
            ".accoWrap .accodian > dl, .accodian.con.ico > dl"
        )

        if self.page_type == "faq" or faq_dls:
            faq_nodes = {id(node) for node in faq_dls}

            for child in root.find_all(recursive=False):
                if not isinstance(child, Tag):
                    self.process_node(child)
                    continue

                child_dls = child.select("dl")
                if any(id(dl) in faq_nodes for dl in child_dls):
                    continue
                self.process_node(child)

            for dl in faq_dls:
                self.parse_faq(dl)
        else:
            for child in root.find_all(recursive=False):
                self.process_node(child)

        return self.blocks

    def parse_faq(self, dl: Tag) -> None:
        dt = dl.find("dt", recursive=False)
        dd = dl.find("dd", recursive=False)
        if not dt or not dd:
            return

        question = fixed_clean_question(fixed_safe_text(dt))
        answer_parser = FixedStructureParser("static_page")
        answer_parser.process_node(dd)

        if not answer_parser.blocks:
            answer_text = fixed_clean_answer(fixed_safe_text(dd))
            answer_parser.add({"type": "paragraph", "text": answer_text})
        else:
            for block in answer_parser.blocks:
                if "text" in block:
                    block["text"] = fixed_clean_answer(block["text"])
                    break

        answer_text = fixed_blocks_text(answer_parser.blocks)
        self.add(
            {
                "type": "faq",
                "question": question,
                "answer_blocks": answer_parser.blocks,
                "answer_text": answer_text,
            }
        )

    def parse_definition_list(self, dl: Tag) -> None:
        children = [
            child
            for child in dl.find_all(recursive=False)
            if isinstance(child, Tag)
        ]
        index = 0

        while index < len(children):
            if children[index].name != "dt":
                self.process_node(children[index])
                index += 1
                continue

            dt = children[index]
            dd = (
                children[index + 1]
                if index + 1 < len(children)
                and children[index + 1].name == "dd"
                else None
            )
            term = fixed_clean_term(fixed_safe_text(dt))

            if dd:
                definition_parser = FixedStructureParser("static_page")
                definition_parser.process_node(dd)
                if not definition_parser.blocks:
                    definition_parser.add(
                        {"type": "paragraph", "text": fixed_safe_text(dd)}
                    )

                self.add(
                    {
                        "type": "definition",
                        "term": term,
                        "definition_blocks": definition_parser.blocks,
                        "definition_text": fixed_blocks_text(
                            definition_parser.blocks
                        ),
                    }
                )
                index += 2
            else:
                self.add({"type": "heading", "level": 3, "text": term})
                index += 1

    def process_node(self, node: Any) -> None:
        if isinstance(node, Comment):
            return

        if isinstance(node, NavigableString):
            text = fixed_norm(str(node))
            if text and text not in FIXED_NOISE_TEXTS:
                self.add({"type": "paragraph", "text": text})
            return

        if not isinstance(node, Tag):
            return

        name = node.name.lower()
        classes = set(node.get("class", []))

        if (
            name in {"script", "style", "noscript", "template"}
            or classes & {
                "floatTop", "floatBottom", "paging", "pagination", "sr-only"
            }
        ):
            return

        if name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
            text = fixed_safe_text(node)
            if text and text not in FIXED_NOISE_TEXTS:
                self.add(
                    {"type": "heading", "level": int(name[1]), "text": text}
                )
            return

        if name == "p":
            text = fixed_safe_text(node)
            if text and text not in FIXED_NOISE_TEXTS:
                self.add({"type": "paragraph", "text": text})
            return

        if name == "table":
            headers, rows = fixed_expand_table(node)
            if headers or rows:
                self.add(
                    {
                        "type": "table",
                        "headers": headers,
                        "rows": rows,
                        "row_count": len(rows),
                    }
                )
            return

        if name in {"ul", "ol"}:
            items: list[str] = []
            for li in node.find_all("li", recursive=False):
                fragment = BeautifulSoup(str(li), "lxml")
                copied_li = fragment.body.find("li") if fragment.body else None
                if not copied_li:
                    continue
                for nested in copied_li.find_all(["ul", "ol"]):
                    nested.decompose()
                text = fixed_safe_text(copied_li)
                if text:
                    items.append(text)

            if items:
                self.add(
                    {"type": "list", "ordered": name == "ol", "items": items}
                )

            for li in node.find_all("li", recursive=False):
                for nested in li.find_all(["ul", "ol"], recursive=False):
                    self.process_node(nested)
            return

        if name == "dl":
            self.parse_definition_list(node)
            return

        if name == "figure":
            caption = node.find("figcaption")
            if caption:
                text = fixed_safe_text(caption)
                if text:
                    self.add({"type": "paragraph", "text": text})
            return

        # div, span 등이 가진 직접 텍스트와 하위 블록의 순서를 보존합니다.
        inline_parts: list[str] = []

        def flush_inline() -> None:
            nonlocal inline_parts
            text = fixed_norm(" ".join(inline_parts))
            inline_parts = []
            if text and text not in FIXED_NOISE_TEXTS:
                self.add({"type": "paragraph", "text": text})

        for child in node.children:
            if isinstance(child, Comment):
                continue
            if isinstance(child, NavigableString):
                text = fixed_norm(str(child))
                if text:
                    inline_parts.append(text)
            elif isinstance(child, Tag):
                child_name = child.name.lower()
                if child_name == "br":
                    inline_parts.append(" ")
                elif child_name in BLOCK_TAGS:
                    flush_inline()
                    self.process_node(child)
                else:
                    text = fixed_safe_text(child)
                    if text:
                        inline_parts.append(text)

        flush_inline()


def fixed_render_blocks(blocks: list[dict[str, Any]]) -> str:
    lines: list[str] = []

    for block in blocks:
        block_type = block.get("type")

        if block_type == "heading":
            level = min(max(int(block.get("level", 2)), 2), 6)
            lines.append(f"{'#' * level} {block.get('text', '')}")
        elif block_type == "paragraph":
            lines.append(block.get("text", ""))
        elif block_type == "list":
            for index, item in enumerate(block.get("items", []), start=1):
                prefix = f"{index}. " if block.get("ordered") else "- "
                lines.append(prefix + item)
        elif block_type == "definition":
            lines.append("### " + block.get("term", ""))
            lines.append(fixed_render_blocks(block.get("definition_blocks", [])))
        elif block_type == "faq":
            lines.append("### Q. " + block.get("question", ""))
            lines.append(fixed_render_blocks(block.get("answer_blocks", [])))
        elif block_type == "table":
            headers = [
                fixed_norm(header).replace("|", "\\|")
                for header in block.get("headers", [])
            ]
            if headers:
                lines.append("| " + " | ".join(headers) + " |")
                lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
                for row in block.get("rows", []):
                    normalized_row = [
                        fixed_norm(value).replace("|", "\\|")
                        for value in row
                    ]
                    normalized_row.extend(
                        [""] * (len(headers) - len(normalized_row))
                    )
                    lines.append(
                        "| " + " | ".join(normalized_row[: len(headers)]) + " |"
                    )

        lines.append("")

    return re.sub(r"\n{3,}", "\n\n", "\n".join(lines)).strip()


def fixed_blocks_text(blocks: list[dict[str, Any]]) -> str:
    values: list[str] = []

    for block in blocks:
        block_type = block.get("type")
        if block_type in {"heading", "paragraph"}:
            values.append(block.get("text", ""))
        elif block_type == "list":
            values.extend(block.get("items", []))
        elif block_type == "definition":
            values.extend(
                [block.get("term", ""), block.get("definition_text", "")]
            )
        elif block_type == "faq":
            values.extend(
                [block.get("question", ""), block.get("answer_text", "")]
            )
        elif block_type == "table":
            values.extend(block.get("headers", []))
            for row in block.get("rows", []):
                values.extend(row)

    return fixed_norm(" ".join(values))


def fixed_manifest_title(manifest_title: str, business_function: str) -> str:
    # Manifest에서 사람이 확인한 전체 경로명을 사용합니다.
    # HTML 내부의 "퀵메뉴" 같은 잘못된 제목을 선택하지 않습니다.
    return fixed_norm(manifest_title) or business_function


def fixed_manifest_breadcrumb(
    manifest_title: str,
    business_function: str,
) -> list[str]:
    parts = [
        fixed_norm(part)
        for part in re.split(r"\s*>\s*", manifest_title or "")
        if fixed_norm(part)
    ]
    result: list[str] = []
    for value in [business_function] + parts:
        if value and value not in result:
            result.append(value)
    return result


def parse_html_document(
    html_bytes: bytes,
    final_url: str,
    manifest_row: dict[str, str],
    encoding: str,
) -> dict[str, Any]:
    html_text = html_bytes.decode(encoding, errors="replace")
    soup = BeautifulSoup(html_text, "lxml")

    # 실제 KDIC 본문 컨테이너를 우선합니다.
    root = (
        soup.select_one(".contents")
        or soup.select_one("#contents")
        or soup.select_one("#content")
        or soup.select_one("main")
        or soup.body
    )
    if not root:
        raise ValueError("본문 루트를 찾지 못했습니다.")

    # 버튼은 노이즈 제거 전에 추출해야 합니다.
    attachments = fixed_extract_attachments(root, final_url)
    images = fixed_extract_images(root, final_url)

    for selector in FIXED_NOISE_SELECTORS:
        for node in root.select(selector):
            node.decompose()

    links = fixed_extract_links(root, final_url)
    source_text = fixed_norm(root.get_text(" ", strip=True))

    structure_parser = FixedStructureParser(
        manifest_row.get("page_type", "static_page")
    )
    blocks = structure_parser.parse(root)
    markdown = fixed_render_blocks(blocks)
    content_text = fixed_blocks_text(blocks)

    # UI 문구 마지막 안전 제거
    for phrase in FIXED_NOISE_TEXTS:
        markdown = re.sub(
            rf"(?m)^\s*{re.escape(phrase)}\s*$",
            "",
            markdown,
        )
    markdown = re.sub(r"\n{3,}", "\n\n", markdown).strip()

    content_text = fixed_norm(
        re.sub(
            "|".join(re.escape(phrase) for phrase in FIXED_NOISE_TEXTS),
            " ",
            content_text,
        )
    )

    faq_count = sum(1 for block in blocks if block.get("type") == "faq")
    table_count = sum(1 for block in blocks if block.get("type") == "table")
    noise_hits = [
        phrase for phrase in FIXED_NOISE_TEXTS if phrase in markdown
    ]
    retention_ratio = round(
        len(content_text) / max(1, len(source_text)),
        3,
    )

    quality_reasons: list[str] = []
    if len(content_text) < 80:
        quality_reasons.append("본문이 80자 미만")
    if retention_ratio < 0.70:
        quality_reasons.append("본문 보존율 70% 미만")
    if noise_hits:
        quality_reasons.append("공통 UI 문구 잔존")
    if manifest_row.get("page_type") == "faq" and faq_count == 0:
        quality_reasons.append("FAQ 질문-답변 추출 실패")

    if any(
        reason in {"공통 UI 문구 잔존", "FAQ 질문-답변 추출 실패"}
        for reason in quality_reasons
    ):
        quality_status = "fail"
    elif quality_reasons:
        quality_status = "review"
    else:
        quality_status = "pass"

    parsed_hash = sha256_text(markdown)

    document = {
        "doc_id": manifest_row["url_id"],
        "parent_doc_id": manifest_row["url_id"],
        "record_type": "page",
        "title": fixed_manifest_title(
            manifest_row["page_title"],
            manifest_row["business_function"],
        ),
        "manifest_title": manifest_row["page_title"],
        "html_title": (
            fixed_norm(soup.title.get_text(" ", strip=True))
            if soup.title
            else ""
        ),
        "source_url": manifest_row["source_url"],
        "final_url": final_url,
        "site_name": manifest_row["site_name"],
        "business_function": manifest_row["business_function"],
        "target_business_function": manifest_row.get(
            "target_business_function",
            manifest_row["business_function"],
        ),
        "page_type": manifest_row["page_type"],
        "parser_profile": manifest_row["parser_profile"],
        "breadcrumb": fixed_manifest_breadcrumb(
            manifest_row["page_title"],
            manifest_row["business_function"],
        ),
        "content_markdown": markdown,
        "content_text": content_text,
        "parsed_content_sha256": parsed_hash,
        "content_hash": parsed_hash,
        "blocks": blocks,
        "links": links,
        "attachments": attachments,
        "images": images,
        "videos": [],
        "quality": {
            "status": quality_status,
            "reasons": quality_reasons,
            "source_text_chars": len(source_text),
            "parsed_text_chars": len(content_text),
            "retention_ratio": retention_ratio,
            "faq_count": faq_count,
            "table_count": table_count,
            "attachment_button_count": len(attachments),
            "noise_hits": noise_hits,
        },
        "parsed_at": now_kst_iso(),
    }

    if manifest_row["url_id"] == "BI-004":
        document["dynamic_collection"] = {
            "collection_scope": "initial_public_page_only",
            "is_complete": False,
            "current_page_count": 1,
        }

    return document

## 7. 첨부파일·이미지 다운로드

In [ ]:
CONTENT_TYPE_EXTENSIONS = {
    "application/pdf": ".pdf",
    "application/zip": ".zip",
    "application/vnd.openxmlformats-officedocument.wordprocessingml.document": ".docx",
    "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet": ".xlsx",
    "application/vnd.openxmlformats-officedocument.presentationml.presentation": ".pptx",
    "image/jpeg": ".jpg",
    "image/png": ".png",
    "image/gif": ".gif",
    "image/webp": ".webp",
    "image/svg+xml": ".svg",
}


def filename_from_response(
    url: str,
    response: requests.Response,
    fallback_stem: str,
) -> str:
    disposition = response.headers.get("Content-Disposition", "")
    filename_star = re.search(
        r"filename\*=UTF-8''([^;]+)",
        disposition,
        flags=re.IGNORECASE,
    )
    filename_plain = re.search(
        r'filename="?([^";]+)"?',
        disposition,
        flags=re.IGNORECASE,
    )

    if filename_star:
        from urllib.parse import unquote
        name = unquote(filename_star.group(1))
    elif filename_plain:
        name = filename_plain.group(1)
    else:
        name = Path(urlparse(url).path).name

    name = safe_name(name, max_length=120)
    extension = Path(name).suffix.lower()

    if not extension:
        content_type = response.headers.get("Content-Type", "").split(";")[0].strip()
        extension = CONTENT_TYPE_EXTENSIONS.get(content_type)
        if not extension:
            extension = mimetypes.guess_extension(content_type) or ""
        name = f"{safe_name(fallback_stem)}{extension}"

    return name


def download_binary_asset(
    url: str,
    output_dir: Path,
    fallback_stem: str,
) -> dict[str, Any]:
    output_dir.mkdir(parents=True, exist_ok=True)

    with SESSION.get(
        url,
        timeout=REQUEST_TIMEOUT_SECONDS,
        allow_redirects=True,
        stream=True,
    ) as response:
        response.raise_for_status()

        content_length = response.headers.get("Content-Length")
        if content_length and int(content_length) > MAX_ASSET_BYTES:
            raise ValueError("파일이 제한 용량을 초과합니다.")

        filename = filename_from_response(url, response, fallback_stem)
        output_path = output_dir / filename

        hasher = hashlib.sha256()
        written = 0

        with output_path.open("wb") as file:
            for chunk in response.iter_content(chunk_size=64 * 1024):
                if not chunk:
                    continue
                written += len(chunk)
                if written > MAX_ASSET_BYTES:
                    output_path.unlink(missing_ok=True)
                    raise ValueError("다운로드 중 제한 용량을 초과했습니다.")
                hasher.update(chunk)
                file.write(chunk)

        return {
            "source_url": url,
            "final_url": response.url,
            "filename": filename,
            "relative_path": str(output_path.relative_to(OUTPUT_ROOT)),
            "content_type": response.headers.get("Content-Type", ""),
            "size_bytes": written,
            "sha256": hasher.hexdigest(),
            "downloaded_at": now_kst_iso(),
            "download_status": "downloaded",
        }


def ensure_attachment_playwright() -> None:
    try:
        import playwright  # noqa: F401
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "playwright"],
            check=True,
        )
    subprocess.run(
        [sys.executable, "-m", "playwright", "install", "chromium"],
        check=True,
    )


def download_js_attachments_for_document(
    document: dict[str, Any],
    output_dir: Path,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    js_attachments = [
        item
        for item in document.get("attachments", [])
        if item.get("download_method") == "gfn_downloadFile"
    ]
    if not js_attachments:
        return [], []

    ensure_attachment_playwright()
    from playwright.sync_api import sync_playwright

    downloaded: list[dict[str, Any]] = []
    failures: list[dict[str, Any]] = []
    output_dir.mkdir(parents=True, exist_ok=True)

    with sync_playwright() as playwright:
        browser = playwright.chromium.launch(headless=True)
        context = browser.new_context(
            accept_downloads=True,
            user_agent=USER_AGENT,
            locale="ko-KR",
        )
        page = context.new_page()

        try:
            page.goto(
                document["source_url"],
                wait_until="domcontentloaded",
                timeout=REQUEST_TIMEOUT_SECONDS * 1000,
            )
            page.wait_for_function(
                "typeof gfn_downloadFile === 'function'",
                timeout=REQUEST_TIMEOUT_SECONDS * 1000,
            )

            for item in js_attachments:
                try:
                    with page.expect_download(timeout=60_000) as download_info:
                        page.evaluate(
                            """
                            ([token1, token2]) => {
                                gfn_downloadFile(token1, token2);
                            }
                            """,
                            [item["token1"], item["token2"]],
                        )

                    download = download_info.value
                    filename = safe_name(download.suggested_filename, max_length=120)
                    output_path = output_dir / filename
                    download.save_as(str(output_path))

                    downloaded.append(
                        {
                            **item,
                            "filename": filename,
                            "relative_path": str(output_path.relative_to(OUTPUT_ROOT)),
                            "size_bytes": output_path.stat().st_size,
                            "sha256": sha256_bytes(output_path.read_bytes()),
                            "download_status": "downloaded",
                            "error": "",
                        }
                    )
                except Exception as error:
                    failures.append(
                        {
                            "asset_type": "attachment",
                            "display_name": item.get("display_name", ""),
                            "download_method": "gfn_downloadFile",
                            "error_type": type(error).__name__,
                            "error": str(error),
                        }
                    )
        finally:
            context.close()
            browser.close()

    return downloaded, failures


def download_document_assets(
    document: dict[str, Any],
    manifest_row: dict[str, str],
) -> dict[str, list[dict[str, Any]]]:
    business_dir = safe_name(manifest_row["business_function"])
    asset_root = PATHS["raw_assets"] / business_dir / manifest_row["url_id"]

    results = {
        "attachments": [],
        "attachment_metadata": [],
        "images": [],
        "videos": [],
        "failures": [],
    }

    should_download_attachments = (
        DOWNLOAD_ATTACHMENTS
        and (
            manifest_row["asset_policy"] == "download_attachments"
            or manifest_row["page_type"] == "attachment_page"
        )
    )

    # JS 버튼은 URL이 없으므로 우선 메타데이터로 항상 보존합니다.
    for item in document.get("attachments", []):
        if item.get("download_method") == "gfn_downloadFile":
            results["attachment_metadata"].append(item)

    # 직접 href 첨부파일
    if should_download_attachments:
        for index, item in enumerate(document.get("attachments", []), start=1):
            if item.get("download_method") != "href" or not item.get("url"):
                continue
            try:
                result = download_binary_asset(
                    item["url"],
                    asset_root / "attachments",
                    f"{manifest_row['url_id']}_attachment_{index:03d}",
                )
                result.update(item)
                results["attachments"].append(result)
            except Exception as error:
                results["failures"].append(
                    {
                        "asset_type": "attachment",
                        "url": item.get("url", ""),
                        "error_type": type(error).__name__,
                        "error": str(error),
                    }
                )

    # 선택적으로 JavaScript 버튼 실제 다운로드
    if should_download_attachments and DOWNLOAD_JS_ATTACHMENTS_WITH_PLAYWRIGHT:
        downloaded, failures = download_js_attachments_for_document(
            document,
            asset_root / "attachments",
        )
        results["attachments"].extend(downloaded)
        results["failures"].extend(failures)

    should_download_images = (
        DOWNLOAD_IMAGES
        and (
            manifest_row["crawl_wave"] == "W1_ASSET"
            or manifest_row["page_type"] in {
                "process_page", "video_page", "attachment_page"
            }
        )
    )

    if should_download_images:
        for index, item in enumerate(document.get("images", []), start=1):
            if item.get("image_role") == "decorative":
                continue
            try:
                result = download_binary_asset(
                    item["url"],
                    asset_root / "images",
                    f"{manifest_row['url_id']}_image_{index:03d}",
                )
                result.update(item)
                results["images"].append(result)
            except Exception as error:
                results["failures"].append(
                    {
                        "asset_type": "image",
                        "url": item.get("url", ""),
                        "error_type": type(error).__name__,
                        "error": str(error),
                    }
                )

    return results

## 8. 동적 조회 페이지 진단

In [ ]:
DYNAMIC_SCRIPT_KEYWORD_PATTERN = re.compile(
    r"(?:ajax|fetch\s*\(|XMLHttpRequest|\.do\b|pageIndex|pageNo|search)",
    flags=re.IGNORECASE,
)


def extract_dynamic_diagnostic(
    html_bytes: bytes,
    final_url: str,
    encoding: str,
) -> dict[str, Any]:
    """공개 동적 페이지의 폼과 스크립트 구조를 진단합니다."""
    html_text = html_bytes.decode(encoding, errors="replace")
    soup = BeautifulSoup(html_text, "lxml")

    forms = []

    for form_index, form in enumerate(
        soup.find_all("form"),
        start=1,
    ):
        inputs = []

        for field in form.select(
            "input, select, textarea, button"
        ):
            name = normalize_space(field.get("name", ""))
            field_type = normalize_space(
                field.get("type", field.name or "")
            )

            if not name and field.name != "button":
                continue

            record = {
                "tag": field.name,
                "name": name,
                "type": field_type,
                "value": normalize_space(
                    field.get("value", "")
                ),
            }

            if field.name == "select":
                record["options"] = [
                    {
                        "value": normalize_space(
                            option.get("value", "")
                        ),
                        "text": normalize_space(
                            option.get_text(" ", strip=True)
                        ),
                    }
                    for option in field.find_all("option")
                ]

            inputs.append(record)

        forms.append(
            {
                "form_index": form_index,
                "method": normalize_space(
                    form.get("method", "GET")
                ).upper(),
                "action": (
                    absolute_url(
                        final_url,
                        form.get("action"),
                    )
                    or final_url
                ),
                "id": normalize_space(form.get("id", "")),
                "name": normalize_space(
                    form.get("name", "")
                ),
                "inputs": inputs,
            }
        )

    script_sources = []

    for script in soup.find_all("script", src=True):
        source = absolute_url(
            final_url,
            script.get("src"),
        )

        if source and source not in script_sources:
            script_sources.append(source)

    inline_script_hints = []

    for script in soup.find_all("script"):
        if script.get("src"):
            continue

        text = script.get_text("\n", strip=True)

        if (
            text
            and DYNAMIC_SCRIPT_KEYWORD_PATTERN.search(text)
        ):
            inline_script_hints.append(text[:1200])

    return {
        "final_url": final_url,
        "forms": forms,
        "script_sources": script_sources,
        "inline_script_hints": inline_script_hints[:20],
        "diagnosed_at": now_kst_iso(),
        "safety_note": (
            "공개 페이지 구조만 진단합니다. "
            "로그인·본인인증·개인정보 조회·신청 제출은 수행하지 않습니다."
        ),
    }


def bi004_extract_base_payload(
    html_bytes: bytes,
    encoding: str,
) -> dict[str, str]:
    """BI-004 검색 폼의 기본 공개 검색 조건을 추출합니다."""
    soup = BeautifulSoup(
        html_bytes.decode(encoding, errors="replace"),
        "lxml",
    )
    form = soup.select_one("form#srchForm")

    if not form:
        raise ValueError(
            "BI-004 검색 폼 form#srchForm을 찾지 못했습니다."
        )

    payload: dict[str, str] = {}

    for field in form.select(
        "input[name], select[name], textarea[name]"
    ):
        name = normalize_space(field.get("name", ""))
        if not name:
            continue

        if field.name == "select":
            selected = field.find("option", selected=True)
            if selected is None:
                selected = field.find("option")
            value = (
                selected.get("value", "")
                if selected
                else ""
            )

        elif field.name == "textarea":
            value = field.get_text("", strip=False)

        else:
            field_type = normalize_space(
                field.get("type", "text")
            ).lower()

            if (
                field_type in {"checkbox", "radio"}
                and not field.has_attr("checked")
            ):
                continue

            value = field.get("value", "")

        payload[name] = str(value or "")

    # 전체 금융권역·전체 회사 조회
    payload["fncSareaCd"] = ""
    payload["searchInfnstNm"] = ""

    return payload


def bi004_last_internal_page_index(
    html_bytes: bytes,
    encoding: str,
) -> int:
    """
    BI-004는 curPage가 0부터 시작합니다.
    화면의 1페이지는 curPage=0, 화면의 2페이지는 curPage=1입니다.
    """
    soup = BeautifulSoup(
        html_bytes.decode(encoding, errors="replace"),
        "lxml",
    )

    page_indexes: list[int] = []

    for node in soup.select(
        ".paging [onclick*='chgPagingNo']"
    ):
        onclick = node.get("onclick", "")
        match = re.search(
            r"chgPagingNo\(\s*(\d+)\s*\)",
            onclick,
        )
        if match:
            page_indexes.append(int(match.group(1)))

    return max(page_indexes, default=0)


def bi004_displayed_page_number(
    html_bytes: bytes,
    encoding: str,
) -> int | None:
    soup = BeautifulSoup(
        html_bytes.decode(encoding, errors="replace"),
        "lxml",
    )

    current = soup.select_one(
        ".paging strong[title*='현재 페이지'] span"
    ) or soup.select_one(".paging strong span")

    if not current:
        return None

    text = normalize_space(current.get_text(" ", strip=True))
    return int(text) if text.isdigit() else None


def bi004_parse_result_table(
    html_bytes: bytes,
    encoding: str,
) -> tuple[list[str], list[list[str]]]:
    """금융회사 조회 결과 표 하나를 구조화합니다."""
    soup = BeautifulSoup(
        html_bytes.decode(encoding, errors="replace"),
        "lxml",
    )

    root = (
        soup.select_one(".contents")
        or soup.select_one("#contents")
        or soup.select_one("main")
        or soup.body
    )

    if not root:
        raise ValueError("BI-004 본문 영역을 찾지 못했습니다.")

    candidate_tables = root.find_all("table")
    target_table = None

    for table in candidate_tables:
        table_text = normalize_space(
            table.get_text(" ", strip=True)
        )
        if (
            "금융권역" in table_text
            and "금융회사명" in table_text
        ):
            target_table = table
            break

    if target_table is None:
        raise ValueError(
            "BI-004 금융회사 결과 표를 찾지 못했습니다."
        )

    headers, rows = fixed_expand_table(target_table)

    cleaned_rows = [
        [normalize_space(value) for value in row]
        for row in rows
        if any(normalize_space(value) for value in row)
    ]

    if not headers:
        raise ValueError("BI-004 결과 표의 헤더가 없습니다.")

    return headers, cleaned_rows


def bi004_replace_document_table(
    document: dict[str, Any],
    headers: list[str],
    rows: list[list[str]],
) -> None:
    """첫 페이지 표를 전체 페이지 통합 표로 교체합니다."""
    new_blocks: list[dict[str, Any]] = []
    replaced = False

    for block in document.get("blocks", []):
        if block.get("type") != "table":
            new_blocks.append(block)
            continue

        block_headers = " ".join(
            block.get("headers", [])
        )

        if (
            not replaced
            and "금융권역" in block_headers
            and "금융회사명" in block_headers
        ):
            new_blocks.append(
                {
                    "type": "table",
                    "headers": headers,
                    "rows": rows,
                    "row_count": len(rows),
                    "table_role": (
                        "deposit_insurance_payment_target_companies"
                    ),
                }
            )
            replaced = True
        else:
            new_blocks.append(block)

    if not replaced:
        new_blocks.append(
            {
                "type": "table",
                "headers": headers,
                "rows": rows,
                "row_count": len(rows),
                "table_role": (
                    "deposit_insurance_payment_target_companies"
                ),
            }
        )

    document["blocks"] = new_blocks
    document["content_markdown"] = fixed_render_blocks(
        new_blocks
    )
    document["content_text"] = fixed_blocks_text(
        new_blocks
    )

    parsed_hash = sha256_text(
        document["content_markdown"]
    )
    document["parsed_content_sha256"] = parsed_hash
    document["content_hash"] = parsed_hash

    quality = document.setdefault("quality", {})
    quality["parsed_text_chars"] = len(
        document["content_text"]
    )
    quality["table_count"] = sum(
        1
        for block in new_blocks
        if block.get("type") == "table"
    )


def collect_bi004_all_pages(
    first_result: FetchResult,
    manifest_row: dict[str, str],
) -> dict[str, Any]:
    """
    첫 GET 응답을 1페이지로 사용하고,
    curPage=1부터 마지막 내부 인덱스까지 POST 요청합니다.
    """
    first_headers, first_rows = bi004_parse_result_table(
        first_result.body,
        first_result.encoding,
    )

    last_index = bi004_last_internal_page_index(
        first_result.body,
        first_result.encoding,
    )
    page_count = last_index + 1

    if page_count < 1:
        raise ValueError("BI-004 페이지 수가 1보다 작습니다.")

    if page_count > BI004_MAX_PAGES:
        raise ValueError(
            f"BI-004 페이지 수 {page_count}가 "
            f"안전 한도 {BI004_MAX_PAGES}를 초과했습니다."
        )

    base_payload = bi004_extract_base_payload(
        first_result.body,
        first_result.encoding,
    )

    pages_dir = (
        PATHS["dynamic_diagnostics"]
        / "BI-004_pages"
    )
    pages_dir.mkdir(parents=True, exist_ok=True)

    combined_rows: list[list[str]] = []
    seen_rows: set[tuple[str, ...]] = set()
    seen_page_hashes: dict[str, int] = {}
    page_records: list[dict[str, Any]] = []
    failures: list[dict[str, Any]] = []

    def register_page(
        internal_index: int,
        body: bytes,
        encoding: str,
        status_code: int,
        final_url: str,
        request_method: str,
        request_payload: dict[str, str] | None,
    ) -> None:
        displayed_page = bi004_displayed_page_number(
            body,
            encoding,
        )
        headers, rows = bi004_parse_result_table(
            body,
            encoding,
        )

        if headers != first_headers:
            failures.append(
                {
                    "internal_page_index": internal_index,
                    "reason": "페이지별 표 헤더가 서로 다름",
                    "headers": headers,
                }
            )

        row_hash = sha256_text(
            json.dumps(
                rows,
                ensure_ascii=False,
                sort_keys=True,
            )
        )

        if row_hash in seen_page_hashes:
            failures.append(
                {
                    "internal_page_index": internal_index,
                    "reason": "다른 페이지와 동일한 표 결과 반복",
                    "same_as_internal_index": (
                        seen_page_hashes[row_hash]
                    ),
                }
            )
        else:
            seen_page_hashes[row_hash] = internal_index

        page_filename = (
            f"page_{internal_index + 1:03d}.html"
        )
        page_path = pages_dir / page_filename
        page_path.write_bytes(body)

        page_meta = {
            "internal_page_index": internal_index,
            "displayed_page_number": displayed_page,
            "status_code": status_code,
            "request_method": request_method,
            "request_payload": request_payload,
            "final_url": final_url,
            "row_count": len(rows),
            "row_hash": row_hash,
            "raw_sha256": sha256_bytes(body),
            "raw_html_path": str(
                page_path.relative_to(OUTPUT_ROOT)
            ),
        }
        page_records.append(page_meta)

        expected_displayed_page = internal_index + 1
        if (
            displayed_page is not None
            and displayed_page != expected_displayed_page
        ):
            failures.append(
                {
                    "internal_page_index": internal_index,
                    "reason": (
                        "요청 페이지와 화면 현재 페이지 불일치"
                    ),
                    "expected_displayed_page": (
                        expected_displayed_page
                    ),
                    "actual_displayed_page": displayed_page,
                }
            )

        for row in rows:
            key = tuple(row)
            if key not in seen_rows:
                seen_rows.add(key)
                combined_rows.append(row)

    # GET으로 확보한 첫 페이지
    register_page(
        internal_index=0,
        body=first_result.body,
        encoding=first_result.encoding,
        status_code=first_result.status_code,
        final_url=first_result.final_url,
        request_method="GET",
        request_payload=None,
    )

    # 화면 2페이지부터 마지막 페이지까지 반복 POST
    for internal_index in range(1, page_count):
        payload = dict(base_payload)
        payload["curPage"] = str(internal_index)
        payload["pageSize"] = str(BI004_PAGE_SIZE)

        try:
            response = SESSION.post(
                manifest_row["source_url"],
                data=payload,
                headers={
                    "Referer": manifest_row["source_url"],
                },
                timeout=REQUEST_TIMEOUT_SECONDS,
                allow_redirects=True,
            )

            response.raise_for_status()

            encoding = choose_response_encoding(response)
            response.encoding = encoding

            register_page(
                internal_index=internal_index,
                body=response.content,
                encoding=encoding,
                status_code=response.status_code,
                final_url=response.url,
                request_method="POST",
                request_payload=payload,
            )

        except Exception as error:
            failures.append(
                {
                    "internal_page_index": internal_index,
                    "reason": "페이지 요청 또는 파싱 실패",
                    "error_type": type(error).__name__,
                    "error": str(error),
                }
            )

        time.sleep(REQUEST_DELAY_SECONDS)

    fetched_indexes = {
        record["internal_page_index"]
        for record in page_records
    }
    expected_indexes = set(range(page_count))

    missing_indexes = sorted(
        expected_indexes - fetched_indexes
    )

    if missing_indexes:
        failures.append(
            {
                "reason": "수집되지 않은 페이지 존재",
                "missing_internal_indexes": missing_indexes,
            }
        )

    is_complete = (
        not failures
        and len(page_records) == page_count
        and len(combined_rows) > 0
    )

    all_rows_csv = (
        PATHS["dynamic_diagnostics"]
        / "BI-004_all_rows.csv"
    )
    pd.DataFrame(
        combined_rows,
        columns=first_headers,
    ).to_csv(
        all_rows_csv,
        index=False,
        encoding="utf-8-sig",
    )

    return {
        "headers": first_headers,
        "rows": combined_rows,
        "collection": {
            "collection_scope": (
                "all_public_pages"
                if is_complete
                else "partial_public_pages"
            ),
            "is_complete": is_complete,
            "page_index_base": 0,
            "expected_page_count": page_count,
            "fetched_page_count": len(page_records),
            "row_count": len(combined_rows),
            "page_size": BI004_PAGE_SIZE,
            "search_filters": {
                "fncSareaCd": "",
                "searchInfnstNm": "",
            },
            "pages": page_records,
            "failures": failures,
            "all_rows_csv_path": str(
                all_rows_csv.relative_to(OUTPUT_ROOT)
            ),
            "collected_at": now_kst_iso(),
        },
    }


def ensure_playwright_installed() -> None:
    try:
        import playwright  # noqa: F401
        return
    except ImportError:
        pass

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "playwright"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "playwright", "install", "chromium"],
        check=True,
    )


def save_playwright_snapshot(
    url: str,
    url_id: str,
) -> dict[str, Any] | None:
    if not ENABLE_PLAYWRIGHT_SNAPSHOT:
        return None

    ensure_playwright_installed()
    from playwright.sync_api import sync_playwright

    html_path = PATHS["dynamic_diagnostics"] / f"{url_id}_rendered.html"
    screenshot_path = PATHS["screenshots"] / f"{url_id}.png"
    ensure_parent(html_path)
    ensure_parent(screenshot_path)

    with sync_playwright() as playwright:
        browser = playwright.chromium.launch(headless=True)
        page = browser.new_page(
            user_agent=USER_AGENT,
            locale="ko-KR",
            viewport={"width": 1440, "height": 1200},
        )

        response = page.goto(
            url,
            wait_until="domcontentloaded",
            timeout=REQUEST_TIMEOUT_SECONDS * 1000,
        )
        page.wait_for_timeout(PLAYWRIGHT_WAIT_MS)

        html = page.content()
        html_path.write_text(html, encoding="utf-8")
        page.screenshot(path=str(screenshot_path), full_page=True)

        result = {
            "status_code": response.status if response else None,
            "final_url": page.url,
            "rendered_html_path": str(html_path.relative_to(OUTPUT_ROOT)),
            "screenshot_path": str(screenshot_path.relative_to(OUTPUT_ROOT)),
            "captured_at": now_kst_iso(),
        }

        browser.close()
        return result

### 동적 페이지 진단 코드 자체 테스트

실제 웹 요청 전에 다음을 검사합니다.

- `fetch(...)`
- `XMLHttpRequest`
- `.do` 엔드포인트
- `pageIndex`, `pageNo`, `search`
- 합성 HTML의 폼·인라인 스크립트 추출

이 셀이 실패하면 실제 크롤링을 실행하지 마십시오.

In [ ]:
_DYNAMIC_PATTERN_TEST_CASES = {
    "fetch('/api/list')": True,
    "const xhr = new XMLHttpRequest();": True,
    "url: '/board/selectList.do'": True,
    "pageIndex = 2": True,
    "pageNo = 3": True,
    "searchKeyword": True,
    "일반 안내 문장입니다.": False,
}

for sample_text, expected in _DYNAMIC_PATTERN_TEST_CASES.items():
    actual = bool(DYNAMIC_SCRIPT_KEYWORD_PATTERN.search(sample_text))
    assert actual == expected, (
        f"동적 스크립트 정규표현식 테스트 실패: "
        f"text={sample_text!r}, expected={expected}, actual={actual}"
    )

_DYNAMIC_TEST_HTML = """
<!doctype html>
<html lang="ko">
<body>
  <form id="searchForm" method="post" action="/search/list.do">
    <input type="hidden" name="pageIndex" value="1">
    <input type="text" name="searchKeyword" value="">
    <select name="category">
      <option value="">전체</option>
      <option value="bank">은행</option>
    </select>
  </form>

  <script>
    function loadData() {
      fetch('/api/list?pageNo=1');
    }
  </script>
</body>
</html>
""".encode("utf-8")

_dynamic_test_result = extract_dynamic_diagnostic(
    html_bytes=_DYNAMIC_TEST_HTML,
    final_url="https://example.org/dynamic/page",
    encoding="utf-8",
)

assert len(_dynamic_test_result["forms"]) == 1
assert _dynamic_test_result["forms"][0]["method"] == "POST"
assert _dynamic_test_result["forms"][0]["action"] == (
    "https://example.org/search/list.do"
)
assert any(
    item["name"] == "pageIndex"
    for item in _dynamic_test_result["forms"][0]["inputs"]
)
assert len(_dynamic_test_result["inline_script_hints"]) == 1

print("동적 페이지 진단 자체 테스트: 통과")
print("정규표현식:", DYNAMIC_SCRIPT_KEYWORD_PATTERN.pattern)


_BI004_PAGINATION_TEST_HTML = """
<!doctype html>
<html lang="ko">
<body>
  <form id="srchForm" method="post">
    <select name="fncSareaCd">
      <option value="" selected>전체</option>
    </select>
    <input name="searchInfnstNm" type="text" value="">
  </form>

  <div class="contents">
    <table>
      <thead>
        <tr>
          <th>금융권역</th>
          <th>금융회사명</th>
          <th>상태</th>
        </tr>
      </thead>
      <tbody>
        <tr>
          <td>상호저축은행</td>
          <td>예시은행</td>
          <td>파산</td>
        </tr>
      </tbody>
    </table>

    <div class="paging">
      <strong title="현재 페이지"><span>1</span></strong>
      <a onclick="chgPagingNo(1);">2</a>
      <a onclick="chgPagingNo(2);">3</a>
      <a class="btnLast" onclick="chgPagingNo(2);">
        마지막 페이지
      </a>
    </div>
  </div>
</body>
</html>
""".encode("utf-8")

assert bi004_last_internal_page_index(
    _BI004_PAGINATION_TEST_HTML,
    "utf-8",
) == 2

assert bi004_displayed_page_number(
    _BI004_PAGINATION_TEST_HTML,
    "utf-8",
) == 1

_test_headers, _test_rows = bi004_parse_result_table(
    _BI004_PAGINATION_TEST_HTML,
    "utf-8",
)

assert _test_headers == [
    "금융권역",
    "금융회사명",
    "상태",
]
assert len(_test_rows) == 1
assert _test_rows[0][1] == "예시은행"

_test_payload = bi004_extract_base_payload(
    _BI004_PAGINATION_TEST_HTML,
    "utf-8",
)
assert _test_payload["fncSareaCd"] == ""
assert _test_payload["searchInfnstNm"] == ""

print("BI-004 페이지네이션 자체 테스트: 통과")


## 9. URL 유형별 처리기

In [ ]:

def save_parsed_document(
    document: dict[str, Any],
    manifest_row: dict[str, str],
) -> tuple[Path, Path]:
    business_dir = safe_name(manifest_row["business_function"])
    url_id = manifest_row["url_id"]

    markdown_path = (
        PATHS["parsed_markdown"]
        / business_dir
        / f"{url_id}.md"
    )
    json_path = (
        PATHS["parsed_json"]
        / business_dir
        / f"{url_id}.json"
    )

    ensure_parent(markdown_path)
    markdown_path.write_text(
        document["content_markdown"],
        encoding="utf-8",
    )
    write_json(json_path, document)

    return markdown_path, json_path


def process_link_only(
    manifest_row: dict[str, str],
) -> dict[str, Any]:
    record = {
        "doc_id": manifest_row["url_id"],
        "record_type": "link_only",
        "title": manifest_row["page_title"],
        "source_url": manifest_row["source_url"],
        "site_name": manifest_row["site_name"],
        "business_function": manifest_row["business_function"],
        "target_business_function": manifest_row.get(
            "target_business_function",
            manifest_row["business_function"],
        ),
        "page_type": manifest_row["page_type"],
        "auth_required": manifest_row["auth_required"],
        "content_scope": manifest_row["content_scope"],
        "description": manifest_row.get("content_summary", ""),
        "decision_reason": manifest_row["decision_reason"],
        "created_at": now_kst_iso(),
    }

    output_path = (
        PATHS["parsed_json"]
        / safe_name(manifest_row["business_function"])
        / f"{manifest_row['url_id']}.json"
    )
    write_json(output_path, record)

    return {
        "url_id": manifest_row["url_id"],
        "business_function": manifest_row["business_function"],
        "page_title": manifest_row["page_title"],
        "source_url": manifest_row["source_url"],
        "normalized_decision": manifest_row["normalized_decision"],
        "crawl_wave": manifest_row["crawl_wave"],
        "status": "link_metadata_created",
        "status_code": None,
        "content_chars": len(record["description"]),
        "attachment_count": 0,
        "image_count": 0,
        "asset_failure_count": 0,
        "error_type": "",
        "error": "",
        "finished_at": now_kst_iso(),
    }


def process_static_or_asset(
    manifest_row: dict[str, str],
) -> dict[str, Any]:
    result = fetch_url(manifest_row["source_url"])
    html_path, meta_path = save_fetch_result(manifest_row, result)

    if not result.ok:
        raise requests.HTTPError(
            f"HTTP {result.status_code}: {result.final_url}"
        )

    if "html" not in result.content_type.lower():
        raise ValueError(
            f"HTML 응답이 아닙니다: {result.content_type}"
        )

    document = parse_html_document(
        html_bytes=result.body,
        final_url=result.final_url,
        manifest_row=manifest_row,
        encoding=result.encoding,
    )

    asset_results = download_document_assets(
        document=document,
        manifest_row=manifest_row,
    )
    document["downloaded_assets"] = asset_results

    markdown_path, json_path = save_parsed_document(
        document=document,
        manifest_row=manifest_row,
    )

    return {
        "url_id": manifest_row["url_id"],
        "business_function": manifest_row["business_function"],
        "page_title": document["title"],
        "source_url": manifest_row["source_url"],
        "final_url": result.final_url,
        "normalized_decision": manifest_row["normalized_decision"],
        "crawl_wave": manifest_row["crawl_wave"],
        "status": (
            "parse_success"
            if document["content_text"]
            else "empty_content"
        ),
        "quality_status": document.get("quality", {}).get("status", ""),
        "quality_reasons": " | ".join(document.get("quality", {}).get("reasons", [])),
        "retention_ratio": document.get("quality", {}).get("retention_ratio", 0),
        "faq_count": document.get("quality", {}).get("faq_count", 0),
        "table_count": document.get("quality", {}).get("table_count", 0),
        "attachment_button_count": document.get("quality", {}).get("attachment_button_count", 0),
        "status_code": result.status_code,
        "content_chars": len(document["content_text"]),
        "block_count": len(document["blocks"]),
        "link_count": len(document["links"]),
        "attachment_count": len(document["attachments"]),
        "image_count": len(document["images"]),
        "downloaded_attachment_count": len(asset_results["attachments"]),
        "downloaded_image_count": len(asset_results["images"]),
        "asset_failure_count": len(asset_results["failures"]),
        "raw_html_path": str(html_path.relative_to(OUTPUT_ROOT)),
        "response_meta_path": str(meta_path.relative_to(OUTPUT_ROOT)),
        "parsed_markdown_path": str(markdown_path.relative_to(OUTPUT_ROOT)),
        "parsed_json_path": str(json_path.relative_to(OUTPUT_ROOT)),
        "raw_sha256": result.raw_sha256,
        "parsed_content_sha256": document["parsed_content_sha256"],
        "error_type": "",
        "error": "",
        "finished_at": now_kst_iso(),
    }


def process_dynamic_diagnostic(
    manifest_row: dict[str, str],
) -> dict[str, Any]:
    result = fetch_url(manifest_row["source_url"])
    html_path, meta_path = save_fetch_result(
        manifest_row,
        result,
    )

    if not result.ok:
        raise requests.HTTPError(
            f"HTTP {result.status_code}: {result.final_url}"
        )

    diagnostic = extract_dynamic_diagnostic(
        html_bytes=result.body,
        final_url=result.final_url,
        encoding=result.encoding,
    )
    diagnostic["url_id"] = manifest_row["url_id"]
    diagnostic["page_title"] = manifest_row["page_title"]
    diagnostic["business_function"] = (
        manifest_row["business_function"]
    )
    diagnostic["source_url"] = manifest_row["source_url"]

    playwright_snapshot = save_playwright_snapshot(
        url=manifest_row["source_url"],
        url_id=manifest_row["url_id"],
    )
    if playwright_snapshot:
        diagnostic["playwright_snapshot"] = (
            playwright_snapshot
        )

    document = parse_html_document(
        html_bytes=result.body,
        final_url=result.final_url,
        manifest_row=manifest_row,
        encoding=result.encoding,
    )

    dynamic_status = "dynamic_diagnostic_created"
    dynamic_page_count = 1
    dynamic_row_count = 0
    dynamic_is_complete = False

    if (
        manifest_row["url_id"] == "BI-004"
        and FETCH_BI004_ALL_PAGES
    ):
        collection_result = collect_bi004_all_pages(
            first_result=result,
            manifest_row=manifest_row,
        )

        bi004_replace_document_table(
            document=document,
            headers=collection_result["headers"],
            rows=collection_result["rows"],
        )

        document["dynamic_collection"] = (
            collection_result["collection"]
        )
        diagnostic["bi004_full_collection"] = (
            collection_result["collection"]
        )

        dynamic_page_count = collection_result[
            "collection"
        ]["fetched_page_count"]
        dynamic_row_count = collection_result[
            "collection"
        ]["row_count"]
        dynamic_is_complete = collection_result[
            "collection"
        ]["is_complete"]

        dynamic_status = (
            "dynamic_full_collection_created"
            if dynamic_is_complete
            else "dynamic_collection_incomplete"
        )

    else:
        document["dynamic_collection"] = {
            "collection_scope": (
                "initial_public_page_only"
            ),
            "is_complete": False,
            "expected_page_count": None,
            "fetched_page_count": 1,
            "row_count": None,
            "page_size": BI004_PAGE_SIZE,
        }
        diagnostic["bi004_full_collection"] = (
            document["dynamic_collection"]
        )

    diagnostic_path = (
        PATHS["dynamic_diagnostics"]
        / f"{manifest_row['url_id']}.json"
    )
    write_json(diagnostic_path, diagnostic)

    markdown_path, json_path = save_parsed_document(
        document=document,
        manifest_row=manifest_row,
    )

    collection_failures = (
        document.get("dynamic_collection", {})
        .get("failures", [])
    )

    return {
        "url_id": manifest_row["url_id"],
        "business_function": (
            manifest_row["business_function"]
        ),
        "page_title": document["title"],
        "source_url": manifest_row["source_url"],
        "final_url": result.final_url,
        "normalized_decision": (
            manifest_row["normalized_decision"]
        ),
        "crawl_wave": manifest_row["crawl_wave"],
        "status": dynamic_status,
        "quality_status": (
            document.get("quality", {}).get(
                "status",
                "",
            )
        ),
        "quality_reasons": " | ".join(
            document.get("quality", {}).get(
                "reasons",
                [],
            )
        ),
        "retention_ratio": (
            document.get("quality", {}).get(
                "retention_ratio",
                0,
            )
        ),
        "faq_count": (
            document.get("quality", {}).get(
                "faq_count",
                0,
            )
        ),
        "table_count": (
            document.get("quality", {}).get(
                "table_count",
                0,
            )
        ),
        "attachment_button_count": (
            document.get("quality", {}).get(
                "attachment_button_count",
                0,
            )
        ),
        "status_code": result.status_code,
        "content_chars": len(
            document["content_text"]
        ),
        "form_count": len(diagnostic["forms"]),
        "script_source_count": len(
            diagnostic["script_sources"]
        ),
        "dynamic_is_complete": (
            dynamic_is_complete
        ),
        "dynamic_page_count": (
            dynamic_page_count
        ),
        "dynamic_row_count": (
            dynamic_row_count
        ),
        "dynamic_failure_count": len(
            collection_failures
        ),
        "attachment_count": len(
            document["attachments"]
        ),
        "image_count": len(
            document["images"]
        ),
        "asset_failure_count": 0,
        "raw_html_path": str(
            html_path.relative_to(OUTPUT_ROOT)
        ),
        "response_meta_path": str(
            meta_path.relative_to(OUTPUT_ROOT)
        ),
        "parsed_markdown_path": str(
            markdown_path.relative_to(OUTPUT_ROOT)
        ),
        "parsed_json_path": str(
            json_path.relative_to(OUTPUT_ROOT)
        ),
        "dynamic_diagnostic_path": str(
            diagnostic_path.relative_to(OUTPUT_ROOT)
        ),
        "error_type": (
            ""
            if dynamic_is_complete
            or not FETCH_BI004_ALL_PAGES
            else "IncompleteDynamicCollection"
        ),
        "error": (
            ""
            if dynamic_is_complete
            or not FETCH_BI004_ALL_PAGES
            else json.dumps(
                collection_failures,
                ensure_ascii=False,
            )
        ),
        "finished_at": now_kst_iso(),
    }

def process_manifest_row(
    manifest_row: dict[str, str],
) -> dict[str, Any]:
    decision = manifest_row["normalized_decision"]
    strategy = manifest_row["fetch_strategy"]

    if decision == "link_only":
        return process_link_only(manifest_row)

    if strategy == "dynamic_http_then_playwright":
        return process_dynamic_diagnostic(manifest_row)

    return process_static_or_asset(manifest_row)


## 10. 크롤링 실행

In [ ]:

CRAWL_RESULTS_JSONL = PATHS["logs"] / "crawl_results.jsonl"
CRAWL_RESULTS_CSV = PATHS["logs"] / "crawl_results.csv"
USED_MANIFEST_PATH = PATHS["manifest"] / "manifest_used.csv"

target_df.to_csv(
    USED_MANIFEST_PATH,
    index=False,
    encoding="utf-8-sig",
)

run_results: list[dict[str, Any]] = []

for _, row in tqdm(
    target_df.iterrows(),
    total=len(target_df),
    desc="크롤링 진행",
):
    manifest_row = row_to_clean_dict(row)
    started_at = now_kst_iso()

    try:
        result = process_manifest_row(manifest_row)
        result["started_at"] = started_at

    except Exception as error:
        result = {
            "url_id": manifest_row["url_id"],
            "business_function": manifest_row["business_function"],
            "page_title": manifest_row["page_title"],
            "source_url": manifest_row["source_url"],
            "normalized_decision": manifest_row["normalized_decision"],
            "crawl_wave": manifest_row["crawl_wave"],
            "status": "failed",
            "status_code": None,
            "content_chars": 0,
            "attachment_count": 0,
            "image_count": 0,
            "asset_failure_count": 0,
            "error_type": type(error).__name__,
            "error": str(error),
            "started_at": started_at,
            "finished_at": now_kst_iso(),
        }

    run_results.append(result)
    append_jsonl(CRAWL_RESULTS_JSONL, result)

    # link_only는 실제 HTTP 요청을 보내지 않으므로 대기하지 않습니다.
    if manifest_row["normalized_decision"] != "link_only":
        time.sleep(REQUEST_DELAY_SECONDS)

results_df = pd.DataFrame(run_results)
results_df.to_csv(
    CRAWL_RESULTS_CSV,
    index=False,
    encoding="utf-8-sig",
)

print("실행 완료")

for optional_column, default_value in {
    "dynamic_is_complete": None,
    "dynamic_page_count": None,
    "dynamic_row_count": None,
    "dynamic_failure_count": None,
}.items():
    if optional_column not in results_df.columns:
        results_df[optional_column] = default_value

display(
    results_df[
        [
            "url_id",
            "business_function",
            "status",
            "status_code",
            "content_chars",
            "attachment_count",
            "image_count",
            "asset_failure_count",
            "dynamic_is_complete",
            "dynamic_page_count",
            "dynamic_row_count",
            "dynamic_failure_count",
            "error_type",
        ]
    ]
)


## 11. 결과 검수

In [ ]:
QUALITY_REPORT_CSV = PATHS["logs"] / "quality_report.csv"

if results_df.empty:
    print("실행 결과가 없습니다.")
else:
    print("수집 상태별 건수")
    display(
        results_df.groupby("status")
        .size()
        .rename("count")
        .reset_index()
        .sort_values("count", ascending=False)
    )

    failures = results_df[results_df["status"] == "failed"].copy()
    empty_contents = results_df[
        results_df["status"].isin({"empty_content"})
        | (
            results_df["status"].isin(
                {"parse_success", "dynamic_diagnostic_created"}
            )
            & (results_df["content_chars"].fillna(0).astype(int) < 80)
        )
    ].copy()

    quality_columns = [
        "url_id", "business_function", "page_title", "status",
        "quality_status", "quality_reasons", "retention_ratio",
        "content_chars", "faq_count", "table_count",
        "attachment_button_count", "attachment_count",
        "image_count", "asset_failure_count", "error_type", "error",
    ]
    available_quality_columns = [
        column for column in quality_columns if column in results_df.columns
    ]
    quality_report_df = results_df[available_quality_columns].copy()
    quality_report_df.to_csv(
        QUALITY_REPORT_CSV,
        index=False,
        encoding="utf-8-sig",
    )

    print("실패:", len(failures))
    print("빈 본문·매우 짧은 본문:", len(empty_contents))

    if "quality_status" in results_df.columns:
        print("파싱 품질 상태")
        display(
            results_df.groupby("quality_status", dropna=False)
            .size()
            .rename("count")
            .reset_index()
        )

        review_required = results_df[
            results_df["quality_status"].isin({"review", "fail"})
        ]
        if not review_required.empty:
            print("사람 검수 또는 파서 수정 필요")
            display(
                review_required[
                    [
                        "url_id", "page_title", "quality_status",
                        "quality_reasons", "retention_ratio",
                        "content_chars",
                    ]
                ]
            )

    if not failures.empty:
        display(
            failures[
                ["url_id", "page_title", "source_url", "error_type", "error"]
            ]
        )

    success_statuses = {
        "parse_success",
        "dynamic_diagnostic_created",
        "link_metadata_created",
    }
    success_count = results_df["status"].isin(success_statuses).sum()
    success_rate = success_count / len(results_df) * 100
    print(f"수집 처리 성공률: {success_count}/{len(results_df)} ({success_rate:.1f}%)")
    print("품질 보고서:", QUALITY_REPORT_CSV)

## 12. 페이지 문서 JSONL과 기본 청크 생성

In [ ]:
DOCUMENTS_JSONL = PATHS["processed"] / "documents.jsonl"
CHUNKS_JSONL = PATHS["processed"] / "chunks.jsonl"


def iter_parsed_json_files() -> Iterable[Path]:
    yield from sorted(PATHS["parsed_json"].rglob("*.json"))


def build_document_record(data: dict[str, Any]) -> dict[str, Any] | None:
    if data.get("record_type") == "link_only":
        return {
            "doc_id": data["doc_id"],
            "parent_doc_id": data["doc_id"],
            "record_type": "link_only",
            "indexable": False,
            "title": data["title"],
            "source_url": data["source_url"],
            "site_name": data["site_name"],
            "business_function": data["business_function"],
            "target_business_function": data["target_business_function"],
            "page_type": data["page_type"],
            "content_markdown": data.get("description", ""),
            "content_text": data.get("description", ""),
            "content_hash": sha256_text(data.get("description", "")),
            "blocks": [],
            "attachments": [],
            "images": [],
            "links": [
                {
                    "url": data["source_url"],
                    "anchor_text": data["title"],
                    "link_type": "official_service",
                }
            ],
            "metadata": {
                "auth_required": data.get("auth_required", ""),
                "content_scope": data.get("content_scope", ""),
                "decision_reason": data.get("decision_reason", ""),
            },
        }

    if not data.get("content_markdown"):
        return None

    return {
        "doc_id": data["doc_id"],
        "parent_doc_id": data.get("parent_doc_id", data["doc_id"]),
        "record_type": "page",
        "indexable": True,
        "title": data.get("title", ""),
        "source_url": data.get("source_url", ""),
        "final_url": data.get("final_url", ""),
        "site_name": data.get("site_name", ""),
        "business_function": data.get("business_function", ""),
        "target_business_function": data.get(
            "target_business_function",
            data.get("business_function", ""),
        ),
        "page_type": data.get("page_type", ""),
        "breadcrumb": data.get("breadcrumb", []),
        "content_markdown": data.get("content_markdown", ""),
        "content_text": data.get("content_text", ""),
        "content_hash": data.get(
            "parsed_content_sha256",
            sha256_text(data.get("content_markdown", "")),
        ),
        "blocks": data.get("blocks", []),
        "attachments": data.get("attachments", []),
        "images": data.get("images", []),
        "links": data.get("links", []),
        "quality": data.get("quality", {}),
        "dynamic_collection": data.get("dynamic_collection"),
        "metadata": {
            "parser_profile": data.get("parser_profile", ""),
            "attachment_count": len(data.get("attachments", [])),
            "image_count": len(data.get("images", [])),
            "parsed_at": data.get("parsed_at", ""),
        },
    }


def render_table_chunk(
    headers: list[str],
    rows: list[list[str]],
) -> str:
    block = {"type": "table", "headers": headers, "rows": rows}
    return fixed_render_blocks([block])


def split_table_block(
    block: dict[str, Any],
    max_chars: int,
) -> list[str]:
    headers = block.get("headers", [])
    rows = block.get("rows", [])
    all_text = render_table_chunk(headers, rows)
    if len(all_text) <= max_chars:
        return [all_text]

    chunks: list[str] = []
    current_rows: list[list[str]] = []

    for row in rows:
        candidate_rows = current_rows + [row]
        candidate_text = render_table_chunk(headers, candidate_rows)

        if current_rows and len(candidate_text) > max_chars:
            chunks.append(render_table_chunk(headers, current_rows))
            current_rows = [row]
        else:
            current_rows = candidate_rows

    if current_rows:
        chunks.append(render_table_chunk(headers, current_rows))

    return chunks


def structure_aware_chunks(
    document: dict[str, Any],
    max_chars: int = 1600,
) -> list[dict[str, Any]]:
    if not document.get("indexable", True):
        return []

    intermediate: list[dict[str, str]] = []
    current_parts: list[str] = []
    current_heading = ""

    def flush_section() -> None:
        nonlocal current_parts
        content = "\n\n".join(
            part for part in current_parts if part
        ).strip()
        current_parts = []
        if content:
            intermediate.append(
                {
                    "section_title": current_heading,
                    "chunk_type": "section",
                    "content": content,
                }
            )

    for block in document.get("blocks", []):
        block_type = block.get("type")

        if block_type == "heading":
            flush_section()
            current_heading = block.get("text", "")
            current_parts = [fixed_render_blocks([block])]
            continue

        if block_type == "faq":
            flush_section()
            intermediate.append(
                {
                    "section_title": block.get("question", ""),
                    "chunk_type": "faq",
                    "content": fixed_render_blocks([block]),
                }
            )
            continue

        if block_type == "table":
            flush_section()
            for table_text in split_table_block(block, max_chars):
                intermediate.append(
                    {
                        "section_title": current_heading,
                        "chunk_type": "table",
                        "content": table_text,
                    }
                )
            continue

        block_text = fixed_render_blocks([block])
        candidate = "\n\n".join(current_parts + [block_text]).strip()
        if current_parts and len(candidate) > max_chars:
            flush_section()
        current_parts.append(block_text)

    flush_section()

    records: list[dict[str, Any]] = []
    for index, item in enumerate(intermediate):
        content = item["content"].strip()
        if not content:
            continue

        records.append(
            {
                "chunk_id": f"{document['doc_id']}_chunk_{index:03d}",
                "parent_doc_id": document["doc_id"],
                "chunk_index": index,
                "title": document["title"],
                "section_title": item["section_title"],
                "chunk_type": item["chunk_type"],
                "business_function": document["business_function"],
                "target_business_function": document[
                    "target_business_function"
                ],
                "page_type": document["page_type"],
                "source_url": document["source_url"],
                "content": content,
                "content_hash": sha256_text(content),
            }
        )

    return records


documents: list[dict[str, Any]] = []

for json_path in iter_parsed_json_files():
    data = json.loads(json_path.read_text(encoding="utf-8"))
    document = build_document_record(data)
    if document:
        documents.append(document)

with DOCUMENTS_JSONL.open("w", encoding="utf-8") as file:
    for document in documents:
        file.write(json.dumps(document, ensure_ascii=False) + "\n")

chunk_records: list[dict[str, Any]] = []

if CREATE_BASELINE_CHUNKS:
    for document in documents:
        chunk_records.extend(
            structure_aware_chunks(
                document,
                max_chars=max(1600, CHUNK_MAX_CHARS),
            )
        )

with CHUNKS_JSONL.open("w", encoding="utf-8") as file:
    for chunk in chunk_records:
        file.write(json.dumps(chunk, ensure_ascii=False) + "\n")

print("페이지 문서:", len(documents))
print("구조 기반 청크:", len(chunk_records))

if chunk_records:
    chunk_df = pd.DataFrame(chunk_records)
    display(
        chunk_df.groupby("chunk_type")
        .size()
        .rename("count")
        .reset_index()
    )

# 핵심 회귀 검사: 기존 분석에서 누락됐던 내용 확인
by_id = {document["doc_id"]: document for document in documents}
validation_rows: list[dict[str, Any]] = []

if "DP-001" in by_id:
    validation_rows.append(
        {
            "검증": "DP-001 보호한도 핵심 문구",
            "통과": "1억원" in by_id["DP-001"].get("content_text", ""),
        }
    )

if "UN-003" in by_id:
    un003_text = by_id["UN-003"].get("content_text", "")
    validation_rows.append(
        {
            "검증": "UN-003 미수령금 종류",
            "통과": all(
                keyword in un003_text
                for keyword in ["예금보험금", "개산지급금", "파산배당금"]
            ),
        }
    )

for faq_id in ["DP-005", "DP-006", "MT-005", "HP-002"]:
    if faq_id in by_id:
        faq_count = sum(
            1
            for block in by_id[faq_id].get("blocks", [])
            if block.get("type") == "faq"
        )
        validation_rows.append(
            {"검증": f"{faq_id} FAQ 질문-답변", "통과": faq_count > 0}
        )

if validation_rows:
    validation_df = pd.DataFrame(validation_rows)
    display(validation_df)
    failed = validation_df[~validation_df["통과"]]
    if not failed.empty:
        print("주의: 아래 항목은 Markdown 직접 검수가 필요합니다.")
        display(failed)

print("documents.jsonl:", DOCUMENTS_JSONL)
print("chunks.jsonl:", CHUNKS_JSONL)

## 13. 결과 ZIP 생성 및 다운로드

In [ ]:

archive_path = Path(
    shutil.make_archive(
        base_name=str(OUTPUT_ROOT),
        format="zip",
        root_dir=OUTPUT_ROOT.parent,
        base_dir=OUTPUT_ROOT.name,
    )
)

print("결과 ZIP:", archive_path)
print("ZIP 크기:", f"{archive_path.stat().st_size / 1024 / 1024:.2f} MB")

try:
    from google.colab import files
    files.download(str(archive_path))
except Exception as error:
    print("자동 다운로드를 시작하지 못했습니다.")
    print("Colab 왼쪽 파일 패널에서 직접 다운로드하세요:", archive_path)
    print("오류:", error)



## 결과 폴더 구조

```text
kdic_crawl_output_<실행시각>/
├── manifest/
│   └── manifest_used.csv
├── raw/
│   ├── html/
│   ├── assets/
│   └── response_meta/
├── parsed/
│   ├── markdown/
│   └── json/
├── diagnostics/
│   ├── dynamic/
│   └── screenshots/
├── processed/
│   ├── documents.jsonl
│   └── chunks.jsonl
└── logs/
    ├── crawl_results.csv
    └── crawl_results.jsonl
```

## 권장 실행 순서

1. `MAX_URLS = 3`으로 테스트
2. 실패·빈 본문·자산 오류 확인
3. 사이트별 본문 선택자와 제거 선택자 조정
4. `MAX_URLS = None`으로 전체 실행
5. `Review_Queue` 결정 후 Manifest 수정본을 업로드하여 재실행
6. 동적 조회 페이지는 진단 JSON을 검토한 뒤 공개 HTTP 요청 전용 수집기를 별도로 작성

## BI-004만 먼저 재검증하는 방법

설정 셀:

```python
RUN_ONLY_URL_IDS = ["BI-004"]
MAX_URLS = None
```

정상 결과:

```text
status = dynamic_diagnostic_created
status_code = 200
```

생성되어야 하는 파일:

```text
parsed/markdown/예금보험금 안내/BI-004.md
parsed/json/예금보험금 안내/BI-004.json
diagnostics/dynamic/BI-004.json
raw/html/예금보험금 안내/BI-004.html
raw/response_meta/예금보험금 안내/BI-004.json
```

전체를 다시 실행할 때:

```python
RUN_ONLY_URL_IDS = []
MAX_URLS = None
```

## FIXED 버전에서 추가로 확인할 파일

```text
logs/quality_report.csv
```

이 파일은 HTTP 성공 여부와 별개로 다음을 보여줍니다.

- 본문 보존율
- FAQ 질문·답변 개수
- 표 개수
- 첨부파일 버튼 인식 개수
- 공통 UI 문구 잔존 여부
- 자동 판정 `pass`, `review`, `fail`

첨부파일 버튼은 `parsed/json/*/*.json`의 `attachments`에서 확인할 수 있습니다.
`DOWNLOAD_JS_ATTACHMENTS_WITH_PLAYWRIGHT = False`인 경우 실제 파일은 내려받지 않고 다운로드 메타데이터만 저장합니다.

## BI-004 전체 페이지 수집 결과 확인

기본 설정은 다음과 같습니다.

```python
FETCH_BI004_ALL_PAGES = True
```

정상 실행 시 `crawl_results.csv`의 BI-004 행:

```text
status = dynamic_full_collection_created
dynamic_is_complete = True
dynamic_page_count = 실제 전체 페이지 수
dynamic_row_count = 중복 제거 후 전체 금융회사 수
dynamic_failure_count = 0
```

페이지별 원본:

```text
diagnostics/dynamic/BI-004_pages/
├── page_001.html
├── page_002.html
└── ...
```

통합 목록:

```text
diagnostics/dynamic/BI-004_all_rows.csv
```

최종 JSON의 `dynamic_collection.is_complete`가 `true`일 때만
BI-004를 전체 공개 목록으로 사용합니다.

In [ ]:

# Colab 런타임에 필요한 라이브러리 설치
%pip -q install "requests>=2.32,<3" "beautifulsoup4>=4.12,<5" "lxml>=5,<7" "pandas>=2.2,<4" "tqdm>=4.66,<5"



## 1. 실행 환경과 설정

기본 설정은 정적 페이지·자산 페이지·링크 전용 레코드를 처리합니다.

- 특정 업무만 실행하려면 `SELECT_BUSINESS_FUNCTIONS`에 업무명을 입력합니다.
- 테스트 실행은 `MAX_URLS = 3`처럼 제한합니다.
- 동적 공개 페이지를 브라우저로 렌더링한 초기 화면까지 저장하려면 `ENABLE_PLAYWRIGHT_SNAPSHOT = True`로 바꿉니다.
- Playwright도 검색 폼 제출이나 로그인은 수행하지 않습니다.


In [ ]:

from __future__ import annotations

import hashlib
import json
import mimetypes
import os
import re
import shutil
import subprocess
import sys
import time
from dataclasses import dataclass, asdict
from datetime import datetime, timezone, timedelta
from pathlib import Path
from typing import Any, Iterable
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup, Tag
from IPython.display import display
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

KST = timezone(timedelta(hours=9))

# -------------------------
# 사용자 실행 설정
# -------------------------
SELECT_BUSINESS_FUNCTIONS: list[str] = []
# 예: ["예금자보호제도", "예금보험금 안내"]

RUN_ONLY_URL_IDS: list[str] = []
# BI-004 오류 수정만 먼저 검증하려면:
# RUN_ONLY_URL_IDS = ["BI-004"]
# 전체 실행하려면 빈 리스트 []를 사용합니다.

RUN_DECISIONS = {"include_full", "include_partial", "link_only"}
RUN_WAVES = {"W1_STATIC", "W1_ASSET", "W2_DYNAMIC", "META"}

MAX_URLS: int | None = None
# 최초 테스트 권장: 3
# 전체 실행: None

REQUEST_DELAY_SECONDS = 1.2
REQUEST_TIMEOUT_SECONDS = 25
MAX_RETRIES = 3
MAX_ASSET_BYTES = 25 * 1024 * 1024

RESPECT_ROBOTS_TXT = True
DOWNLOAD_ATTACHMENTS = True
DOWNLOAD_IMAGES = True
DOWNLOAD_VIDEOS = False

ENABLE_PLAYWRIGHT_SNAPSHOT = False
PLAYWRIGHT_WAIT_MS = 1500

# JavaScript gfn_downloadFile(...) 버튼을 실제 파일로 내려받을지 여부
# 처음에는 False로 파싱 품질부터 검증하고, 이후 True로 시험하는 것을 권장합니다.
DOWNLOAD_JS_ATTACHMENTS_WITH_PLAYWRIGHT = False

# BI-004 지급대상 금융회사 공개 목록의 전체 페이지를 POST로 수집합니다.
# 전체 수집을 원하지 않을 때만 False로 변경합니다.
FETCH_BI004_ALL_PAGES = True

# 사이트 오류로 비정상적으로 많은 페이지가 계산되는 상황을 막는 안전 한도
BI004_MAX_PAGES = 100
BI004_PAGE_SIZE = 10

CREATE_BASELINE_CHUNKS = True
CHUNK_MAX_CHARS = 1200
CHUNK_OVERLAP_CHARS = 150

USER_AGENT = (
    "KDIC-RAG-Academic-Crawler/0.1 "
    "(low-rate public-document collection; no authentication automation)"
)

RUN_ID = datetime.now(KST).strftime("%Y%m%d_%H%M%S")
RUNTIME_ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
OUTPUT_ROOT = RUNTIME_ROOT / f"kdic_crawl_output_{RUN_ID}"

PATHS = {
    "raw_html": OUTPUT_ROOT / "raw" / "html",
    "raw_assets": OUTPUT_ROOT / "raw" / "assets",
    "response_meta": OUTPUT_ROOT / "raw" / "response_meta",
    "parsed_markdown": OUTPUT_ROOT / "parsed" / "markdown",
    "parsed_json": OUTPUT_ROOT / "parsed" / "json",
    "dynamic_diagnostics": OUTPUT_ROOT / "diagnostics" / "dynamic",
    "screenshots": OUTPUT_ROOT / "diagnostics" / "screenshots",
    "logs": OUTPUT_ROOT / "logs",
    "processed": OUTPUT_ROOT / "processed",
    "manifest": OUTPUT_ROOT / "manifest",
}

for path in PATHS.values():
    path.mkdir(parents=True, exist_ok=True)

print("실행 ID:", RUN_ID)
print("결과 경로:", OUTPUT_ROOT)


## 2. 42개 URL Manifest 불러오기

In [ ]:

# 현재 프로젝트의 42개 Manifest를 노트북 내부에 압축하여 포함했습니다.
# 별도 CSV 업로드 없이 기본 Manifest를 자동 생성합니다.

import base64
import gzip

EMBEDDED_MANIFEST_GZIP_B64 = """H4sIAHmDVGoC/+1dbW8TWbL+PtL8h1akXSXaXhzbCS/3w1xlgJmJBmaZwOzofrIcuxO8OG3T3Q6T/TAy0EGGZJawNwYH7FznEkiCgtYJDnF2g66Un+Pu1v6FW3XqdLvbbjs2hLCajYRiu19On656TtVTdeoc/vmP/0to0lRETokZJRlJxMXxjJqQJVWNTGTkmJZIyaIWVSYlLdJ6Ih2dlCJaQktKoprKKDEpAm2IKjQYkaNTkphSEpMJOZqMpJUEfNdmRDmlTEWTiT9L8UhciiVUbMX+ElGkqOq0OpOWxAlJi12PqJoS1aTJGTihqJICjaUmEvDIaEa7DvfczCQUKS5GVRX6mE4lE7EZMZaSNUnWImosBc0o0nRCutW4NKZEbyUjt6LTEv+qalEto4rST2kppkHXUhktndFEWYLvvFNOi5mpqaiCL6JJ6uefBcULV34/OBgUzUKuXsuZywvGm6pVqJnlovFQb3OUflh59uO6pqXV/wgEbt26depGPBE7lVJO3VACajoQT8Obw8tqgSvw5+qMquHnpSlNDahSEnp6NabIp+Ip/hRs9Umh/mbHvLMpWnndzBXEhBxLZuISaCyZFM0ns8ZmVTDXcsZmzSwV4Ntts/xIqO/u17cqxkpRgDvNpxtCvZI1HryA/gnY2lxJMPM5405VsP5SMEtVcy0rosASsQhqSkSxSqqmRq5rU0kRXyHCT08HQUzwTwZNQgci46k4Ck78MRi5em3k2uh5lGIE5AgyF81nIKSa8M21y5eE3wn1nU1zpSLAO5krVfioQZfhsLGxaN3NGr9UoB+WXsE3N+8VQ0PCD2OXxKBZqhlze0JDHLaQhaD5eAceUH+zL5jVHHygVvIrgrGn03UCnDX3lgRTXzVezYqffxYixYZ6Uqz/0V40fKLZZs3WaxXoclAw5rPWo6KVf2282kA91itFY3ZBMEv75l7BR1DwqgL9Np7PGy9A8eXbxst3Vh4e+mjBfJIz5lbxGlRYccd6Og/34EPMuTJHkFnSQZDZeu2+lS8YDxYRI8bLzQZGwoSR8HsMfnzQ3dvCF97Hf4FvZT5d7AoyhBS3abiamVJOUOPYA7cdELjAPeI2/3fWeFHoN0tZ6/H9g10SGJw42LVym6i3tVX4B2KC74Lx31sDgnGnYD6pguqHSPVDH656+G79Nfehqr+mTMJnPHaCgW4xQHJHS4JOgM7QsX7SPhgCC46h3lEw8HTjrY7myFzNNmzAMAFhuCcg9NXflOuVkqnvQ4dB9V+NfA9/jb/VQD9GeQ/usO7tWPkN6IK5/bofVQDv9gDe4K0OiMWuwtPNvdeiYM6tQqMAzCpoCZDs8X4DfQ6cJhKy6sZTbCowPm4Tia+iN79TolPaSDo5g+DhEsrnzLlN6+Gmpdc+NngmojebQIMdjsDhT+Br+t3cgeyCPWwHWl1PXyvfEOE9dTAU4HOy5ts8GRhUnnmnQlcLdDnor3AbDUw5D1IQrGe643tq5luQXzlnVtbRe4G3Mp5vIlL6AHWnCXWnP5ydYIs9MRTAyuEkBaBirj06RrQwE/Np0MJI5fN9FCdwhHplCXporM0LocHQsDGrC5yCom15/ALNiWC3YD4BED25Z/wCstgpMuJxZ9NYyTr6rol2I+fMZ4t+xNbdKjWBtsTdTNUszQNizhBiznwIYuiZ/UChzKXaQDvUxBOB8T+nx5Oy46fGY8oV/G2HL3+YVtQ2wJF+YohxPulZaKLrOwvYEeBygJTTMARBsIAO8OIMVy5AGXM7xpssxxVSPDOvG3MFQMyEpEgyRKmx1BTEkgmM6+wHqTcSaQc5/I949dvRK+xMGgLB8Rk7uiR0AynEMWvmH6CEfXTq6btAnRfByNd3Xh/smn9fADIBWoavpX0AG3wWbrNTxsNZNC34Qi0KZzplDeL4AEIMqj0rfjnqikG5PMHG0Ljp7TD4IfI65lyR+bw93dSL4HS6MhIX0urU6Dj3JFdSqnohNiWfBDUtpiN0KmiLmQlXaBfXQFPgCgAT81kIPegr0AR9C4wCZ6V2QyUIYotlq5A33ugezdkK//yzc4SU0IcihT2QOycEjO95Ov0esFFSsU+OGOhWDFNd7w+ZkatXL147SsSEmuTeHjI8jAGSKRgQVNwtGW+zSDQpmIVXhjNgblhr8ElH9+bhCxMaNe8w3OAgoSb8YaiBFzKWs//ZGx4u/pROyifmw595hHrJidRy/fCiXNe13MEuhpvARu0DA6T6PPtwVB8k1Q8dlWtxHWIP9onK3yMA/hKQ8nViWsIg+CtZ1dqgBEYBUnDmvIX623lz+YUDGaADWiIKqFmpQDcE+KhvVQ92HQwY+ip+mQXhPF1k1lZnfF4vIXCMv64ieJCU1LcroAnkf/Q4gVPh+IwcnQKUJFOpG5m08xNfNqJdl+RIOhmduaUkJq9rhCrvDW50AQCVmYgiqZmkpoozkir+GIpc+K/vRi57McZ7ZGwvmkVdIMQFqH8+4OprRpe5lPXjtaJgzYOeKm6dMY+zDaq8z2/mgWgzI6IotU/0b6EFNW6ImE8g/M46T2FjFhEaEn/4jpEfCqk9QTQ3mJ1OeY7xoJtBIMCN7RcC3e2+sKd4iQXWo/JNdG3s+9eZ+IlJa2PSwi0mDeH3aoOZK7ePc2mjYbgYYks1aNfQd4EoVTbwoVyR+Gs73+BCwTAhJ3QEyHFna5AP3ZtH64q9YYc8j3b4EqNvHNxl3VpatJVKPWyXuMlEA1GA0iSi6AKiDa3eNSn5fUb1g9bxp3LaQ4uldI4fWmGgTl4suOBVLMOL4euQDgT79bd0Q18QMcsGvyCcrG+/EzAk4udJpSx76MZGvZo1yuu8LdAshN9cQJ5gPDhE2At/Aux1jy6yXNqkgqZrVFPil7Xr8ROIdWu9ACMPAXvBg92ww925NtpSd4+u2ewE02Rh3cCEAMsRrFqLOv1qtmwOaxsmdA19PJ8I0Dbv/YJ9IOWjApcLdFVP3vEbJS19k1DRQXaVVfzVO8RmGhZuk14UXTqwvQhx164YGFKutRK4oH5rcd9paUCwDZ7Hv+qNZ0Ev+XPsOVG9iImGB6sIvNPi5WuUiapsAFLNeyss1V0pWEsFG129ncKY9NkCT1UD7lxXuC8Q6AoIMDug78aUorI/aMu6ZWD9mKDH+RZbQwNeCJIimbTtrCDG3XQXHv5h7BIMW85iaIw2Kx1iioqpl8XpRFxK+UAwwmpbVB8k0h1TkhaNR7WoiHCEQCKj0h2NEx8lJUFAGzosCm2nVK4zNmx31/EYmDqaBHXZsjMEqdCngNQh+SsPnk6yVn4QGToVFHyE6nKA0AA0ygoxtirM9DxcAuSI3F9S4gpjxFwBGT02RakqYdiYW69j2NicrzpLkAl/Asi4nHy3wEFyhcTqJCRsBU+XtOmQmI/CyEdon7ddyfDgOcLJ0HvixHcOvx00+D3dzsZfVuPK19PjNzpMyLfMlBV0c3kTnTOWtlCaBxw/+GcnA1SGuKCGmICzEDEbC+tsJvWop8Bw1B/sovaaFcbYA9MZsoeNClYvMPjqNRjpLBB7jfioV/IikRqq04Pwv1apV3TmSOEdENPFMk1+tHUvoBBP0BUaJH0PH6W+r/3hihAc7Fqv11Lp4OC/c4XFId5W6J1F2CH3TtGYW7XTt0xeoJj6VpUlbPPvEABBAsDpI3AMni54hz0ODeAzwMu/YOmFUoGA3RYkCSUQjccVVQqMsI8RTZMva5pyEmZ3Qk9DzNhbEHK5CEbiYBcJQu1+S6KnuWZnecFc07l+4T4Qg+hpiPsOOrhdxaflsJJUoPYd1sFkhiU5oRCh68xHQJeT06FOOckc9yxvI8PjJ6SO4LMTPuh1CILO1H0v6Z5+c3sdCdv8PEi5KUzqrC1nZlMvAhAx4LTy6/V/FOzJk6imRWPXp7D+3i86cs6qBEzX5QTOeOqWnExF455LXcGS6+hHocEdEOqZqG9PcBr3w6VYMuIBpbtF5vDRMTJZotOH4Y34DBM+z/5L4bMHZKaVo0elT+UEhLIs58bu+vUi0quEZp5dQL7YHTRbRdjvV7QCqMBMFRZRu9tFhgYSZ24JGOnLTUTqECH13DEi1RvneQL/jrjUFBV5+tW0ovlkAI7fZXfIAHTps48YaJ1Jnu1Fc0WAA4/rPcCyK1eWgcOtdAj/2QKPvy8YL1+zyx+sWvfKgCZrdtEAWRb3mRCdtT3DDGDA24/RFDKaCkeM8h6bGajvFgHxXaNsTEpeit5KjU1mkoeDrI+bPnocSMC6vSlyOGEENVdkc1nm20VWHlnWITAGaf3Wjup04MwDfX6JTpay4S/BmmWCd2c+6bCdk0ZA6yxzDYYBsyf17SyoRrBuL2BWGuyBsVISk9JkNIkVDbSIMCHHpZ+aTGoyId/gxpSuZhdxIONJXFGYimWYzYzAnRnJZU/xgk+QDeWlqCzspSGNyftXG2wEU9xCwoQAl8EBmzfuV5sLb+0kRjvc0c1Y0lW0Ht+HaJotZYPXwVSIQwAoOx8MHiPqsTx5acPUy9bdUtdQv6pltPPXY93DnNIHNvBWivVd9D8EY9HG6do8t5bNwKY+YjUP0n4yEi35fKdVgZq1faQz/cEg7jwJwYC4jsrxhDwZUVIZTVLaI5rOczDD9/FURo7TBX7J/WNAcxM/AGj5hl5caNQ6z9+3idrh8kCb+I1AguZgsWKU9vkgYMkb1r491bSmm8uzWCixgfm7EE0NBEPHShMwvciTiZ1n0ptR/UdVG4ulT0o02qR6h9qWaBCkAIHP90EvzlS5SxOcAzDa/rSKU29zRZYtRCKK6/z4nXbmN0QzBMHwR04E9ZQCmmIhDo90TlJA7xm4IGjYG9NhRkK8WeNWK9Xn0aFf3pkXomH2nJ0VcQ0GCNB4wdYKMy7VSATRvEJw6DjQ5TP9hB6rVnHWcVCPefmktZQ/JOTmMCTc4fTD98mJ2Ghc1mbG1KTWOdJu8bSMmEVScnJGBC7D6o91oO12kSwaeAgR/+c+uICSqW81phPQ5XJIEq/ivpUq/LCEQUBFl2+zhxHDScjgQ6MxLTEtRaD7E5HYdSl2Q/Q4S+qL91Bnr+tzO1x9+eK1EQ+mHRrwahbQ6+ki+10zX5bsGtbf2a+GC1Qb9R5ElRkhB74I5wCeIKm7dvTCCSRcDv4SF/8wR4hl9RsVex0oXMhWhfnPmhHe+8QztHqKxgpcvV0F0bKF8T6TmZ7yW7cWcQE8TakEhz/ZVCsNxx6mWrFasotKIzbYP3BuzW07j2qOrbk8aKhtlTYa1u0q4o9smb2alZLdh5UH8eQk8w+eBvqHofWf+X4aA406Q2q2aYLO8bjhoHhhhCqDtnRcrbeCQnKWCxx6bHkTM5+OGXSd67zHhjIeSI4nteifprTAJfgyAl8wV+N872rLDXonoT9sbOf4wwV6Oo6qsz+fG8SaPByjMO5xOcObOzDOWETA1gAMOOr/duy35z3db7haxjbBGvJVA6uPjDc7xkLBfFzlISU3EBx3HKEfA2NNEBs+rMDHoylKB4KZ2MaibXf+14EKSCXHzFwzSEIEktBRggSG4hu7ZE345/49gapym+bje4DNqDyRGpVvXlBjqowGpeu6jYYvpPwoOQW09RRSsiJk1k8YRbx02HYraOX/lmXmhdwhhalMGeRLXL63nMWVLbg7kiQDOKK4o5EqKdOJmNSVN8QVJQw2/KZIXFJjSiKNWz59BA/owG0Y6z1odBFPEVo8GBUHsSU3rGVaJLtu18Pyom23eMl5ti44Odh1jUXeL5bLclqjPti64IABlIYJpeH3QmmHRdOKlMSdswK4XPpbRY4B2KIAxJOyoKYIYBirS4ZPhfEPc38ECqfERBeYZp03YBsB/bLIalj1mrWcswljn51idlkj0bvmGRpdQJJVXO/HAoP5rHBmEMz9bwZEATqHwUE1a6/HflRslJtgHBAeIqAMHY05g74+3QDsglsQ6Giv1uu8IsXHYtPKDP44HFf28jaWRaMt1ER3L4D2QCdYuUXZXprO1uU3FuTzlfjEkZBmYLyQX7fyZXQ3xJS49ROj8T9FY2w+jTxTWuK5M3oy38ONH3XA1nQVmq6xi38cvfijc8bZZs6aX2CxBxsCPENHO27QikBXiATjtJloDbclWv4ysQqPzPw7bpK6YFstLbCZMSy+L/rpnsvc5TyHCW3DR4I2speI/K3ZXmF2RZW/VN4PXphqw3R2pYj5W9aHg13qBQtUWt/iV44tEgibfiYouEVjT309WOwEro5N2NL1lrMgnE4TnE4fCZz4I8HxuqYBuofT2ORRwIk9/QRGbgwwkXwYjNxN9Id/ZpvVgC5QfpjWLwy4MHWGMHWme0xRMgCiiVoO58te4hZ5WCm6pQO3w3maZwsBzwokRvc7k3s1Df8UbSIjxwMIsK/k+AVpXLuQlCeRz3e3PJdPNjUINCW5qFOYFqgUG1ViLOXI6RUPrLffYaoG+SZ7Uz7D0cR53Gw+nU4ilUcK3guX75DZOh5233covW+7wMkWmb9QD8FrK+O3x/3yAqMmXhRhXqQZRs7zHOp/lgB8tmsA+9eIuyLTpiVzh9UPuwLQNqlYX9bmiZCag2K7TvrhrPHynW17+Lw9K8u5Ddac20NuoHjzWFL8L2MDO8HLDhazHt7WkAqvGu6Gq1nFKqZDHlYEO353LGJnMfOZxJLuHfK4/Rog65z4Dd9HuJQ15u6DxJmDnysDgvyPtTONc6vIlpZ1nqy0t5mkU1R0gCvP3W1SkxRkwBeCaw/m87wcm7oQSyonOxu0Lww53bqzwZKO68QxHqVEGoxgmnkSUCflPG01aQee4UGec6VMfhWeB/dhKaiz+MV8WrH+UnMvggFtNu97MDRIUAt1DTV/I9aCn66NGMLlipLWEDL/zmshHH2D5lj9FTk20N+ebsA7QDvLL9oWXrrQwvfwQbTY1as0on0bpJ2uWbkFIBKkRCZoKEi4CLfBBZtrZ0Fnv/E2B5DCSiOgzOz8wAfbo9Y2m+wRL8rowSyNTqsT0eQPspIcTU4muzdQfbj3IlKvVjmwdBOcfVFzVq/aiVkcnWvsW1+fz51wECWuP0c3aqwtoFDZlZ3l2teHLfLaOetusb432+e49Y79czfWwIO3z7xdTN6gNanxJzQ22/GNfmz++cldf4sJosLtah1eC9kh39ujYNwpiIL1dIUV2hmVCqCRlcjz1+d2kubJcV8n3HqT+CXTDhdbn+jdKtwLYDRIfo/w1QIWreCQC9GQG+raFPsMKHNNh+atgo5FbJxr9FNwZhMQJj2bs2OytNB2887pVCyAGzTACAmcjyqSpIxJkwlVG2H/4UMgCV99wyG2XwAWxtuTUJ4e0ENZ2atfd/lMNqXqx0a+tm0nV7utmaq9kWcsJatO5f3Rzay2ypvHaVQysoTM0E8BuIPva75NGE8IMpfJNx6dFzpoyt6soYp1b5y30njrO0zZjEraWzGw0sWi3gjMPbqnBSOO9gWnhu7/AYEZ+Ef+YwAA"""

MANIFEST_PATH = RUNTIME_ROOT / "kdic_crawl_manifest_42.csv"
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH.write_bytes(
    gzip.decompress(base64.b64decode(EMBEDDED_MANIFEST_GZIP_B64))
)

print("Manifest 생성 완료:", MANIFEST_PATH)
print("크기:", MANIFEST_PATH.stat().st_size, "bytes")


In [ ]:

# 필요하면 사용자 수정본 CSV로 교체할 수 있습니다.
USE_CUSTOM_MANIFEST_UPLOAD = False

if USE_CUSTOM_MANIFEST_UPLOAD:
    from google.colab import files

    uploaded = files.upload()
    csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
    if len(csv_names) != 1:
        raise ValueError("CSV 파일을 정확히 1개 업로드해야 합니다.")

    MANIFEST_PATH = Path("/content") / csv_names[0]
    MANIFEST_PATH.write_bytes(uploaded[csv_names[0]])
    print("사용자 Manifest로 교체:", MANIFEST_PATH)


In [ ]:

REQUIRED_COLUMNS = {
    "url_id",
    "business_function",
    "page_title",
    "source_url",
    "site_name",
    "normalized_decision",
    "decision_reason",
    "page_type",
    "fetch_strategy",
    "parser_profile",
    "auth_required",
    "asset_policy",
    "content_scope",
    "crawl_wave",
}

manifest_df = pd.read_csv(MANIFEST_PATH, encoding="utf-8-sig", dtype=str).fillna("")

missing_columns = sorted(REQUIRED_COLUMNS - set(manifest_df.columns))
if missing_columns:
    raise ValueError(f"Manifest 필수 열이 없습니다: {missing_columns}")

if manifest_df["url_id"].duplicated().any():
    duplicates = manifest_df.loc[
        manifest_df["url_id"].duplicated(keep=False), ["url_id", "source_url"]
    ]
    raise ValueError(f"url_id 중복:\n{duplicates}")

if manifest_df["source_url"].duplicated().any():
    duplicates = manifest_df.loc[
        manifest_df["source_url"].duplicated(keep=False), ["url_id", "source_url"]
    ]
    raise ValueError(f"source_url 중복:\n{duplicates}")

invalid_urls = manifest_df[
    ~manifest_df["source_url"].str.match(r"^https?://", na=False)
]
if not invalid_urls.empty:
    raise ValueError(
        "HTTP/HTTPS가 아닌 URL이 있습니다:\n"
        + invalid_urls[["url_id", "source_url"]].to_string(index=False)
    )

# 실행 대상 필터
target_df = manifest_df.copy()

if SELECT_BUSINESS_FUNCTIONS:
    target_df = target_df[
        target_df["business_function"].isin(SELECT_BUSINESS_FUNCTIONS)
    ]

if RUN_ONLY_URL_IDS:
    known_url_ids = set(manifest_df["url_id"])
    unknown_url_ids = sorted(set(RUN_ONLY_URL_IDS) - known_url_ids)

    if unknown_url_ids:
        raise ValueError(
            f"Manifest에 존재하지 않는 url_id입니다: {unknown_url_ids}"
        )

    target_df = target_df[
        target_df["url_id"].isin(RUN_ONLY_URL_IDS)
    ]

target_df = target_df[
    target_df["normalized_decision"].isin(RUN_DECISIONS)
    & target_df["crawl_wave"].isin(RUN_WAVES)
].copy()

if MAX_URLS is not None:
    target_df = target_df.head(MAX_URLS).copy()

print("전체 Manifest:", len(manifest_df))
print("이번 실행 대상:", len(target_df))

display(
    manifest_df.groupby(
        ["business_function", "normalized_decision"],
        dropna=False,
    ).size().rename("count").reset_index()
)

display(
    target_df[
        [
            "url_id",
            "business_function",
            "page_title",
            "normalized_decision",
            "crawl_wave",
            "fetch_strategy",
        ]
    ].head(20)
)


## 3. 공통 유틸리티

In [ ]:

ATTACHMENT_EXTENSIONS = {
    ".pdf", ".hwp", ".hwpx", ".doc", ".docx",
    ".xls", ".xlsx", ".ppt", ".pptx", ".zip",
    ".csv", ".txt",
}

IMAGE_EXTENSIONS = {
    ".png", ".jpg", ".jpeg", ".gif", ".webp", ".svg", ".bmp",
}

VIDEO_EXTENSIONS = {
    ".mp4", ".webm", ".mov", ".avi", ".mkv",
}

NOISE_SELECTORS = [
    "script", "style", "noscript", "template",
    "header", "footer", "nav", "aside",
    ".skip", ".skip-navigation", ".skipnav",
    ".gnb", ".lnb", ".snb", ".breadcrumb-wrap",
    ".footer", ".header", ".quick-menu", ".quick",
    ".util", ".utility", ".pagination",
    "[aria-hidden='true']",
]

MAIN_SELECTORS_BY_SITE = {
    "예금보험공사": [
        "#contents", "#content", ".contents", ".content",
        ".sub-content", ".sub_contents", "main", "article",
    ],
    "금융안심포털": [
        "#contents", "#content", ".contents", ".content",
        ".sub-content", ".sub_contents", "main", "article",
    ],
}

BREADCRUMB_SELECTORS = [
    ".breadcrumb", ".breadcrumbs", ".location", ".location-wrap",
    ".path", ".sub-location", "nav[aria-label*='breadcrumb' i]",
]


def now_kst_iso() -> str:
    return datetime.now(KST).isoformat(timespec="seconds")


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def sha256_text(text: str) -> str:
    return sha256_bytes(text.encode("utf-8"))


def normalize_space(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


def normalize_multiline(text: str) -> str:
    lines = [normalize_space(line) for line in (text or "").splitlines()]
    cleaned: list[str] = []

    for line in lines:
        if not line:
            if cleaned and cleaned[-1] != "":
                cleaned.append("")
            continue
        cleaned.append(line)

    while cleaned and cleaned[-1] == "":
        cleaned.pop()

    return "\n".join(cleaned)


def safe_name(value: str, max_length: int = 90) -> str:
    value = normalize_space(value)
    value = re.sub(r'[\\/:*?"<>|]+', "_", value)
    value = re.sub(r"_+", "_", value).strip("._ ")
    return (value or "untitled")[:max_length]


def ensure_parent(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)


def write_json(path: Path, data: Any) -> None:
    ensure_parent(path)
    path.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def append_jsonl(path: Path, data: dict[str, Any]) -> None:
    ensure_parent(path)
    with path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(data, ensure_ascii=False) + "\n")


def absolute_url(base_url: str, candidate: str | None) -> str | None:
    if not candidate:
        return None

    candidate = candidate.strip()
    if not candidate or candidate.startswith(("#", "javascript:", "mailto:", "tel:")):
        return None

    return urljoin(base_url, candidate)


def extension_from_url(url: str) -> str:
    path = urlparse(url).path
    return Path(path).suffix.lower()


def classify_asset_url(url: str) -> str:
    extension = extension_from_url(url)

    if extension in ATTACHMENT_EXTENSIONS:
        return "attachment"
    if extension in IMAGE_EXTENSIONS:
        return "image"
    if extension in VIDEO_EXTENSIONS:
        return "video"
    return "link"


def row_to_clean_dict(row: pd.Series) -> dict[str, str]:
    return {
        str(key): "" if pd.isna(value) else str(value)
        for key, value in row.to_dict().items()
    }


## 4. HTTP 세션·재시도·robots.txt 검사

In [ ]:

def create_session() -> requests.Session:
    retry_policy = Retry(
        total=MAX_RETRIES,
        connect=MAX_RETRIES,
        read=MAX_RETRIES,
        status=MAX_RETRIES,
        backoff_factor=0.8,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset({"GET", "HEAD"}),
        respect_retry_after_header=True,
        raise_on_status=False,
    )

    adapter = HTTPAdapter(
        max_retries=retry_policy,
        pool_connections=2,
        pool_maxsize=2,
    )

    session = requests.Session()
    session.headers.update(
        {
            "User-Agent": USER_AGENT,
            "Accept": (
                "text/html,application/xhtml+xml,application/xml;q=0.9,"
                "image/avif,image/webp,*/*;q=0.8"
            ),
            "Accept-Language": "ko-KR,ko;q=0.9,en;q=0.5",
            "Connection": "keep-alive",
        }
    )
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


SESSION = create_session()
ROBOTS_CACHE: dict[str, RobotFileParser | None] = {}


def get_robots_parser(url: str) -> RobotFileParser | None:
    parsed = urlparse(url)
    base = f"{parsed.scheme}://{parsed.netloc}"

    if base in ROBOTS_CACHE:
        return ROBOTS_CACHE[base]

    robots_url = urljoin(base, "/robots.txt")

    try:
        response = SESSION.get(
            robots_url,
            timeout=REQUEST_TIMEOUT_SECONDS,
            allow_redirects=True,
        )

        if response.status_code >= 400:
            ROBOTS_CACHE[base] = None
            return None

        parser = RobotFileParser()
        parser.set_url(robots_url)
        parser.parse(response.text.splitlines())
        ROBOTS_CACHE[base] = parser
        return parser

    except requests.RequestException:
        ROBOTS_CACHE[base] = None
        return None


def robots_allows(url: str) -> tuple[bool, str]:
    if not RESPECT_ROBOTS_TXT:
        return True, "robots_check_disabled"

    parser = get_robots_parser(url)
    if parser is None:
        return True, "robots_unavailable_assumed_allowed"

    allowed = parser.can_fetch(USER_AGENT, url)
    return allowed, "allowed" if allowed else "disallowed"


def choose_response_encoding(response: requests.Response) -> str:
    content_type = response.headers.get("Content-Type", "")
    declared = response.encoding

    # requests의 기본 ISO-8859-1 추정은 한국어 HTML에서 잘못될 수 있습니다.
    if not declared or declared.lower() == "iso-8859-1":
        apparent = response.apparent_encoding
        if apparent:
            return apparent

    return declared or "utf-8"


## 5. HTML 수집과 응답 메타데이터 저장

In [ ]:

@dataclass
class FetchResult:
    requested_url: str
    final_url: str
    status_code: int
    ok: bool
    content_type: str
    encoding: str
    elapsed_seconds: float
    collected_at: str
    content_length: int
    raw_sha256: str
    redirect_history: list[dict[str, Any]]
    selected_headers: dict[str, str]
    body: bytes


def fetch_url(url: str) -> FetchResult:
    allowed, robots_status = robots_allows(url)
    if not allowed:
        raise PermissionError(f"robots.txt에서 수집을 허용하지 않습니다: {url}")

    started = time.perf_counter()

    response = SESSION.get(
        url,
        timeout=REQUEST_TIMEOUT_SECONDS,
        allow_redirects=True,
    )

    elapsed = time.perf_counter() - started
    encoding = choose_response_encoding(response)
    response.encoding = encoding

    selected_header_names = {
        "Content-Type",
        "Content-Length",
        "Last-Modified",
        "ETag",
        "Cache-Control",
        "Content-Disposition",
    }

    selected_headers = {
        key: value
        for key, value in response.headers.items()
        if key in selected_header_names
    }
    selected_headers["Robots-Check"] = robots_status

    return FetchResult(
        requested_url=url,
        final_url=response.url,
        status_code=response.status_code,
        ok=response.ok,
        content_type=response.headers.get("Content-Type", ""),
        encoding=encoding,
        elapsed_seconds=round(elapsed, 3),
        collected_at=now_kst_iso(),
        content_length=len(response.content),
        raw_sha256=sha256_bytes(response.content),
        redirect_history=[
            {
                "status_code": item.status_code,
                "url": item.url,
                "location": item.headers.get("Location"),
            }
            for item in response.history
        ],
        selected_headers=selected_headers,
        body=response.content,
    )


def save_fetch_result(
    manifest_row: dict[str, str],
    result: FetchResult,
) -> tuple[Path, Path]:
    business_dir = safe_name(manifest_row["business_function"])
    url_id = manifest_row["url_id"]

    html_path = PATHS["raw_html"] / business_dir / f"{url_id}.html"
    meta_path = PATHS["response_meta"] / business_dir / f"{url_id}.json"

    ensure_parent(html_path)
    html_path.write_bytes(result.body)

    metadata = asdict(result)
    metadata.pop("body", None)
    metadata["url_id"] = url_id
    metadata["business_function"] = manifest_row["business_function"]
    metadata["page_title_manifest"] = manifest_row["page_title"]
    metadata["fetch_strategy"] = manifest_row["fetch_strategy"]
    metadata["parser_profile"] = manifest_row["parser_profile"]
    metadata["raw_html_path"] = str(html_path.relative_to(OUTPUT_ROOT))

    write_json(meta_path, metadata)
    return html_path, meta_path


## 6. 결정론적 본문·표·링크·이미지 추출

In [ ]:
from bs4 import Comment, NavigableString

# 실제 본문 밖에서 반복되는 UI 요소
FIXED_NOISE_SELECTORS = [
    "script", "style", "noscript", "template",
    ".floatTop", ".floatBottom", ".paging", ".pagination",
    ".sr-only", ".skip", ".skipnav", ".skip-navigation",
    ".loading", ".waitPage", "#waitPage",
]

FIXED_NOISE_TEXTS = {
    "퀵메뉴", "글자 크기", "글자확대", "글자축소",
    "KOR", "ENG", "상단으로 이동", "검색 초기화",
    "좌우로 움직여보세요", "표 더보기",
}

FAQ_PREFIX_RE = re.compile(r"^\s*질문\s*")
ANSWER_PREFIX_RE = re.compile(r"^\s*답변\s*")
OPEN_SUFFIX_RE = re.compile(r"\s*열기\s*$")
DOWNLOAD_CALL_RE = re.compile(
    r"gfn_downloadFile\(\s*'([^']+)'\s*,\s*'([^']+)'\s*\)"
)
MOVE_URL_RE = re.compile(r"gfn_moveUrl\(\s*'([^']+)'\s*\)")

BLOCK_TAGS = {
    "div", "section", "article", "aside", "header", "footer", "main",
    "p", "ul", "ol", "dl", "table", "figure",
    "h1", "h2", "h3", "h4", "h5", "h6",
}


def fixed_norm(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


def fixed_safe_text(node: Tag) -> str:
    """원본 DOM을 훼손하지 않고 노이즈를 제거한 텍스트를 반환합니다."""
    fragment = BeautifulSoup(str(node), "lxml")
    root = fragment.body.find() if fragment.body else fragment.find()
    if not root:
        return ""

    for child in root.select(
        ".sr-only,.hide,script,style,noscript,template,.floatTop,.floatBottom"
    ):
        child.decompose()

    for image in root.find_all("img"):
        image.replace_with(fixed_norm(image.get("alt", "")))

    for br in root.find_all("br"):
        br.replace_with(" ")

    return fixed_norm(root.get_text(" ", strip=True))


def fixed_clean_question(text: str) -> str:
    text = FAQ_PREFIX_RE.sub("", fixed_norm(text))
    text = OPEN_SUFFIX_RE.sub("", text)
    return fixed_norm(text)


def fixed_clean_answer(text: str) -> str:
    return fixed_norm(ANSWER_PREFIX_RE.sub("", fixed_norm(text)))


def fixed_clean_term(text: str) -> str:
    text = OPEN_SUFFIX_RE.sub("", fixed_norm(text))
    text = re.sub(r"^\s*\d{1,2}\s+(?=\D)", "", text)
    return fixed_norm(text)


def fixed_infer_file_format(button: Tag) -> str:
    search_text = (
        " ".join(button.get("class", [])).lower()
        + " "
        + fixed_norm(button.get_text(" ", strip=True)).lower()
    )
    for keyword, file_format in [
        ("icohwp", "HWP"), ("hwp", "HWP"),
        ("icopdf", "PDF"), ("pdf", "PDF"),
        ("xlsx", "XLSX"), ("excel", "XLSX"),
        ("docx", "DOCX"), ("doc", "DOC"),
    ]:
        if keyword in search_text:
            return file_format
    return "FILE"


def fixed_cell_text(cell: Tag) -> str:
    fragment = BeautifulSoup(str(cell), "lxml")
    root = fragment.body.find() if fragment.body else fragment.find()
    if not root:
        return ""

    for child in root.select("script,style,noscript,template,.sr-only"):
        child.decompose()

    for button in root.find_all("button"):
        file_format = fixed_infer_file_format(button)
        label = fixed_norm(button.get_text(" ", strip=True))
        label = re.sub(r"다운로드$", "", label).strip()
        replacement = (
            f"{label} {file_format} 다운로드".strip()
            if label
            else f"{file_format} 다운로드"
        )
        button.replace_with(replacement)

    for image in root.find_all("img"):
        image.replace_with(fixed_norm(image.get("alt", "")))

    for br in root.find_all("br"):
        br.replace_with(" ")

    return fixed_norm(root.get_text(" ", strip=True))


def fixed_expand_table(table: Tag) -> tuple[list[str], list[list[str]]]:
    """rowspan/colspan을 펼쳐 각 행이 같은 열 수를 갖도록 만듭니다."""
    grid: list[list[str]] = []
    active_rowspans: dict[int, list[Any]] = {}
    header_flags: list[bool] = []
    max_columns = 0

    for tr in table.find_all("tr"):
        row: list[str] = []
        column = 0
        cells = tr.find_all(["th", "td"], recursive=False)

        def fill_active() -> None:
            nonlocal column
            while column in active_rowspans:
                remaining, value = active_rowspans[column]
                while len(row) <= column:
                    row.append("")
                row[column] = value
                remaining -= 1
                if remaining <= 0:
                    del active_rowspans[column]
                else:
                    active_rowspans[column] = [remaining, value]
                column += 1

        fill_active()
        header_flags.append(any(cell.name == "th" for cell in cells))

        for cell in cells:
            fill_active()
            text = fixed_cell_text(cell)
            rowspan = max(1, int(cell.get("rowspan", 1) or 1))
            colspan = max(1, int(cell.get("colspan", 1) or 1))

            for offset in range(colspan):
                target_column = column + offset
                while len(row) <= target_column:
                    row.append("")
                row[target_column] = text
                if rowspan > 1:
                    active_rowspans[target_column] = [rowspan - 1, text]

            column += colspan

        if active_rowspans:
            final_active_column = max(active_rowspans)
            while column <= final_active_column:
                if column in active_rowspans:
                    fill_active()
                else:
                    while len(row) <= column:
                        row.append("")
                    column += 1

        max_columns = max(max_columns, len(row))
        if row and any(row):
            grid.append(row)

    if not grid:
        return [], []

    for row in grid:
        row.extend([""] * (max_columns - len(row)))

    if table.thead:
        header_count = len(table.thead.find_all("tr", recursive=False))
    else:
        header_count = 1 if header_flags and header_flags[0] else 0

    if header_count == 0:
        header_count = 1

    header_rows = grid[:header_count]
    rows = grid[header_count:]
    headers: list[str] = []

    for column in range(max_columns):
        values: list[str] = []
        for header_row in header_rows:
            value = fixed_norm(header_row[column])
            if value and value not in values:
                values.append(value)
        headers.append(" / ".join(values) if values else f"열{column + 1}")

    # 중복 헤더 이름을 고유하게 만듭니다.
    counts: dict[str, int] = {}
    unique_headers: list[str] = []
    for raw_header in headers:
        header = raw_header or "열"
        counts[header] = counts.get(header, 0) + 1
        unique_headers.append(
            header if counts[header] == 1 else f"{header} {counts[header]}"
        )

    return unique_headers, rows


def fixed_extract_attachments(root: Tag, base_url: str) -> list[dict[str, Any]]:
    attachments: list[dict[str, Any]] = []
    seen: set[Any] = set()

    # KDIC 사이트의 JavaScript 다운로드 버튼
    for button in root.find_all("button", onclick=True):
        match = DOWNLOAD_CALL_RE.search(button.get("onclick", ""))
        if not match:
            continue

        key = (match.group(1), match.group(2))
        if key in seen:
            continue
        seen.add(key)

        file_format = fixed_infer_file_format(button)
        label = fixed_norm(button.get_text(" ", strip=True))
        label = re.sub(r"다운로드$", "", label).strip()

        row_context = ""
        parent_row = button.find_parent("tr")
        if parent_row:
            values: list[str] = []
            for cell in parent_row.find_all(["th", "td"], recursive=False):
                if button in cell.descendants:
                    continue
                value = fixed_cell_text(cell)
                if value and "다운로드" not in value and value not in values:
                    values.append(value)
            row_context = " | ".join(values[:4])

        display_name = label or row_context or f"{file_format} 첨부파일"
        if file_format not in display_name.upper():
            display_name = f"{display_name} ({file_format})"

        attachment_id = sha256_text(
            f"{match.group(1)}|{match.group(2)}"
        )[:16]

        attachments.append(
            {
                "attachment_id": attachment_id,
                "display_name": display_name,
                "file_format": file_format,
                "download_method": "gfn_downloadFile",
                "token1": match.group(1),
                "token2": match.group(2),
                "row_context": row_context,
                "button_text": fixed_norm(button.get_text(" ", strip=True)),
                "source_url": base_url,
                "download_status": "metadata_only",
            }
        )

    # href에 직접 연결된 첨부파일
    attachment_extensions = {
        ".pdf", ".hwp", ".hwpx", ".doc", ".docx",
        ".xls", ".xlsx", ".ppt", ".pptx", ".zip", ".csv",
    }

    for anchor in root.find_all("a", href=True):
        href = anchor.get("href", "")
        url = absolute_url(base_url, href)
        if not url:
            continue

        extension = Path(urlparse(url).path).suffix.lower()
        if extension not in attachment_extensions:
            continue

        key = ("href", url)
        if key in seen:
            continue
        seen.add(key)

        attachments.append(
            {
                "attachment_id": sha256_text(url)[:16],
                "display_name": (
                    fixed_norm(anchor.get_text(" ", strip=True))
                    or Path(urlparse(url).path).name
                ),
                "file_format": extension.lstrip(".").upper(),
                "download_method": "href",
                "url": url,
                "source_url": base_url,
                "download_status": "metadata_only",
            }
        )

    return attachments


def fixed_extract_links(root: Tag, base_url: str) -> list[dict[str, Any]]:
    links: list[dict[str, Any]] = []
    seen: set[Any] = set()

    for anchor in root.find_all("a", href=True):
        href = anchor.get("href", "").strip()
        if not href or href.startswith(("#", "javascript:", "mailto:", "tel:")):
            continue

        url = urljoin(base_url, href)
        text = fixed_norm(anchor.get_text(" ", strip=True))
        if not text or text in FIXED_NOISE_TEXTS:
            continue

        key = (url, text)
        if key in seen:
            continue
        seen.add(key)
        links.append({"url": url, "anchor_text": text})

    for button in root.find_all("button", onclick=True):
        match = MOVE_URL_RE.search(button.get("onclick", ""))
        if not match:
            continue
        url = urljoin(base_url, match.group(1))
        text = fixed_norm(button.get_text(" ", strip=True))
        key = (url, text)
        if key not in seen:
            seen.add(key)
            links.append(
                {"url": url, "anchor_text": text, "link_type": "button"}
            )

    return links


def fixed_extract_images(root: Tag, base_url: str) -> list[dict[str, Any]]:
    images: list[dict[str, Any]] = []
    seen: set[str] = set()

    for image in root.find_all("img"):
        source = (
            image.get("src")
            or image.get("data-src")
            or image.get("data-original")
        )
        url = absolute_url(base_url, source)
        if not url or url in seen:
            continue
        seen.add(url)

        alt = fixed_norm(image.get("alt", ""))
        filename = Path(urlparse(url).path).name
        lowered = filename.lower()
        image_role = "decorative"

        if "webtoon" in lowered:
            image_role = "supplementary_visual"
        elif any(keyword in lowered for keyword in ["proc", "process", "step", "flow"]):
            image_role = "process_diagram"
        elif alt:
            image_role = "content_image"

        images.append(
            {
                "url": url,
                "alt": alt,
                "filename": filename,
                "image_role": image_role,
            }
        )

    return images


class FixedStructureParser:
    def __init__(self, page_type: str = "static_page"):
        self.blocks: list[dict[str, Any]] = []
        self.page_type = page_type

    def add(self, block: dict[str, Any] | None) -> None:
        if not block:
            return

        if (
            self.blocks
            and block.get("type") in {"paragraph", "heading"}
            and self.blocks[-1].get("type") == block.get("type")
            and self.blocks[-1].get("text") == block.get("text")
        ):
            return

        self.blocks.append(block)

    def parse(self, root: Tag) -> list[dict[str, Any]]:
        faq_dls = root.select(
            ".accoWrap .accodian > dl, .accodian.con.ico > dl"
        )

        if self.page_type == "faq" or faq_dls:
            faq_nodes = {id(node) for node in faq_dls}

            for child in root.find_all(recursive=False):
                if not isinstance(child, Tag):
                    self.process_node(child)
                    continue

                child_dls = child.select("dl")
                if any(id(dl) in faq_nodes for dl in child_dls):
                    continue
                self.process_node(child)

            for dl in faq_dls:
                self.parse_faq(dl)
        else:
            for child in root.find_all(recursive=False):
                self.process_node(child)

        return self.blocks

    def parse_faq(self, dl: Tag) -> None:
        dt = dl.find("dt", recursive=False)
        dd = dl.find("dd", recursive=False)
        if not dt or not dd:
            return

        question = fixed_clean_question(fixed_safe_text(dt))
        answer_parser = FixedStructureParser("static_page")
        answer_parser.process_node(dd)

        if not answer_parser.blocks:
            answer_text = fixed_clean_answer(fixed_safe_text(dd))
            answer_parser.add({"type": "paragraph", "text": answer_text})
        else:
            for block in answer_parser.blocks:
                if "text" in block:
                    block["text"] = fixed_clean_answer(block["text"])
                    break

        answer_text = fixed_blocks_text(answer_parser.blocks)
        self.add(
            {
                "type": "faq",
                "question": question,
                "answer_blocks": answer_parser.blocks,
                "answer_text": answer_text,
            }
        )

    def parse_definition_list(self, dl: Tag) -> None:
        children = [
            child
            for child in dl.find_all(recursive=False)
            if isinstance(child, Tag)
        ]
        index = 0

        while index < len(children):
            if children[index].name != "dt":
                self.process_node(children[index])
                index += 1
                continue

            dt = children[index]
            dd = (
                children[index + 1]
                if index + 1 < len(children)
                and children[index + 1].name == "dd"
                else None
            )
            term = fixed_clean_term(fixed_safe_text(dt))

            if dd:
                definition_parser = FixedStructureParser("static_page")
                definition_parser.process_node(dd)
                if not definition_parser.blocks:
                    definition_parser.add(
                        {"type": "paragraph", "text": fixed_safe_text(dd)}
                    )

                self.add(
                    {
                        "type": "definition",
                        "term": term,
                        "definition_blocks": definition_parser.blocks,
                        "definition_text": fixed_blocks_text(
                            definition_parser.blocks
                        ),
                    }
                )
                index += 2
            else:
                self.add({"type": "heading", "level": 3, "text": term})
                index += 1

    def process_node(self, node: Any) -> None:
        if isinstance(node, Comment):
            return

        if isinstance(node, NavigableString):
            text = fixed_norm(str(node))
            if text and text not in FIXED_NOISE_TEXTS:
                self.add({"type": "paragraph", "text": text})
            return

        if not isinstance(node, Tag):
            return

        name = node.name.lower()
        classes = set(node.get("class", []))

        if (
            name in {"script", "style", "noscript", "template"}
            or classes & {
                "floatTop", "floatBottom", "paging", "pagination", "sr-only"
            }
        ):
            return

        if name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
            text = fixed_safe_text(node)
            if text and text not in FIXED_NOISE_TEXTS:
                self.add(
                    {"type": "heading", "level": int(name[1]), "text": text}
                )
            return

        if name == "p":
            text = fixed_safe_text(node)
            if text and text not in FIXED_NOISE_TEXTS:
                self.add({"type": "paragraph", "text": text})
            return

        if name == "table":
            headers, rows = fixed_expand_table(node)
            if headers or rows:
                self.add(
                    {
                        "type": "table",
                        "headers": headers,
                        "rows": rows,
                        "row_count": len(rows),
                    }
                )
            return

        if name in {"ul", "ol"}:
            items: list[str] = []
            for li in node.find_all("li", recursive=False):
                fragment = BeautifulSoup(str(li), "lxml")
                copied_li = fragment.body.find("li") if fragment.body else None
                if not copied_li:
                    continue
                for nested in copied_li.find_all(["ul", "ol"]):
                    nested.decompose()
                text = fixed_safe_text(copied_li)
                if text:
                    items.append(text)

            if items:
                self.add(
                    {"type": "list", "ordered": name == "ol", "items": items}
                )

            for li in node.find_all("li", recursive=False):
                for nested in li.find_all(["ul", "ol"], recursive=False):
                    self.process_node(nested)
            return

        if name == "dl":
            self.parse_definition_list(node)
            return

        if name == "figure":
            caption = node.find("figcaption")
            if caption:
                text = fixed_safe_text(caption)
                if text:
                    self.add({"type": "paragraph", "text": text})
            return

        # div, span 등이 가진 직접 텍스트와 하위 블록의 순서를 보존합니다.
        inline_parts: list[str] = []

        def flush_inline() -> None:
            nonlocal inline_parts
            text = fixed_norm(" ".join(inline_parts))
            inline_parts = []
            if text and text not in FIXED_NOISE_TEXTS:
                self.add({"type": "paragraph", "text": text})

        for child in node.children:
            if isinstance(child, Comment):
                continue
            if isinstance(child, NavigableString):
                text = fixed_norm(str(child))
                if text:
                    inline_parts.append(text)
            elif isinstance(child, Tag):
                child_name = child.name.lower()
                if child_name == "br":
                    inline_parts.append(" ")
                elif child_name in BLOCK_TAGS:
                    flush_inline()
                    self.process_node(child)
                else:
                    text = fixed_safe_text(child)
                    if text:
                        inline_parts.append(text)

        flush_inline()


def fixed_render_blocks(blocks: list[dict[str, Any]]) -> str:
    lines: list[str] = []

    for block in blocks:
        block_type = block.get("type")

        if block_type == "heading":
            level = min(max(int(block.get("level", 2)), 2), 6)
            lines.append(f"{'#' * level} {block.get('text', '')}")
        elif block_type == "paragraph":
            lines.append(block.get("text", ""))
        elif block_type == "list":
            for index, item in enumerate(block.get("items", []), start=1):
                prefix = f"{index}. " if block.get("ordered") else "- "
                lines.append(prefix + item)
        elif block_type == "definition":
            lines.append("### " + block.get("term", ""))
            lines.append(fixed_render_blocks(block.get("definition_blocks", [])))
        elif block_type == "faq":
            lines.append("### Q. " + block.get("question", ""))
            lines.append(fixed_render_blocks(block.get("answer_blocks", [])))
        elif block_type == "table":
            headers = [
                fixed_norm(header).replace("|", "\\|")
                for header in block.get("headers", [])
            ]
            if headers:
                lines.append("| " + " | ".join(headers) + " |")
                lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
                for row in block.get("rows", []):
                    normalized_row = [
                        fixed_norm(value).replace("|", "\\|")
                        for value in row
                    ]
                    normalized_row.extend(
                        [""] * (len(headers) - len(normalized_row))
                    )
                    lines.append(
                        "| " + " | ".join(normalized_row[: len(headers)]) + " |"
                    )

        lines.append("")

    return re.sub(r"\n{3,}", "\n\n", "\n".join(lines)).strip()


def fixed_blocks_text(blocks: list[dict[str, Any]]) -> str:
    values: list[str] = []

    for block in blocks:
        block_type = block.get("type")
        if block_type in {"heading", "paragraph"}:
            values.append(block.get("text", ""))
        elif block_type == "list":
            values.extend(block.get("items", []))
        elif block_type == "definition":
            values.extend(
                [block.get("term", ""), block.get("definition_text", "")]
            )
        elif block_type == "faq":
            values.extend(
                [block.get("question", ""), block.get("answer_text", "")]
            )
        elif block_type == "table":
            values.extend(block.get("headers", []))
            for row in block.get("rows", []):
                values.extend(row)

    return fixed_norm(" ".join(values))


def fixed_manifest_title(manifest_title: str, business_function: str) -> str:
    # Manifest에서 사람이 확인한 전체 경로명을 사용합니다.
    # HTML 내부의 "퀵메뉴" 같은 잘못된 제목을 선택하지 않습니다.
    return fixed_norm(manifest_title) or business_function


def fixed_manifest_breadcrumb(
    manifest_title: str,
    business_function: str,
) -> list[str]:
    parts = [
        fixed_norm(part)
        for part in re.split(r"\s*>\s*", manifest_title or "")
        if fixed_norm(part)
    ]
    result: list[str] = []
    for value in [business_function] + parts:
        if value and value not in result:
            result.append(value)
    return result


def parse_html_document(
    html_bytes: bytes,
    final_url: str,
    manifest_row: dict[str, str],
    encoding: str,
) -> dict[str, Any]:
    html_text = html_bytes.decode(encoding, errors="replace")
    soup = BeautifulSoup(html_text, "lxml")

    # 실제 KDIC 본문 컨테이너를 우선합니다.
    root = (
        soup.select_one(".contents")
        or soup.select_one("#contents")
        or soup.select_one("#content")
        or soup.select_one("main")
        or soup.body
    )
    if not root:
        raise ValueError("본문 루트를 찾지 못했습니다.")

    # 버튼은 노이즈 제거 전에 추출해야 합니다.
    attachments = fixed_extract_attachments(root, final_url)
    images = fixed_extract_images(root, final_url)

    for selector in FIXED_NOISE_SELECTORS:
        for node in root.select(selector):
            node.decompose()

    links = fixed_extract_links(root, final_url)
    source_text = fixed_norm(root.get_text(" ", strip=True))

    structure_parser = FixedStructureParser(
        manifest_row.get("page_type", "static_page")
    )
    blocks = structure_parser.parse(root)
    markdown = fixed_render_blocks(blocks)
    content_text = fixed_blocks_text(blocks)

    # UI 문구 마지막 안전 제거
    for phrase in FIXED_NOISE_TEXTS:
        markdown = re.sub(
            rf"(?m)^\s*{re.escape(phrase)}\s*$",
            "",
            markdown,
        )
    markdown = re.sub(r"\n{3,}", "\n\n", markdown).strip()

    content_text = fixed_norm(
        re.sub(
            "|".join(re.escape(phrase) for phrase in FIXED_NOISE_TEXTS),
            " ",
            content_text,
        )
    )

    faq_count = sum(1 for block in blocks if block.get("type") == "faq")
    table_count = sum(1 for block in blocks if block.get("type") == "table")
    noise_hits = [
        phrase for phrase in FIXED_NOISE_TEXTS if phrase in markdown
    ]
    retention_ratio = round(
        len(content_text) / max(1, len(source_text)),
        3,
    )

    quality_reasons: list[str] = []
    if len(content_text) < 80:
        quality_reasons.append("본문이 80자 미만")
    if retention_ratio < 0.70:
        quality_reasons.append("본문 보존율 70% 미만")
    if noise_hits:
        quality_reasons.append("공통 UI 문구 잔존")
    if manifest_row.get("page_type") == "faq" and faq_count == 0:
        quality_reasons.append("FAQ 질문-답변 추출 실패")

    if any(
        reason in {"공통 UI 문구 잔존", "FAQ 질문-답변 추출 실패"}
        for reason in quality_reasons
    ):
        quality_status = "fail"
    elif quality_reasons:
        quality_status = "review"
    else:
        quality_status = "pass"

    parsed_hash = sha256_text(markdown)

    document = {
        "doc_id": manifest_row["url_id"],
        "parent_doc_id": manifest_row["url_id"],
        "record_type": "page",
        "title": fixed_manifest_title(
            manifest_row["page_title"],
            manifest_row["business_function"],
        ),
        "manifest_title": manifest_row["page_title"],
        "html_title": (
            fixed_norm(soup.title.get_text(" ", strip=True))
            if soup.title
            else ""
        ),
        "source_url": manifest_row["source_url"],
        "final_url": final_url,
        "site_name": manifest_row["site_name"],
        "business_function": manifest_row["business_function"],
        "target_business_function": manifest_row.get(
            "target_business_function",
            manifest_row["business_function"],
        ),
        "page_type": manifest_row["page_type"],
        "parser_profile": manifest_row["parser_profile"],
        "breadcrumb": fixed_manifest_breadcrumb(
            manifest_row["page_title"],
            manifest_row["business_function"],
        ),
        "content_markdown": markdown,
        "content_text": content_text,
        "parsed_content_sha256": parsed_hash,
        "content_hash": parsed_hash,
        "blocks": blocks,
        "links": links,
        "attachments": attachments,
        "images": images,
        "videos": [],
        "quality": {
            "status": quality_status,
            "reasons": quality_reasons,
            "source_text_chars": len(source_text),
            "parsed_text_chars": len(content_text),
            "retention_ratio": retention_ratio,
            "faq_count": faq_count,
            "table_count": table_count,
            "attachment_button_count": len(attachments),
            "noise_hits": noise_hits,
        },
        "parsed_at": now_kst_iso(),
    }

    if manifest_row["url_id"] == "BI-004":
        document["dynamic_collection"] = {
            "collection_scope": "initial_public_page_only",
            "is_complete": False,
            "current_page_count": 1,
        }

    return document

## 7. 첨부파일·이미지 다운로드

In [ ]:
CONTENT_TYPE_EXTENSIONS = {
    "application/pdf": ".pdf",
    "application/zip": ".zip",
    "application/vnd.openxmlformats-officedocument.wordprocessingml.document": ".docx",
    "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet": ".xlsx",
    "application/vnd.openxmlformats-officedocument.presentationml.presentation": ".pptx",
    "image/jpeg": ".jpg",
    "image/png": ".png",
    "image/gif": ".gif",
    "image/webp": ".webp",
    "image/svg+xml": ".svg",
}


def filename_from_response(
    url: str,
    response: requests.Response,
    fallback_stem: str,
) -> str:
    disposition = response.headers.get("Content-Disposition", "")
    filename_star = re.search(
        r"filename\*=UTF-8''([^;]+)",
        disposition,
        flags=re.IGNORECASE,
    )
    filename_plain = re.search(
        r'filename="?([^";]+)"?',
        disposition,
        flags=re.IGNORECASE,
    )

    if filename_star:
        from urllib.parse import unquote
        name = unquote(filename_star.group(1))
    elif filename_plain:
        name = filename_plain.group(1)
    else:
        name = Path(urlparse(url).path).name

    name = safe_name(name, max_length=120)
    extension = Path(name).suffix.lower()

    if not extension:
        content_type = response.headers.get("Content-Type", "").split(";")[0].strip()
        extension = CONTENT_TYPE_EXTENSIONS.get(content_type)
        if not extension:
            extension = mimetypes.guess_extension(content_type) or ""
        name = f"{safe_name(fallback_stem)}{extension}"

    return name


def download_binary_asset(
    url: str,
    output_dir: Path,
    fallback_stem: str,
) -> dict[str, Any]:
    output_dir.mkdir(parents=True, exist_ok=True)

    with SESSION.get(
        url,
        timeout=REQUEST_TIMEOUT_SECONDS,
        allow_redirects=True,
        stream=True,
    ) as response:
        response.raise_for_status()

        content_length = response.headers.get("Content-Length")
        if content_length and int(content_length) > MAX_ASSET_BYTES:
            raise ValueError("파일이 제한 용량을 초과합니다.")

        filename = filename_from_response(url, response, fallback_stem)
        output_path = output_dir / filename

        hasher = hashlib.sha256()
        written = 0

        with output_path.open("wb") as file:
            for chunk in response.iter_content(chunk_size=64 * 1024):
                if not chunk:
                    continue
                written += len(chunk)
                if written > MAX_ASSET_BYTES:
                    output_path.unlink(missing_ok=True)
                    raise ValueError("다운로드 중 제한 용량을 초과했습니다.")
                hasher.update(chunk)
                file.write(chunk)

        return {
            "source_url": url,
            "final_url": response.url,
            "filename": filename,
            "relative_path": str(output_path.relative_to(OUTPUT_ROOT)),
            "content_type": response.headers.get("Content-Type", ""),
            "size_bytes": written,
            "sha256": hasher.hexdigest(),
            "downloaded_at": now_kst_iso(),
            "download_status": "downloaded",
        }


def ensure_attachment_playwright() -> None:
    try:
        import playwright  # noqa: F401
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "playwright"],
            check=True,
        )
    subprocess.run(
        [sys.executable, "-m", "playwright", "install", "chromium"],
        check=True,
    )


def download_js_attachments_for_document(
    document: dict[str, Any],
    output_dir: Path,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    js_attachments = [
        item
        for item in document.get("attachments", [])
        if item.get("download_method") == "gfn_downloadFile"
    ]
    if not js_attachments:
        return [], []

    ensure_attachment_playwright()
    from playwright.sync_api import sync_playwright

    downloaded: list[dict[str, Any]] = []
    failures: list[dict[str, Any]] = []
    output_dir.mkdir(parents=True, exist_ok=True)

    with sync_playwright() as playwright:
        browser = playwright.chromium.launch(headless=True)
        context = browser.new_context(
            accept_downloads=True,
            user_agent=USER_AGENT,
            locale="ko-KR",
        )
        page = context.new_page()

        try:
            page.goto(
                document["source_url"],
                wait_until="domcontentloaded",
                timeout=REQUEST_TIMEOUT_SECONDS * 1000,
            )
            page.wait_for_function(
                "typeof gfn_downloadFile === 'function'",
                timeout=REQUEST_TIMEOUT_SECONDS * 1000,
            )

            for item in js_attachments:
                try:
                    with page.expect_download(timeout=60_000) as download_info:
                        page.evaluate(
                            """
                            ([token1, token2]) => {
                                gfn_downloadFile(token1, token2);
                            }
                            """,
                            [item["token1"], item["token2"]],
                        )

                    download = download_info.value
                    filename = safe_name(download.suggested_filename, max_length=120)
                    output_path = output_dir / filename
                    download.save_as(str(output_path))

                    downloaded.append(
                        {
                            **item,
                            "filename": filename,
                            "relative_path": str(output_path.relative_to(OUTPUT_ROOT)),
                            "size_bytes": output_path.stat().st_size,
                            "sha256": sha256_bytes(output_path.read_bytes()),
                            "download_status": "downloaded",
                            "error": "",
                        }
                    )
                except Exception as error:
                    failures.append(
                        {
                            "asset_type": "attachment",
                            "display_name": item.get("display_name", ""),
                            "download_method": "gfn_downloadFile",
                            "error_type": type(error).__name__,
                            "error": str(error),
                        }
                    )
        finally:
            context.close()
            browser.close()

    return downloaded, failures


def download_document_assets(
    document: dict[str, Any],
    manifest_row: dict[str, str],
) -> dict[str, list[dict[str, Any]]]:
    business_dir = safe_name(manifest_row["business_function"])
    asset_root = PATHS["raw_assets"] / business_dir / manifest_row["url_id"]

    results = {
        "attachments": [],
        "attachment_metadata": [],
        "images": [],
        "videos": [],
        "failures": [],
    }

    should_download_attachments = (
        DOWNLOAD_ATTACHMENTS
        and (
            manifest_row["asset_policy"] == "download_attachments"
            or manifest_row["page_type"] == "attachment_page"
        )
    )

    # JS 버튼은 URL이 없으므로 우선 메타데이터로 항상 보존합니다.
    for item in document.get("attachments", []):
        if item.get("download_method") == "gfn_downloadFile":
            results["attachment_metadata"].append(item)

    # 직접 href 첨부파일
    if should_download_attachments:
        for index, item in enumerate(document.get("attachments", []), start=1):
            if item.get("download_method") != "href" or not item.get("url"):
                continue
            try:
                result = download_binary_asset(
                    item["url"],
                    asset_root / "attachments",
                    f"{manifest_row['url_id']}_attachment_{index:03d}",
                )
                result.update(item)
                results["attachments"].append(result)
            except Exception as error:
                results["failures"].append(
                    {
                        "asset_type": "attachment",
                        "url": item.get("url", ""),
                        "error_type": type(error).__name__,
                        "error": str(error),
                    }
                )

    # 선택적으로 JavaScript 버튼 실제 다운로드
    if should_download_attachments and DOWNLOAD_JS_ATTACHMENTS_WITH_PLAYWRIGHT:
        downloaded, failures = download_js_attachments_for_document(
            document,
            asset_root / "attachments",
        )
        results["attachments"].extend(downloaded)
        results["failures"].extend(failures)

    should_download_images = (
        DOWNLOAD_IMAGES
        and (
            manifest_row["crawl_wave"] == "W1_ASSET"
            or manifest_row["page_type"] in {
                "process_page", "video_page", "attachment_page"
            }
        )
    )

    if should_download_images:
        for index, item in enumerate(document.get("images", []), start=1):
            if item.get("image_role") == "decorative":
                continue
            try:
                result = download_binary_asset(
                    item["url"],
                    asset_root / "images",
                    f"{manifest_row['url_id']}_image_{index:03d}",
                )
                result.update(item)
                results["images"].append(result)
            except Exception as error:
                results["failures"].append(
                    {
                        "asset_type": "image",
                        "url": item.get("url", ""),
                        "error_type": type(error).__name__,
                        "error": str(error),
                    }
                )

    return results

## 8. 동적 조회 페이지 진단

In [ ]:
DYNAMIC_SCRIPT_KEYWORD_PATTERN = re.compile(
    r"(?:ajax|fetch\s*\(|XMLHttpRequest|\.do\b|pageIndex|pageNo|search)",
    flags=re.IGNORECASE,
)


def extract_dynamic_diagnostic(
    html_bytes: bytes,
    final_url: str,
    encoding: str,
) -> dict[str, Any]:
    """공개 동적 페이지의 폼과 스크립트 구조를 진단합니다."""
    html_text = html_bytes.decode(encoding, errors="replace")
    soup = BeautifulSoup(html_text, "lxml")

    forms = []

    for form_index, form in enumerate(
        soup.find_all("form"),
        start=1,
    ):
        inputs = []

        for field in form.select(
            "input, select, textarea, button"
        ):
            name = normalize_space(field.get("name", ""))
            field_type = normalize_space(
                field.get("type", field.name or "")
            )

            if not name and field.name != "button":
                continue

            record = {
                "tag": field.name,
                "name": name,
                "type": field_type,
                "value": normalize_space(
                    field.get("value", "")
                ),
            }

            if field.name == "select":
                record["options"] = [
                    {
                        "value": normalize_space(
                            option.get("value", "")
                        ),
                        "text": normalize_space(
                            option.get_text(" ", strip=True)
                        ),
                    }
                    for option in field.find_all("option")
                ]

            inputs.append(record)

        forms.append(
            {
                "form_index": form_index,
                "method": normalize_space(
                    form.get("method", "GET")
                ).upper(),
                "action": (
                    absolute_url(
                        final_url,
                        form.get("action"),
                    )
                    or final_url
                ),
                "id": normalize_space(form.get("id", "")),
                "name": normalize_space(
                    form.get("name", "")
                ),
                "inputs": inputs,
            }
        )

    script_sources = []

    for script in soup.find_all("script", src=True):
        source = absolute_url(
            final_url,
            script.get("src"),
        )

        if source and source not in script_sources:
            script_sources.append(source)

    inline_script_hints = []

    for script in soup.find_all("script"):
        if script.get("src"):
            continue

        text = script.get_text("\n", strip=True)

        if (
            text
            and DYNAMIC_SCRIPT_KEYWORD_PATTERN.search(text)
        ):
            inline_script_hints.append(text[:1200])

    return {
        "final_url": final_url,
        "forms": forms,
        "script_sources": script_sources,
        "inline_script_hints": inline_script_hints[:20],
        "diagnosed_at": now_kst_iso(),
        "safety_note": (
            "공개 페이지 구조만 진단합니다. "
            "로그인·본인인증·개인정보 조회·신청 제출은 수행하지 않습니다."
        ),
    }


def bi004_extract_base_payload(
    html_bytes: bytes,
    encoding: str,
) -> dict[str, str]:
    """BI-004 검색 폼의 기본 공개 검색 조건을 추출합니다."""
    soup = BeautifulSoup(
        html_bytes.decode(encoding, errors="replace"),
        "lxml",
    )
    form = soup.select_one("form#srchForm")

    if not form:
        raise ValueError(
            "BI-004 검색 폼 form#srchForm을 찾지 못했습니다."
        )

    payload: dict[str, str] = {}

    for field in form.select(
        "input[name], select[name], textarea[name]"
    ):
        name = normalize_space(field.get("name", ""))
        if not name:
            continue

        if field.name == "select":
            selected = field.find("option", selected=True)
            if selected is None:
                selected = field.find("option")
            value = (
                selected.get("value", "")
                if selected
                else ""
            )

        elif field.name == "textarea":
            value = field.get_text("", strip=False)

        else:
            field_type = normalize_space(
                field.get("type", "text")
            ).lower()

            if (
                field_type in {"checkbox", "radio"}
                and not field.has_attr("checked")
            ):
                continue

            value = field.get("value", "")

        payload[name] = str(value or "")

    # 전체 금융권역·전체 회사 조회
    payload["fncSareaCd"] = ""
    payload["searchInfnstNm"] = ""

    return payload


def bi004_last_internal_page_index(
    html_bytes: bytes,
    encoding: str,
) -> int:
    """
    BI-004는 curPage가 0부터 시작합니다.
    화면의 1페이지는 curPage=0, 화면의 2페이지는 curPage=1입니다.
    """
    soup = BeautifulSoup(
        html_bytes.decode(encoding, errors="replace"),
        "lxml",
    )

    page_indexes: list[int] = []

    for node in soup.select(
        ".paging [onclick*='chgPagingNo']"
    ):
        onclick = node.get("onclick", "")
        match = re.search(
            r"chgPagingNo\(\s*(\d+)\s*\)",
            onclick,
        )
        if match:
            page_indexes.append(int(match.group(1)))

    return max(page_indexes, default=0)


def bi004_displayed_page_number(
    html_bytes: bytes,
    encoding: str,
) -> int | None:
    soup = BeautifulSoup(
        html_bytes.decode(encoding, errors="replace"),
        "lxml",
    )

    current = soup.select_one(
        ".paging strong[title*='현재 페이지'] span"
    ) or soup.select_one(".paging strong span")

    if not current:
        return None

    text = normalize_space(current.get_text(" ", strip=True))
    return int(text) if text.isdigit() else None


def bi004_parse_result_table(
    html_bytes: bytes,
    encoding: str,
) -> tuple[list[str], list[list[str]]]:
    """금융회사 조회 결과 표 하나를 구조화합니다."""
    soup = BeautifulSoup(
        html_bytes.decode(encoding, errors="replace"),
        "lxml",
    )

    root = (
        soup.select_one(".contents")
        or soup.select_one("#contents")
        or soup.select_one("main")
        or soup.body
    )

    if not root:
        raise ValueError("BI-004 본문 영역을 찾지 못했습니다.")

    candidate_tables = root.find_all("table")
    target_table = None

    for table in candidate_tables:
        table_text = normalize_space(
            table.get_text(" ", strip=True)
        )
        if (
            "금융권역" in table_text
            and "금융회사명" in table_text
        ):
            target_table = table
            break

    if target_table is None:
        raise ValueError(
            "BI-004 금융회사 결과 표를 찾지 못했습니다."
        )

    headers, rows = fixed_expand_table(target_table)

    cleaned_rows = [
        [normalize_space(value) for value in row]
        for row in rows
        if any(normalize_space(value) for value in row)
    ]

    if not headers:
        raise ValueError("BI-004 결과 표의 헤더가 없습니다.")

    return headers, cleaned_rows


def bi004_replace_document_table(
    document: dict[str, Any],
    headers: list[str],
    rows: list[list[str]],
) -> None:
    """첫 페이지 표를 전체 페이지 통합 표로 교체합니다."""
    new_blocks: list[dict[str, Any]] = []
    replaced = False

    for block in document.get("blocks", []):
        if block.get("type") != "table":
            new_blocks.append(block)
            continue

        block_headers = " ".join(
            block.get("headers", [])
        )

        if (
            not replaced
            and "금융권역" in block_headers
            and "금융회사명" in block_headers
        ):
            new_blocks.append(
                {
                    "type": "table",
                    "headers": headers,
                    "rows": rows,
                    "row_count": len(rows),
                    "table_role": (
                        "deposit_insurance_payment_target_companies"
                    ),
                }
            )
            replaced = True
        else:
            new_blocks.append(block)

    if not replaced:
        new_blocks.append(
            {
                "type": "table",
                "headers": headers,
                "rows": rows,
                "row_count": len(rows),
                "table_role": (
                    "deposit_insurance_payment_target_companies"
                ),
            }
        )

    document["blocks"] = new_blocks
    document["content_markdown"] = fixed_render_blocks(
        new_blocks
    )
    document["content_text"] = fixed_blocks_text(
        new_blocks
    )

    parsed_hash = sha256_text(
        document["content_markdown"]
    )
    document["parsed_content_sha256"] = parsed_hash
    document["content_hash"] = parsed_hash

    quality = document.setdefault("quality", {})
    quality["parsed_text_chars"] = len(
        document["content_text"]
    )
    quality["table_count"] = sum(
        1
        for block in new_blocks
        if block.get("type") == "table"
    )


def collect_bi004_all_pages(
    first_result: FetchResult,
    manifest_row: dict[str, str],
) -> dict[str, Any]:
    """
    첫 GET 응답을 1페이지로 사용하고,
    curPage=1부터 마지막 내부 인덱스까지 POST 요청합니다.
    """
    first_headers, first_rows = bi004_parse_result_table(
        first_result.body,
        first_result.encoding,
    )

    last_index = bi004_last_internal_page_index(
        first_result.body,
        first_result.encoding,
    )
    page_count = last_index + 1

    if page_count < 1:
        raise ValueError("BI-004 페이지 수가 1보다 작습니다.")

    if page_count > BI004_MAX_PAGES:
        raise ValueError(
            f"BI-004 페이지 수 {page_count}가 "
            f"안전 한도 {BI004_MAX_PAGES}를 초과했습니다."
        )

    base_payload = bi004_extract_base_payload(
        first_result.body,
        first_result.encoding,
    )

    pages_dir = (
        PATHS["dynamic_diagnostics"]
        / "BI-004_pages"
    )
    pages_dir.mkdir(parents=True, exist_ok=True)

    combined_rows: list[list[str]] = []
    seen_rows: set[tuple[str, ...]] = set()
    seen_page_hashes: dict[str, int] = {}
    page_records: list[dict[str, Any]] = []
    failures: list[dict[str, Any]] = []

    def register_page(
        internal_index: int,
        body: bytes,
        encoding: str,
        status_code: int,
        final_url: str,
        request_method: str,
        request_payload: dict[str, str] | None,
    ) -> None:
        displayed_page = bi004_displayed_page_number(
            body,
            encoding,
        )
        headers, rows = bi004_parse_result_table(
            body,
            encoding,
        )

        if headers != first_headers:
            failures.append(
                {
                    "internal_page_index": internal_index,
                    "reason": "페이지별 표 헤더가 서로 다름",
                    "headers": headers,
                }
            )

        row_hash = sha256_text(
            json.dumps(
                rows,
                ensure_ascii=False,
                sort_keys=True,
            )
        )

        if row_hash in seen_page_hashes:
            failures.append(
                {
                    "internal_page_index": internal_index,
                    "reason": "다른 페이지와 동일한 표 결과 반복",
                    "same_as_internal_index": (
                        seen_page_hashes[row_hash]
                    ),
                }
            )
        else:
            seen_page_hashes[row_hash] = internal_index

        page_filename = (
            f"page_{internal_index + 1:03d}.html"
        )
        page_path = pages_dir / page_filename
        page_path.write_bytes(body)

        page_meta = {
            "internal_page_index": internal_index,
            "displayed_page_number": displayed_page,
            "status_code": status_code,
            "request_method": request_method,
            "request_payload": request_payload,
            "final_url": final_url,
            "row_count": len(rows),
            "row_hash": row_hash,
            "raw_sha256": sha256_bytes(body),
            "raw_html_path": str(
                page_path.relative_to(OUTPUT_ROOT)
            ),
        }
        page_records.append(page_meta)

        expected_displayed_page = internal_index + 1
        if (
            displayed_page is not None
            and displayed_page != expected_displayed_page
        ):
            failures.append(
                {
                    "internal_page_index": internal_index,
                    "reason": (
                        "요청 페이지와 화면 현재 페이지 불일치"
                    ),
                    "expected_displayed_page": (
                        expected_displayed_page
                    ),
                    "actual_displayed_page": displayed_page,
                }
            )

        for row in rows:
            key = tuple(row)
            if key not in seen_rows:
                seen_rows.add(key)
                combined_rows.append(row)

    # GET으로 확보한 첫 페이지
    register_page(
        internal_index=0,
        body=first_result.body,
        encoding=first_result.encoding,
        status_code=first_result.status_code,
        final_url=first_result.final_url,
        request_method="GET",
        request_payload=None,
    )

    # 화면 2페이지부터 마지막 페이지까지 반복 POST
    for internal_index in range(1, page_count):
        payload = dict(base_payload)
        payload["curPage"] = str(internal_index)
        payload["pageSize"] = str(BI004_PAGE_SIZE)

        try:
            response = SESSION.post(
                manifest_row["source_url"],
                data=payload,
                headers={
                    "Referer": manifest_row["source_url"],
                },
                timeout=REQUEST_TIMEOUT_SECONDS,
                allow_redirects=True,
            )

            response.raise_for_status()

            encoding = choose_response_encoding(response)
            response.encoding = encoding

            register_page(
                internal_index=internal_index,
                body=response.content,
                encoding=encoding,
                status_code=response.status_code,
                final_url=response.url,
                request_method="POST",
                request_payload=payload,
            )

        except Exception as error:
            failures.append(
                {
                    "internal_page_index": internal_index,
                    "reason": "페이지 요청 또는 파싱 실패",
                    "error_type": type(error).__name__,
                    "error": str(error),
                }
            )

        time.sleep(REQUEST_DELAY_SECONDS)

    fetched_indexes = {
        record["internal_page_index"]
        for record in page_records
    }
    expected_indexes = set(range(page_count))

    missing_indexes = sorted(
        expected_indexes - fetched_indexes
    )

    if missing_indexes:
        failures.append(
            {
                "reason": "수집되지 않은 페이지 존재",
                "missing_internal_indexes": missing_indexes,
            }
        )

    is_complete = (
        not failures
        and len(page_records) == page_count
        and len(combined_rows) > 0
    )

    all_rows_csv = (
        PATHS["dynamic_diagnostics"]
        / "BI-004_all_rows.csv"
    )
    pd.DataFrame(
        combined_rows,
        columns=first_headers,
    ).to_csv(
        all_rows_csv,
        index=False,
        encoding="utf-8-sig",
    )

    return {
        "headers": first_headers,
        "rows": combined_rows,
        "collection": {
            "collection_scope": (
                "all_public_pages"
                if is_complete
                else "partial_public_pages"
            ),
            "is_complete": is_complete,
            "page_index_base": 0,
            "expected_page_count": page_count,
            "fetched_page_count": len(page_records),
            "row_count": len(combined_rows),
            "page_size": BI004_PAGE_SIZE,
            "search_filters": {
                "fncSareaCd": "",
                "searchInfnstNm": "",
            },
            "pages": page_records,
            "failures": failures,
            "all_rows_csv_path": str(
                all_rows_csv.relative_to(OUTPUT_ROOT)
            ),
            "collected_at": now_kst_iso(),
        },
    }


def ensure_playwright_installed() -> None:
    try:
        import playwright  # noqa: F401
        return
    except ImportError:
        pass

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "playwright"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "playwright", "install", "chromium"],
        check=True,
    )


def save_playwright_snapshot(
    url: str,
    url_id: str,
) -> dict[str, Any] | None:
    if not ENABLE_PLAYWRIGHT_SNAPSHOT:
        return None

    ensure_playwright_installed()
    from playwright.sync_api import sync_playwright

    html_path = PATHS["dynamic_diagnostics"] / f"{url_id}_rendered.html"
    screenshot_path = PATHS["screenshots"] / f"{url_id}.png"
    ensure_parent(html_path)
    ensure_parent(screenshot_path)

    with sync_playwright() as playwright:
        browser = playwright.chromium.launch(headless=True)
        page = browser.new_page(
            user_agent=USER_AGENT,
            locale="ko-KR",
            viewport={"width": 1440, "height": 1200},
        )

        response = page.goto(
            url,
            wait_until="domcontentloaded",
            timeout=REQUEST_TIMEOUT_SECONDS * 1000,
        )
        page.wait_for_timeout(PLAYWRIGHT_WAIT_MS)

        html = page.content()
        html_path.write_text(html, encoding="utf-8")
        page.screenshot(path=str(screenshot_path), full_page=True)

        result = {
            "status_code": response.status if response else None,
            "final_url": page.url,
            "rendered_html_path": str(html_path.relative_to(OUTPUT_ROOT)),
            "screenshot_path": str(screenshot_path.relative_to(OUTPUT_ROOT)),
            "captured_at": now_kst_iso(),
        }

        browser.close()
        return result

### 동적 페이지 진단 코드 자체 테스트

실제 웹 요청 전에 다음을 검사합니다.

- `fetch(...)`
- `XMLHttpRequest`
- `.do` 엔드포인트
- `pageIndex`, `pageNo`, `search`
- 합성 HTML의 폼·인라인 스크립트 추출

이 셀이 실패하면 실제 크롤링을 실행하지 마십시오.

In [ ]:
_DYNAMIC_PATTERN_TEST_CASES = {
    "fetch('/api/list')": True,
    "const xhr = new XMLHttpRequest();": True,
    "url: '/board/selectList.do'": True,
    "pageIndex = 2": True,
    "pageNo = 3": True,
    "searchKeyword": True,
    "일반 안내 문장입니다.": False,
}

for sample_text, expected in _DYNAMIC_PATTERN_TEST_CASES.items():
    actual = bool(DYNAMIC_SCRIPT_KEYWORD_PATTERN.search(sample_text))
    assert actual == expected, (
        f"동적 스크립트 정규표현식 테스트 실패: "
        f"text={sample_text!r}, expected={expected}, actual={actual}"
    )

_DYNAMIC_TEST_HTML = """
<!doctype html>
<html lang="ko">
<body>
  <form id="searchForm" method="post" action="/search/list.do">
    <input type="hidden" name="pageIndex" value="1">
    <input type="text" name="searchKeyword" value="">
    <select name="category">
      <option value="">전체</option>
      <option value="bank">은행</option>
    </select>
  </form>

  <script>
    function loadData() {
      fetch('/api/list?pageNo=1');
    }
  </script>
</body>
</html>
""".encode("utf-8")

_dynamic_test_result = extract_dynamic_diagnostic(
    html_bytes=_DYNAMIC_TEST_HTML,
    final_url="https://example.org/dynamic/page",
    encoding="utf-8",
)

assert len(_dynamic_test_result["forms"]) == 1
assert _dynamic_test_result["forms"][0]["method"] == "POST"
assert _dynamic_test_result["forms"][0]["action"] == (
    "https://example.org/search/list.do"
)
assert any(
    item["name"] == "pageIndex"
    for item in _dynamic_test_result["forms"][0]["inputs"]
)
assert len(_dynamic_test_result["inline_script_hints"]) == 1

print("동적 페이지 진단 자체 테스트: 통과")
print("정규표현식:", DYNAMIC_SCRIPT_KEYWORD_PATTERN.pattern)


_BI004_PAGINATION_TEST_HTML = """
<!doctype html>
<html lang="ko">
<body>
  <form id="srchForm" method="post">
    <select name="fncSareaCd">
      <option value="" selected>전체</option>
    </select>
    <input name="searchInfnstNm" type="text" value="">
  </form>

  <div class="contents">
    <table>
      <thead>
        <tr>
          <th>금융권역</th>
          <th>금융회사명</th>
          <th>상태</th>
        </tr>
      </thead>
      <tbody>
        <tr>
          <td>상호저축은행</td>
          <td>예시은행</td>
          <td>파산</td>
        </tr>
      </tbody>
    </table>

    <div class="paging">
      <strong title="현재 페이지"><span>1</span></strong>
      <a onclick="chgPagingNo(1);">2</a>
      <a onclick="chgPagingNo(2);">3</a>
      <a class="btnLast" onclick="chgPagingNo(2);">
        마지막 페이지
      </a>
    </div>
  </div>
</body>
</html>
""".encode("utf-8")

assert bi004_last_internal_page_index(
    _BI004_PAGINATION_TEST_HTML,
    "utf-8",
) == 2

assert bi004_displayed_page_number(
    _BI004_PAGINATION_TEST_HTML,
    "utf-8",
) == 1

_test_headers, _test_rows = bi004_parse_result_table(
    _BI004_PAGINATION_TEST_HTML,
    "utf-8",
)

assert _test_headers == [
    "금융권역",
    "금융회사명",
    "상태",
]
assert len(_test_rows) == 1
assert _test_rows[0][1] == "예시은행"

_test_payload = bi004_extract_base_payload(
    _BI004_PAGINATION_TEST_HTML,
    "utf-8",
)
assert _test_payload["fncSareaCd"] == ""
assert _test_payload["searchInfnstNm"] == ""

print("BI-004 페이지네이션 자체 테스트: 통과")


## 9. URL 유형별 처리기

In [ ]:

def save_parsed_document(
    document: dict[str, Any],
    manifest_row: dict[str, str],
) -> tuple[Path, Path]:
    business_dir = safe_name(manifest_row["business_function"])
    url_id = manifest_row["url_id"]

    markdown_path = (
        PATHS["parsed_markdown"]
        / business_dir
        / f"{url_id}.md"
    )
    json_path = (
        PATHS["parsed_json"]
        / business_dir
        / f"{url_id}.json"
    )

    ensure_parent(markdown_path)
    markdown_path.write_text(
        document["content_markdown"],
        encoding="utf-8",
    )
    write_json(json_path, document)

    return markdown_path, json_path


def process_link_only(
    manifest_row: dict[str, str],
) -> dict[str, Any]:
    record = {
        "doc_id": manifest_row["url_id"],
        "record_type": "link_only",
        "title": manifest_row["page_title"],
        "source_url": manifest_row["source_url"],
        "site_name": manifest_row["site_name"],
        "business_function": manifest_row["business_function"],
        "target_business_function": manifest_row.get(
            "target_business_function",
            manifest_row["business_function"],
        ),
        "page_type": manifest_row["page_type"],
        "auth_required": manifest_row["auth_required"],
        "content_scope": manifest_row["content_scope"],
        "description": manifest_row.get("content_summary", ""),
        "decision_reason": manifest_row["decision_reason"],
        "created_at": now_kst_iso(),
    }

    output_path = (
        PATHS["parsed_json"]
        / safe_name(manifest_row["business_function"])
        / f"{manifest_row['url_id']}.json"
    )
    write_json(output_path, record)

    return {
        "url_id": manifest_row["url_id"],
        "business_function": manifest_row["business_function"],
        "page_title": manifest_row["page_title"],
        "source_url": manifest_row["source_url"],
        "normalized_decision": manifest_row["normalized_decision"],
        "crawl_wave": manifest_row["crawl_wave"],
        "status": "link_metadata_created",
        "status_code": None,
        "content_chars": len(record["description"]),
        "attachment_count": 0,
        "image_count": 0,
        "asset_failure_count": 0,
        "error_type": "",
        "error": "",
        "finished_at": now_kst_iso(),
    }


def process_static_or_asset(
    manifest_row: dict[str, str],
) -> dict[str, Any]:
    result = fetch_url(manifest_row["source_url"])
    html_path, meta_path = save_fetch_result(manifest_row, result)

    if not result.ok:
        raise requests.HTTPError(
            f"HTTP {result.status_code}: {result.final_url}"
        )

    if "html" not in result.content_type.lower():
        raise ValueError(
            f"HTML 응답이 아닙니다: {result.content_type}"
        )

    document = parse_html_document(
        html_bytes=result.body,
        final_url=result.final_url,
        manifest_row=manifest_row,
        encoding=result.encoding,
    )

    asset_results = download_document_assets(
        document=document,
        manifest_row=manifest_row,
    )
    document["downloaded_assets"] = asset_results

    markdown_path, json_path = save_parsed_document(
        document=document,
        manifest_row=manifest_row,
    )

    return {
        "url_id": manifest_row["url_id"],
        "business_function": manifest_row["business_function"],
        "page_title": document["title"],
        "source_url": manifest_row["source_url"],
        "final_url": result.final_url,
        "normalized_decision": manifest_row["normalized_decision"],
        "crawl_wave": manifest_row["crawl_wave"],
        "status": (
            "parse_success"
            if document["content_text"]
            else "empty_content"
        ),
        "quality_status": document.get("quality", {}).get("status", ""),
        "quality_reasons": " | ".join(document.get("quality", {}).get("reasons", [])),
        "retention_ratio": document.get("quality", {}).get("retention_ratio", 0),
        "faq_count": document.get("quality", {}).get("faq_count", 0),
        "table_count": document.get("quality", {}).get("table_count", 0),
        "attachment_button_count": document.get("quality", {}).get("attachment_button_count", 0),
        "status_code": result.status_code,
        "content_chars": len(document["content_text"]),
        "block_count": len(document["blocks"]),
        "link_count": len(document["links"]),
        "attachment_count": len(document["attachments"]),
        "image_count": len(document["images"]),
        "downloaded_attachment_count": len(asset_results["attachments"]),
        "downloaded_image_count": len(asset_results["images"]),
        "asset_failure_count": len(asset_results["failures"]),
        "raw_html_path": str(html_path.relative_to(OUTPUT_ROOT)),
        "response_meta_path": str(meta_path.relative_to(OUTPUT_ROOT)),
        "parsed_markdown_path": str(markdown_path.relative_to(OUTPUT_ROOT)),
        "parsed_json_path": str(json_path.relative_to(OUTPUT_ROOT)),
        "raw_sha256": result.raw_sha256,
        "parsed_content_sha256": document["parsed_content_sha256"],
        "error_type": "",
        "error": "",
        "finished_at": now_kst_iso(),
    }


def process_dynamic_diagnostic(
    manifest_row: dict[str, str],
) -> dict[str, Any]:
    result = fetch_url(manifest_row["source_url"])
    html_path, meta_path = save_fetch_result(
        manifest_row,
        result,
    )

    if not result.ok:
        raise requests.HTTPError(
            f"HTTP {result.status_code}: {result.final_url}"
        )

    diagnostic = extract_dynamic_diagnostic(
        html_bytes=result.body,
        final_url=result.final_url,
        encoding=result.encoding,
    )
    diagnostic["url_id"] = manifest_row["url_id"]
    diagnostic["page_title"] = manifest_row["page_title"]
    diagnostic["business_function"] = (
        manifest_row["business_function"]
    )
    diagnostic["source_url"] = manifest_row["source_url"]

    playwright_snapshot = save_playwright_snapshot(
        url=manifest_row["source_url"],
        url_id=manifest_row["url_id"],
    )
    if playwright_snapshot:
        diagnostic["playwright_snapshot"] = (
            playwright_snapshot
        )

    document = parse_html_document(
        html_bytes=result.body,
        final_url=result.final_url,
        manifest_row=manifest_row,
        encoding=result.encoding,
    )

    dynamic_status = "dynamic_diagnostic_created"
    dynamic_page_count = 1
    dynamic_row_count = 0
    dynamic_is_complete = False

    if (
        manifest_row["url_id"] == "BI-004"
        and FETCH_BI004_ALL_PAGES
    ):
        collection_result = collect_bi004_all_pages(
            first_result=result,
            manifest_row=manifest_row,
        )

        bi004_replace_document_table(
            document=document,
            headers=collection_result["headers"],
            rows=collection_result["rows"],
        )

        document["dynamic_collection"] = (
            collection_result["collection"]
        )
        diagnostic["bi004_full_collection"] = (
            collection_result["collection"]
        )

        dynamic_page_count = collection_result[
            "collection"
        ]["fetched_page_count"]
        dynamic_row_count = collection_result[
            "collection"
        ]["row_count"]
        dynamic_is_complete = collection_result[
            "collection"
        ]["is_complete"]

        dynamic_status = (
            "dynamic_full_collection_created"
            if dynamic_is_complete
            else "dynamic_collection_incomplete"
        )

    else:
        document["dynamic_collection"] = {
            "collection_scope": (
                "initial_public_page_only"
            ),
            "is_complete": False,
            "expected_page_count": None,
            "fetched_page_count": 1,
            "row_count": None,
            "page_size": BI004_PAGE_SIZE,
        }
        diagnostic["bi004_full_collection"] = (
            document["dynamic_collection"]
        )

    diagnostic_path = (
        PATHS["dynamic_diagnostics"]
        / f"{manifest_row['url_id']}.json"
    )
    write_json(diagnostic_path, diagnostic)

    markdown_path, json_path = save_parsed_document(
        document=document,
        manifest_row=manifest_row,
    )

    collection_failures = (
        document.get("dynamic_collection", {})
        .get("failures", [])
    )

    return {
        "url_id": manifest_row["url_id"],
        "business_function": (
            manifest_row["business_function"]
        ),
        "page_title": document["title"],
        "source_url": manifest_row["source_url"],
        "final_url": result.final_url,
        "normalized_decision": (
            manifest_row["normalized_decision"]
        ),
        "crawl_wave": manifest_row["crawl_wave"],
        "status": dynamic_status,
        "quality_status": (
            document.get("quality", {}).get(
                "status",
                "",
            )
        ),
        "quality_reasons": " | ".join(
            document.get("quality", {}).get(
                "reasons",
                [],
            )
        ),
        "retention_ratio": (
            document.get("quality", {}).get(
                "retention_ratio",
                0,
            )
        ),
        "faq_count": (
            document.get("quality", {}).get(
                "faq_count",
                0,
            )
        ),
        "table_count": (
            document.get("quality", {}).get(
                "table_count",
                0,
            )
        ),
        "attachment_button_count": (
            document.get("quality", {}).get(
                "attachment_button_count",
                0,
            )
        ),
        "status_code": result.status_code,
        "content_chars": len(
            document["content_text"]
        ),
        "form_count": len(diagnostic["forms"]),
        "script_source_count": len(
            diagnostic["script_sources"]
        ),
        "dynamic_is_complete": (
            dynamic_is_complete
        ),
        "dynamic_page_count": (
            dynamic_page_count
        ),
        "dynamic_row_count": (
            dynamic_row_count
        ),
        "dynamic_failure_count": len(
            collection_failures
        ),
        "attachment_count": len(
            document["attachments"]
        ),
        "image_count": len(
            document["images"]
        ),
        "asset_failure_count": 0,
        "raw_html_path": str(
            html_path.relative_to(OUTPUT_ROOT)
        ),
        "response_meta_path": str(
            meta_path.relative_to(OUTPUT_ROOT)
        ),
        "parsed_markdown_path": str(
            markdown_path.relative_to(OUTPUT_ROOT)
        ),
        "parsed_json_path": str(
            json_path.relative_to(OUTPUT_ROOT)
        ),
        "dynamic_diagnostic_path": str(
            diagnostic_path.relative_to(OUTPUT_ROOT)
        ),
        "error_type": (
            ""
            if dynamic_is_complete
            or not FETCH_BI004_ALL_PAGES
            else "IncompleteDynamicCollection"
        ),
        "error": (
            ""
            if dynamic_is_complete
            or not FETCH_BI004_ALL_PAGES
            else json.dumps(
                collection_failures,
                ensure_ascii=False,
            )
        ),
        "finished_at": now_kst_iso(),
    }

def process_manifest_row(
    manifest_row: dict[str, str],
) -> dict[str, Any]:
    decision = manifest_row["normalized_decision"]
    strategy = manifest_row["fetch_strategy"]

    if decision == "link_only":
        return process_link_only(manifest_row)

    if strategy == "dynamic_http_then_playwright":
        return process_dynamic_diagnostic(manifest_row)

    return process_static_or_asset(manifest_row)


## 10. 크롤링 실행

In [ ]:

CRAWL_RESULTS_JSONL = PATHS["logs"] / "crawl_results.jsonl"
CRAWL_RESULTS_CSV = PATHS["logs"] / "crawl_results.csv"
USED_MANIFEST_PATH = PATHS["manifest"] / "manifest_used.csv"

target_df.to_csv(
    USED_MANIFEST_PATH,
    index=False,
    encoding="utf-8-sig",
)

run_results: list[dict[str, Any]] = []

for _, row in tqdm(
    target_df.iterrows(),
    total=len(target_df),
    desc="크롤링 진행",
):
    manifest_row = row_to_clean_dict(row)
    started_at = now_kst_iso()

    try:
        result = process_manifest_row(manifest_row)
        result["started_at"] = started_at

    except Exception as error:
        result = {
            "url_id": manifest_row["url_id"],
            "business_function": manifest_row["business_function"],
            "page_title": manifest_row["page_title"],
            "source_url": manifest_row["source_url"],
            "normalized_decision": manifest_row["normalized_decision"],
            "crawl_wave": manifest_row["crawl_wave"],
            "status": "failed",
            "status_code": None,
            "content_chars": 0,
            "attachment_count": 0,
            "image_count": 0,
            "asset_failure_count": 0,
            "error_type": type(error).__name__,
            "error": str(error),
            "started_at": started_at,
            "finished_at": now_kst_iso(),
        }

    run_results.append(result)
    append_jsonl(CRAWL_RESULTS_JSONL, result)

    # link_only는 실제 HTTP 요청을 보내지 않으므로 대기하지 않습니다.
    if manifest_row["normalized_decision"] != "link_only":
        time.sleep(REQUEST_DELAY_SECONDS)

results_df = pd.DataFrame(run_results)
results_df.to_csv(
    CRAWL_RESULTS_CSV,
    index=False,
    encoding="utf-8-sig",
)

print("실행 완료")

for optional_column, default_value in {
    "dynamic_is_complete": None,
    "dynamic_page_count": None,
    "dynamic_row_count": None,
    "dynamic_failure_count": None,
}.items():
    if optional_column not in results_df.columns:
        results_df[optional_column] = default_value

display(
    results_df[
        [
            "url_id",
            "business_function",
            "status",
            "status_code",
            "content_chars",
            "attachment_count",
            "image_count",
            "asset_failure_count",
            "dynamic_is_complete",
            "dynamic_page_count",
            "dynamic_row_count",
            "dynamic_failure_count",
            "error_type",
        ]
    ]
)


## 11. 결과 검수

In [ ]:
QUALITY_REPORT_CSV = PATHS["logs"] / "quality_report.csv"

if results_df.empty:
    print("실행 결과가 없습니다.")
else:
    print("수집 상태별 건수")
    display(
        results_df.groupby("status")
        .size()
        .rename("count")
        .reset_index()
        .sort_values("count", ascending=False)
    )

    failures = results_df[results_df["status"] == "failed"].copy()
    empty_contents = results_df[
        results_df["status"].isin({"empty_content"})
        | (
            results_df["status"].isin(
                {"parse_success", "dynamic_diagnostic_created"}
            )
            & (results_df["content_chars"].fillna(0).astype(int) < 80)
        )
    ].copy()

    quality_columns = [
        "url_id", "business_function", "page_title", "status",
        "quality_status", "quality_reasons", "retention_ratio",
        "content_chars", "faq_count", "table_count",
        "attachment_button_count", "attachment_count",
        "image_count", "asset_failure_count", "error_type", "error",
    ]
    available_quality_columns = [
        column for column in quality_columns if column in results_df.columns
    ]
    quality_report_df = results_df[available_quality_columns].copy()
    quality_report_df.to_csv(
        QUALITY_REPORT_CSV,
        index=False,
        encoding="utf-8-sig",
    )

    print("실패:", len(failures))
    print("빈 본문·매우 짧은 본문:", len(empty_contents))

    if "quality_status" in results_df.columns:
        print("파싱 품질 상태")
        display(
            results_df.groupby("quality_status", dropna=False)
            .size()
            .rename("count")
            .reset_index()
        )

        review_required = results_df[
            results_df["quality_status"].isin({"review", "fail"})
        ]
        if not review_required.empty:
            print("사람 검수 또는 파서 수정 필요")
            display(
                review_required[
                    [
                        "url_id", "page_title", "quality_status",
                        "quality_reasons", "retention_ratio",
                        "content_chars",
                    ]
                ]
            )

    if not failures.empty:
        display(
            failures[
                ["url_id", "page_title", "source_url", "error_type", "error"]
            ]
        )

    success_statuses = {
        "parse_success",
        "dynamic_diagnostic_created",
        "link_metadata_created",
    }
    success_count = results_df["status"].isin(success_statuses).sum()
    success_rate = success_count / len(results_df) * 100
    print(f"수집 처리 성공률: {success_count}/{len(results_df)} ({success_rate:.1f}%)")
    print("품질 보고서:", QUALITY_REPORT_CSV)

## 12. 페이지 문서 JSONL과 기본 청크 생성

In [ ]:
DOCUMENTS_JSONL = PATHS["processed"] / "documents.jsonl"
CHUNKS_JSONL = PATHS["processed"] / "chunks.jsonl"


def iter_parsed_json_files() -> Iterable[Path]:
    yield from sorted(PATHS["parsed_json"].rglob("*.json"))


def build_document_record(data: dict[str, Any]) -> dict[str, Any] | None:
    if data.get("record_type") == "link_only":
        return {
            "doc_id": data["doc_id"],
            "parent_doc_id": data["doc_id"],
            "record_type": "link_only",
            "indexable": False,
            "title": data["title"],
            "source_url": data["source_url"],
            "site_name": data["site_name"],
            "business_function": data["business_function"],
            "target_business_function": data["target_business_function"],
            "page_type": data["page_type"],
            "content_markdown": data.get("description", ""),
            "content_text": data.get("description", ""),
            "content_hash": sha256_text(data.get("description", "")),
            "blocks": [],
            "attachments": [],
            "images": [],
            "links": [
                {
                    "url": data["source_url"],
                    "anchor_text": data["title"],
                    "link_type": "official_service",
                }
            ],
            "metadata": {
                "auth_required": data.get("auth_required", ""),
                "content_scope": data.get("content_scope", ""),
                "decision_reason": data.get("decision_reason", ""),
            },
        }

    if not data.get("content_markdown"):
        return None

    return {
        "doc_id": data["doc_id"],
        "parent_doc_id": data.get("parent_doc_id", data["doc_id"]),
        "record_type": "page",
        "indexable": True,
        "title": data.get("title", ""),
        "source_url": data.get("source_url", ""),
        "final_url": data.get("final_url", ""),
        "site_name": data.get("site_name", ""),
        "business_function": data.get("business_function", ""),
        "target_business_function": data.get(
            "target_business_function",
            data.get("business_function", ""),
        ),
        "page_type": data.get("page_type", ""),
        "breadcrumb": data.get("breadcrumb", []),
        "content_markdown": data.get("content_markdown", ""),
        "content_text": data.get("content_text", ""),
        "content_hash": data.get(
            "parsed_content_sha256",
            sha256_text(data.get("content_markdown", "")),
        ),
        "blocks": data.get("blocks", []),
        "attachments": data.get("attachments", []),
        "images": data.get("images", []),
        "links": data.get("links", []),
        "quality": data.get("quality", {}),
        "dynamic_collection": data.get("dynamic_collection"),
        "metadata": {
            "parser_profile": data.get("parser_profile", ""),
            "attachment_count": len(data.get("attachments", [])),
            "image_count": len(data.get("images", [])),
            "parsed_at": data.get("parsed_at", ""),
        },
    }


def render_table_chunk(
    headers: list[str],
    rows: list[list[str]],
) -> str:
    block = {"type": "table", "headers": headers, "rows": rows}
    return fixed_render_blocks([block])


def split_table_block(
    block: dict[str, Any],
    max_chars: int,
) -> list[str]:
    headers = block.get("headers", [])
    rows = block.get("rows", [])
    all_text = render_table_chunk(headers, rows)
    if len(all_text) <= max_chars:
        return [all_text]

    chunks: list[str] = []
    current_rows: list[list[str]] = []

    for row in rows:
        candidate_rows = current_rows + [row]
        candidate_text = render_table_chunk(headers, candidate_rows)

        if current_rows and len(candidate_text) > max_chars:
            chunks.append(render_table_chunk(headers, current_rows))
            current_rows = [row]
        else:
            current_rows = candidate_rows

    if current_rows:
        chunks.append(render_table_chunk(headers, current_rows))

    return chunks


def structure_aware_chunks(
    document: dict[str, Any],
    max_chars: int = 1600,
) -> list[dict[str, Any]]:
    if not document.get("indexable", True):
        return []

    intermediate: list[dict[str, str]] = []
    current_parts: list[str] = []
    current_heading = ""

    def flush_section() -> None:
        nonlocal current_parts
        content = "\n\n".join(
            part for part in current_parts if part
        ).strip()
        current_parts = []
        if content:
            intermediate.append(
                {
                    "section_title": current_heading,
                    "chunk_type": "section",
                    "content": content,
                }
            )

    for block in document.get("blocks", []):
        block_type = block.get("type")

        if block_type == "heading":
            flush_section()
            current_heading = block.get("text", "")
            current_parts = [fixed_render_blocks([block])]
            continue

        if block_type == "faq":
            flush_section()
            intermediate.append(
                {
                    "section_title": block.get("question", ""),
                    "chunk_type": "faq",
                    "content": fixed_render_blocks([block]),
                }
            )
            continue

        if block_type == "table":
            flush_section()
            for table_text in split_table_block(block, max_chars):
                intermediate.append(
                    {
                        "section_title": current_heading,
                        "chunk_type": "table",
                        "content": table_text,
                    }
                )
            continue

        block_text = fixed_render_blocks([block])
        candidate = "\n\n".join(current_parts + [block_text]).strip()
        if current_parts and len(candidate) > max_chars:
            flush_section()
        current_parts.append(block_text)

    flush_section()

    records: list[dict[str, Any]] = []
    for index, item in enumerate(intermediate):
        content = item["content"].strip()
        if not content:
            continue

        records.append(
            {
                "chunk_id": f"{document['doc_id']}_chunk_{index:03d}",
                "parent_doc_id": document["doc_id"],
                "chunk_index": index,
                "title": document["title"],
                "section_title": item["section_title"],
                "chunk_type": item["chunk_type"],
                "business_function": document["business_function"],
                "target_business_function": document[
                    "target_business_function"
                ],
                "page_type": document["page_type"],
                "source_url": document["source_url"],
                "content": content,
                "content_hash": sha256_text(content),
            }
        )

    return records


documents: list[dict[str, Any]] = []

for json_path in iter_parsed_json_files():
    data = json.loads(json_path.read_text(encoding="utf-8"))
    document = build_document_record(data)
    if document:
        documents.append(document)

with DOCUMENTS_JSONL.open("w", encoding="utf-8") as file:
    for document in documents:
        file.write(json.dumps(document, ensure_ascii=False) + "\n")

chunk_records: list[dict[str, Any]] = []

if CREATE_BASELINE_CHUNKS:
    for document in documents:
        chunk_records.extend(
            structure_aware_chunks(
                document,
                max_chars=max(1600, CHUNK_MAX_CHARS),
            )
        )

with CHUNKS_JSONL.open("w", encoding="utf-8") as file:
    for chunk in chunk_records:
        file.write(json.dumps(chunk, ensure_ascii=False) + "\n")

print("페이지 문서:", len(documents))
print("구조 기반 청크:", len(chunk_records))

if chunk_records:
    chunk_df = pd.DataFrame(chunk_records)
    display(
        chunk_df.groupby("chunk_type")
        .size()
        .rename("count")
        .reset_index()
    )

# 핵심 회귀 검사: 기존 분석에서 누락됐던 내용 확인
by_id = {document["doc_id"]: document for document in documents}
validation_rows: list[dict[str, Any]] = []

if "DP-001" in by_id:
    validation_rows.append(
        {
            "검증": "DP-001 보호한도 핵심 문구",
            "통과": "1억원" in by_id["DP-001"].get("content_text", ""),
        }
    )

if "UN-003" in by_id:
    un003_text = by_id["UN-003"].get("content_text", "")
    validation_rows.append(
        {
            "검증": "UN-003 미수령금 종류",
            "통과": all(
                keyword in un003_text
                for keyword in ["예금보험금", "개산지급금", "파산배당금"]
            ),
        }
    )

for faq_id in ["DP-005", "DP-006", "MT-005", "HP-002"]:
    if faq_id in by_id:
        faq_count = sum(
            1
            for block in by_id[faq_id].get("blocks", [])
            if block.get("type") == "faq"
        )
        validation_rows.append(
            {"검증": f"{faq_id} FAQ 질문-답변", "통과": faq_count > 0}
        )

if validation_rows:
    validation_df = pd.DataFrame(validation_rows)
    display(validation_df)
    failed = validation_df[~validation_df["통과"]]
    if not failed.empty:
        print("주의: 아래 항목은 Markdown 직접 검수가 필요합니다.")
        display(failed)

print("documents.jsonl:", DOCUMENTS_JSONL)
print("chunks.jsonl:", CHUNKS_JSONL)

## 13. 결과 ZIP 생성 및 다운로드

In [ ]:

archive_path = Path(
    shutil.make_archive(
        base_name=str(OUTPUT_ROOT),
        format="zip",
        root_dir=OUTPUT_ROOT.parent,
        base_dir=OUTPUT_ROOT.name,
    )
)

print("결과 ZIP:", archive_path)
print("ZIP 크기:", f"{archive_path.stat().st_size / 1024 / 1024:.2f} MB")

try:
    from google.colab import files
    files.download(str(archive_path))
except Exception as error:
    print("자동 다운로드를 시작하지 못했습니다.")
    print("Colab 왼쪽 파일 패널에서 직접 다운로드하세요:", archive_path)
    print("오류:", error)



## 결과 폴더 구조

```text
kdic_crawl_output_<실행시각>/
├── manifest/
│   └── manifest_used.csv
├── raw/
│   ├── html/
│   ├── assets/
│   └── response_meta/
├── parsed/
│   ├── markdown/
│   └── json/
├── diagnostics/
│   ├── dynamic/
│   └── screenshots/
├── processed/
│   ├── documents.jsonl
│   └── chunks.jsonl
└── logs/
    ├── crawl_results.csv
    └── crawl_results.jsonl
```

## 권장 실행 순서

1. `MAX_URLS = 3`으로 테스트
2. 실패·빈 본문·자산 오류 확인
3. 사이트별 본문 선택자와 제거 선택자 조정
4. `MAX_URLS = None`으로 전체 실행
5. `Review_Queue` 결정 후 Manifest 수정본을 업로드하여 재실행
6. 동적 조회 페이지는 진단 JSON을 검토한 뒤 공개 HTTP 요청 전용 수집기를 별도로 작성

## BI-004만 먼저 재검증하는 방법

설정 셀:

```python
RUN_ONLY_URL_IDS = ["BI-004"]
MAX_URLS = None
```

정상 결과:

```text
status = dynamic_diagnostic_created
status_code = 200
```

생성되어야 하는 파일:

```text
parsed/markdown/예금보험금 안내/BI-004.md
parsed/json/예금보험금 안내/BI-004.json
diagnostics/dynamic/BI-004.json
raw/html/예금보험금 안내/BI-004.html
raw/response_meta/예금보험금 안내/BI-004.json
```

전체를 다시 실행할 때:

```python
RUN_ONLY_URL_IDS = []
MAX_URLS = None
```

## FIXED 버전에서 추가로 확인할 파일

```text
logs/quality_report.csv
```

이 파일은 HTTP 성공 여부와 별개로 다음을 보여줍니다.

- 본문 보존율
- FAQ 질문·답변 개수
- 표 개수
- 첨부파일 버튼 인식 개수
- 공통 UI 문구 잔존 여부
- 자동 판정 `pass`, `review`, `fail`

첨부파일 버튼은 `parsed/json/*/*.json`의 `attachments`에서 확인할 수 있습니다.
`DOWNLOAD_JS_ATTACHMENTS_WITH_PLAYWRIGHT = False`인 경우 실제 파일은 내려받지 않고 다운로드 메타데이터만 저장합니다.

## BI-004 전체 페이지 수집 결과 확인

기본 설정은 다음과 같습니다.

```python
FETCH_BI004_ALL_PAGES = True
```

정상 실행 시 `crawl_results.csv`의 BI-004 행:

```text
status = dynamic_full_collection_created
dynamic_is_complete = True
dynamic_page_count = 실제 전체 페이지 수
dynamic_row_count = 중복 제거 후 전체 금융회사 수
dynamic_failure_count = 0
```

페이지별 원본:

```text
diagnostics/dynamic/BI-004_pages/
├── page_001.html
├── page_002.html
└── ...
```

통합 목록:

```text
diagnostics/dynamic/BI-004_all_rows.csv
```

최종 JSON의 `dynamic_collection.is_complete`가 `true`일 때만
BI-004를 전체 공개 목록으로 사용합니다.


# 예금보험공사 RAG 챗봇 데이터 수집 파이프라인

이 노트북은 예금보험공사·금융안심포털의 **42개 URL Manifest**를 기준으로 다음 작업을 수행합니다.

1. Manifest 검증 및 실행 대상 선택
2. 정적 HTML 수집
3. 이미지·첨부파일·링크 메타데이터 추출
4. 동적 조회 페이지의 초기 HTML·폼·스크립트 구조 진단
5. 결정론적 HTML → Markdown/JSON 변환
6. 응답 메타데이터·해시·실행 로그 저장
7. 페이지 문서와 기본 청크 JSONL 생성
8. 결과 전체 ZIP 다운로드

## 수집 정책

- `include_full`: 공개 본문 전체 수집
- `include_partial`: 공개 페이지의 초기 구조만 진단하며, 검색 조건을 임의 생성하지 않음
- `link_only`: 로그인·본인인증·신청·자가진단 결과를 수집하지 않고 서비스 설명과 공식 URL만 저장
- `review`: 사람의 최종 결정 전에는 실행하지 않음
- `exclude`: 요청 자체를 보내지 않음

> 이 노트북은 로그인 우회, 본인인증 자동화, 개인정보 조회, 신청 제출을 수행하지 않습니다.  
> 기본 요청 간격과 낮은 동시성으로 실행되며, `robots.txt` 정책을 확인합니다.

## v2 수정 사항

이번 버전에서는 `BI-004` 동적 페이지 처리 중 발생한 아래 오류를 수정했습니다.

```text
missing ), unterminated subpattern at position 0
```

원인은 동적 스크립트 탐지용 정규표현식에 역슬래시가 중복 입력된 것이었습니다.

수정 전:

```python
r"(ajax|fetch\\s*\\(|XMLHttpRequest|\\.do\\b|pageIndex|pageNo|search)"
```

수정 후:

```python
r"(?:ajax|fetch\s*\(|XMLHttpRequest|\.do\b|pageIndex|pageNo|search)"
```

추가 기능:

- `RUN_ONLY_URL_IDS` 특정 URL 단독 실행
- 동적 페이지 정규표현식 자체 테스트
- `BI-004` 진단 함수 합성 HTML 테스트
- 존재하지 않는 `url_id` 입력 검증

## FIXED 버전의 수정 범위

이 노트북은 기존 실행 순서를 변경하지 않았습니다.

```text
설정 → Manifest → HTTP 수집 → 원본 저장 → 파싱 → 자산 처리
→ 동적 페이지 진단 → 로그 → 검수 → 문서/청크 생성 → ZIP
```

기존 결과 분석에서 확인된 다음 문제만 수정했습니다.

- `div`, `span`, `dd` 직접 텍스트 누락
- FAQ의 질문 `dt` 누락
- 제목이 `퀵메뉴`로 저장되는 문제
- 공통 UI 문구 혼입
- `rowspan`, `colspan` 표 구조 붕괴
- `gfn_downloadFile()` 첨부파일 버튼 미인식
- 문자 수만 기준으로 FAQ·표가 잘리는 청킹
- BI-004가 첫 페이지만 수집됐다는 사실이 메타데이터에 남지 않는 문제

In [ ]:

# Colab 런타임에 필요한 라이브러리 설치
%pip -q install "requests>=2.32,<3" "beautifulsoup4>=4.12,<5" "lxml>=5,<7" "pandas>=2.2,<4" "tqdm>=4.66,<5"



## 1. 실행 환경과 설정

기본 설정은 정적 페이지·자산 페이지·링크 전용 레코드를 처리합니다.

- 특정 업무만 실행하려면 `SELECT_BUSINESS_FUNCTIONS`에 업무명을 입력합니다.
- 테스트 실행은 `MAX_URLS = 3`처럼 제한합니다.
- 동적 공개 페이지를 브라우저로 렌더링한 초기 화면까지 저장하려면 `ENABLE_PLAYWRIGHT_SNAPSHOT = True`로 바꿉니다.
- Playwright도 검색 폼 제출이나 로그인은 수행하지 않습니다.


In [ ]:

from __future__ import annotations

import hashlib
import json
import mimetypes
import os
import re
import shutil
import subprocess
import sys
import time
from dataclasses import dataclass, asdict
from datetime import datetime, timezone, timedelta
from pathlib import Path
from typing import Any, Iterable
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup, Tag
from IPython.display import display
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

KST = timezone(timedelta(hours=9))

# -------------------------
# 사용자 실행 설정
# -------------------------
SELECT_BUSINESS_FUNCTIONS: list[str] = []
# 예: ["예금자보호제도", "예금보험금 안내"]

RUN_ONLY_URL_IDS: list[str] = []
# BI-004 오류 수정만 먼저 검증하려면:
# RUN_ONLY_URL_IDS = ["BI-004"]
# 전체 실행하려면 빈 리스트 []를 사용합니다.

RUN_DECISIONS = {"include_full", "include_partial", "link_only"}
RUN_WAVES = {"W1_STATIC", "W1_ASSET", "W2_DYNAMIC", "META"}

MAX_URLS: int | None = None
# 최초 테스트 권장: 3
# 전체 실행: None

REQUEST_DELAY_SECONDS = 1.2
REQUEST_TIMEOUT_SECONDS = 25
MAX_RETRIES = 3
MAX_ASSET_BYTES = 25 * 1024 * 1024

RESPECT_ROBOTS_TXT = True
DOWNLOAD_ATTACHMENTS = True
DOWNLOAD_IMAGES = True
DOWNLOAD_VIDEOS = False

ENABLE_PLAYWRIGHT_SNAPSHOT = False
PLAYWRIGHT_WAIT_MS = 1500

# JavaScript gfn_downloadFile(...) 버튼을 실제 파일로 내려받을지 여부
# 처음에는 False로 파싱 품질부터 검증하고, 이후 True로 시험하는 것을 권장합니다.
DOWNLOAD_JS_ATTACHMENTS_WITH_PLAYWRIGHT = False

# BI-004 지급대상 금융회사 공개 목록의 전체 페이지를 POST로 수집할지 여부
# False이면 첫 화면만 저장하고 is_complete=False를 메타데이터에 기록합니다.
FETCH_BI004_ALL_PAGES = False

CREATE_BASELINE_CHUNKS = True
CHUNK_MAX_CHARS = 1200
CHUNK_OVERLAP_CHARS = 150

USER_AGENT = (
    "KDIC-RAG-Academic-Crawler/0.1 "
    "(low-rate public-document collection; no authentication automation)"
)

RUN_ID = datetime.now(KST).strftime("%Y%m%d_%H%M%S")
RUNTIME_ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
OUTPUT_ROOT = RUNTIME_ROOT / f"kdic_crawl_output_{RUN_ID}"

PATHS = {
    "raw_html": OUTPUT_ROOT / "raw" / "html",
    "raw_assets": OUTPUT_ROOT / "raw" / "assets",
    "response_meta": OUTPUT_ROOT / "raw" / "response_meta",
    "parsed_markdown": OUTPUT_ROOT / "parsed" / "markdown",
    "parsed_json": OUTPUT_ROOT / "parsed" / "json",
    "dynamic_diagnostics": OUTPUT_ROOT / "diagnostics" / "dynamic",
    "screenshots": OUTPUT_ROOT / "diagnostics" / "screenshots",
    "logs": OUTPUT_ROOT / "logs",
    "processed": OUTPUT_ROOT / "processed",
    "manifest": OUTPUT_ROOT / "manifest",
}

for path in PATHS.values():
    path.mkdir(parents=True, exist_ok=True)

print("실행 ID:", RUN_ID)
print("결과 경로:", OUTPUT_ROOT)


## 2. 42개 URL Manifest 불러오기

In [ ]:

# 현재 프로젝트의 42개 Manifest를 노트북 내부에 압축하여 포함했습니다.
# 별도 CSV 업로드 없이 기본 Manifest를 자동 생성합니다.

import base64
import gzip

EMBEDDED_MANIFEST_GZIP_B64 = """H4sIAHmDVGoC/+1dbW8TWbL+PtL8h1akXSXaXhzbCS/3w1xlgJmJBmaZwOzofrIcuxO8OG3T3Q6T/TAy0EGGZJawNwYH7FznEkiCgtYJDnF2g66Un+Pu1v6FW3XqdLvbbjs2hLCajYRiu19On656TtVTdeoc/vmP/0to0lRETokZJRlJxMXxjJqQJVWNTGTkmJZIyaIWVSYlLdJ6Ih2dlCJaQktKoprKKDEpAm2IKjQYkaNTkphSEpMJOZqMpJUEfNdmRDmlTEWTiT9L8UhciiVUbMX+ElGkqOq0OpOWxAlJi12PqJoS1aTJGTihqJICjaUmEvDIaEa7DvfczCQUKS5GVRX6mE4lE7EZMZaSNUnWImosBc0o0nRCutW4NKZEbyUjt6LTEv+qalEto4rST2kppkHXUhktndFEWYLvvFNOi5mpqaiCL6JJ6uefBcULV34/OBgUzUKuXsuZywvGm6pVqJnlovFQb3OUflh59uO6pqXV/wgEbt26depGPBE7lVJO3VACajoQT8Obw8tqgSvw5+qMquHnpSlNDahSEnp6NabIp+Ip/hRs9Umh/mbHvLMpWnndzBXEhBxLZuISaCyZFM0ns8ZmVTDXcsZmzSwV4Ntts/xIqO/u17cqxkpRgDvNpxtCvZI1HryA/gnY2lxJMPM5405VsP5SMEtVcy0rosASsQhqSkSxSqqmRq5rU0kRXyHCT08HQUzwTwZNQgci46k4Ck78MRi5em3k2uh5lGIE5AgyF81nIKSa8M21y5eE3wn1nU1zpSLAO5krVfioQZfhsLGxaN3NGr9UoB+WXsE3N+8VQ0PCD2OXxKBZqhlze0JDHLaQhaD5eAceUH+zL5jVHHygVvIrgrGn03UCnDX3lgRTXzVezYqffxYixYZ6Uqz/0V40fKLZZs3WaxXoclAw5rPWo6KVf2282kA91itFY3ZBMEv75l7BR1DwqgL9Np7PGy9A8eXbxst3Vh4e+mjBfJIz5lbxGlRYccd6Og/34EPMuTJHkFnSQZDZeu2+lS8YDxYRI8bLzQZGwoSR8HsMfnzQ3dvCF97Hf4FvZT5d7AoyhBS3abiamVJOUOPYA7cdELjAPeI2/3fWeFHoN0tZ6/H9g10SGJw42LVym6i3tVX4B2KC74Lx31sDgnGnYD6pguqHSPVDH656+G79Nfehqr+mTMJnPHaCgW4xQHJHS4JOgM7QsX7SPhgCC46h3lEw8HTjrY7myFzNNmzAMAFhuCcg9NXflOuVkqnvQ4dB9V+NfA9/jb/VQD9GeQ/usO7tWPkN6IK5/bofVQDv9gDe4K0OiMWuwtPNvdeiYM6tQqMAzCpoCZDs8X4DfQ6cJhKy6sZTbCowPm4Tia+iN79TolPaSDo5g+DhEsrnzLlN6+Gmpdc+NngmojebQIMdjsDhT+Br+t3cgeyCPWwHWl1PXyvfEOE9dTAU4HOy5ts8GRhUnnmnQlcLdDnor3AbDUw5D1IQrGe643tq5luQXzlnVtbRe4G3Mp5vIlL6AHWnCXWnP5ydYIs9MRTAyuEkBaBirj06RrQwE/Np0MJI5fN9FCdwhHplCXporM0LocHQsDGrC5yCom15/ALNiWC3YD4BED25Z/wCstgpMuJxZ9NYyTr6rol2I+fMZ4t+xNbdKjWBtsTdTNUszQNizhBiznwIYuiZ/UChzKXaQDvUxBOB8T+nx5Oy46fGY8oV/G2HL3+YVtQ2wJF+YohxPulZaKLrOwvYEeBygJTTMARBsIAO8OIMVy5AGXM7xpssxxVSPDOvG3MFQMyEpEgyRKmx1BTEkgmM6+wHqTcSaQc5/I949dvRK+xMGgLB8Rk7uiR0AynEMWvmH6CEfXTq6btAnRfByNd3Xh/smn9fADIBWoavpX0AG3wWbrNTxsNZNC34Qi0KZzplDeL4AEIMqj0rfjnqikG5PMHG0Ljp7TD4IfI65lyR+bw93dSL4HS6MhIX0urU6Dj3JFdSqnohNiWfBDUtpiN0KmiLmQlXaBfXQFPgCgAT81kIPegr0AR9C4wCZ6V2QyUIYotlq5A33ugezdkK//yzc4SU0IcihT2QOycEjO95Ov0esFFSsU+OGOhWDFNd7w+ZkatXL147SsSEmuTeHjI8jAGSKRgQVNwtGW+zSDQpmIVXhjNgblhr8ElH9+bhCxMaNe8w3OAgoSb8YaiBFzKWs//ZGx4u/pROyifmw595hHrJidRy/fCiXNe13MEuhpvARu0DA6T6PPtwVB8k1Q8dlWtxHWIP9onK3yMA/hKQ8nViWsIg+CtZ1dqgBEYBUnDmvIX623lz+YUDGaADWiIKqFmpQDcE+KhvVQ92HQwY+ip+mQXhPF1k1lZnfF4vIXCMv64ieJCU1LcroAnkf/Q4gVPh+IwcnQKUJFOpG5m08xNfNqJdl+RIOhmduaUkJq9rhCrvDW50AQCVmYgiqZmkpoozkir+GIpc+K/vRi57McZ7ZGwvmkVdIMQFqH8+4OprRpe5lPXjtaJgzYOeKm6dMY+zDaq8z2/mgWgzI6IotU/0b6EFNW6ImE8g/M46T2FjFhEaEn/4jpEfCqk9QTQ3mJ1OeY7xoJtBIMCN7RcC3e2+sKd4iQXWo/JNdG3s+9eZ+IlJa2PSwi0mDeH3aoOZK7ePc2mjYbgYYks1aNfQd4EoVTbwoVyR+Gs73+BCwTAhJ3QEyHFna5AP3ZtH64q9YYc8j3b4EqNvHNxl3VpatJVKPWyXuMlEA1GA0iSi6AKiDa3eNSn5fUb1g9bxp3LaQ4uldI4fWmGgTl4suOBVLMOL4euQDgT79bd0Q18QMcsGvyCcrG+/EzAk4udJpSx76MZGvZo1yuu8LdAshN9cQJ5gPDhE2At/Aux1jy6yXNqkgqZrVFPil7Xr8ROIdWu9ACMPAXvBg92ww925NtpSd4+u2ewE02Rh3cCEAMsRrFqLOv1qtmwOaxsmdA19PJ8I0Dbv/YJ9IOWjApcLdFVP3vEbJS19k1DRQXaVVfzVO8RmGhZuk14UXTqwvQhx164YGFKutRK4oH5rcd9paUCwDZ7Hv+qNZ0Ev+XPsOVG9iImGB6sIvNPi5WuUiapsAFLNeyss1V0pWEsFG129ncKY9NkCT1UD7lxXuC8Q6AoIMDug78aUorI/aMu6ZWD9mKDH+RZbQwNeCJIimbTtrCDG3XQXHv5h7BIMW85iaIw2Kx1iioqpl8XpRFxK+UAwwmpbVB8k0h1TkhaNR7WoiHCEQCKj0h2NEx8lJUFAGzosCm2nVK4zNmx31/EYmDqaBHXZsjMEqdCngNQh+SsPnk6yVn4QGToVFHyE6nKA0AA0ygoxtirM9DxcAuSI3F9S4gpjxFwBGT02RakqYdiYW69j2NicrzpLkAl/Asi4nHy3wEFyhcTqJCRsBU+XtOmQmI/CyEdon7ddyfDgOcLJ0HvixHcOvx00+D3dzsZfVuPK19PjNzpMyLfMlBV0c3kTnTOWtlCaBxw/+GcnA1SGuKCGmICzEDEbC+tsJvWop8Bw1B/sovaaFcbYA9MZsoeNClYvMPjqNRjpLBB7jfioV/IikRqq04Pwv1apV3TmSOEdENPFMk1+tHUvoBBP0BUaJH0PH6W+r/3hihAc7Fqv11Lp4OC/c4XFId5W6J1F2CH3TtGYW7XTt0xeoJj6VpUlbPPvEABBAsDpI3AMni54hz0ODeAzwMu/YOmFUoGA3RYkCSUQjccVVQqMsI8RTZMva5pyEmZ3Qk9DzNhbEHK5CEbiYBcJQu1+S6KnuWZnecFc07l+4T4Qg+hpiPsOOrhdxaflsJJUoPYd1sFkhiU5oRCh68xHQJeT06FOOckc9yxvI8PjJ6SO4LMTPuh1CILO1H0v6Z5+c3sdCdv8PEi5KUzqrC1nZlMvAhAx4LTy6/V/FOzJk6imRWPXp7D+3i86cs6qBEzX5QTOeOqWnExF455LXcGS6+hHocEdEOqZqG9PcBr3w6VYMuIBpbtF5vDRMTJZotOH4Y34DBM+z/5L4bMHZKaVo0elT+UEhLIs58bu+vUi0quEZp5dQL7YHTRbRdjvV7QCqMBMFRZRu9tFhgYSZ24JGOnLTUTqECH13DEi1RvneQL/jrjUFBV5+tW0ovlkAI7fZXfIAHTps48YaJ1Jnu1Fc0WAA4/rPcCyK1eWgcOtdAj/2QKPvy8YL1+zyx+sWvfKgCZrdtEAWRb3mRCdtT3DDGDA24/RFDKaCkeM8h6bGajvFgHxXaNsTEpeit5KjU1mkoeDrI+bPnocSMC6vSlyOGEENVdkc1nm20VWHlnWITAGaf3Wjup04MwDfX6JTpay4S/BmmWCd2c+6bCdk0ZA6yxzDYYBsyf17SyoRrBuL2BWGuyBsVISk9JkNIkVDbSIMCHHpZ+aTGoyId/gxpSuZhdxIONJXFGYimWYzYzAnRnJZU/xgk+QDeWlqCzspSGNyftXG2wEU9xCwoQAl8EBmzfuV5sLb+0kRjvc0c1Y0lW0Ht+HaJotZYPXwVSIQwAoOx8MHiPqsTx5acPUy9bdUtdQv6pltPPXY93DnNIHNvBWivVd9D8EY9HG6do8t5bNwKY+YjUP0n4yEi35fKdVgZq1faQz/cEg7jwJwYC4jsrxhDwZUVIZTVLaI5rOczDD9/FURo7TBX7J/WNAcxM/AGj5hl5caNQ6z9+3idrh8kCb+I1AguZgsWKU9vkgYMkb1r491bSmm8uzWCixgfm7EE0NBEPHShMwvciTiZ1n0ptR/UdVG4ulT0o02qR6h9qWaBCkAIHP90EvzlS5SxOcAzDa/rSKU29zRZYtRCKK6/z4nXbmN0QzBMHwR04E9ZQCmmIhDo90TlJA7xm4IGjYG9NhRkK8WeNWK9Xn0aFf3pkXomH2nJ0VcQ0GCNB4wdYKMy7VSATRvEJw6DjQ5TP9hB6rVnHWcVCPefmktZQ/JOTmMCTc4fTD98mJ2Ghc1mbG1KTWOdJu8bSMmEVScnJGBC7D6o91oO12kSwaeAgR/+c+uICSqW81phPQ5XJIEq/ivpUq/LCEQUBFl2+zhxHDScjgQ6MxLTEtRaD7E5HYdSl2Q/Q4S+qL91Bnr+tzO1x9+eK1EQ+mHRrwahbQ6+ki+10zX5bsGtbf2a+GC1Qb9R5ElRkhB74I5wCeIKm7dvTCCSRcDv4SF/8wR4hl9RsVex0oXMhWhfnPmhHe+8QztHqKxgpcvV0F0bKF8T6TmZ7yW7cWcQE8TakEhz/ZVCsNxx6mWrFasotKIzbYP3BuzW07j2qOrbk8aKhtlTYa1u0q4o9smb2alZLdh5UH8eQk8w+eBvqHofWf+X4aA406Q2q2aYLO8bjhoHhhhCqDtnRcrbeCQnKWCxx6bHkTM5+OGXSd67zHhjIeSI4nteifprTAJfgyAl8wV+N872rLDXonoT9sbOf4wwV6Oo6qsz+fG8SaPByjMO5xOcObOzDOWETA1gAMOOr/duy35z3db7haxjbBGvJVA6uPjDc7xkLBfFzlISU3EBx3HKEfA2NNEBs+rMDHoylKB4KZ2MaibXf+14EKSCXHzFwzSEIEktBRggSG4hu7ZE345/49gapym+bje4DNqDyRGpVvXlBjqowGpeu6jYYvpPwoOQW09RRSsiJk1k8YRbx02HYraOX/lmXmhdwhhalMGeRLXL63nMWVLbg7kiQDOKK4o5EqKdOJmNSVN8QVJQw2/KZIXFJjSiKNWz59BA/owG0Y6z1odBFPEVo8GBUHsSU3rGVaJLtu18Pyom23eMl5ti44Odh1jUXeL5bLclqjPti64IABlIYJpeH3QmmHRdOKlMSdswK4XPpbRY4B2KIAxJOyoKYIYBirS4ZPhfEPc38ECqfERBeYZp03YBsB/bLIalj1mrWcswljn51idlkj0bvmGRpdQJJVXO/HAoP5rHBmEMz9bwZEATqHwUE1a6/HflRslJtgHBAeIqAMHY05g74+3QDsglsQ6Giv1uu8IsXHYtPKDP44HFf28jaWRaMt1ER3L4D2QCdYuUXZXprO1uU3FuTzlfjEkZBmYLyQX7fyZXQ3xJS49ROj8T9FY2w+jTxTWuK5M3oy38ONH3XA1nQVmq6xi38cvfijc8bZZs6aX2CxBxsCPENHO27QikBXiATjtJloDbclWv4ysQqPzPw7bpK6YFstLbCZMSy+L/rpnsvc5TyHCW3DR4I2speI/K3ZXmF2RZW/VN4PXphqw3R2pYj5W9aHg13qBQtUWt/iV44tEgibfiYouEVjT309WOwEro5N2NL1lrMgnE4TnE4fCZz4I8HxuqYBuofT2ORRwIk9/QRGbgwwkXwYjNxN9Id/ZpvVgC5QfpjWLwy4MHWGMHWme0xRMgCiiVoO58te4hZ5WCm6pQO3w3maZwsBzwokRvc7k3s1Df8UbSIjxwMIsK/k+AVpXLuQlCeRz3e3PJdPNjUINCW5qFOYFqgUG1ViLOXI6RUPrLffYaoG+SZ7Uz7D0cR53Gw+nU4ilUcK3guX75DZOh5233covW+7wMkWmb9QD8FrK+O3x/3yAqMmXhRhXqQZRs7zHOp/lgB8tmsA+9eIuyLTpiVzh9UPuwLQNqlYX9bmiZCag2K7TvrhrPHynW17+Lw9K8u5Ddac20NuoHjzWFL8L2MDO8HLDhazHt7WkAqvGu6Gq1nFKqZDHlYEO353LGJnMfOZxJLuHfK4/Rog65z4Dd9HuJQ15u6DxJmDnysDgvyPtTONc6vIlpZ1nqy0t5mkU1R0gCvP3W1SkxRkwBeCaw/m87wcm7oQSyonOxu0Lww53bqzwZKO68QxHqVEGoxgmnkSUCflPG01aQee4UGec6VMfhWeB/dhKaiz+MV8WrH+UnMvggFtNu97MDRIUAt1DTV/I9aCn66NGMLlipLWEDL/zmshHH2D5lj9FTk20N+ebsA7QDvLL9oWXrrQwvfwQbTY1as0on0bpJ2uWbkFIBKkRCZoKEi4CLfBBZtrZ0Fnv/E2B5DCSiOgzOz8wAfbo9Y2m+wRL8rowSyNTqsT0eQPspIcTU4muzdQfbj3IlKvVjmwdBOcfVFzVq/aiVkcnWvsW1+fz51wECWuP0c3aqwtoFDZlZ3l2teHLfLaOetusb432+e49Y79czfWwIO3z7xdTN6gNanxJzQ22/GNfmz++cldf4sJosLtah1eC9kh39ujYNwpiIL1dIUV2hmVCqCRlcjz1+d2kubJcV8n3HqT+CXTDhdbn+jdKtwLYDRIfo/w1QIWreCQC9GQG+raFPsMKHNNh+atgo5FbJxr9FNwZhMQJj2bs2OytNB2887pVCyAGzTACAmcjyqSpIxJkwlVG2H/4UMgCV99wyG2XwAWxtuTUJ4e0ENZ2atfd/lMNqXqx0a+tm0nV7utmaq9kWcsJatO5f3Rzay2ypvHaVQysoTM0E8BuIPva75NGE8IMpfJNx6dFzpoyt6soYp1b5y30njrO0zZjEraWzGw0sWi3gjMPbqnBSOO9gWnhu7/AYEZ+Ef+YwAA"""

MANIFEST_PATH = RUNTIME_ROOT / "kdic_crawl_manifest_42.csv"
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH.write_bytes(
    gzip.decompress(base64.b64decode(EMBEDDED_MANIFEST_GZIP_B64))
)

print("Manifest 생성 완료:", MANIFEST_PATH)
print("크기:", MANIFEST_PATH.stat().st_size, "bytes")


In [ ]:

# 필요하면 사용자 수정본 CSV로 교체할 수 있습니다.
USE_CUSTOM_MANIFEST_UPLOAD = False

if USE_CUSTOM_MANIFEST_UPLOAD:
    from google.colab import files

    uploaded = files.upload()
    csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
    if len(csv_names) != 1:
        raise ValueError("CSV 파일을 정확히 1개 업로드해야 합니다.")

    MANIFEST_PATH = Path("/content") / csv_names[0]
    MANIFEST_PATH.write_bytes(uploaded[csv_names[0]])
    print("사용자 Manifest로 교체:", MANIFEST_PATH)


In [ ]:

REQUIRED_COLUMNS = {
    "url_id",
    "business_function",
    "page_title",
    "source_url",
    "site_name",
    "normalized_decision",
    "decision_reason",
    "page_type",
    "fetch_strategy",
    "parser_profile",
    "auth_required",
    "asset_policy",
    "content_scope",
    "crawl_wave",
}

manifest_df = pd.read_csv(MANIFEST_PATH, encoding="utf-8-sig", dtype=str).fillna("")

missing_columns = sorted(REQUIRED_COLUMNS - set(manifest_df.columns))
if missing_columns:
    raise ValueError(f"Manifest 필수 열이 없습니다: {missing_columns}")

if manifest_df["url_id"].duplicated().any():
    duplicates = manifest_df.loc[
        manifest_df["url_id"].duplicated(keep=False), ["url_id", "source_url"]
    ]
    raise ValueError(f"url_id 중복:\n{duplicates}")

if manifest_df["source_url"].duplicated().any():
    duplicates = manifest_df.loc[
        manifest_df["source_url"].duplicated(keep=False), ["url_id", "source_url"]
    ]
    raise ValueError(f"source_url 중복:\n{duplicates}")

invalid_urls = manifest_df[
    ~manifest_df["source_url"].str.match(r"^https?://", na=False)
]
if not invalid_urls.empty:
    raise ValueError(
        "HTTP/HTTPS가 아닌 URL이 있습니다:\n"
        + invalid_urls[["url_id", "source_url"]].to_string(index=False)
    )

# 실행 대상 필터
target_df = manifest_df.copy()

if SELECT_BUSINESS_FUNCTIONS:
    target_df = target_df[
        target_df["business_function"].isin(SELECT_BUSINESS_FUNCTIONS)
    ]

if RUN_ONLY_URL_IDS:
    known_url_ids = set(manifest_df["url_id"])
    unknown_url_ids = sorted(set(RUN_ONLY_URL_IDS) - known_url_ids)

    if unknown_url_ids:
        raise ValueError(
            f"Manifest에 존재하지 않는 url_id입니다: {unknown_url_ids}"
        )

    target_df = target_df[
        target_df["url_id"].isin(RUN_ONLY_URL_IDS)
    ]

target_df = target_df[
    target_df["normalized_decision"].isin(RUN_DECISIONS)
    & target_df["crawl_wave"].isin(RUN_WAVES)
].copy()

if MAX_URLS is not None:
    target_df = target_df.head(MAX_URLS).copy()

print("전체 Manifest:", len(manifest_df))
print("이번 실행 대상:", len(target_df))

display(
    manifest_df.groupby(
        ["business_function", "normalized_decision"],
        dropna=False,
    ).size().rename("count").reset_index()
)

display(
    target_df[
        [
            "url_id",
            "business_function",
            "page_title",
            "normalized_decision",
            "crawl_wave",
            "fetch_strategy",
        ]
    ].head(20)
)


## 3. 공통 유틸리티

In [ ]:

ATTACHMENT_EXTENSIONS = {
    ".pdf", ".hwp", ".hwpx", ".doc", ".docx",
    ".xls", ".xlsx", ".ppt", ".pptx", ".zip",
    ".csv", ".txt",
}

IMAGE_EXTENSIONS = {
    ".png", ".jpg", ".jpeg", ".gif", ".webp", ".svg", ".bmp",
}

VIDEO_EXTENSIONS = {
    ".mp4", ".webm", ".mov", ".avi", ".mkv",
}

NOISE_SELECTORS = [
    "script", "style", "noscript", "template",
    "header", "footer", "nav", "aside",
    ".skip", ".skip-navigation", ".skipnav",
    ".gnb", ".lnb", ".snb", ".breadcrumb-wrap",
    ".footer", ".header", ".quick-menu", ".quick",
    ".util", ".utility", ".pagination",
    "[aria-hidden='true']",
]

MAIN_SELECTORS_BY_SITE = {
    "예금보험공사": [
        "#contents", "#content", ".contents", ".content",
        ".sub-content", ".sub_contents", "main", "article",
    ],
    "금융안심포털": [
        "#contents", "#content", ".contents", ".content",
        ".sub-content", ".sub_contents", "main", "article",
    ],
}

BREADCRUMB_SELECTORS = [
    ".breadcrumb", ".breadcrumbs", ".location", ".location-wrap",
    ".path", ".sub-location", "nav[aria-label*='breadcrumb' i]",
]


def now_kst_iso() -> str:
    return datetime.now(KST).isoformat(timespec="seconds")


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def sha256_text(text: str) -> str:
    return sha256_bytes(text.encode("utf-8"))


def normalize_space(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


def normalize_multiline(text: str) -> str:
    lines = [normalize_space(line) for line in (text or "").splitlines()]
    cleaned: list[str] = []

    for line in lines:
        if not line:
            if cleaned and cleaned[-1] != "":
                cleaned.append("")
            continue
        cleaned.append(line)

    while cleaned and cleaned[-1] == "":
        cleaned.pop()

    return "\n".join(cleaned)


def safe_name(value: str, max_length: int = 90) -> str:
    value = normalize_space(value)
    value = re.sub(r'[\\/:*?"<>|]+', "_", value)
    value = re.sub(r"_+", "_", value).strip("._ ")
    return (value or "untitled")[:max_length]


def ensure_parent(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)


def write_json(path: Path, data: Any) -> None:
    ensure_parent(path)
    path.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def append_jsonl(path: Path, data: dict[str, Any]) -> None:
    ensure_parent(path)
    with path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(data, ensure_ascii=False) + "\n")


def absolute_url(base_url: str, candidate: str | None) -> str | None:
    if not candidate:
        return None

    candidate = candidate.strip()
    if not candidate or candidate.startswith(("#", "javascript:", "mailto:", "tel:")):
        return None

    return urljoin(base_url, candidate)


def extension_from_url(url: str) -> str:
    path = urlparse(url).path
    return Path(path).suffix.lower()


def classify_asset_url(url: str) -> str:
    extension = extension_from_url(url)

    if extension in ATTACHMENT_EXTENSIONS:
        return "attachment"
    if extension in IMAGE_EXTENSIONS:
        return "image"
    if extension in VIDEO_EXTENSIONS:
        return "video"
    return "link"


def row_to_clean_dict(row: pd.Series) -> dict[str, str]:
    return {
        str(key): "" if pd.isna(value) else str(value)
        for key, value in row.to_dict().items()
    }


## 4. HTTP 세션·재시도·robots.txt 검사

In [ ]:

def create_session() -> requests.Session:
    retry_policy = Retry(
        total=MAX_RETRIES,
        connect=MAX_RETRIES,
        read=MAX_RETRIES,
        status=MAX_RETRIES,
        backoff_factor=0.8,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset({"GET", "HEAD"}),
        respect_retry_after_header=True,
        raise_on_status=False,
    )

    adapter = HTTPAdapter(
        max_retries=retry_policy,
        pool_connections=2,
        pool_maxsize=2,
    )

    session = requests.Session()
    session.headers.update(
        {
            "User-Agent": USER_AGENT,
            "Accept": (
                "text/html,application/xhtml+xml,application/xml;q=0.9,"
                "image/avif,image/webp,*/*;q=0.8"
            ),
            "Accept-Language": "ko-KR,ko;q=0.9,en;q=0.5",
            "Connection": "keep-alive",
        }
    )
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


SESSION = create_session()
ROBOTS_CACHE: dict[str, RobotFileParser | None] = {}


def get_robots_parser(url: str) -> RobotFileParser | None:
    parsed = urlparse(url)
    base = f"{parsed.scheme}://{parsed.netloc}"

    if base in ROBOTS_CACHE:
        return ROBOTS_CACHE[base]

    robots_url = urljoin(base, "/robots.txt")

    try:
        response = SESSION.get(
            robots_url,
            timeout=REQUEST_TIMEOUT_SECONDS,
            allow_redirects=True,
        )

        if response.status_code >= 400:
            ROBOTS_CACHE[base] = None
            return None

        parser = RobotFileParser()
        parser.set_url(robots_url)
        parser.parse(response.text.splitlines())
        ROBOTS_CACHE[base] = parser
        return parser

    except requests.RequestException:
        ROBOTS_CACHE[base] = None
        return None


def robots_allows(url: str) -> tuple[bool, str]:
    if not RESPECT_ROBOTS_TXT:
        return True, "robots_check_disabled"

    parser = get_robots_parser(url)
    if parser is None:
        return True, "robots_unavailable_assumed_allowed"

    allowed = parser.can_fetch(USER_AGENT, url)
    return allowed, "allowed" if allowed else "disallowed"


def choose_response_encoding(response: requests.Response) -> str:
    content_type = response.headers.get("Content-Type", "")
    declared = response.encoding

    # requests의 기본 ISO-8859-1 추정은 한국어 HTML에서 잘못될 수 있습니다.
    if not declared or declared.lower() == "iso-8859-1":
        apparent = response.apparent_encoding
        if apparent:
            return apparent

    return declared or "utf-8"


## 5. HTML 수집과 응답 메타데이터 저장

In [ ]:

@dataclass
class FetchResult:
    requested_url: str
    final_url: str
    status_code: int
    ok: bool
    content_type: str
    encoding: str
    elapsed_seconds: float
    collected_at: str
    content_length: int
    raw_sha256: str
    redirect_history: list[dict[str, Any]]
    selected_headers: dict[str, str]
    body: bytes


def fetch_url(url: str) -> FetchResult:
    allowed, robots_status = robots_allows(url)
    if not allowed:
        raise PermissionError(f"robots.txt에서 수집을 허용하지 않습니다: {url}")

    started = time.perf_counter()

    response = SESSION.get(
        url,
        timeout=REQUEST_TIMEOUT_SECONDS,
        allow_redirects=True,
    )

    elapsed = time.perf_counter() - started
    encoding = choose_response_encoding(response)
    response.encoding = encoding

    selected_header_names = {
        "Content-Type",
        "Content-Length",
        "Last-Modified",
        "ETag",
        "Cache-Control",
        "Content-Disposition",
    }

    selected_headers = {
        key: value
        for key, value in response.headers.items()
        if key in selected_header_names
    }
    selected_headers["Robots-Check"] = robots_status

    return FetchResult(
        requested_url=url,
        final_url=response.url,
        status_code=response.status_code,
        ok=response.ok,
        content_type=response.headers.get("Content-Type", ""),
        encoding=encoding,
        elapsed_seconds=round(elapsed, 3),
        collected_at=now_kst_iso(),
        content_length=len(response.content),
        raw_sha256=sha256_bytes(response.content),
        redirect_history=[
            {
                "status_code": item.status_code,
                "url": item.url,
                "location": item.headers.get("Location"),
            }
            for item in response.history
        ],
        selected_headers=selected_headers,
        body=response.content,
    )


def save_fetch_result(
    manifest_row: dict[str, str],
    result: FetchResult,
) -> tuple[Path, Path]:
    business_dir = safe_name(manifest_row["business_function"])
    url_id = manifest_row["url_id"]

    html_path = PATHS["raw_html"] / business_dir / f"{url_id}.html"
    meta_path = PATHS["response_meta"] / business_dir / f"{url_id}.json"

    ensure_parent(html_path)
    html_path.write_bytes(result.body)

    metadata = asdict(result)
    metadata.pop("body", None)
    metadata["url_id"] = url_id
    metadata["business_function"] = manifest_row["business_function"]
    metadata["page_title_manifest"] = manifest_row["page_title"]
    metadata["fetch_strategy"] = manifest_row["fetch_strategy"]
    metadata["parser_profile"] = manifest_row["parser_profile"]
    metadata["raw_html_path"] = str(html_path.relative_to(OUTPUT_ROOT))

    write_json(meta_path, metadata)
    return html_path, meta_path


## 6. 결정론적 본문·표·링크·이미지 추출

In [ ]:
from bs4 import Comment, NavigableString

# 실제 본문 밖에서 반복되는 UI 요소
FIXED_NOISE_SELECTORS = [
    "script", "style", "noscript", "template",
    ".floatTop", ".floatBottom", ".paging", ".pagination",
    ".sr-only", ".skip", ".skipnav", ".skip-navigation",
    ".loading", ".waitPage", "#waitPage",
]

FIXED_NOISE_TEXTS = {
    "퀵메뉴", "글자 크기", "글자확대", "글자축소",
    "KOR", "ENG", "상단으로 이동", "검색 초기화",
    "좌우로 움직여보세요", "표 더보기",
}

FAQ_PREFIX_RE = re.compile(r"^\s*질문\s*")
ANSWER_PREFIX_RE = re.compile(r"^\s*답변\s*")
OPEN_SUFFIX_RE = re.compile(r"\s*열기\s*$")
DOWNLOAD_CALL_RE = re.compile(
    r"gfn_downloadFile\(\s*'([^']+)'\s*,\s*'([^']+)'\s*\)"
)
MOVE_URL_RE = re.compile(r"gfn_moveUrl\(\s*'([^']+)'\s*\)")

BLOCK_TAGS = {
    "div", "section", "article", "aside", "header", "footer", "main",
    "p", "ul", "ol", "dl", "table", "figure",
    "h1", "h2", "h3", "h4", "h5", "h6",
}


def fixed_norm(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


def fixed_safe_text(node: Tag) -> str:
    """원본 DOM을 훼손하지 않고 노이즈를 제거한 텍스트를 반환합니다."""
    fragment = BeautifulSoup(str(node), "lxml")
    root = fragment.body.find() if fragment.body else fragment.find()
    if not root:
        return ""

    for child in root.select(
        ".sr-only,.hide,script,style,noscript,template,.floatTop,.floatBottom"
    ):
        child.decompose()

    for image in root.find_all("img"):
        image.replace_with(fixed_norm(image.get("alt", "")))

    for br in root.find_all("br"):
        br.replace_with(" ")

    return fixed_norm(root.get_text(" ", strip=True))


def fixed_clean_question(text: str) -> str:
    text = FAQ_PREFIX_RE.sub("", fixed_norm(text))
    text = OPEN_SUFFIX_RE.sub("", text)
    return fixed_norm(text)


def fixed_clean_answer(text: str) -> str:
    return fixed_norm(ANSWER_PREFIX_RE.sub("", fixed_norm(text)))


def fixed_clean_term(text: str) -> str:
    text = OPEN_SUFFIX_RE.sub("", fixed_norm(text))
    text = re.sub(r"^\s*\d{1,2}\s+(?=\D)", "", text)
    return fixed_norm(text)


def fixed_infer_file_format(button: Tag) -> str:
    search_text = (
        " ".join(button.get("class", [])).lower()
        + " "
        + fixed_norm(button.get_text(" ", strip=True)).lower()
    )
    for keyword, file_format in [
        ("icohwp", "HWP"), ("hwp", "HWP"),
        ("icopdf", "PDF"), ("pdf", "PDF"),
        ("xlsx", "XLSX"), ("excel", "XLSX"),
        ("docx", "DOCX"), ("doc", "DOC"),
    ]:
        if keyword in search_text:
            return file_format
    return "FILE"


def fixed_cell_text(cell: Tag) -> str:
    fragment = BeautifulSoup(str(cell), "lxml")
    root = fragment.body.find() if fragment.body else fragment.find()
    if not root:
        return ""

    for child in root.select("script,style,noscript,template,.sr-only"):
        child.decompose()

    for button in root.find_all("button"):
        file_format = fixed_infer_file_format(button)
        label = fixed_norm(button.get_text(" ", strip=True))
        label = re.sub(r"다운로드$", "", label).strip()
        replacement = (
            f"{label} {file_format} 다운로드".strip()
            if label
            else f"{file_format} 다운로드"
        )
        button.replace_with(replacement)

    for image in root.find_all("img"):
        image.replace_with(fixed_norm(image.get("alt", "")))

    for br in root.find_all("br"):
        br.replace_with(" ")

    return fixed_norm(root.get_text(" ", strip=True))


def fixed_expand_table(table: Tag) -> tuple[list[str], list[list[str]]]:
    """rowspan/colspan을 펼쳐 각 행이 같은 열 수를 갖도록 만듭니다."""
    grid: list[list[str]] = []
    active_rowspans: dict[int, list[Any]] = {}
    header_flags: list[bool] = []
    max_columns = 0

    for tr in table.find_all("tr"):
        row: list[str] = []
        column = 0
        cells = tr.find_all(["th", "td"], recursive=False)

        def fill_active() -> None:
            nonlocal column
            while column in active_rowspans:
                remaining, value = active_rowspans[column]
                while len(row) <= column:
                    row.append("")
                row[column] = value
                remaining -= 1
                if remaining <= 0:
                    del active_rowspans[column]
                else:
                    active_rowspans[column] = [remaining, value]
                column += 1

        fill_active()
        header_flags.append(any(cell.name == "th" for cell in cells))

        for cell in cells:
            fill_active()
            text = fixed_cell_text(cell)
            rowspan = max(1, int(cell.get("rowspan", 1) or 1))
            colspan = max(1, int(cell.get("colspan", 1) or 1))

            for offset in range(colspan):
                target_column = column + offset
                while len(row) <= target_column:
                    row.append("")
                row[target_column] = text
                if rowspan > 1:
                    active_rowspans[target_column] = [rowspan - 1, text]

            column += colspan

        if active_rowspans:
            final_active_column = max(active_rowspans)
            while column <= final_active_column:
                if column in active_rowspans:
                    fill_active()
                else:
                    while len(row) <= column:
                        row.append("")
                    column += 1

        max_columns = max(max_columns, len(row))
        if row and any(row):
            grid.append(row)

    if not grid:
        return [], []

    for row in grid:
        row.extend([""] * (max_columns - len(row)))

    if table.thead:
        header_count = len(table.thead.find_all("tr", recursive=False))
    else:
        header_count = 1 if header_flags and header_flags[0] else 0

    if header_count == 0:
        header_count = 1

    header_rows = grid[:header_count]
    rows = grid[header_count:]
    headers: list[str] = []

    for column in range(max_columns):
        values: list[str] = []
        for header_row in header_rows:
            value = fixed_norm(header_row[column])
            if value and value not in values:
                values.append(value)
        headers.append(" / ".join(values) if values else f"열{column + 1}")

    # 중복 헤더 이름을 고유하게 만듭니다.
    counts: dict[str, int] = {}
    unique_headers: list[str] = []
    for raw_header in headers:
        header = raw_header or "열"
        counts[header] = counts.get(header, 0) + 1
        unique_headers.append(
            header if counts[header] == 1 else f"{header} {counts[header]}"
        )

    return unique_headers, rows


def fixed_extract_attachments(root: Tag, base_url: str) -> list[dict[str, Any]]:
    attachments: list[dict[str, Any]] = []
    seen: set[Any] = set()

    # KDIC 사이트의 JavaScript 다운로드 버튼
    for button in root.find_all("button", onclick=True):
        match = DOWNLOAD_CALL_RE.search(button.get("onclick", ""))
        if not match:
            continue

        key = (match.group(1), match.group(2))
        if key in seen:
            continue
        seen.add(key)

        file_format = fixed_infer_file_format(button)
        label = fixed_norm(button.get_text(" ", strip=True))
        label = re.sub(r"다운로드$", "", label).strip()

        row_context = ""
        parent_row = button.find_parent("tr")
        if parent_row:
            values: list[str] = []
            for cell in parent_row.find_all(["th", "td"], recursive=False):
                if button in cell.descendants:
                    continue
                value = fixed_cell_text(cell)
                if value and "다운로드" not in value and value not in values:
                    values.append(value)
            row_context = " | ".join(values[:4])

        display_name = label or row_context or f"{file_format} 첨부파일"
        if file_format not in display_name.upper():
            display_name = f"{display_name} ({file_format})"

        attachment_id = sha256_text(
            f"{match.group(1)}|{match.group(2)}"
        )[:16]

        attachments.append(
            {
                "attachment_id": attachment_id,
                "display_name": display_name,
                "file_format": file_format,
                "download_method": "gfn_downloadFile",
                "token1": match.group(1),
                "token2": match.group(2),
                "row_context": row_context,
                "button_text": fixed_norm(button.get_text(" ", strip=True)),
                "source_url": base_url,
                "download_status": "metadata_only",
            }
        )

    # href에 직접 연결된 첨부파일
    attachment_extensions = {
        ".pdf", ".hwp", ".hwpx", ".doc", ".docx",
        ".xls", ".xlsx", ".ppt", ".pptx", ".zip", ".csv",
    }

    for anchor in root.find_all("a", href=True):
        href = anchor.get("href", "")
        url = absolute_url(base_url, href)
        if not url:
            continue

        extension = Path(urlparse(url).path).suffix.lower()
        if extension not in attachment_extensions:
            continue

        key = ("href", url)
        if key in seen:
            continue
        seen.add(key)

        attachments.append(
            {
                "attachment_id": sha256_text(url)[:16],
                "display_name": (
                    fixed_norm(anchor.get_text(" ", strip=True))
                    or Path(urlparse(url).path).name
                ),
                "file_format": extension.lstrip(".").upper(),
                "download_method": "href",
                "url": url,
                "source_url": base_url,
                "download_status": "metadata_only",
            }
        )

    return attachments


def fixed_extract_links(root: Tag, base_url: str) -> list[dict[str, Any]]:
    links: list[dict[str, Any]] = []
    seen: set[Any] = set()

    for anchor in root.find_all("a", href=True):
        href = anchor.get("href", "").strip()
        if not href or href.startswith(("#", "javascript:", "mailto:", "tel:")):
            continue

        url = urljoin(base_url, href)
        text = fixed_norm(anchor.get_text(" ", strip=True))
        if not text or text in FIXED_NOISE_TEXTS:
            continue

        key = (url, text)
        if key in seen:
            continue
        seen.add(key)
        links.append({"url": url, "anchor_text": text})

    for button in root.find_all("button", onclick=True):
        match = MOVE_URL_RE.search(button.get("onclick", ""))
        if not match:
            continue
        url = urljoin(base_url, match.group(1))
        text = fixed_norm(button.get_text(" ", strip=True))
        key = (url, text)
        if key not in seen:
            seen.add(key)
            links.append(
                {"url": url, "anchor_text": text, "link_type": "button"}
            )

    return links


def fixed_extract_images(root: Tag, base_url: str) -> list[dict[str, Any]]:
    images: list[dict[str, Any]] = []
    seen: set[str] = set()

    for image in root.find_all("img"):
        source = (
            image.get("src")
            or image.get("data-src")
            or image.get("data-original")
        )
        url = absolute_url(base_url, source)
        if not url or url in seen:
            continue
        seen.add(url)

        alt = fixed_norm(image.get("alt", ""))
        filename = Path(urlparse(url).path).name
        lowered = filename.lower()
        image_role = "decorative"

        if "webtoon" in lowered:
            image_role = "supplementary_visual"
        elif any(keyword in lowered for keyword in ["proc", "process", "step", "flow"]):
            image_role = "process_diagram"
        elif alt:
            image_role = "content_image"

        images.append(
            {
                "url": url,
                "alt": alt,
                "filename": filename,
                "image_role": image_role,
            }
        )

    return images


class FixedStructureParser:
    def __init__(self, page_type: str = "static_page"):
        self.blocks: list[dict[str, Any]] = []
        self.page_type = page_type

    def add(self, block: dict[str, Any] | None) -> None:
        if not block:
            return

        if (
            self.blocks
            and block.get("type") in {"paragraph", "heading"}
            and self.blocks[-1].get("type") == block.get("type")
            and self.blocks[-1].get("text") == block.get("text")
        ):
            return

        self.blocks.append(block)

    def parse(self, root: Tag) -> list[dict[str, Any]]:
        faq_dls = root.select(
            ".accoWrap .accodian > dl, .accodian.con.ico > dl"
        )

        if self.page_type == "faq" or faq_dls:
            faq_nodes = {id(node) for node in faq_dls}

            for child in root.find_all(recursive=False):
                if not isinstance(child, Tag):
                    self.process_node(child)
                    continue

                child_dls = child.select("dl")
                if any(id(dl) in faq_nodes for dl in child_dls):
                    continue
                self.process_node(child)

            for dl in faq_dls:
                self.parse_faq(dl)
        else:
            for child in root.find_all(recursive=False):
                self.process_node(child)

        return self.blocks

    def parse_faq(self, dl: Tag) -> None:
        dt = dl.find("dt", recursive=False)
        dd = dl.find("dd", recursive=False)
        if not dt or not dd:
            return

        question = fixed_clean_question(fixed_safe_text(dt))
        answer_parser = FixedStructureParser("static_page")
        answer_parser.process_node(dd)

        if not answer_parser.blocks:
            answer_text = fixed_clean_answer(fixed_safe_text(dd))
            answer_parser.add({"type": "paragraph", "text": answer_text})
        else:
            for block in answer_parser.blocks:
                if "text" in block:
                    block["text"] = fixed_clean_answer(block["text"])
                    break

        answer_text = fixed_blocks_text(answer_parser.blocks)
        self.add(
            {
                "type": "faq",
                "question": question,
                "answer_blocks": answer_parser.blocks,
                "answer_text": answer_text,
            }
        )

    def parse_definition_list(self, dl: Tag) -> None:
        children = [
            child
            for child in dl.find_all(recursive=False)
            if isinstance(child, Tag)
        ]
        index = 0

        while index < len(children):
            if children[index].name != "dt":
                self.process_node(children[index])
                index += 1
                continue

            dt = children[index]
            dd = (
                children[index + 1]
                if index + 1 < len(children)
                and children[index + 1].name == "dd"
                else None
            )
            term = fixed_clean_term(fixed_safe_text(dt))

            if dd:
                definition_parser = FixedStructureParser("static_page")
                definition_parser.process_node(dd)
                if not definition_parser.blocks:
                    definition_parser.add(
                        {"type": "paragraph", "text": fixed_safe_text(dd)}
                    )

                self.add(
                    {
                        "type": "definition",
                        "term": term,
                        "definition_blocks": definition_parser.blocks,
                        "definition_text": fixed_blocks_text(
                            definition_parser.blocks
                        ),
                    }
                )
                index += 2
            else:
                self.add({"type": "heading", "level": 3, "text": term})
                index += 1

    def process_node(self, node: Any) -> None:
        if isinstance(node, Comment):
            return

        if isinstance(node, NavigableString):
            text = fixed_norm(str(node))
            if text and text not in FIXED_NOISE_TEXTS:
                self.add({"type": "paragraph", "text": text})
            return

        if not isinstance(node, Tag):
            return

        name = node.name.lower()
        classes = set(node.get("class", []))

        if (
            name in {"script", "style", "noscript", "template"}
            or classes & {
                "floatTop", "floatBottom", "paging", "pagination", "sr-only"
            }
        ):
            return

        if name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
            text = fixed_safe_text(node)
            if text and text not in FIXED_NOISE_TEXTS:
                self.add(
                    {"type": "heading", "level": int(name[1]), "text": text}
                )
            return

        if name == "p":
            text = fixed_safe_text(node)
            if text and text not in FIXED_NOISE_TEXTS:
                self.add({"type": "paragraph", "text": text})
            return

        if name == "table":
            headers, rows = fixed_expand_table(node)
            if headers or rows:
                self.add(
                    {
                        "type": "table",
                        "headers": headers,
                        "rows": rows,
                        "row_count": len(rows),
                    }
                )
            return

        if name in {"ul", "ol"}:
            items: list[str] = []
            for li in node.find_all("li", recursive=False):
                fragment = BeautifulSoup(str(li), "lxml")
                copied_li = fragment.body.find("li") if fragment.body else None
                if not copied_li:
                    continue
                for nested in copied_li.find_all(["ul", "ol"]):
                    nested.decompose()
                text = fixed_safe_text(copied_li)
                if text:
                    items.append(text)

            if items:
                self.add(
                    {"type": "list", "ordered": name == "ol", "items": items}
                )

            for li in node.find_all("li", recursive=False):
                for nested in li.find_all(["ul", "ol"], recursive=False):
                    self.process_node(nested)
            return

        if name == "dl":
            self.parse_definition_list(node)
            return

        if name == "figure":
            caption = node.find("figcaption")
            if caption:
                text = fixed_safe_text(caption)
                if text:
                    self.add({"type": "paragraph", "text": text})
            return

        # div, span 등이 가진 직접 텍스트와 하위 블록의 순서를 보존합니다.
        inline_parts: list[str] = []

        def flush_inline() -> None:
            nonlocal inline_parts
            text = fixed_norm(" ".join(inline_parts))
            inline_parts = []
            if text and text not in FIXED_NOISE_TEXTS:
                self.add({"type": "paragraph", "text": text})

        for child in node.children:
            if isinstance(child, Comment):
                continue
            if isinstance(child, NavigableString):
                text = fixed_norm(str(child))
                if text:
                    inline_parts.append(text)
            elif isinstance(child, Tag):
                child_name = child.name.lower()
                if child_name == "br":
                    inline_parts.append(" ")
                elif child_name in BLOCK_TAGS:
                    flush_inline()
                    self.process_node(child)
                else:
                    text = fixed_safe_text(child)
                    if text:
                        inline_parts.append(text)

        flush_inline()


def fixed_render_blocks(blocks: list[dict[str, Any]]) -> str:
    lines: list[str] = []

    for block in blocks:
        block_type = block.get("type")

        if block_type == "heading":
            level = min(max(int(block.get("level", 2)), 2), 6)
            lines.append(f"{'#' * level} {block.get('text', '')}")
        elif block_type == "paragraph":
            lines.append(block.get("text", ""))
        elif block_type == "list":
            for index, item in enumerate(block.get("items", []), start=1):
                prefix = f"{index}. " if block.get("ordered") else "- "
                lines.append(prefix + item)
        elif block_type == "definition":
            lines.append("### " + block.get("term", ""))
            lines.append(fixed_render_blocks(block.get("definition_blocks", [])))
        elif block_type == "faq":
            lines.append("### Q. " + block.get("question", ""))
            lines.append(fixed_render_blocks(block.get("answer_blocks", [])))
        elif block_type == "table":
            headers = [
                fixed_norm(header).replace("|", "\\|")
                for header in block.get("headers", [])
            ]
            if headers:
                lines.append("| " + " | ".join(headers) + " |")
                lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
                for row in block.get("rows", []):
                    normalized_row = [
                        fixed_norm(value).replace("|", "\\|")
                        for value in row
                    ]
                    normalized_row.extend(
                        [""] * (len(headers) - len(normalized_row))
                    )
                    lines.append(
                        "| " + " | ".join(normalized_row[: len(headers)]) + " |"
                    )

        lines.append("")

    return re.sub(r"\n{3,}", "\n\n", "\n".join(lines)).strip()


def fixed_blocks_text(blocks: list[dict[str, Any]]) -> str:
    values: list[str] = []

    for block in blocks:
        block_type = block.get("type")
        if block_type in {"heading", "paragraph"}:
            values.append(block.get("text", ""))
        elif block_type == "list":
            values.extend(block.get("items", []))
        elif block_type == "definition":
            values.extend(
                [block.get("term", ""), block.get("definition_text", "")]
            )
        elif block_type == "faq":
            values.extend(
                [block.get("question", ""), block.get("answer_text", "")]
            )
        elif block_type == "table":
            values.extend(block.get("headers", []))
            for row in block.get("rows", []):
                values.extend(row)

    return fixed_norm(" ".join(values))


def fixed_manifest_title(manifest_title: str, business_function: str) -> str:
    # Manifest에서 사람이 확인한 전체 경로명을 사용합니다.
    # HTML 내부의 "퀵메뉴" 같은 잘못된 제목을 선택하지 않습니다.
    return fixed_norm(manifest_title) or business_function


def fixed_manifest_breadcrumb(
    manifest_title: str,
    business_function: str,
) -> list[str]:
    parts = [
        fixed_norm(part)
        for part in re.split(r"\s*>\s*", manifest_title or "")
        if fixed_norm(part)
    ]
    result: list[str] = []
    for value in [business_function] + parts:
        if value and value not in result:
            result.append(value)
    return result


def parse_html_document(
    html_bytes: bytes,
    final_url: str,
    manifest_row: dict[str, str],
    encoding: str,
) -> dict[str, Any]:
    html_text = html_bytes.decode(encoding, errors="replace")
    soup = BeautifulSoup(html_text, "lxml")

    # 실제 KDIC 본문 컨테이너를 우선합니다.
    root = (
        soup.select_one(".contents")
        or soup.select_one("#contents")
        or soup.select_one("#content")
        or soup.select_one("main")
        or soup.body
    )
    if not root:
        raise ValueError("본문 루트를 찾지 못했습니다.")

    # 버튼은 노이즈 제거 전에 추출해야 합니다.
    attachments = fixed_extract_attachments(root, final_url)
    images = fixed_extract_images(root, final_url)

    for selector in FIXED_NOISE_SELECTORS:
        for node in root.select(selector):
            node.decompose()

    links = fixed_extract_links(root, final_url)
    source_text = fixed_norm(root.get_text(" ", strip=True))

    structure_parser = FixedStructureParser(
        manifest_row.get("page_type", "static_page")
    )
    blocks = structure_parser.parse(root)
    markdown = fixed_render_blocks(blocks)
    content_text = fixed_blocks_text(blocks)

    # UI 문구 마지막 안전 제거
    for phrase in FIXED_NOISE_TEXTS:
        markdown = re.sub(
            rf"(?m)^\s*{re.escape(phrase)}\s*$",
            "",
            markdown,
        )
    markdown = re.sub(r"\n{3,}", "\n\n", markdown).strip()

    content_text = fixed_norm(
        re.sub(
            "|".join(re.escape(phrase) for phrase in FIXED_NOISE_TEXTS),
            " ",
            content_text,
        )
    )

    faq_count = sum(1 for block in blocks if block.get("type") == "faq")
    table_count = sum(1 for block in blocks if block.get("type") == "table")
    noise_hits = [
        phrase for phrase in FIXED_NOISE_TEXTS if phrase in markdown
    ]
    retention_ratio = round(
        len(content_text) / max(1, len(source_text)),
        3,
    )

    quality_reasons: list[str] = []
    if len(content_text) < 80:
        quality_reasons.append("본문이 80자 미만")
    if retention_ratio < 0.70:
        quality_reasons.append("본문 보존율 70% 미만")
    if noise_hits:
        quality_reasons.append("공통 UI 문구 잔존")
    if manifest_row.get("page_type") == "faq" and faq_count == 0:
        quality_reasons.append("FAQ 질문-답변 추출 실패")

    if any(
        reason in {"공통 UI 문구 잔존", "FAQ 질문-답변 추출 실패"}
        for reason in quality_reasons
    ):
        quality_status = "fail"
    elif quality_reasons:
        quality_status = "review"
    else:
        quality_status = "pass"

    parsed_hash = sha256_text(markdown)

    document = {
        "doc_id": manifest_row["url_id"],
        "parent_doc_id": manifest_row["url_id"],
        "record_type": "page",
        "title": fixed_manifest_title(
            manifest_row["page_title"],
            manifest_row["business_function"],
        ),
        "manifest_title": manifest_row["page_title"],
        "html_title": (
            fixed_norm(soup.title.get_text(" ", strip=True))
            if soup.title
            else ""
        ),
        "source_url": manifest_row["source_url"],
        "final_url": final_url,
        "site_name": manifest_row["site_name"],
        "business_function": manifest_row["business_function"],
        "target_business_function": manifest_row.get(
            "target_business_function",
            manifest_row["business_function"],
        ),
        "page_type": manifest_row["page_type"],
        "parser_profile": manifest_row["parser_profile"],
        "breadcrumb": fixed_manifest_breadcrumb(
            manifest_row["page_title"],
            manifest_row["business_function"],
        ),
        "content_markdown": markdown,
        "content_text": content_text,
        "parsed_content_sha256": parsed_hash,
        "content_hash": parsed_hash,
        "blocks": blocks,
        "links": links,
        "attachments": attachments,
        "images": images,
        "videos": [],
        "quality": {
            "status": quality_status,
            "reasons": quality_reasons,
            "source_text_chars": len(source_text),
            "parsed_text_chars": len(content_text),
            "retention_ratio": retention_ratio,
            "faq_count": faq_count,
            "table_count": table_count,
            "attachment_button_count": len(attachments),
            "noise_hits": noise_hits,
        },
        "parsed_at": now_kst_iso(),
    }

    if manifest_row["url_id"] == "BI-004":
        document["dynamic_collection"] = {
            "collection_scope": "initial_public_page_only",
            "is_complete": False,
            "current_page_count": 1,
        }

    return document

## 7. 첨부파일·이미지 다운로드

In [ ]:
CONTENT_TYPE_EXTENSIONS = {
    "application/pdf": ".pdf",
    "application/zip": ".zip",
    "application/vnd.openxmlformats-officedocument.wordprocessingml.document": ".docx",
    "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet": ".xlsx",
    "application/vnd.openxmlformats-officedocument.presentationml.presentation": ".pptx",
    "image/jpeg": ".jpg",
    "image/png": ".png",
    "image/gif": ".gif",
    "image/webp": ".webp",
    "image/svg+xml": ".svg",
}


def filename_from_response(
    url: str,
    response: requests.Response,
    fallback_stem: str,
) -> str:
    disposition = response.headers.get("Content-Disposition", "")
    filename_star = re.search(
        r"filename\*=UTF-8''([^;]+)",
        disposition,
        flags=re.IGNORECASE,
    )
    filename_plain = re.search(
        r'filename="?([^";]+)"?',
        disposition,
        flags=re.IGNORECASE,
    )

    if filename_star:
        from urllib.parse import unquote
        name = unquote(filename_star.group(1))
    elif filename_plain:
        name = filename_plain.group(1)
    else:
        name = Path(urlparse(url).path).name

    name = safe_name(name, max_length=120)
    extension = Path(name).suffix.lower()

    if not extension:
        content_type = response.headers.get("Content-Type", "").split(";")[0].strip()
        extension = CONTENT_TYPE_EXTENSIONS.get(content_type)
        if not extension:
            extension = mimetypes.guess_extension(content_type) or ""
        name = f"{safe_name(fallback_stem)}{extension}"

    return name


def download_binary_asset(
    url: str,
    output_dir: Path,
    fallback_stem: str,
) -> dict[str, Any]:
    output_dir.mkdir(parents=True, exist_ok=True)

    with SESSION.get(
        url,
        timeout=REQUEST_TIMEOUT_SECONDS,
        allow_redirects=True,
        stream=True,
    ) as response:
        response.raise_for_status()

        content_length = response.headers.get("Content-Length")
        if content_length and int(content_length) > MAX_ASSET_BYTES:
            raise ValueError("파일이 제한 용량을 초과합니다.")

        filename = filename_from_response(url, response, fallback_stem)
        output_path = output_dir / filename

        hasher = hashlib.sha256()
        written = 0

        with output_path.open("wb") as file:
            for chunk in response.iter_content(chunk_size=64 * 1024):
                if not chunk:
                    continue
                written += len(chunk)
                if written > MAX_ASSET_BYTES:
                    output_path.unlink(missing_ok=True)
                    raise ValueError("다운로드 중 제한 용량을 초과했습니다.")
                hasher.update(chunk)
                file.write(chunk)

        return {
            "source_url": url,
            "final_url": response.url,
            "filename": filename,
            "relative_path": str(output_path.relative_to(OUTPUT_ROOT)),
            "content_type": response.headers.get("Content-Type", ""),
            "size_bytes": written,
            "sha256": hasher.hexdigest(),
            "downloaded_at": now_kst_iso(),
            "download_status": "downloaded",
        }


def ensure_attachment_playwright() -> None:
    try:
        import playwright  # noqa: F401
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "playwright"],
            check=True,
        )
    subprocess.run(
        [sys.executable, "-m", "playwright", "install", "chromium"],
        check=True,
    )


def download_js_attachments_for_document(
    document: dict[str, Any],
    output_dir: Path,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    js_attachments = [
        item
        for item in document.get("attachments", [])
        if item.get("download_method") == "gfn_downloadFile"
    ]
    if not js_attachments:
        return [], []

    ensure_attachment_playwright()
    from playwright.sync_api import sync_playwright

    downloaded: list[dict[str, Any]] = []
    failures: list[dict[str, Any]] = []
    output_dir.mkdir(parents=True, exist_ok=True)

    with sync_playwright() as playwright:
        browser = playwright.chromium.launch(headless=True)
        context = browser.new_context(
            accept_downloads=True,
            user_agent=USER_AGENT,
            locale="ko-KR",
        )
        page = context.new_page()

        try:
            page.goto(
                document["source_url"],
                wait_until="domcontentloaded",
                timeout=REQUEST_TIMEOUT_SECONDS * 1000,
            )
            page.wait_for_function(
                "typeof gfn_downloadFile === 'function'",
                timeout=REQUEST_TIMEOUT_SECONDS * 1000,
            )

            for item in js_attachments:
                try:
                    with page.expect_download(timeout=60_000) as download_info:
                        page.evaluate(
                            """
                            ([token1, token2]) => {
                                gfn_downloadFile(token1, token2);
                            }
                            """,
                            [item["token1"], item["token2"]],
                        )

                    download = download_info.value
                    filename = safe_name(download.suggested_filename, max_length=120)
                    output_path = output_dir / filename
                    download.save_as(str(output_path))

                    downloaded.append(
                        {
                            **item,
                            "filename": filename,
                            "relative_path": str(output_path.relative_to(OUTPUT_ROOT)),
                            "size_bytes": output_path.stat().st_size,
                            "sha256": sha256_bytes(output_path.read_bytes()),
                            "download_status": "downloaded",
                            "error": "",
                        }
                    )
                except Exception as error:
                    failures.append(
                        {
                            "asset_type": "attachment",
                            "display_name": item.get("display_name", ""),
                            "download_method": "gfn_downloadFile",
                            "error_type": type(error).__name__,
                            "error": str(error),
                        }
                    )
        finally:
            context.close()
            browser.close()

    return downloaded, failures


def download_document_assets(
    document: dict[str, Any],
    manifest_row: dict[str, str],
) -> dict[str, list[dict[str, Any]]]:
    business_dir = safe_name(manifest_row["business_function"])
    asset_root = PATHS["raw_assets"] / business_dir / manifest_row["url_id"]

    results = {
        "attachments": [],
        "attachment_metadata": [],
        "images": [],
        "videos": [],
        "failures": [],
    }

    should_download_attachments = (
        DOWNLOAD_ATTACHMENTS
        and (
            manifest_row["asset_policy"] == "download_attachments"
            or manifest_row["page_type"] == "attachment_page"
        )
    )

    # JS 버튼은 URL이 없으므로 우선 메타데이터로 항상 보존합니다.
    for item in document.get("attachments", []):
        if item.get("download_method") == "gfn_downloadFile":
            results["attachment_metadata"].append(item)

    # 직접 href 첨부파일
    if should_download_attachments:
        for index, item in enumerate(document.get("attachments", []), start=1):
            if item.get("download_method") != "href" or not item.get("url"):
                continue
            try:
                result = download_binary_asset(
                    item["url"],
                    asset_root / "attachments",
                    f"{manifest_row['url_id']}_attachment_{index:03d}",
                )
                result.update(item)
                results["attachments"].append(result)
            except Exception as error:
                results["failures"].append(
                    {
                        "asset_type": "attachment",
                        "url": item.get("url", ""),
                        "error_type": type(error).__name__,
                        "error": str(error),
                    }
                )

    # 선택적으로 JavaScript 버튼 실제 다운로드
    if should_download_attachments and DOWNLOAD_JS_ATTACHMENTS_WITH_PLAYWRIGHT:
        downloaded, failures = download_js_attachments_for_document(
            document,
            asset_root / "attachments",
        )
        results["attachments"].extend(downloaded)
        results["failures"].extend(failures)

    should_download_images = (
        DOWNLOAD_IMAGES
        and (
            manifest_row["crawl_wave"] == "W1_ASSET"
            or manifest_row["page_type"] in {
                "process_page", "video_page", "attachment_page"
            }
        )
    )

    if should_download_images:
        for index, item in enumerate(document.get("images", []), start=1):
            if item.get("image_role") == "decorative":
                continue
            try:
                result = download_binary_asset(
                    item["url"],
                    asset_root / "images",
                    f"{manifest_row['url_id']}_image_{index:03d}",
                )
                result.update(item)
                results["images"].append(result)
            except Exception as error:
                results["failures"].append(
                    {
                        "asset_type": "image",
                        "url": item.get("url", ""),
                        "error_type": type(error).__name__,
                        "error": str(error),
                    }
                )

    return results

## 8. 동적 조회 페이지 진단

In [ ]:

DYNAMIC_SCRIPT_KEYWORD_PATTERN = re.compile(
    r"(?:ajax|fetch\s*\(|XMLHttpRequest|\.do\b|pageIndex|pageNo|search)",
    flags=re.IGNORECASE,
)



def extract_dynamic_diagnostic(
    html_bytes: bytes,
    final_url: str,
    encoding: str,
) -> dict[str, Any]:
    html_text = html_bytes.decode(encoding, errors="replace")
    soup = BeautifulSoup(html_text, "lxml")

    forms = []
    for form_index, form in enumerate(soup.find_all("form"), start=1):
        inputs = []

        for field in form.select("input,select,textarea,button"):
            name = normalize_space(field.get("name", ""))
            field_type = normalize_space(
                field.get("type", field.name or "")
            )

            if not name and field.name != "button":
                continue

            record = {
                "tag": field.name,
                "name": name,
                "type": field_type,
                "value": normalize_space(field.get("value", "")),
            }

            if field.name == "select":
                record["options"] = [
                    {
                        "value": normalize_space(option.get("value", "")),
                        "text": element_text(option),
                    }
                    for option in field.find_all("option")
                ]

            inputs.append(record)

        forms.append(
            {
                "form_index": form_index,
                "method": normalize_space(form.get("method", "GET")).upper(),
                "action": absolute_url(final_url, form.get("action")) or final_url,
                "id": normalize_space(form.get("id", "")),
                "name": normalize_space(form.get("name", "")),
                "inputs": inputs,
            }
        )

    script_sources = []
    for script in soup.find_all("script", src=True):
        source = absolute_url(final_url, script.get("src"))
        if source and source not in script_sources:
            script_sources.append(source)

    inline_script_hints = []

    for script in soup.find_all("script"):
        if script.get("src"):
            continue

        text = script.get_text("\n", strip=True)
        if text and DYNAMIC_SCRIPT_KEYWORD_PATTERN.search(text):
            inline_script_hints.append(text[:1200])

    return {
        "final_url": final_url,
        "forms": forms,
        "script_sources": script_sources,
        "inline_script_hints": inline_script_hints[:20],
        "diagnosed_at": now_kst_iso(),
        "safety_note": (
            "초기 공개 페이지 구조만 진단했습니다. "
            "검색 폼 제출·로그인·본인인증·신청 실행은 수행하지 않았습니다."
        ),
    }


def ensure_playwright_installed() -> None:
    try:
        import playwright  # noqa: F401
        return
    except ImportError:
        pass

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "playwright"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "playwright", "install", "chromium"],
        check=True,
    )


def save_playwright_snapshot(
    url: str,
    url_id: str,
) -> dict[str, Any] | None:
    if not ENABLE_PLAYWRIGHT_SNAPSHOT:
        return None

    ensure_playwright_installed()
    from playwright.sync_api import sync_playwright

    html_path = PATHS["dynamic_diagnostics"] / f"{url_id}_rendered.html"
    screenshot_path = PATHS["screenshots"] / f"{url_id}.png"
    ensure_parent(html_path)
    ensure_parent(screenshot_path)

    with sync_playwright() as playwright:
        browser = playwright.chromium.launch(headless=True)
        page = browser.new_page(
            user_agent=USER_AGENT,
            locale="ko-KR",
            viewport={"width": 1440, "height": 1200},
        )

        response = page.goto(
            url,
            wait_until="domcontentloaded",
            timeout=REQUEST_TIMEOUT_SECONDS * 1000,
        )
        page.wait_for_timeout(PLAYWRIGHT_WAIT_MS)

        html = page.content()
        html_path.write_text(html, encoding="utf-8")
        page.screenshot(path=str(screenshot_path), full_page=True)

        result = {
            "status_code": response.status if response else None,
            "final_url": page.url,
            "rendered_html_path": str(html_path.relative_to(OUTPUT_ROOT)),
            "screenshot_path": str(screenshot_path.relative_to(OUTPUT_ROOT)),
            "captured_at": now_kst_iso(),
        }

        browser.close()
        return result


### 동적 페이지 진단 코드 자체 테스트

실제 웹 요청 전에 다음을 검사합니다.

- `fetch(...)`
- `XMLHttpRequest`
- `.do` 엔드포인트
- `pageIndex`, `pageNo`, `search`
- 합성 HTML의 폼·인라인 스크립트 추출

이 셀이 실패하면 실제 크롤링을 실행하지 마십시오.

In [ ]:
_DYNAMIC_PATTERN_TEST_CASES = {
    "fetch('/api/list')": True,
    "const xhr = new XMLHttpRequest();": True,
    "url: '/board/selectList.do'": True,
    "pageIndex = 2": True,
    "pageNo = 3": True,
    "searchKeyword": True,
    "일반 안내 문장입니다.": False,
}

for sample_text, expected in _DYNAMIC_PATTERN_TEST_CASES.items():
    actual = bool(DYNAMIC_SCRIPT_KEYWORD_PATTERN.search(sample_text))
    assert actual == expected, (
        f"동적 스크립트 정규표현식 테스트 실패: "
        f"text={sample_text!r}, expected={expected}, actual={actual}"
    )

_DYNAMIC_TEST_HTML = """
<!doctype html>
<html lang="ko">
<body>
  <form id="searchForm" method="post" action="/search/list.do">
    <input type="hidden" name="pageIndex" value="1">
    <input type="text" name="searchKeyword" value="">
    <select name="category">
      <option value="">전체</option>
      <option value="bank">은행</option>
    </select>
  </form>

  <script>
    function loadData() {
      fetch('/api/list?pageNo=1');
    }
  </script>
</body>
</html>
""".encode("utf-8")

_dynamic_test_result = extract_dynamic_diagnostic(
    html_bytes=_DYNAMIC_TEST_HTML,
    final_url="https://example.org/dynamic/page",
    encoding="utf-8",
)

assert len(_dynamic_test_result["forms"]) == 1
assert _dynamic_test_result["forms"][0]["method"] == "POST"
assert _dynamic_test_result["forms"][0]["action"] == (
    "https://example.org/search/list.do"
)
assert any(
    item["name"] == "pageIndex"
    for item in _dynamic_test_result["forms"][0]["inputs"]
)
assert len(_dynamic_test_result["inline_script_hints"]) == 1

print("동적 페이지 진단 자체 테스트: 통과")
print("정규표현식:", DYNAMIC_SCRIPT_KEYWORD_PATTERN.pattern)

## 9. URL 유형별 처리기

In [ ]:

def save_parsed_document(
    document: dict[str, Any],
    manifest_row: dict[str, str],
) -> tuple[Path, Path]:
    business_dir = safe_name(manifest_row["business_function"])
    url_id = manifest_row["url_id"]

    markdown_path = (
        PATHS["parsed_markdown"]
        / business_dir
        / f"{url_id}.md"
    )
    json_path = (
        PATHS["parsed_json"]
        / business_dir
        / f"{url_id}.json"
    )

    ensure_parent(markdown_path)
    markdown_path.write_text(
        document["content_markdown"],
        encoding="utf-8",
    )
    write_json(json_path, document)

    return markdown_path, json_path


def process_link_only(
    manifest_row: dict[str, str],
) -> dict[str, Any]:
    record = {
        "doc_id": manifest_row["url_id"],
        "record_type": "link_only",
        "title": manifest_row["page_title"],
        "source_url": manifest_row["source_url"],
        "site_name": manifest_row["site_name"],
        "business_function": manifest_row["business_function"],
        "target_business_function": manifest_row.get(
            "target_business_function",
            manifest_row["business_function"],
        ),
        "page_type": manifest_row["page_type"],
        "auth_required": manifest_row["auth_required"],
        "content_scope": manifest_row["content_scope"],
        "description": manifest_row.get("content_summary", ""),
        "decision_reason": manifest_row["decision_reason"],
        "created_at": now_kst_iso(),
    }

    output_path = (
        PATHS["parsed_json"]
        / safe_name(manifest_row["business_function"])
        / f"{manifest_row['url_id']}.json"
    )
    write_json(output_path, record)

    return {
        "url_id": manifest_row["url_id"],
        "business_function": manifest_row["business_function"],
        "page_title": manifest_row["page_title"],
        "source_url": manifest_row["source_url"],
        "normalized_decision": manifest_row["normalized_decision"],
        "crawl_wave": manifest_row["crawl_wave"],
        "status": "link_metadata_created",
        "status_code": None,
        "content_chars": len(record["description"]),
        "attachment_count": 0,
        "image_count": 0,
        "asset_failure_count": 0,
        "error_type": "",
        "error": "",
        "finished_at": now_kst_iso(),
    }


def process_static_or_asset(
    manifest_row: dict[str, str],
) -> dict[str, Any]:
    result = fetch_url(manifest_row["source_url"])
    html_path, meta_path = save_fetch_result(manifest_row, result)

    if not result.ok:
        raise requests.HTTPError(
            f"HTTP {result.status_code}: {result.final_url}"
        )

    if "html" not in result.content_type.lower():
        raise ValueError(
            f"HTML 응답이 아닙니다: {result.content_type}"
        )

    document = parse_html_document(
        html_bytes=result.body,
        final_url=result.final_url,
        manifest_row=manifest_row,
        encoding=result.encoding,
    )

    asset_results = download_document_assets(
        document=document,
        manifest_row=manifest_row,
    )
    document["downloaded_assets"] = asset_results

    markdown_path, json_path = save_parsed_document(
        document=document,
        manifest_row=manifest_row,
    )

    return {
        "url_id": manifest_row["url_id"],
        "business_function": manifest_row["business_function"],
        "page_title": document["title"],
        "source_url": manifest_row["source_url"],
        "final_url": result.final_url,
        "normalized_decision": manifest_row["normalized_decision"],
        "crawl_wave": manifest_row["crawl_wave"],
        "status": (
            "parse_success"
            if document["content_text"]
            else "empty_content"
        ),
        "quality_status": document.get("quality", {}).get("status", ""),
        "quality_reasons": " | ".join(document.get("quality", {}).get("reasons", [])),
        "retention_ratio": document.get("quality", {}).get("retention_ratio", 0),
        "faq_count": document.get("quality", {}).get("faq_count", 0),
        "table_count": document.get("quality", {}).get("table_count", 0),
        "attachment_button_count": document.get("quality", {}).get("attachment_button_count", 0),
        "status_code": result.status_code,
        "content_chars": len(document["content_text"]),
        "block_count": len(document["blocks"]),
        "link_count": len(document["links"]),
        "attachment_count": len(document["attachments"]),
        "image_count": len(document["images"]),
        "downloaded_attachment_count": len(asset_results["attachments"]),
        "downloaded_image_count": len(asset_results["images"]),
        "asset_failure_count": len(asset_results["failures"]),
        "raw_html_path": str(html_path.relative_to(OUTPUT_ROOT)),
        "response_meta_path": str(meta_path.relative_to(OUTPUT_ROOT)),
        "parsed_markdown_path": str(markdown_path.relative_to(OUTPUT_ROOT)),
        "parsed_json_path": str(json_path.relative_to(OUTPUT_ROOT)),
        "raw_sha256": result.raw_sha256,
        "parsed_content_sha256": document["parsed_content_sha256"],
        "error_type": "",
        "error": "",
        "finished_at": now_kst_iso(),
    }


def process_dynamic_diagnostic(
    manifest_row: dict[str, str],
) -> dict[str, Any]:
    result = fetch_url(manifest_row["source_url"])
    html_path, meta_path = save_fetch_result(manifest_row, result)

    if not result.ok:
        raise requests.HTTPError(
            f"HTTP {result.status_code}: {result.final_url}"
        )

    diagnostic = extract_dynamic_diagnostic(
        html_bytes=result.body,
        final_url=result.final_url,
        encoding=result.encoding,
    )
    diagnostic["url_id"] = manifest_row["url_id"]
    diagnostic["page_title"] = manifest_row["page_title"]
    diagnostic["business_function"] = manifest_row["business_function"]
    diagnostic["source_url"] = manifest_row["source_url"]

    playwright_snapshot = save_playwright_snapshot(
        url=manifest_row["source_url"],
        url_id=manifest_row["url_id"],
    )
    if playwright_snapshot:
        diagnostic["playwright_snapshot"] = playwright_snapshot

    diagnostic_path = (
        PATHS["dynamic_diagnostics"]
        / f"{manifest_row['url_id']}.json"
    )
    write_json(diagnostic_path, diagnostic)

    # 초기 HTML에 공개 본문이 있을 수 있으므로 파싱 결과도 참고용으로 저장합니다.
    document = parse_html_document(
        html_bytes=result.body,
        final_url=result.final_url,
        manifest_row=manifest_row,
        encoding=result.encoding,
    )
    document["collection_scope"] = "initial_public_page_only"
    markdown_path, json_path = save_parsed_document(
        document=document,
        manifest_row=manifest_row,
    )

    return {
        "url_id": manifest_row["url_id"],
        "business_function": manifest_row["business_function"],
        "page_title": document["title"],
        "source_url": manifest_row["source_url"],
        "final_url": result.final_url,
        "normalized_decision": manifest_row["normalized_decision"],
        "crawl_wave": manifest_row["crawl_wave"],
        "status": "dynamic_diagnostic_created",
        "quality_status": document.get("quality", {}).get("status", ""),
        "quality_reasons": " | ".join(document.get("quality", {}).get("reasons", [])),
        "retention_ratio": document.get("quality", {}).get("retention_ratio", 0),
        "faq_count": document.get("quality", {}).get("faq_count", 0),
        "table_count": document.get("quality", {}).get("table_count", 0),
        "attachment_button_count": document.get("quality", {}).get("attachment_button_count", 0),
        "status_code": result.status_code,
        "content_chars": len(document["content_text"]),
        "form_count": len(diagnostic["forms"]),
        "script_source_count": len(diagnostic["script_sources"]),
        "attachment_count": len(document["attachments"]),
        "image_count": len(document["images"]),
        "asset_failure_count": 0,
        "raw_html_path": str(html_path.relative_to(OUTPUT_ROOT)),
        "response_meta_path": str(meta_path.relative_to(OUTPUT_ROOT)),
        "parsed_markdown_path": str(markdown_path.relative_to(OUTPUT_ROOT)),
        "parsed_json_path": str(json_path.relative_to(OUTPUT_ROOT)),
        "dynamic_diagnostic_path": str(
            diagnostic_path.relative_to(OUTPUT_ROOT)
        ),
        "error_type": "",
        "error": "",
        "finished_at": now_kst_iso(),
    }


def process_manifest_row(
    manifest_row: dict[str, str],
) -> dict[str, Any]:
    decision = manifest_row["normalized_decision"]
    strategy = manifest_row["fetch_strategy"]

    if decision == "link_only":
        return process_link_only(manifest_row)

    if strategy == "dynamic_http_then_playwright":
        return process_dynamic_diagnostic(manifest_row)

    return process_static_or_asset(manifest_row)


## 10. 크롤링 실행

In [ ]:

CRAWL_RESULTS_JSONL = PATHS["logs"] / "crawl_results.jsonl"
CRAWL_RESULTS_CSV = PATHS["logs"] / "crawl_results.csv"
USED_MANIFEST_PATH = PATHS["manifest"] / "manifest_used.csv"

target_df.to_csv(
    USED_MANIFEST_PATH,
    index=False,
    encoding="utf-8-sig",
)

run_results: list[dict[str, Any]] = []

for _, row in tqdm(
    target_df.iterrows(),
    total=len(target_df),
    desc="크롤링 진행",
):
    manifest_row = row_to_clean_dict(row)
    started_at = now_kst_iso()

    try:
        result = process_manifest_row(manifest_row)
        result["started_at"] = started_at

    except Exception as error:
        result = {
            "url_id": manifest_row["url_id"],
            "business_function": manifest_row["business_function"],
            "page_title": manifest_row["page_title"],
            "source_url": manifest_row["source_url"],
            "normalized_decision": manifest_row["normalized_decision"],
            "crawl_wave": manifest_row["crawl_wave"],
            "status": "failed",
            "status_code": None,
            "content_chars": 0,
            "attachment_count": 0,
            "image_count": 0,
            "asset_failure_count": 0,
            "error_type": type(error).__name__,
            "error": str(error),
            "started_at": started_at,
            "finished_at": now_kst_iso(),
        }

    run_results.append(result)
    append_jsonl(CRAWL_RESULTS_JSONL, result)

    # link_only는 실제 HTTP 요청을 보내지 않으므로 대기하지 않습니다.
    if manifest_row["normalized_decision"] != "link_only":
        time.sleep(REQUEST_DELAY_SECONDS)

results_df = pd.DataFrame(run_results)
results_df.to_csv(
    CRAWL_RESULTS_CSV,
    index=False,
    encoding="utf-8-sig",
)

print("실행 완료")
display(
    results_df[
        [
            "url_id",
            "business_function",
            "status",
            "status_code",
            "content_chars",
            "attachment_count",
            "image_count",
            "asset_failure_count",
            "error_type",
        ]
    ]
)


## 11. 결과 검수

In [ ]:
QUALITY_REPORT_CSV = PATHS["logs"] / "quality_report.csv"

if results_df.empty:
    print("실행 결과가 없습니다.")
else:
    print("수집 상태별 건수")
    display(
        results_df.groupby("status")
        .size()
        .rename("count")
        .reset_index()
        .sort_values("count", ascending=False)
    )

    failures = results_df[results_df["status"] == "failed"].copy()
    empty_contents = results_df[
        results_df["status"].isin({"empty_content"})
        | (
            results_df["status"].isin(
                {"parse_success", "dynamic_diagnostic_created"}
            )
            & (results_df["content_chars"].fillna(0).astype(int) < 80)
        )
    ].copy()

    quality_columns = [
        "url_id", "business_function", "page_title", "status",
        "quality_status", "quality_reasons", "retention_ratio",
        "content_chars", "faq_count", "table_count",
        "attachment_button_count", "attachment_count",
        "image_count", "asset_failure_count", "error_type", "error",
    ]
    available_quality_columns = [
        column for column in quality_columns if column in results_df.columns
    ]
    quality_report_df = results_df[available_quality_columns].copy()
    quality_report_df.to_csv(
        QUALITY_REPORT_CSV,
        index=False,
        encoding="utf-8-sig",
    )

    print("실패:", len(failures))
    print("빈 본문·매우 짧은 본문:", len(empty_contents))

    if "quality_status" in results_df.columns:
        print("파싱 품질 상태")
        display(
            results_df.groupby("quality_status", dropna=False)
            .size()
            .rename("count")
            .reset_index()
        )

        review_required = results_df[
            results_df["quality_status"].isin({"review", "fail"})
        ]
        if not review_required.empty:
            print("사람 검수 또는 파서 수정 필요")
            display(
                review_required[
                    [
                        "url_id", "page_title", "quality_status",
                        "quality_reasons", "retention_ratio",
                        "content_chars",
                    ]
                ]
            )

    if not failures.empty:
        display(
            failures[
                ["url_id", "page_title", "source_url", "error_type", "error"]
            ]
        )

    success_statuses = {
        "parse_success",
        "dynamic_diagnostic_created",
        "link_metadata_created",
    }
    success_count = results_df["status"].isin(success_statuses).sum()
    success_rate = success_count / len(results_df) * 100
    print(f"수집 처리 성공률: {success_count}/{len(results_df)} ({success_rate:.1f}%)")
    print("품질 보고서:", QUALITY_REPORT_CSV)

## 12. 페이지 문서 JSONL과 기본 청크 생성

In [ ]:
DOCUMENTS_JSONL = PATHS["processed"] / "documents.jsonl"
CHUNKS_JSONL = PATHS["processed"] / "chunks.jsonl"


def iter_parsed_json_files() -> Iterable[Path]:
    yield from sorted(PATHS["parsed_json"].rglob("*.json"))


def build_document_record(data: dict[str, Any]) -> dict[str, Any] | None:
    if data.get("record_type") == "link_only":
        return {
            "doc_id": data["doc_id"],
            "parent_doc_id": data["doc_id"],
            "record_type": "link_only",
            "indexable": False,
            "title": data["title"],
            "source_url": data["source_url"],
            "site_name": data["site_name"],
            "business_function": data["business_function"],
            "target_business_function": data["target_business_function"],
            "page_type": data["page_type"],
            "content_markdown": data.get("description", ""),
            "content_text": data.get("description", ""),
            "content_hash": sha256_text(data.get("description", "")),
            "blocks": [],
            "attachments": [],
            "images": [],
            "links": [
                {
                    "url": data["source_url"],
                    "anchor_text": data["title"],
                    "link_type": "official_service",
                }
            ],
            "metadata": {
                "auth_required": data.get("auth_required", ""),
                "content_scope": data.get("content_scope", ""),
                "decision_reason": data.get("decision_reason", ""),
            },
        }

    if not data.get("content_markdown"):
        return None

    return {
        "doc_id": data["doc_id"],
        "parent_doc_id": data.get("parent_doc_id", data["doc_id"]),
        "record_type": "page",
        "indexable": True,
        "title": data.get("title", ""),
        "source_url": data.get("source_url", ""),
        "final_url": data.get("final_url", ""),
        "site_name": data.get("site_name", ""),
        "business_function": data.get("business_function", ""),
        "target_business_function": data.get(
            "target_business_function",
            data.get("business_function", ""),
        ),
        "page_type": data.get("page_type", ""),
        "breadcrumb": data.get("breadcrumb", []),
        "content_markdown": data.get("content_markdown", ""),
        "content_text": data.get("content_text", ""),
        "content_hash": data.get(
            "parsed_content_sha256",
            sha256_text(data.get("content_markdown", "")),
        ),
        "blocks": data.get("blocks", []),
        "attachments": data.get("attachments", []),
        "images": data.get("images", []),
        "links": data.get("links", []),
        "quality": data.get("quality", {}),
        "dynamic_collection": data.get("dynamic_collection"),
        "metadata": {
            "parser_profile": data.get("parser_profile", ""),
            "attachment_count": len(data.get("attachments", [])),
            "image_count": len(data.get("images", [])),
            "parsed_at": data.get("parsed_at", ""),
        },
    }


def render_table_chunk(
    headers: list[str],
    rows: list[list[str]],
) -> str:
    block = {"type": "table", "headers": headers, "rows": rows}
    return fixed_render_blocks([block])


def split_table_block(
    block: dict[str, Any],
    max_chars: int,
) -> list[str]:
    headers = block.get("headers", [])
    rows = block.get("rows", [])
    all_text = render_table_chunk(headers, rows)
    if len(all_text) <= max_chars:
        return [all_text]

    chunks: list[str] = []
    current_rows: list[list[str]] = []

    for row in rows:
        candidate_rows = current_rows + [row]
        candidate_text = render_table_chunk(headers, candidate_rows)

        if current_rows and len(candidate_text) > max_chars:
            chunks.append(render_table_chunk(headers, current_rows))
            current_rows = [row]
        else:
            current_rows = candidate_rows

    if current_rows:
        chunks.append(render_table_chunk(headers, current_rows))

    return chunks


def structure_aware_chunks(
    document: dict[str, Any],
    max_chars: int = 1600,
) -> list[dict[str, Any]]:
    if not document.get("indexable", True):
        return []

    intermediate: list[dict[str, str]] = []
    current_parts: list[str] = []
    current_heading = ""

    def flush_section() -> None:
        nonlocal current_parts
        content = "\n\n".join(
            part for part in current_parts if part
        ).strip()
        current_parts = []
        if content:
            intermediate.append(
                {
                    "section_title": current_heading,
                    "chunk_type": "section",
                    "content": content,
                }
            )

    for block in document.get("blocks", []):
        block_type = block.get("type")

        if block_type == "heading":
            flush_section()
            current_heading = block.get("text", "")
            current_parts = [fixed_render_blocks([block])]
            continue

        if block_type == "faq":
            flush_section()
            intermediate.append(
                {
                    "section_title": block.get("question", ""),
                    "chunk_type": "faq",
                    "content": fixed_render_blocks([block]),
                }
            )
            continue

        if block_type == "table":
            flush_section()
            for table_text in split_table_block(block, max_chars):
                intermediate.append(
                    {
                        "section_title": current_heading,
                        "chunk_type": "table",
                        "content": table_text,
                    }
                )
            continue

        block_text = fixed_render_blocks([block])
        candidate = "\n\n".join(current_parts + [block_text]).strip()
        if current_parts and len(candidate) > max_chars:
            flush_section()
        current_parts.append(block_text)

    flush_section()

    records: list[dict[str, Any]] = []
    for index, item in enumerate(intermediate):
        content = item["content"].strip()
        if not content:
            continue

        records.append(
            {
                "chunk_id": f"{document['doc_id']}_chunk_{index:03d}",
                "parent_doc_id": document["doc_id"],
                "chunk_index": index,
                "title": document["title"],
                "section_title": item["section_title"],
                "chunk_type": item["chunk_type"],
                "business_function": document["business_function"],
                "target_business_function": document[
                    "target_business_function"
                ],
                "page_type": document["page_type"],
                "source_url": document["source_url"],
                "content": content,
                "content_hash": sha256_text(content),
            }
        )

    return records


documents: list[dict[str, Any]] = []

for json_path in iter_parsed_json_files():
    data = json.loads(json_path.read_text(encoding="utf-8"))
    document = build_document_record(data)
    if document:
        documents.append(document)

with DOCUMENTS_JSONL.open("w", encoding="utf-8") as file:
    for document in documents:
        file.write(json.dumps(document, ensure_ascii=False) + "\n")

chunk_records: list[dict[str, Any]] = []

if CREATE_BASELINE_CHUNKS:
    for document in documents:
        chunk_records.extend(
            structure_aware_chunks(
                document,
                max_chars=max(1600, CHUNK_MAX_CHARS),
            )
        )

with CHUNKS_JSONL.open("w", encoding="utf-8") as file:
    for chunk in chunk_records:
        file.write(json.dumps(chunk, ensure_ascii=False) + "\n")

print("페이지 문서:", len(documents))
print("구조 기반 청크:", len(chunk_records))

if chunk_records:
    chunk_df = pd.DataFrame(chunk_records)
    display(
        chunk_df.groupby("chunk_type")
        .size()
        .rename("count")
        .reset_index()
    )

# 핵심 회귀 검사: 기존 분석에서 누락됐던 내용 확인
by_id = {document["doc_id"]: document for document in documents}
validation_rows: list[dict[str, Any]] = []

if "DP-001" in by_id:
    validation_rows.append(
        {
            "검증": "DP-001 보호한도 핵심 문구",
            "통과": "1억원" in by_id["DP-001"].get("content_text", ""),
        }
    )

if "UN-003" in by_id:
    un003_text = by_id["UN-003"].get("content_text", "")
    validation_rows.append(
        {
            "검증": "UN-003 미수령금 종류",
            "통과": all(
                keyword in un003_text
                for keyword in ["예금보험금", "개산지급금", "파산배당금"]
            ),
        }
    )

for faq_id in ["DP-005", "DP-006", "MT-005", "HP-002"]:
    if faq_id in by_id:
        faq_count = sum(
            1
            for block in by_id[faq_id].get("blocks", [])
            if block.get("type") == "faq"
        )
        validation_rows.append(
            {"검증": f"{faq_id} FAQ 질문-답변", "통과": faq_count > 0}
        )

if validation_rows:
    validation_df = pd.DataFrame(validation_rows)
    display(validation_df)
    failed = validation_df[~validation_df["통과"]]
    if not failed.empty:
        print("주의: 아래 항목은 Markdown 직접 검수가 필요합니다.")
        display(failed)

print("documents.jsonl:", DOCUMENTS_JSONL)
print("chunks.jsonl:", CHUNKS_JSONL)

## 13. 결과 ZIP 생성 및 다운로드

In [ ]:

archive_path = Path(
    shutil.make_archive(
        base_name=str(OUTPUT_ROOT),
        format="zip",
        root_dir=OUTPUT_ROOT.parent,
        base_dir=OUTPUT_ROOT.name,
    )
)

print("결과 ZIP:", archive_path)
print("ZIP 크기:", f"{archive_path.stat().st_size / 1024 / 1024:.2f} MB")

try:
    from google.colab import files
    files.download(str(archive_path))
except Exception as error:
    print("자동 다운로드를 시작하지 못했습니다.")
    print("Colab 왼쪽 파일 패널에서 직접 다운로드하세요:", archive_path)
    print("오류:", error)



## 결과 폴더 구조

```text
kdic_crawl_output_<실행시각>/
├── manifest/
│   └── manifest_used.csv
├── raw/
│   ├── html/
│   ├── assets/
│   └── response_meta/
├── parsed/
│   ├── markdown/
│   └── json/
├── diagnostics/
│   ├── dynamic/
│   └── screenshots/
├── processed/
│   ├── documents.jsonl
│   └── chunks.jsonl
└── logs/
    ├── crawl_results.csv
    └── crawl_results.jsonl
```

## 권장 실행 순서

1. `MAX_URLS = 3`으로 테스트
2. 실패·빈 본문·자산 오류 확인
3. 사이트별 본문 선택자와 제거 선택자 조정
4. `MAX_URLS = None`으로 전체 실행
5. `Review_Queue` 결정 후 Manifest 수정본을 업로드하여 재실행
6. 동적 조회 페이지는 진단 JSON을 검토한 뒤 공개 HTTP 요청 전용 수집기를 별도로 작성

## BI-004만 먼저 재검증하는 방법

설정 셀:

```python
RUN_ONLY_URL_IDS = ["BI-004"]
MAX_URLS = None
```

정상 결과:

```text
status = dynamic_diagnostic_created
status_code = 200
```

생성되어야 하는 파일:

```text
parsed/markdown/예금보험금 안내/BI-004.md
parsed/json/예금보험금 안내/BI-004.json
diagnostics/dynamic/BI-004.json
raw/html/예금보험금 안내/BI-004.html
raw/response_meta/예금보험금 안내/BI-004.json
```

전체를 다시 실행할 때:

```python
RUN_ONLY_URL_IDS = []
MAX_URLS = None
```

## FIXED 버전에서 추가로 확인할 파일

```text
logs/quality_report.csv
```

이 파일은 HTTP 성공 여부와 별개로 다음을 보여줍니다.

- 본문 보존율
- FAQ 질문·답변 개수
- 표 개수
- 첨부파일 버튼 인식 개수
- 공통 UI 문구 잔존 여부
- 자동 판정 `pass`, `review`, `fail`

첨부파일 버튼은 `parsed/json/*/*.json`의 `attachments`에서 확인할 수 있습니다.
`DOWNLOAD_JS_ATTACHMENTS_WITH_PLAYWRIGHT = False`인 경우 실제 파일은 내려받지 않고 다운로드 메타데이터만 저장합니다.

In [ ]:

# Colab 런타임에 필요한 라이브러리 설치
%pip -q install "requests>=2.32,<3" "beautifulsoup4>=4.12,<5" "lxml>=5,<7" "pandas>=2.2,<4" "tqdm>=4.66,<5"



## 1. 실행 환경과 설정

기본 설정은 정적 페이지·자산 페이지·링크 전용 레코드를 처리합니다.

- 특정 업무만 실행하려면 `SELECT_BUSINESS_FUNCTIONS`에 업무명을 입력합니다.
- 테스트 실행은 `MAX_URLS = 3`처럼 제한합니다.
- 동적 공개 페이지를 브라우저로 렌더링한 초기 화면까지 저장하려면 `ENABLE_PLAYWRIGHT_SNAPSHOT = True`로 바꿉니다.
- Playwright도 검색 폼 제출이나 로그인은 수행하지 않습니다.


In [ ]:

from __future__ import annotations

import hashlib
import json
import mimetypes
import os
import re
import shutil
import subprocess
import sys
import time
from dataclasses import dataclass, asdict
from datetime import datetime, timezone, timedelta
from pathlib import Path
from typing import Any, Iterable
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup, Tag
from IPython.display import display
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

KST = timezone(timedelta(hours=9))

# -------------------------
# 사용자 실행 설정
# -------------------------
SELECT_BUSINESS_FUNCTIONS: list[str] = []
# 예: ["예금자보호제도", "예금보험금 안내"]

RUN_DECISIONS = {"include_full", "include_partial", "link_only"}
RUN_WAVES = {"W1_STATIC", "W1_ASSET", "W2_DYNAMIC", "META"}

MAX_URLS: int | None = None
# 최초 테스트 권장: 3
# 전체 실행: None

REQUEST_DELAY_SECONDS = 1.2
REQUEST_TIMEOUT_SECONDS = 25
MAX_RETRIES = 3
MAX_ASSET_BYTES = 25 * 1024 * 1024

RESPECT_ROBOTS_TXT = True
DOWNLOAD_ATTACHMENTS = True
DOWNLOAD_IMAGES = True
DOWNLOAD_VIDEOS = False

ENABLE_PLAYWRIGHT_SNAPSHOT = False
PLAYWRIGHT_WAIT_MS = 1500

CREATE_BASELINE_CHUNKS = True
CHUNK_MAX_CHARS = 1200
CHUNK_OVERLAP_CHARS = 150

USER_AGENT = (
    "KDIC-RAG-Academic-Crawler/0.1 "
    "(low-rate public-document collection; no authentication automation)"
)

RUN_ID = datetime.now(KST).strftime("%Y%m%d_%H%M%S")
RUNTIME_ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
OUTPUT_ROOT = RUNTIME_ROOT / f"kdic_crawl_output_{RUN_ID}"

PATHS = {
    "raw_html": OUTPUT_ROOT / "raw" / "html",
    "raw_assets": OUTPUT_ROOT / "raw" / "assets",
    "response_meta": OUTPUT_ROOT / "raw" / "response_meta",
    "parsed_markdown": OUTPUT_ROOT / "parsed" / "markdown",
    "parsed_json": OUTPUT_ROOT / "parsed" / "json",
    "dynamic_diagnostics": OUTPUT_ROOT / "diagnostics" / "dynamic",
    "screenshots": OUTPUT_ROOT / "diagnostics" / "screenshots",
    "logs": OUTPUT_ROOT / "logs",
    "processed": OUTPUT_ROOT / "processed",
    "manifest": OUTPUT_ROOT / "manifest",
}

for path in PATHS.values():
    path.mkdir(parents=True, exist_ok=True)

print("실행 ID:", RUN_ID)
print("결과 경로:", OUTPUT_ROOT)


## 2. 42개 URL Manifest 불러오기

In [ ]:

# 현재 프로젝트의 42개 Manifest를 노트북 내부에 압축하여 포함했습니다.
# 별도 CSV 업로드 없이 기본 Manifest를 자동 생성합니다.

import base64
import gzip

EMBEDDED_MANIFEST_GZIP_B64 = """H4sIAHmDVGoC/+1dbW8TWbL+PtL8h1akXSXaXhzbCS/3w1xlgJmJBmaZwOzofrIcuxO8OG3T3Q6T/TAy0EGGZJawNwYH7FznEkiCgtYJDnF2g66Un+Pu1v6FW3XqdLvbbjs2hLCajYRiu19On656TtVTdeoc/vmP/0to0lRETokZJRlJxMXxjJqQJVWNTGTkmJZIyaIWVSYlLdJ6Ih2dlCJaQktKoprKKDEpAm2IKjQYkaNTkphSEpMJOZqMpJUEfNdmRDmlTEWTiT9L8UhciiVUbMX+ElGkqOq0OpOWxAlJi12PqJoS1aTJGTihqJICjaUmEvDIaEa7DvfczCQUKS5GVRX6mE4lE7EZMZaSNUnWImosBc0o0nRCutW4NKZEbyUjt6LTEv+qalEto4rST2kppkHXUhktndFEWYLvvFNOi5mpqaiCL6JJ6uefBcULV34/OBgUzUKuXsuZywvGm6pVqJnlovFQb3OUflh59uO6pqXV/wgEbt26depGPBE7lVJO3VACajoQT8Obw8tqgSvw5+qMquHnpSlNDahSEnp6NabIp+Ip/hRs9Umh/mbHvLMpWnndzBXEhBxLZuISaCyZFM0ns8ZmVTDXcsZmzSwV4Ntts/xIqO/u17cqxkpRgDvNpxtCvZI1HryA/gnY2lxJMPM5405VsP5SMEtVcy0rosASsQhqSkSxSqqmRq5rU0kRXyHCT08HQUzwTwZNQgci46k4Ck78MRi5em3k2uh5lGIE5AgyF81nIKSa8M21y5eE3wn1nU1zpSLAO5krVfioQZfhsLGxaN3NGr9UoB+WXsE3N+8VQ0PCD2OXxKBZqhlze0JDHLaQhaD5eAceUH+zL5jVHHygVvIrgrGn03UCnDX3lgRTXzVezYqffxYixYZ6Uqz/0V40fKLZZs3WaxXoclAw5rPWo6KVf2282kA91itFY3ZBMEv75l7BR1DwqgL9Np7PGy9A8eXbxst3Vh4e+mjBfJIz5lbxGlRYccd6Og/34EPMuTJHkFnSQZDZeu2+lS8YDxYRI8bLzQZGwoSR8HsMfnzQ3dvCF97Hf4FvZT5d7AoyhBS3abiamVJOUOPYA7cdELjAPeI2/3fWeFHoN0tZ6/H9g10SGJw42LVym6i3tVX4B2KC74Lx31sDgnGnYD6pguqHSPVDH656+G79Nfehqr+mTMJnPHaCgW4xQHJHS4JOgM7QsX7SPhgCC46h3lEw8HTjrY7myFzNNmzAMAFhuCcg9NXflOuVkqnvQ4dB9V+NfA9/jb/VQD9GeQ/usO7tWPkN6IK5/bofVQDv9gDe4K0OiMWuwtPNvdeiYM6tQqMAzCpoCZDs8X4DfQ6cJhKy6sZTbCowPm4Tia+iN79TolPaSDo5g+DhEsrnzLlN6+Gmpdc+NngmojebQIMdjsDhT+Br+t3cgeyCPWwHWl1PXyvfEOE9dTAU4HOy5ts8GRhUnnmnQlcLdDnor3AbDUw5D1IQrGe643tq5luQXzlnVtbRe4G3Mp5vIlL6AHWnCXWnP5ydYIs9MRTAyuEkBaBirj06RrQwE/Np0MJI5fN9FCdwhHplCXporM0LocHQsDGrC5yCom15/ALNiWC3YD4BED25Z/wCstgpMuJxZ9NYyTr6rol2I+fMZ4t+xNbdKjWBtsTdTNUszQNizhBiznwIYuiZ/UChzKXaQDvUxBOB8T+nx5Oy46fGY8oV/G2HL3+YVtQ2wJF+YohxPulZaKLrOwvYEeBygJTTMARBsIAO8OIMVy5AGXM7xpssxxVSPDOvG3MFQMyEpEgyRKmx1BTEkgmM6+wHqTcSaQc5/I949dvRK+xMGgLB8Rk7uiR0AynEMWvmH6CEfXTq6btAnRfByNd3Xh/smn9fADIBWoavpX0AG3wWbrNTxsNZNC34Qi0KZzplDeL4AEIMqj0rfjnqikG5PMHG0Ljp7TD4IfI65lyR+bw93dSL4HS6MhIX0urU6Dj3JFdSqnohNiWfBDUtpiN0KmiLmQlXaBfXQFPgCgAT81kIPegr0AR9C4wCZ6V2QyUIYotlq5A33ugezdkK//yzc4SU0IcihT2QOycEjO95Ov0esFFSsU+OGOhWDFNd7w+ZkatXL147SsSEmuTeHjI8jAGSKRgQVNwtGW+zSDQpmIVXhjNgblhr8ElH9+bhCxMaNe8w3OAgoSb8YaiBFzKWs//ZGx4u/pROyifmw595hHrJidRy/fCiXNe13MEuhpvARu0DA6T6PPtwVB8k1Q8dlWtxHWIP9onK3yMA/hKQ8nViWsIg+CtZ1dqgBEYBUnDmvIX623lz+YUDGaADWiIKqFmpQDcE+KhvVQ92HQwY+ip+mQXhPF1k1lZnfF4vIXCMv64ieJCU1LcroAnkf/Q4gVPh+IwcnQKUJFOpG5m08xNfNqJdl+RIOhmduaUkJq9rhCrvDW50AQCVmYgiqZmkpoozkir+GIpc+K/vRi57McZ7ZGwvmkVdIMQFqH8+4OprRpe5lPXjtaJgzYOeKm6dMY+zDaq8z2/mgWgzI6IotU/0b6EFNW6ImE8g/M46T2FjFhEaEn/4jpEfCqk9QTQ3mJ1OeY7xoJtBIMCN7RcC3e2+sKd4iQXWo/JNdG3s+9eZ+IlJa2PSwi0mDeH3aoOZK7ePc2mjYbgYYks1aNfQd4EoVTbwoVyR+Gs73+BCwTAhJ3QEyHFna5AP3ZtH64q9YYc8j3b4EqNvHNxl3VpatJVKPWyXuMlEA1GA0iSi6AKiDa3eNSn5fUb1g9bxp3LaQ4uldI4fWmGgTl4suOBVLMOL4euQDgT79bd0Q18QMcsGvyCcrG+/EzAk4udJpSx76MZGvZo1yuu8LdAshN9cQJ5gPDhE2At/Aux1jy6yXNqkgqZrVFPil7Xr8ROIdWu9ACMPAXvBg92ww925NtpSd4+u2ewE02Rh3cCEAMsRrFqLOv1qtmwOaxsmdA19PJ8I0Dbv/YJ9IOWjApcLdFVP3vEbJS19k1DRQXaVVfzVO8RmGhZuk14UXTqwvQhx164YGFKutRK4oH5rcd9paUCwDZ7Hv+qNZ0Ev+XPsOVG9iImGB6sIvNPi5WuUiapsAFLNeyss1V0pWEsFG129ncKY9NkCT1UD7lxXuC8Q6AoIMDug78aUorI/aMu6ZWD9mKDH+RZbQwNeCJIimbTtrCDG3XQXHv5h7BIMW85iaIw2Kx1iioqpl8XpRFxK+UAwwmpbVB8k0h1TkhaNR7WoiHCEQCKj0h2NEx8lJUFAGzosCm2nVK4zNmx31/EYmDqaBHXZsjMEqdCngNQh+SsPnk6yVn4QGToVFHyE6nKA0AA0ygoxtirM9DxcAuSI3F9S4gpjxFwBGT02RakqYdiYW69j2NicrzpLkAl/Asi4nHy3wEFyhcTqJCRsBU+XtOmQmI/CyEdon7ddyfDgOcLJ0HvixHcOvx00+D3dzsZfVuPK19PjNzpMyLfMlBV0c3kTnTOWtlCaBxw/+GcnA1SGuKCGmICzEDEbC+tsJvWop8Bw1B/sovaaFcbYA9MZsoeNClYvMPjqNRjpLBB7jfioV/IikRqq04Pwv1apV3TmSOEdENPFMk1+tHUvoBBP0BUaJH0PH6W+r/3hihAc7Fqv11Lp4OC/c4XFId5W6J1F2CH3TtGYW7XTt0xeoJj6VpUlbPPvEABBAsDpI3AMni54hz0ODeAzwMu/YOmFUoGA3RYkCSUQjccVVQqMsI8RTZMva5pyEmZ3Qk9DzNhbEHK5CEbiYBcJQu1+S6KnuWZnecFc07l+4T4Qg+hpiPsOOrhdxaflsJJUoPYd1sFkhiU5oRCh68xHQJeT06FOOckc9yxvI8PjJ6SO4LMTPuh1CILO1H0v6Z5+c3sdCdv8PEi5KUzqrC1nZlMvAhAx4LTy6/V/FOzJk6imRWPXp7D+3i86cs6qBEzX5QTOeOqWnExF455LXcGS6+hHocEdEOqZqG9PcBr3w6VYMuIBpbtF5vDRMTJZotOH4Y34DBM+z/5L4bMHZKaVo0elT+UEhLIs58bu+vUi0quEZp5dQL7YHTRbRdjvV7QCqMBMFRZRu9tFhgYSZ24JGOnLTUTqECH13DEi1RvneQL/jrjUFBV5+tW0ovlkAI7fZXfIAHTps48YaJ1Jnu1Fc0WAA4/rPcCyK1eWgcOtdAj/2QKPvy8YL1+zyx+sWvfKgCZrdtEAWRb3mRCdtT3DDGDA24/RFDKaCkeM8h6bGajvFgHxXaNsTEpeit5KjU1mkoeDrI+bPnocSMC6vSlyOGEENVdkc1nm20VWHlnWITAGaf3Wjup04MwDfX6JTpay4S/BmmWCd2c+6bCdk0ZA6yxzDYYBsyf17SyoRrBuL2BWGuyBsVISk9JkNIkVDbSIMCHHpZ+aTGoyId/gxpSuZhdxIONJXFGYimWYzYzAnRnJZU/xgk+QDeWlqCzspSGNyftXG2wEU9xCwoQAl8EBmzfuV5sLb+0kRjvc0c1Y0lW0Ht+HaJotZYPXwVSIQwAoOx8MHiPqsTx5acPUy9bdUtdQv6pltPPXY93DnNIHNvBWivVd9D8EY9HG6do8t5bNwKY+YjUP0n4yEi35fKdVgZq1faQz/cEg7jwJwYC4jsrxhDwZUVIZTVLaI5rOczDD9/FURo7TBX7J/WNAcxM/AGj5hl5caNQ6z9+3idrh8kCb+I1AguZgsWKU9vkgYMkb1r491bSmm8uzWCixgfm7EE0NBEPHShMwvciTiZ1n0ptR/UdVG4ulT0o02qR6h9qWaBCkAIHP90EvzlS5SxOcAzDa/rSKU29zRZYtRCKK6/z4nXbmN0QzBMHwR04E9ZQCmmIhDo90TlJA7xm4IGjYG9NhRkK8WeNWK9Xn0aFf3pkXomH2nJ0VcQ0GCNB4wdYKMy7VSATRvEJw6DjQ5TP9hB6rVnHWcVCPefmktZQ/JOTmMCTc4fTD98mJ2Ghc1mbG1KTWOdJu8bSMmEVScnJGBC7D6o91oO12kSwaeAgR/+c+uICSqW81phPQ5XJIEq/ivpUq/LCEQUBFl2+zhxHDScjgQ6MxLTEtRaD7E5HYdSl2Q/Q4S+qL91Bnr+tzO1x9+eK1EQ+mHRrwahbQ6+ki+10zX5bsGtbf2a+GC1Qb9R5ElRkhB74I5wCeIKm7dvTCCSRcDv4SF/8wR4hl9RsVex0oXMhWhfnPmhHe+8QztHqKxgpcvV0F0bKF8T6TmZ7yW7cWcQE8TakEhz/ZVCsNxx6mWrFasotKIzbYP3BuzW07j2qOrbk8aKhtlTYa1u0q4o9smb2alZLdh5UH8eQk8w+eBvqHofWf+X4aA406Q2q2aYLO8bjhoHhhhCqDtnRcrbeCQnKWCxx6bHkTM5+OGXSd67zHhjIeSI4nteifprTAJfgyAl8wV+N872rLDXonoT9sbOf4wwV6Oo6qsz+fG8SaPByjMO5xOcObOzDOWETA1gAMOOr/duy35z3db7haxjbBGvJVA6uPjDc7xkLBfFzlISU3EBx3HKEfA2NNEBs+rMDHoylKB4KZ2MaibXf+14EKSCXHzFwzSEIEktBRggSG4hu7ZE345/49gapym+bje4DNqDyRGpVvXlBjqowGpeu6jYYvpPwoOQW09RRSsiJk1k8YRbx02HYraOX/lmXmhdwhhalMGeRLXL63nMWVLbg7kiQDOKK4o5EqKdOJmNSVN8QVJQw2/KZIXFJjSiKNWz59BA/owG0Y6z1odBFPEVo8GBUHsSU3rGVaJLtu18Pyom23eMl5ti44Odh1jUXeL5bLclqjPti64IABlIYJpeH3QmmHRdOKlMSdswK4XPpbRY4B2KIAxJOyoKYIYBirS4ZPhfEPc38ECqfERBeYZp03YBsB/bLIalj1mrWcswljn51idlkj0bvmGRpdQJJVXO/HAoP5rHBmEMz9bwZEATqHwUE1a6/HflRslJtgHBAeIqAMHY05g74+3QDsglsQ6Giv1uu8IsXHYtPKDP44HFf28jaWRaMt1ER3L4D2QCdYuUXZXprO1uU3FuTzlfjEkZBmYLyQX7fyZXQ3xJS49ROj8T9FY2w+jTxTWuK5M3oy38ONH3XA1nQVmq6xi38cvfijc8bZZs6aX2CxBxsCPENHO27QikBXiATjtJloDbclWv4ysQqPzPw7bpK6YFstLbCZMSy+L/rpnsvc5TyHCW3DR4I2speI/K3ZXmF2RZW/VN4PXphqw3R2pYj5W9aHg13qBQtUWt/iV44tEgibfiYouEVjT309WOwEro5N2NL1lrMgnE4TnE4fCZz4I8HxuqYBuofT2ORRwIk9/QRGbgwwkXwYjNxN9Id/ZpvVgC5QfpjWLwy4MHWGMHWme0xRMgCiiVoO58te4hZ5WCm6pQO3w3maZwsBzwokRvc7k3s1Df8UbSIjxwMIsK/k+AVpXLuQlCeRz3e3PJdPNjUINCW5qFOYFqgUG1ViLOXI6RUPrLffYaoG+SZ7Uz7D0cR53Gw+nU4ilUcK3guX75DZOh5233covW+7wMkWmb9QD8FrK+O3x/3yAqMmXhRhXqQZRs7zHOp/lgB8tmsA+9eIuyLTpiVzh9UPuwLQNqlYX9bmiZCag2K7TvrhrPHynW17+Lw9K8u5Ddac20NuoHjzWFL8L2MDO8HLDhazHt7WkAqvGu6Gq1nFKqZDHlYEO353LGJnMfOZxJLuHfK4/Rog65z4Dd9HuJQ15u6DxJmDnysDgvyPtTONc6vIlpZ1nqy0t5mkU1R0gCvP3W1SkxRkwBeCaw/m87wcm7oQSyonOxu0Lww53bqzwZKO68QxHqVEGoxgmnkSUCflPG01aQee4UGec6VMfhWeB/dhKaiz+MV8WrH+UnMvggFtNu97MDRIUAt1DTV/I9aCn66NGMLlipLWEDL/zmshHH2D5lj9FTk20N+ebsA7QDvLL9oWXrrQwvfwQbTY1as0on0bpJ2uWbkFIBKkRCZoKEi4CLfBBZtrZ0Fnv/E2B5DCSiOgzOz8wAfbo9Y2m+wRL8rowSyNTqsT0eQPspIcTU4muzdQfbj3IlKvVjmwdBOcfVFzVq/aiVkcnWvsW1+fz51wECWuP0c3aqwtoFDZlZ3l2teHLfLaOetusb432+e49Y79czfWwIO3z7xdTN6gNanxJzQ22/GNfmz++cldf4sJosLtah1eC9kh39ujYNwpiIL1dIUV2hmVCqCRlcjz1+d2kubJcV8n3HqT+CXTDhdbn+jdKtwLYDRIfo/w1QIWreCQC9GQG+raFPsMKHNNh+atgo5FbJxr9FNwZhMQJj2bs2OytNB2887pVCyAGzTACAmcjyqSpIxJkwlVG2H/4UMgCV99wyG2XwAWxtuTUJ4e0ENZ2atfd/lMNqXqx0a+tm0nV7utmaq9kWcsJatO5f3Rzay2ypvHaVQysoTM0E8BuIPva75NGE8IMpfJNx6dFzpoyt6soYp1b5y30njrO0zZjEraWzGw0sWi3gjMPbqnBSOO9gWnhu7/AYEZ+Ef+YwAA"""

MANIFEST_PATH = RUNTIME_ROOT / "kdic_crawl_manifest_42.csv"
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH.write_bytes(
    gzip.decompress(base64.b64decode(EMBEDDED_MANIFEST_GZIP_B64))
)

print("Manifest 생성 완료:", MANIFEST_PATH)
print("크기:", MANIFEST_PATH.stat().st_size, "bytes")


In [ ]:

# 필요하면 사용자 수정본 CSV로 교체할 수 있습니다.
USE_CUSTOM_MANIFEST_UPLOAD = False

if USE_CUSTOM_MANIFEST_UPLOAD:
    from google.colab import files

    uploaded = files.upload()
    csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
    if len(csv_names) != 1:
        raise ValueError("CSV 파일을 정확히 1개 업로드해야 합니다.")

    MANIFEST_PATH = Path("/content") / csv_names[0]
    MANIFEST_PATH.write_bytes(uploaded[csv_names[0]])
    print("사용자 Manifest로 교체:", MANIFEST_PATH)


In [ ]:

REQUIRED_COLUMNS = {
    "url_id",
    "business_function",
    "page_title",
    "source_url",
    "site_name",
    "normalized_decision",
    "decision_reason",
    "page_type",
    "fetch_strategy",
    "parser_profile",
    "auth_required",
    "asset_policy",
    "content_scope",
    "crawl_wave",
}

manifest_df = pd.read_csv(MANIFEST_PATH, encoding="utf-8-sig", dtype=str).fillna("")

missing_columns = sorted(REQUIRED_COLUMNS - set(manifest_df.columns))
if missing_columns:
    raise ValueError(f"Manifest 필수 열이 없습니다: {missing_columns}")

if manifest_df["url_id"].duplicated().any():
    duplicates = manifest_df.loc[
        manifest_df["url_id"].duplicated(keep=False), ["url_id", "source_url"]
    ]
    raise ValueError(f"url_id 중복:\n{duplicates}")

if manifest_df["source_url"].duplicated().any():
    duplicates = manifest_df.loc[
        manifest_df["source_url"].duplicated(keep=False), ["url_id", "source_url"]
    ]
    raise ValueError(f"source_url 중복:\n{duplicates}")

invalid_urls = manifest_df[
    ~manifest_df["source_url"].str.match(r"^https?://", na=False)
]
if not invalid_urls.empty:
    raise ValueError(
        "HTTP/HTTPS가 아닌 URL이 있습니다:\n"
        + invalid_urls[["url_id", "source_url"]].to_string(index=False)
    )

# 실행 대상 필터
target_df = manifest_df.copy()

if SELECT_BUSINESS_FUNCTIONS:
    target_df = target_df[
        target_df["business_function"].isin(SELECT_BUSINESS_FUNCTIONS)
    ]

target_df = target_df[
    target_df["normalized_decision"].isin(RUN_DECISIONS)
    & target_df["crawl_wave"].isin(RUN_WAVES)
].copy()

if MAX_URLS is not None:
    target_df = target_df.head(MAX_URLS).copy()

print("전체 Manifest:", len(manifest_df))
print("이번 실행 대상:", len(target_df))

display(
    manifest_df.groupby(
        ["business_function", "normalized_decision"],
        dropna=False,
    ).size().rename("count").reset_index()
)

display(
    target_df[
        [
            "url_id",
            "business_function",
            "page_title",
            "normalized_decision",
            "crawl_wave",
            "fetch_strategy",
        ]
    ].head(20)
)


## 3. 공통 유틸리티

In [ ]:

ATTACHMENT_EXTENSIONS = {
    ".pdf", ".hwp", ".hwpx", ".doc", ".docx",
    ".xls", ".xlsx", ".ppt", ".pptx", ".zip",
    ".csv", ".txt",
}

IMAGE_EXTENSIONS = {
    ".png", ".jpg", ".jpeg", ".gif", ".webp", ".svg", ".bmp",
}

VIDEO_EXTENSIONS = {
    ".mp4", ".webm", ".mov", ".avi", ".mkv",
}

NOISE_SELECTORS = [
    "script", "style", "noscript", "template",
    "header", "footer", "nav", "aside",
    ".skip", ".skip-navigation", ".skipnav",
    ".gnb", ".lnb", ".snb", ".breadcrumb-wrap",
    ".footer", ".header", ".quick-menu", ".quick",
    ".util", ".utility", ".pagination",
    "[aria-hidden='true']",
]

MAIN_SELECTORS_BY_SITE = {
    "예금보험공사": [
        "#contents", "#content", ".contents", ".content",
        ".sub-content", ".sub_contents", "main", "article",
    ],
    "금융안심포털": [
        "#contents", "#content", ".contents", ".content",
        ".sub-content", ".sub_contents", "main", "article",
    ],
}

BREADCRUMB_SELECTORS = [
    ".breadcrumb", ".breadcrumbs", ".location", ".location-wrap",
    ".path", ".sub-location", "nav[aria-label*='breadcrumb' i]",
]


def now_kst_iso() -> str:
    return datetime.now(KST).isoformat(timespec="seconds")


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def sha256_text(text: str) -> str:
    return sha256_bytes(text.encode("utf-8"))


def normalize_space(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


def normalize_multiline(text: str) -> str:
    lines = [normalize_space(line) for line in (text or "").splitlines()]
    cleaned: list[str] = []

    for line in lines:
        if not line:
            if cleaned and cleaned[-1] != "":
                cleaned.append("")
            continue
        cleaned.append(line)

    while cleaned and cleaned[-1] == "":
        cleaned.pop()

    return "\n".join(cleaned)


def safe_name(value: str, max_length: int = 90) -> str:
    value = normalize_space(value)
    value = re.sub(r'[\\/:*?"<>|]+', "_", value)
    value = re.sub(r"_+", "_", value).strip("._ ")
    return (value or "untitled")[:max_length]


def ensure_parent(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)


def write_json(path: Path, data: Any) -> None:
    ensure_parent(path)
    path.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def append_jsonl(path: Path, data: dict[str, Any]) -> None:
    ensure_parent(path)
    with path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(data, ensure_ascii=False) + "\n")


def absolute_url(base_url: str, candidate: str | None) -> str | None:
    if not candidate:
        return None

    candidate = candidate.strip()
    if not candidate or candidate.startswith(("#", "javascript:", "mailto:", "tel:")):
        return None

    return urljoin(base_url, candidate)


def extension_from_url(url: str) -> str:
    path = urlparse(url).path
    return Path(path).suffix.lower()


def classify_asset_url(url: str) -> str:
    extension = extension_from_url(url)

    if extension in ATTACHMENT_EXTENSIONS:
        return "attachment"
    if extension in IMAGE_EXTENSIONS:
        return "image"
    if extension in VIDEO_EXTENSIONS:
        return "video"
    return "link"


def row_to_clean_dict(row: pd.Series) -> dict[str, str]:
    return {
        str(key): "" if pd.isna(value) else str(value)
        for key, value in row.to_dict().items()
    }


## 4. HTTP 세션·재시도·robots.txt 검사

In [ ]:

def create_session() -> requests.Session:
    retry_policy = Retry(
        total=MAX_RETRIES,
        connect=MAX_RETRIES,
        read=MAX_RETRIES,
        status=MAX_RETRIES,
        backoff_factor=0.8,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset({"GET", "HEAD"}),
        respect_retry_after_header=True,
        raise_on_status=False,
    )

    adapter = HTTPAdapter(
        max_retries=retry_policy,
        pool_connections=2,
        pool_maxsize=2,
    )

    session = requests.Session()
    session.headers.update(
        {
            "User-Agent": USER_AGENT,
            "Accept": (
                "text/html,application/xhtml+xml,application/xml;q=0.9,"
                "image/avif,image/webp,*/*;q=0.8"
            ),
            "Accept-Language": "ko-KR,ko;q=0.9,en;q=0.5",
            "Connection": "keep-alive",
        }
    )
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


SESSION = create_session()
ROBOTS_CACHE: dict[str, RobotFileParser | None] = {}


def get_robots_parser(url: str) -> RobotFileParser | None:
    parsed = urlparse(url)
    base = f"{parsed.scheme}://{parsed.netloc}"

    if base in ROBOTS_CACHE:
        return ROBOTS_CACHE[base]

    robots_url = urljoin(base, "/robots.txt")

    try:
        response = SESSION.get(
            robots_url,
            timeout=REQUEST_TIMEOUT_SECONDS,
            allow_redirects=True,
        )

        if response.status_code >= 400:
            ROBOTS_CACHE[base] = None
            return None

        parser = RobotFileParser()
        parser.set_url(robots_url)
        parser.parse(response.text.splitlines())
        ROBOTS_CACHE[base] = parser
        return parser

    except requests.RequestException:
        ROBOTS_CACHE[base] = None
        return None


def robots_allows(url: str) -> tuple[bool, str]:
    if not RESPECT_ROBOTS_TXT:
        return True, "robots_check_disabled"

    parser = get_robots_parser(url)
    if parser is None:
        return True, "robots_unavailable_assumed_allowed"

    allowed = parser.can_fetch(USER_AGENT, url)
    return allowed, "allowed" if allowed else "disallowed"


def choose_response_encoding(response: requests.Response) -> str:
    content_type = response.headers.get("Content-Type", "")
    declared = response.encoding

    # requests의 기본 ISO-8859-1 추정은 한국어 HTML에서 잘못될 수 있습니다.
    if not declared or declared.lower() == "iso-8859-1":
        apparent = response.apparent_encoding
        if apparent:
            return apparent

    return declared or "utf-8"


## 5. HTML 수집과 응답 메타데이터 저장

In [ ]:

@dataclass
class FetchResult:
    requested_url: str
    final_url: str
    status_code: int
    ok: bool
    content_type: str
    encoding: str
    elapsed_seconds: float
    collected_at: str
    content_length: int
    raw_sha256: str
    redirect_history: list[dict[str, Any]]
    selected_headers: dict[str, str]
    body: bytes


def fetch_url(url: str) -> FetchResult:
    allowed, robots_status = robots_allows(url)
    if not allowed:
        raise PermissionError(f"robots.txt에서 수집을 허용하지 않습니다: {url}")

    started = time.perf_counter()

    response = SESSION.get(
        url,
        timeout=REQUEST_TIMEOUT_SECONDS,
        allow_redirects=True,
    )

    elapsed = time.perf_counter() - started
    encoding = choose_response_encoding(response)
    response.encoding = encoding

    selected_header_names = {
        "Content-Type",
        "Content-Length",
        "Last-Modified",
        "ETag",
        "Cache-Control",
        "Content-Disposition",
    }

    selected_headers = {
        key: value
        for key, value in response.headers.items()
        if key in selected_header_names
    }
    selected_headers["Robots-Check"] = robots_status

    return FetchResult(
        requested_url=url,
        final_url=response.url,
        status_code=response.status_code,
        ok=response.ok,
        content_type=response.headers.get("Content-Type", ""),
        encoding=encoding,
        elapsed_seconds=round(elapsed, 3),
        collected_at=now_kst_iso(),
        content_length=len(response.content),
        raw_sha256=sha256_bytes(response.content),
        redirect_history=[
            {
                "status_code": item.status_code,
                "url": item.url,
                "location": item.headers.get("Location"),
            }
            for item in response.history
        ],
        selected_headers=selected_headers,
        body=response.content,
    )


def save_fetch_result(
    manifest_row: dict[str, str],
    result: FetchResult,
) -> tuple[Path, Path]:
    business_dir = safe_name(manifest_row["business_function"])
    url_id = manifest_row["url_id"]

    html_path = PATHS["raw_html"] / business_dir / f"{url_id}.html"
    meta_path = PATHS["response_meta"] / business_dir / f"{url_id}.json"

    ensure_parent(html_path)
    html_path.write_bytes(result.body)

    metadata = asdict(result)
    metadata.pop("body", None)
    metadata["url_id"] = url_id
    metadata["business_function"] = manifest_row["business_function"]
    metadata["page_title_manifest"] = manifest_row["page_title"]
    metadata["fetch_strategy"] = manifest_row["fetch_strategy"]
    metadata["parser_profile"] = manifest_row["parser_profile"]
    metadata["raw_html_path"] = str(html_path.relative_to(OUTPUT_ROOT))

    write_json(meta_path, metadata)
    return html_path, meta_path


## 6. 결정론적 본문·표·링크·이미지 추출

In [ ]:

def element_text(tag: Tag) -> str:
    return normalize_space(tag.get_text(" ", strip=True))


def candidate_score(tag: Tag) -> float:
    text = element_text(tag)
    if not text:
        return -1.0

    link_text_length = sum(
        len(element_text(link))
        for link in tag.find_all("a")
    )
    paragraph_count = len(tag.find_all("p"))
    heading_count = len(tag.find_all(re.compile(r"^h[1-6]$")))
    table_count = len(tag.find_all("table"))

    return (
        len(text)
        - link_text_length * 0.55
        + paragraph_count * 25
        + heading_count * 35
        + table_count * 90
    )


def choose_main_root(soup: BeautifulSoup, site_name: str) -> Tag:
    selectors = MAIN_SELECTORS_BY_SITE.get(site_name, []) + ["main", "article"]

    candidates: list[Tag] = []
    seen_ids: set[int] = set()

    for selector in selectors:
        for candidate in soup.select(selector):
            if id(candidate) not in seen_ids:
                seen_ids.add(id(candidate))
                candidates.append(candidate)

    if candidates:
        return max(candidates, key=candidate_score)

    body = soup.body
    return body if isinstance(body, Tag) else soup


def clean_root(root: Tag) -> Tag:
    for selector in NOISE_SELECTORS:
        for node in root.select(selector):
            node.decompose()

    for comment in root.find_all(string=lambda text: isinstance(text, str) and not text.strip()):
        # 공백 노드는 그대로 두어도 결과에 영향이 없으므로 제거하지 않습니다.
        pass

    return root


def extract_title(
    soup: BeautifulSoup,
    root: Tag,
    manifest_title: str,
) -> str:
    for selector in ["h1", ".page-title", ".title", "h2"]:
        node = root.select_one(selector)
        if node:
            text = element_text(node)
            if 2 <= len(text) <= 180:
                return text

    if soup.title:
        text = normalize_space(soup.title.get_text(" ", strip=True))
        if text:
            return text

    return manifest_title


def extract_breadcrumb(soup: BeautifulSoup) -> list[str]:
    for selector in BREADCRUMB_SELECTORS:
        node = soup.select_one(selector)
        if not node:
            continue

        values = []
        for item in node.select("a, span, li"):
            text = element_text(item)
            if text and text not in values:
                values.append(text)

        if values:
            return values

    return []


def markdown_escape_cell(text: str) -> str:
    return normalize_space(text).replace("|", "\\|")


def table_to_markdown(table: Tag) -> tuple[str, list[list[str]]]:
    rows: list[list[str]] = []

    for tr in table.select("tr"):
        cells = [
            markdown_escape_cell(cell.get_text(" ", strip=True))
            for cell in tr.find_all(["th", "td"], recursive=False)
        ]
        if cells and any(cells):
            rows.append(cells)

    if not rows:
        return "", []

    width = max(len(row) for row in rows)
    normalized = [row + [""] * (width - len(row)) for row in rows]

    header = normalized[0]
    body = normalized[1:]

    markdown_lines = [
        "| " + " | ".join(header) + " |",
        "| " + " | ".join(["---"] * width) + " |",
    ]

    markdown_lines.extend(
        "| " + " | ".join(row) + " |"
        for row in body
    )

    return "\n".join(markdown_lines), normalized


def has_ancestor(tag: Tag, names: set[str]) -> bool:
    parent = tag.parent
    while isinstance(parent, Tag):
        if parent.name in names:
            return True
        parent = parent.parent
    return False


def root_to_markdown(root: Tag) -> tuple[str, list[dict[str, Any]]]:
    lines: list[str] = []
    blocks: list[dict[str, Any]] = []
    block_index = 0

    for node in root.select("h1,h2,h3,h4,h5,h6,p,li,table"):
        if node.name == "table":
            if has_ancestor(node, {"table"}):
                continue

            markdown, table_rows = table_to_markdown(node)
            if not markdown:
                continue

            block_index += 1
            lines.extend(["", markdown, ""])
            blocks.append(
                {
                    "block_index": block_index,
                    "block_type": "table",
                    "text": markdown,
                    "table_rows": table_rows,
                }
            )
            continue

        if has_ancestor(node, {"table"}):
            continue

        if node.name == "p" and has_ancestor(node, {"li"}):
            continue

        text = element_text(node)
        if not text:
            continue

        block_index += 1

        if re.fullmatch(r"h[1-6]", node.name or ""):
            level = int(node.name[1])
            markdown = f"{'#' * level} {text}"
            block_type = "heading"
        elif node.name == "li":
            markdown = f"- {text}"
            block_type = "list_item"
        else:
            markdown = text
            block_type = "paragraph"

        lines.extend([markdown, ""])
        blocks.append(
            {
                "block_index": block_index,
                "block_type": block_type,
                "text": text,
                "heading_level": (
                    int(node.name[1])
                    if block_type == "heading"
                    else None
                ),
            }
        )

    markdown = normalize_multiline("\n".join(lines))
    return markdown, blocks


def nearby_context(node: Tag, max_chars: int = 260) -> str:
    values: list[str] = []

    for candidate in [
        node.find_previous(["h1", "h2", "h3", "h4", "p", "li"]),
        node.parent if isinstance(node.parent, Tag) else None,
        node.find_next(["p", "li"]),
    ]:
        if isinstance(candidate, Tag):
            text = element_text(candidate)
            if text and text not in values:
                values.append(text)

    return normalize_space(" ".join(values))[:max_chars]


def extract_links(root: Tag, base_url: str) -> list[dict[str, Any]]:
    links: list[dict[str, Any]] = []
    seen: set[tuple[str, str]] = set()

    for anchor in root.find_all("a", href=True):
        url = absolute_url(base_url, anchor.get("href"))
        if not url:
            continue

        text = element_text(anchor)
        key = (url, text)
        if key in seen:
            continue
        seen.add(key)

        links.append(
            {
                "url": url,
                "anchor_text": text,
                "asset_type": classify_asset_url(url),
                "context": nearby_context(anchor),
            }
        )

    return links


def extract_images(root: Tag, base_url: str) -> list[dict[str, Any]]:
    images: list[dict[str, Any]] = []
    seen_urls: set[str] = set()

    for image in root.find_all("img"):
        candidates = [
            image.get("src"),
            image.get("data-src"),
            image.get("data-original"),
        ]

        srcset = image.get("srcset")
        if srcset:
            candidates.extend(
                item.strip().split(" ")[0]
                for item in srcset.split(",")
                if item.strip()
            )

        url = next(
            (
                absolute_url(base_url, candidate)
                for candidate in candidates
                if absolute_url(base_url, candidate)
            ),
            None,
        )

        if not url or url in seen_urls:
            continue
        seen_urls.add(url)

        caption = ""
        figure = image.find_parent("figure")
        if figure:
            figcaption = figure.find("figcaption")
            if figcaption:
                caption = element_text(figcaption)

        images.append(
            {
                "url": url,
                "alt": normalize_space(image.get("alt", "")),
                "title": normalize_space(image.get("title", "")),
                "caption": caption,
                "context": nearby_context(image),
            }
        )

    return images


def extract_videos(root: Tag, base_url: str) -> list[dict[str, Any]]:
    videos: list[dict[str, Any]] = []
    seen_urls: set[str] = set()

    for node in root.select("video[src], video source[src], iframe[src]"):
        url = absolute_url(base_url, node.get("src"))
        if not url or url in seen_urls:
            continue
        seen_urls.add(url)

        videos.append(
            {
                "url": url,
                "tag": node.name,
                "title": normalize_space(node.get("title", "")),
                "context": nearby_context(node),
            }
        )

    return videos


def parse_html_document(
    html_bytes: bytes,
    final_url: str,
    manifest_row: dict[str, str],
    encoding: str,
) -> dict[str, Any]:
    html_text = html_bytes.decode(encoding, errors="replace")
    soup = BeautifulSoup(html_text, "lxml")

    root = choose_main_root(soup, manifest_row["site_name"])
    root = clean_root(root)

    title = extract_title(
        soup=soup,
        root=root,
        manifest_title=manifest_row["page_title"],
    )
    breadcrumb = extract_breadcrumb(soup)
    markdown, blocks = root_to_markdown(root)
    links = extract_links(root, final_url)
    images = extract_images(root, final_url)
    videos = extract_videos(root, final_url)

    attachments = [
        link for link in links
        if link["asset_type"] == "attachment"
    ]

    return {
        "doc_id": manifest_row["url_id"],
        "parent_doc_id": manifest_row["url_id"],
        "title": title,
        "manifest_title": manifest_row["page_title"],
        "source_url": manifest_row["source_url"],
        "final_url": final_url,
        "site_name": manifest_row["site_name"],
        "business_function": manifest_row["business_function"],
        "target_business_function": manifest_row.get(
            "target_business_function",
            manifest_row["business_function"],
        ),
        "page_type": manifest_row["page_type"],
        "parser_profile": manifest_row["parser_profile"],
        "breadcrumb": breadcrumb,
        "content_markdown": markdown,
        "content_text": normalize_space(root.get_text(" ", strip=True)),
        "parsed_content_sha256": sha256_text(markdown),
        "blocks": blocks,
        "links": links,
        "attachments": attachments,
        "images": images,
        "videos": videos,
        "parsed_at": now_kst_iso(),
    }


## 7. 첨부파일·이미지 다운로드

In [ ]:

CONTENT_TYPE_EXTENSIONS = {
    "application/pdf": ".pdf",
    "application/zip": ".zip",
    "application/vnd.openxmlformats-officedocument.wordprocessingml.document": ".docx",
    "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet": ".xlsx",
    "application/vnd.openxmlformats-officedocument.presentationml.presentation": ".pptx",
    "image/jpeg": ".jpg",
    "image/png": ".png",
    "image/gif": ".gif",
    "image/webp": ".webp",
    "image/svg+xml": ".svg",
}


def filename_from_response(
    url: str,
    response: requests.Response,
    fallback_stem: str,
) -> str:
    disposition = response.headers.get("Content-Disposition", "")

    filename_star = re.search(
        r"filename\*=UTF-8''([^;]+)",
        disposition,
        flags=re.IGNORECASE,
    )
    filename_plain = re.search(
        r'filename="?([^";]+)"?',
        disposition,
        flags=re.IGNORECASE,
    )

    if filename_star:
        from urllib.parse import unquote
        name = unquote(filename_star.group(1))
    elif filename_plain:
        name = filename_plain.group(1)
    else:
        name = Path(urlparse(url).path).name

    name = safe_name(name, max_length=120)
    extension = Path(name).suffix.lower()

    if not extension:
        content_type = response.headers.get("Content-Type", "").split(";")[0].strip()
        extension = CONTENT_TYPE_EXTENSIONS.get(content_type)
        if not extension:
            extension = mimetypes.guess_extension(content_type) or ""
        name = f"{safe_name(fallback_stem)}{extension}"

    return name


def download_binary_asset(
    url: str,
    output_dir: Path,
    fallback_stem: str,
) -> dict[str, Any]:
    output_dir.mkdir(parents=True, exist_ok=True)

    with SESSION.get(
        url,
        timeout=REQUEST_TIMEOUT_SECONDS,
        allow_redirects=True,
        stream=True,
    ) as response:
        response.raise_for_status()

        content_length = response.headers.get("Content-Length")
        if content_length and int(content_length) > MAX_ASSET_BYTES:
            raise ValueError(
                f"파일이 제한 용량을 초과합니다: {content_length} bytes"
            )

        filename = filename_from_response(url, response, fallback_stem)
        output_path = output_dir / filename

        hasher = hashlib.sha256()
        written = 0

        with output_path.open("wb") as file:
            for chunk in response.iter_content(chunk_size=64 * 1024):
                if not chunk:
                    continue

                written += len(chunk)
                if written > MAX_ASSET_BYTES:
                    file.close()
                    output_path.unlink(missing_ok=True)
                    raise ValueError(
                        f"다운로드 중 제한 용량 {MAX_ASSET_BYTES} bytes를 초과했습니다."
                    )

                hasher.update(chunk)
                file.write(chunk)

        return {
            "source_url": url,
            "final_url": response.url,
            "filename": filename,
            "relative_path": str(output_path.relative_to(OUTPUT_ROOT)),
            "content_type": response.headers.get("Content-Type", ""),
            "size_bytes": written,
            "sha256": hasher.hexdigest(),
            "downloaded_at": now_kst_iso(),
        }


def download_document_assets(
    document: dict[str, Any],
    manifest_row: dict[str, str],
) -> dict[str, list[dict[str, Any]]]:
    business_dir = safe_name(manifest_row["business_function"])
    asset_root = (
        PATHS["raw_assets"]
        / business_dir
        / manifest_row["url_id"]
    )

    results = {
        "attachments": [],
        "images": [],
        "videos": [],
        "failures": [],
    }

    should_download_attachments = (
        DOWNLOAD_ATTACHMENTS
        and (
            manifest_row["asset_policy"] == "download_attachments"
            or manifest_row["page_type"] == "attachment_page"
        )
    )

    should_download_images = (
        DOWNLOAD_IMAGES
        and (
            manifest_row["crawl_wave"] == "W1_ASSET"
            or manifest_row["page_type"] in {
                "process_page",
                "video_page",
                "attachment_page",
            }
        )
    )

    if should_download_attachments:
        for index, item in enumerate(document["attachments"], start=1):
            try:
                result = download_binary_asset(
                    item["url"],
                    asset_root / "attachments",
                    f"{manifest_row['url_id']}_attachment_{index:03d}",
                )
                result["anchor_text"] = item.get("anchor_text", "")
                result["context"] = item.get("context", "")
                results["attachments"].append(result)
            except Exception as error:
                results["failures"].append(
                    {
                        "asset_type": "attachment",
                        "url": item["url"],
                        "error_type": type(error).__name__,
                        "error": str(error),
                    }
                )

    if should_download_images:
        for index, item in enumerate(document["images"], start=1):
            try:
                result = download_binary_asset(
                    item["url"],
                    asset_root / "images",
                    f"{manifest_row['url_id']}_image_{index:03d}",
                )
                result.update(
                    {
                        "alt": item.get("alt", ""),
                        "caption": item.get("caption", ""),
                        "context": item.get("context", ""),
                    }
                )
                results["images"].append(result)
            except Exception as error:
                results["failures"].append(
                    {
                        "asset_type": "image",
                        "url": item["url"],
                        "error_type": type(error).__name__,
                        "error": str(error),
                    }
                )

    # 영상은 기본적으로 URL·문맥 메타데이터만 보존합니다.
    if DOWNLOAD_VIDEOS:
        for index, item in enumerate(document["videos"], start=1):
            try:
                result = download_binary_asset(
                    item["url"],
                    asset_root / "videos",
                    f"{manifest_row['url_id']}_video_{index:03d}",
                )
                results["videos"].append(result)
            except Exception as error:
                results["failures"].append(
                    {
                        "asset_type": "video",
                        "url": item["url"],
                        "error_type": type(error).__name__,
                        "error": str(error),
                    }
                )

    return results


## 8. 동적 조회 페이지 진단

In [ ]:

def extract_dynamic_diagnostic(
    html_bytes: bytes,
    final_url: str,
    encoding: str,
) -> dict[str, Any]:
    html_text = html_bytes.decode(encoding, errors="replace")
    soup = BeautifulSoup(html_text, "lxml")

    forms = []
    for form_index, form in enumerate(soup.find_all("form"), start=1):
        inputs = []

        for field in form.select("input,select,textarea,button"):
            name = normalize_space(field.get("name", ""))
            field_type = normalize_space(
                field.get("type", field.name or "")
            )

            if not name and field.name != "button":
                continue

            record = {
                "tag": field.name,
                "name": name,
                "type": field_type,
                "value": normalize_space(field.get("value", "")),
            }

            if field.name == "select":
                record["options"] = [
                    {
                        "value": normalize_space(option.get("value", "")),
                        "text": element_text(option),
                    }
                    for option in field.find_all("option")
                ]

            inputs.append(record)

        forms.append(
            {
                "form_index": form_index,
                "method": normalize_space(form.get("method", "GET")).upper(),
                "action": absolute_url(final_url, form.get("action")) or final_url,
                "id": normalize_space(form.get("id", "")),
                "name": normalize_space(form.get("name", "")),
                "inputs": inputs,
            }
        )

    script_sources = []
    for script in soup.find_all("script", src=True):
        source = absolute_url(final_url, script.get("src"))
        if source and source not in script_sources:
            script_sources.append(source)

    inline_script_hints = []
    keyword_pattern = re.compile(
        r"(ajax|fetch\\s*\\(|XMLHttpRequest|\\.do\\b|pageIndex|pageNo|search)",
        flags=re.IGNORECASE,
    )

    for script in soup.find_all("script"):
        if script.get("src"):
            continue

        text = script.get_text("\\n", strip=True)
        if text and keyword_pattern.search(text):
            inline_script_hints.append(text[:1200])

    return {
        "final_url": final_url,
        "forms": forms,
        "script_sources": script_sources,
        "inline_script_hints": inline_script_hints[:20],
        "diagnosed_at": now_kst_iso(),
        "safety_note": (
            "초기 공개 페이지 구조만 진단했습니다. "
            "검색 폼 제출·로그인·본인인증·신청 실행은 수행하지 않았습니다."
        ),
    }


def ensure_playwright_installed() -> None:
    try:
        import playwright  # noqa: F401
        return
    except ImportError:
        pass

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "playwright"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "playwright", "install", "chromium"],
        check=True,
    )


def save_playwright_snapshot(
    url: str,
    url_id: str,
) -> dict[str, Any] | None:
    if not ENABLE_PLAYWRIGHT_SNAPSHOT:
        return None

    ensure_playwright_installed()
    from playwright.sync_api import sync_playwright

    html_path = PATHS["dynamic_diagnostics"] / f"{url_id}_rendered.html"
    screenshot_path = PATHS["screenshots"] / f"{url_id}.png"
    ensure_parent(html_path)
    ensure_parent(screenshot_path)

    with sync_playwright() as playwright:
        browser = playwright.chromium.launch(headless=True)
        page = browser.new_page(
            user_agent=USER_AGENT,
            locale="ko-KR",
            viewport={"width": 1440, "height": 1200},
        )

        response = page.goto(
            url,
            wait_until="domcontentloaded",
            timeout=REQUEST_TIMEOUT_SECONDS * 1000,
        )
        page.wait_for_timeout(PLAYWRIGHT_WAIT_MS)

        html = page.content()
        html_path.write_text(html, encoding="utf-8")
        page.screenshot(path=str(screenshot_path), full_page=True)

        result = {
            "status_code": response.status if response else None,
            "final_url": page.url,
            "rendered_html_path": str(html_path.relative_to(OUTPUT_ROOT)),
            "screenshot_path": str(screenshot_path.relative_to(OUTPUT_ROOT)),
            "captured_at": now_kst_iso(),
        }

        browser.close()
        return result


## 9. URL 유형별 처리기

In [ ]:

def save_parsed_document(
    document: dict[str, Any],
    manifest_row: dict[str, str],
) -> tuple[Path, Path]:
    business_dir = safe_name(manifest_row["business_function"])
    url_id = manifest_row["url_id"]

    markdown_path = (
        PATHS["parsed_markdown"]
        / business_dir
        / f"{url_id}.md"
    )
    json_path = (
        PATHS["parsed_json"]
        / business_dir
        / f"{url_id}.json"
    )

    ensure_parent(markdown_path)
    markdown_path.write_text(
        document["content_markdown"],
        encoding="utf-8",
    )
    write_json(json_path, document)

    return markdown_path, json_path


def process_link_only(
    manifest_row: dict[str, str],
) -> dict[str, Any]:
    record = {
        "doc_id": manifest_row["url_id"],
        "record_type": "link_only",
        "title": manifest_row["page_title"],
        "source_url": manifest_row["source_url"],
        "site_name": manifest_row["site_name"],
        "business_function": manifest_row["business_function"],
        "target_business_function": manifest_row.get(
            "target_business_function",
            manifest_row["business_function"],
        ),
        "page_type": manifest_row["page_type"],
        "auth_required": manifest_row["auth_required"],
        "content_scope": manifest_row["content_scope"],
        "description": manifest_row.get("content_summary", ""),
        "decision_reason": manifest_row["decision_reason"],
        "created_at": now_kst_iso(),
    }

    output_path = (
        PATHS["parsed_json"]
        / safe_name(manifest_row["business_function"])
        / f"{manifest_row['url_id']}.json"
    )
    write_json(output_path, record)

    return {
        "url_id": manifest_row["url_id"],
        "business_function": manifest_row["business_function"],
        "page_title": manifest_row["page_title"],
        "source_url": manifest_row["source_url"],
        "normalized_decision": manifest_row["normalized_decision"],
        "crawl_wave": manifest_row["crawl_wave"],
        "status": "link_metadata_created",
        "status_code": None,
        "content_chars": len(record["description"]),
        "attachment_count": 0,
        "image_count": 0,
        "asset_failure_count": 0,
        "error_type": "",
        "error": "",
        "finished_at": now_kst_iso(),
    }


def process_static_or_asset(
    manifest_row: dict[str, str],
) -> dict[str, Any]:
    result = fetch_url(manifest_row["source_url"])
    html_path, meta_path = save_fetch_result(manifest_row, result)

    if not result.ok:
        raise requests.HTTPError(
            f"HTTP {result.status_code}: {result.final_url}"
        )

    if "html" not in result.content_type.lower():
        raise ValueError(
            f"HTML 응답이 아닙니다: {result.content_type}"
        )

    document = parse_html_document(
        html_bytes=result.body,
        final_url=result.final_url,
        manifest_row=manifest_row,
        encoding=result.encoding,
    )

    asset_results = download_document_assets(
        document=document,
        manifest_row=manifest_row,
    )
    document["downloaded_assets"] = asset_results

    markdown_path, json_path = save_parsed_document(
        document=document,
        manifest_row=manifest_row,
    )

    return {
        "url_id": manifest_row["url_id"],
        "business_function": manifest_row["business_function"],
        "page_title": document["title"],
        "source_url": manifest_row["source_url"],
        "final_url": result.final_url,
        "normalized_decision": manifest_row["normalized_decision"],
        "crawl_wave": manifest_row["crawl_wave"],
        "status": (
            "parse_success"
            if document["content_text"]
            else "empty_content"
        ),
        "status_code": result.status_code,
        "content_chars": len(document["content_text"]),
        "block_count": len(document["blocks"]),
        "link_count": len(document["links"]),
        "attachment_count": len(document["attachments"]),
        "image_count": len(document["images"]),
        "downloaded_attachment_count": len(asset_results["attachments"]),
        "downloaded_image_count": len(asset_results["images"]),
        "asset_failure_count": len(asset_results["failures"]),
        "raw_html_path": str(html_path.relative_to(OUTPUT_ROOT)),
        "response_meta_path": str(meta_path.relative_to(OUTPUT_ROOT)),
        "parsed_markdown_path": str(markdown_path.relative_to(OUTPUT_ROOT)),
        "parsed_json_path": str(json_path.relative_to(OUTPUT_ROOT)),
        "raw_sha256": result.raw_sha256,
        "parsed_content_sha256": document["parsed_content_sha256"],
        "error_type": "",
        "error": "",
        "finished_at": now_kst_iso(),
    }


def process_dynamic_diagnostic(
    manifest_row: dict[str, str],
) -> dict[str, Any]:
    result = fetch_url(manifest_row["source_url"])
    html_path, meta_path = save_fetch_result(manifest_row, result)

    if not result.ok:
        raise requests.HTTPError(
            f"HTTP {result.status_code}: {result.final_url}"
        )

    diagnostic = extract_dynamic_diagnostic(
        html_bytes=result.body,
        final_url=result.final_url,
        encoding=result.encoding,
    )
    diagnostic["url_id"] = manifest_row["url_id"]
    diagnostic["page_title"] = manifest_row["page_title"]
    diagnostic["business_function"] = manifest_row["business_function"]
    diagnostic["source_url"] = manifest_row["source_url"]

    playwright_snapshot = save_playwright_snapshot(
        url=manifest_row["source_url"],
        url_id=manifest_row["url_id"],
    )
    if playwright_snapshot:
        diagnostic["playwright_snapshot"] = playwright_snapshot

    diagnostic_path = (
        PATHS["dynamic_diagnostics"]
        / f"{manifest_row['url_id']}.json"
    )
    write_json(diagnostic_path, diagnostic)

    # 초기 HTML에 공개 본문이 있을 수 있으므로 파싱 결과도 참고용으로 저장합니다.
    document = parse_html_document(
        html_bytes=result.body,
        final_url=result.final_url,
        manifest_row=manifest_row,
        encoding=result.encoding,
    )
    document["collection_scope"] = "initial_public_page_only"
    markdown_path, json_path = save_parsed_document(
        document=document,
        manifest_row=manifest_row,
    )

    return {
        "url_id": manifest_row["url_id"],
        "business_function": manifest_row["business_function"],
        "page_title": document["title"],
        "source_url": manifest_row["source_url"],
        "final_url": result.final_url,
        "normalized_decision": manifest_row["normalized_decision"],
        "crawl_wave": manifest_row["crawl_wave"],
        "status": "dynamic_diagnostic_created",
        "status_code": result.status_code,
        "content_chars": len(document["content_text"]),
        "form_count": len(diagnostic["forms"]),
        "script_source_count": len(diagnostic["script_sources"]),
        "attachment_count": len(document["attachments"]),
        "image_count": len(document["images"]),
        "asset_failure_count": 0,
        "raw_html_path": str(html_path.relative_to(OUTPUT_ROOT)),
        "response_meta_path": str(meta_path.relative_to(OUTPUT_ROOT)),
        "parsed_markdown_path": str(markdown_path.relative_to(OUTPUT_ROOT)),
        "parsed_json_path": str(json_path.relative_to(OUTPUT_ROOT)),
        "dynamic_diagnostic_path": str(
            diagnostic_path.relative_to(OUTPUT_ROOT)
        ),
        "error_type": "",
        "error": "",
        "finished_at": now_kst_iso(),
    }


def process_manifest_row(
    manifest_row: dict[str, str],
) -> dict[str, Any]:
    decision = manifest_row["normalized_decision"]
    strategy = manifest_row["fetch_strategy"]

    if decision == "link_only":
        return process_link_only(manifest_row)

    if strategy == "dynamic_http_then_playwright":
        return process_dynamic_diagnostic(manifest_row)

    return process_static_or_asset(manifest_row)


## 10. 크롤링 실행

In [ ]:

CRAWL_RESULTS_JSONL = PATHS["logs"] / "crawl_results.jsonl"
CRAWL_RESULTS_CSV = PATHS["logs"] / "crawl_results.csv"
USED_MANIFEST_PATH = PATHS["manifest"] / "manifest_used.csv"

target_df.to_csv(
    USED_MANIFEST_PATH,
    index=False,
    encoding="utf-8-sig",
)

run_results: list[dict[str, Any]] = []

for _, row in tqdm(
    target_df.iterrows(),
    total=len(target_df),
    desc="크롤링 진행",
):
    manifest_row = row_to_clean_dict(row)
    started_at = now_kst_iso()

    try:
        result = process_manifest_row(manifest_row)
        result["started_at"] = started_at

    except Exception as error:
        result = {
            "url_id": manifest_row["url_id"],
            "business_function": manifest_row["business_function"],
            "page_title": manifest_row["page_title"],
            "source_url": manifest_row["source_url"],
            "normalized_decision": manifest_row["normalized_decision"],
            "crawl_wave": manifest_row["crawl_wave"],
            "status": "failed",
            "status_code": None,
            "content_chars": 0,
            "attachment_count": 0,
            "image_count": 0,
            "asset_failure_count": 0,
            "error_type": type(error).__name__,
            "error": str(error),
            "started_at": started_at,
            "finished_at": now_kst_iso(),
        }

    run_results.append(result)
    append_jsonl(CRAWL_RESULTS_JSONL, result)

    # link_only는 실제 HTTP 요청을 보내지 않으므로 대기하지 않습니다.
    if manifest_row["normalized_decision"] != "link_only":
        time.sleep(REQUEST_DELAY_SECONDS)

results_df = pd.DataFrame(run_results)
results_df.to_csv(
    CRAWL_RESULTS_CSV,
    index=False,
    encoding="utf-8-sig",
)

print("실행 완료")
display(
    results_df[
        [
            "url_id",
            "business_function",
            "status",
            "status_code",
            "content_chars",
            "attachment_count",
            "image_count",
            "asset_failure_count",
            "error_type",
        ]
    ]
)


## 11. 결과 검수

In [ ]:

if results_df.empty:
    print("실행 결과가 없습니다.")
else:
    print("상태별 건수")
    display(
        results_df.groupby("status")
        .size()
        .rename("count")
        .reset_index()
        .sort_values("count", ascending=False)
    )

    failures = results_df[results_df["status"] == "failed"].copy()
    empty_contents = results_df[
        results_df["status"].isin({"empty_content"})
        | (
            results_df["status"].isin(
                {"parse_success", "dynamic_diagnostic_created"}
            )
            & (results_df["content_chars"].fillna(0).astype(int) < 80)
        )
    ].copy()

    print("실패:", len(failures))
    print("본문 검수 필요:", len(empty_contents))

    if not failures.empty:
        display(
            failures[
                [
                    "url_id",
                    "page_title",
                    "source_url",
                    "error_type",
                    "error",
                ]
            ]
        )

    if not empty_contents.empty:
        display(
            empty_contents[
                [
                    "url_id",
                    "page_title",
                    "status",
                    "content_chars",
                ]
            ]
        )

    success_statuses = {
        "parse_success",
        "dynamic_diagnostic_created",
        "link_metadata_created",
    }
    success_count = results_df["status"].isin(success_statuses).sum()
    success_rate = (
        success_count / len(results_df) * 100
        if len(results_df)
        else 0
    )

    print(f"기본 성공률: {success_count}/{len(results_df)} ({success_rate:.1f}%)")


## 12. 페이지 문서 JSONL과 기본 청크 생성

In [ ]:

DOCUMENTS_JSONL = PATHS["processed"] / "documents.jsonl"
CHUNKS_JSONL = PATHS["processed"] / "chunks.jsonl"


def iter_parsed_json_files() -> Iterable[Path]:
    yield from sorted(PATHS["parsed_json"].rglob("*.json"))


def build_document_record(data: dict[str, Any]) -> dict[str, Any] | None:
    record_type = data.get("record_type", "page")

    if record_type == "link_only":
        return {
            "doc_id": data["doc_id"],
            "parent_doc_id": data["doc_id"],
            "record_type": "link_only",
            "title": data["title"],
            "source_url": data["source_url"],
            "site_name": data["site_name"],
            "business_function": data["business_function"],
            "target_business_function": data["target_business_function"],
            "page_type": data["page_type"],
            "content": data.get("description", ""),
            "content_hash": sha256_text(data.get("description", "")),
            "metadata": {
                "auth_required": data.get("auth_required", ""),
                "content_scope": data.get("content_scope", ""),
                "decision_reason": data.get("decision_reason", ""),
            },
        }

    content = data.get("content_markdown", "")
    if not content:
        return None

    return {
        "doc_id": data["doc_id"],
        "parent_doc_id": data.get("parent_doc_id", data["doc_id"]),
        "record_type": "page",
        "title": data.get("title", ""),
        "source_url": data.get("source_url", ""),
        "final_url": data.get("final_url", ""),
        "site_name": data.get("site_name", ""),
        "business_function": data.get("business_function", ""),
        "target_business_function": data.get(
            "target_business_function",
            data.get("business_function", ""),
        ),
        "page_type": data.get("page_type", ""),
        "breadcrumb": data.get("breadcrumb", []),
        "content": content,
        "content_hash": data.get(
            "parsed_content_sha256",
            sha256_text(content),
        ),
        "metadata": {
            "parser_profile": data.get("parser_profile", ""),
            "attachment_count": len(data.get("attachments", [])),
            "image_count": len(data.get("images", [])),
            "video_count": len(data.get("videos", [])),
            "parsed_at": data.get("parsed_at", ""),
        },
    }


def split_text_with_overlap(
    text: str,
    max_chars: int,
    overlap_chars: int,
) -> list[str]:
    text = normalize_multiline(text)
    if not text:
        return []

    paragraphs = [
        paragraph.strip()
        for paragraph in re.split(r"\n{2,}", text)
        if paragraph.strip()
    ]

    chunks: list[str] = []
    current = ""

    for paragraph in paragraphs:
        candidate = (
            f"{current}\n\n{paragraph}".strip()
            if current
            else paragraph
        )

        if len(candidate) <= max_chars:
            current = candidate
            continue

        if current:
            chunks.append(current)

        if len(paragraph) <= max_chars:
            overlap = (
                current[-overlap_chars:]
                if current and overlap_chars > 0
                else ""
            )
            current = f"{overlap}\n\n{paragraph}".strip()
            if len(current) > max_chars:
                current = paragraph
            continue

        # 매우 긴 단락은 문자 기준으로 안전하게 분할
        step = max(1, max_chars - overlap_chars)
        start = 0
        while start < len(paragraph):
            piece = paragraph[start:start + max_chars].strip()
            if piece:
                chunks.append(piece)
            start += step
        current = ""

    if current:
        chunks.append(current)

    return chunks


documents: list[dict[str, Any]] = []

for json_path in iter_parsed_json_files():
    data = json.loads(json_path.read_text(encoding="utf-8"))
    document = build_document_record(data)
    if document:
        documents.append(document)

with DOCUMENTS_JSONL.open("w", encoding="utf-8") as file:
    for document in documents:
        file.write(json.dumps(document, ensure_ascii=False) + "\n")

chunk_records: list[dict[str, Any]] = []

if CREATE_BASELINE_CHUNKS:
    for document in documents:
        for index, chunk_text in enumerate(
            split_text_with_overlap(
                document["content"],
                max_chars=CHUNK_MAX_CHARS,
                overlap_chars=CHUNK_OVERLAP_CHARS,
            )
        ):
            chunk_records.append(
                {
                    "chunk_id": f"{document['doc_id']}_chunk_{index:03d}",
                    "parent_doc_id": document["doc_id"],
                    "chunk_index": index,
                    "title": document["title"],
                    "business_function": document["business_function"],
                    "target_business_function": document[
                        "target_business_function"
                    ],
                    "page_type": document["page_type"],
                    "source_url": document["source_url"],
                    "content": chunk_text,
                    "content_hash": sha256_text(chunk_text),
                }
            )

    with CHUNKS_JSONL.open("w", encoding="utf-8") as file:
        for chunk in chunk_records:
            file.write(json.dumps(chunk, ensure_ascii=False) + "\n")

print("페이지 문서:", len(documents))
print("기본 청크:", len(chunk_records))
print("documents.jsonl:", DOCUMENTS_JSONL)
print("chunks.jsonl:", CHUNKS_JSONL)


## 13. 결과 ZIP 생성 및 다운로드

In [ ]:

archive_path = Path(
    shutil.make_archive(
        base_name=str(OUTPUT_ROOT),
        format="zip",
        root_dir=OUTPUT_ROOT.parent,
        base_dir=OUTPUT_ROOT.name,
    )
)

print("결과 ZIP:", archive_path)
print("ZIP 크기:", f"{archive_path.stat().st_size / 1024 / 1024:.2f} MB")

try:
    from google.colab import files
    files.download(str(archive_path))
except Exception as error:
    print("자동 다운로드를 시작하지 못했습니다.")
    print("Colab 왼쪽 파일 패널에서 직접 다운로드하세요:", archive_path)
    print("오류:", error)



## 결과 폴더 구조

```text
kdic_crawl_output_<실행시각>/
├── manifest/
│   └── manifest_used.csv
├── raw/
│   ├── html/
│   ├── assets/
│   └── response_meta/
├── parsed/
│   ├── markdown/
│   └── json/
├── diagnostics/
│   ├── dynamic/
│   └── screenshots/
├── processed/
│   ├── documents.jsonl
│   └── chunks.jsonl
└── logs/
    ├── crawl_results.csv
    └── crawl_results.jsonl
```

## 권장 실행 순서

1. `MAX_URLS = 3`으로 테스트
2. 실패·빈 본문·자산 오류 확인
3. 사이트별 본문 선택자와 제거 선택자 조정
4. `MAX_URLS = None`으로 전체 실행
5. `Review_Queue` 결정 후 Manifest 수정본을 업로드하여 재실행
6. 동적 조회 페이지는 진단 JSON을 검토한 뒤 공개 HTTP 요청 전용 수집기를 별도로 작성
